# Generate Ground-Motion Fields

This notebook calculates event-specific ground motions for the common
stochastic earthquake catalog generated in Notebook 3.

The baseline model uses the Parker et al. NGA-Subduction ground-motion model
for:

- Cascadia interface earthquakes
- Oregon intraslab earthquakes
- PGA and 5%-damped pseudo-spectral acceleration
- building-level site conditions
- shared between-event residuals
- conditionally independent within-event residuals

Spatial correlation is not included in the internship-ready baseline.

Before calculating any ground motions, the notebook inspects the exact Parker
model variants, supported intensity measures, required inputs, and output
structure exposed by the installed USGS `nshmp-lib` implementation.

In [1]:
from pathlib import Path

import re
import shutil
import subprocess

import pandas as pd
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name == "seismic-correlation-insurance-loss":
            return candidate

    raise RuntimeError(
        "Could not locate the seismic-correlation-insurance-loss repository."
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"

CATALOG_DIR = (
    DATA_DIR
    / "processed"
    / "annual_event_catalog"
)

EVENT_CATALOG_PATH = (
    CATALOG_DIR
    / "annual_event_catalog.csv"
)

CATALOG_METADATA_PATH = (
    METADATA_DIR
    / "notebook_3_completion_metadata.json"
)

RUNTIME_CLASSPATH_PATH = (
    METADATA_DIR
    / "nshmp_haz_2_6_5_runtime_classpath.csv"
)

PARKER_INSPECTION_DIR = (
    PROJECT_ROOT
    / "tools"
    / "parker_gmm_inspection"
)

PARKER_INSPECTION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PARKER_ENTRY_PATH = (
    METADATA_DIR
    / "parker_nshmp_lib_entries.csv"
)

PARKER_REPORT_PATH = (
    METADATA_DIR
    / "parker_nshmp_lib_inspection.txt"
)


required_files = pd.DataFrame(
    [
        {
            "description": "Notebook 3 event catalog",
            "path": str(EVENT_CATALOG_PATH),
            "exists": EVENT_CATALOG_PATH.is_file(),
        },
        {
            "description": "Notebook 3 completion metadata",
            "path": str(CATALOG_METADATA_PATH),
            "exists": CATALOG_METADATA_PATH.is_file(),
        },
        {
            "description": "nshmp-haz runtime classpath",
            "path": str(RUNTIME_CLASSPATH_PATH),
            "exists": RUNTIME_CLASSPATH_PATH.is_file(),
        },
    ]
)


print("Required Notebook 4 inputs:")
display(required_files)


if not required_files["exists"].all():
    missing = required_files.loc[
        ~required_files["exists"],
        "path",
    ].tolist()

    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(
            f"  {path}"
            for path in missing
        )
    )


runtime_table = pd.read_csv(
    RUNTIME_CLASSPATH_PATH
)


if "path" not in runtime_table.columns:
    raise RuntimeError(
        "The runtime classpath manifest does not contain a path column."
    )


runtime_paths = [
    Path(path).resolve()
    for path in runtime_table["path"].dropna()
    if Path(path).is_file()
]


nshmp_lib_candidates = [
    path
    for path in runtime_paths
    if path.name.lower().startswith(
        "nshmp-lib"
    )
    and path.suffix.lower() == ".jar"
]


if len(nshmp_lib_candidates) != 1:
    print("Candidate nshmp-lib JAR files:")

    for path in nshmp_lib_candidates:
        print(f"  {path}")

    raise RuntimeError(
        "Expected exactly one nshmp-lib JAR in the runtime classpath."
    )


NSHMP_LIB_JAR = nshmp_lib_candidates[0]


jar_executable = shutil.which(
    "jar"
)

javap_executable = shutil.which(
    "javap"
)


if jar_executable is None:
    raise FileNotFoundError(
        "The JDK jar command was not found through PATH."
    )

if javap_executable is None:
    raise FileNotFoundError(
        "The JDK javap command was not found through PATH."
    )


jar_result = subprocess.run(
    [
        jar_executable,
        "tf",
        str(NSHMP_LIB_JAR),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


if jar_result.returncode != 0:
    raise RuntimeError(
        "Could not inspect the nshmp-lib JAR.\n\n"
        + jar_result.stderr
    )


jar_entries = [
    line.strip()
    for line in jar_result.stdout.splitlines()
    if line.strip()
]


parker_entries = [
    entry
    for entry in jar_entries
    if "parker" in entry.lower()
]


if not parker_entries:
    raise RuntimeError(
        "No Parker-related classes or resources were found "
        "inside nshmp-lib."
    )


entry_table = pd.DataFrame(
    {
        "jar_entry": parker_entries,
        "entry_type": [
            (
                "class"
                if entry.endswith(".class")
                else "resource"
            )
            for entry in parker_entries
        ],
    }
)


entry_table.to_csv(
    PARKER_ENTRY_PATH,
    index=False,
)


explicit_classes = [
    "gov.usgs.earthquake.nshmp.gmm.Gmm",
    "gov.usgs.earthquake.nshmp.gmm.Imt",
    "gov.usgs.earthquake.nshmp.gmm.GmmInput",
    "gov.usgs.earthquake.nshmp.gmm.GmmInput$Builder",
    "gov.usgs.earthquake.nshmp.gmm.ScalarGroundMotion",
]


parker_classes = []

for entry in parker_entries:
    if not entry.endswith(".class"):
        continue

    class_name = (
        entry[
            :-len(".class")
        ]
        .replace(
            "/",
            ".",
        )
    )

    if re.search(
        r"\$\d+$",
        class_name,
    ):
        continue

    parker_classes.append(
        class_name
    )


classes_to_inspect = list(
    dict.fromkeys(
        [
            *explicit_classes,
            *parker_classes,
        ]
    )
)


inspection_outputs = {}


for class_name in classes_to_inspect:
    result = subprocess.run(
        [
            javap_executable,
            "-classpath",
            str(NSHMP_LIB_JAR),
            "-private",
            class_name,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )

    if result.returncode == 0:
        inspection_outputs[class_name] = (
            result.stdout.strip()
        )
    else:
        inspection_outputs[class_name] = (
            "JAVAP FAILED\n"
            + result.stderr.strip()
        )


gmm_output = inspection_outputs.get(
    "gov.usgs.earthquake.nshmp.gmm.Gmm",
    "",
)


gmm_constants = re.findall(
    r"\bGmm\s+([A-Z][A-Z0-9_]+);",
    gmm_output,
)


parker_gmm_constants = [
    constant
    for constant in gmm_constants
    if "PARKER" in constant
]


report_sections = [
    "PARKER GMM INSPECTION",
    "",
    f"Project root: {PROJECT_ROOT}",
    f"nshmp-lib JAR: {NSHMP_LIB_JAR}",
    "",
    "Parker GMM constants:",
]


if parker_gmm_constants:
    report_sections.extend(
        f"  {constant}"
        for constant in parker_gmm_constants
    )
else:
    report_sections.append(
        "  No Parker constants were parsed automatically."
    )


report_sections.extend(
    [
        "",
        "Parker-related JAR entries:",
        *[
            f"  {entry}"
            for entry in parker_entries
        ],
        "",
        "JAVAP INSPECTION OUTPUTS",
    ]
)


for class_name, output in inspection_outputs.items():
    report_sections.extend(
        [
            "",
            "=" * 78,
            class_name,
            "=" * 78,
            output,
        ]
    )


PARKER_REPORT_PATH.write_text(
    "\n".join(
        report_sections
    )
    + "\n",
    encoding="utf-8",
)


print("\nParker GMM constants:")

if parker_gmm_constants:
    for constant in parker_gmm_constants:
        print(f"  {constant}")
else:
    print(
        "  No constants were parsed automatically. "
        "Review the saved report."
    )


print("\nParker-related classes and resources:")
display(entry_table)


print("\nClasses inspected:")

for class_name in classes_to_inspect:
    status = (
        "passed"
        if not inspection_outputs[
            class_name
        ].startswith(
            "JAVAP FAILED"
        )
        else "failed"
    )

    print(
        f"  {status:6s}  {class_name}"
    )


print("\nSelected Gmm API lines:")

for line in gmm_output.splitlines():
    lower_line = line.lower()

    if (
        "parker" in lower_line
        or "supported" in lower_line
        or "instance(" in lower_line
        or "constraints" in lower_line
    ):
        print(line)


print("\nNOTEBOOK 4 MODEL INSPECTION COMPLETE")

print(f"\nnshmp-lib:\n  {NSHMP_LIB_JAR}")

print(
    "\nParker entry inventory:"
    f"\n  {PARKER_ENTRY_PATH}"
)

print(
    "\nComplete API inspection report:"
    f"\n  {PARKER_REPORT_PATH}"
)

print(
    "\nNext step: identify the exact Cascadia interface "
    "and intraslab model constants and query their "
    "supported intensity measures."
)

Required Notebook 4 inputs:


,description,path,exists
0,Notebook 3 event catalog,C:\Users\USER\Documents\GitHub\seismic-correla...,True
1,Notebook 3 completion metadata,C:\Users\USER\Documents\GitHub\seismic-correla...,True
2,nshmp-haz runtime classpath,C:\Users\USER\Documents\GitHub\seismic-correla...,True



Parker GMM constants:
  No constants were parsed automatically. Review the saved report.

Parker-related classes and resources:


,jar_entry,entry_type
0,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
1,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
2,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
3,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
4,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
5,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
6,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
7,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
8,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class
9,gov/usgs/earthquake/nshmp/gmm/ParkerEtAl_2020$...,class



Classes inspected:
  passed  gov.usgs.earthquake.nshmp.gmm.Gmm
  passed  gov.usgs.earthquake.nshmp.gmm.Imt
  passed  gov.usgs.earthquake.nshmp.gmm.GmmInput
  passed  gov.usgs.earthquake.nshmp.gmm.GmmInput$Builder
  failed  gov.usgs.earthquake.nshmp.gmm.ScalarGroundMotion
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$Coefficients
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$Global
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$Cascadia
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$Alaska
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$Prvi
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$GlobalInterface
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$GlobalInterfaceNoEpi
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$GlobalInterfaceAkAdjusted
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$GlobalSlab
  passed  gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$GlobalSlabNoEpi
  passed  gov.usgs.earthquake.nshmp.

In [2]:
from pathlib import Path

import os
import shutil
import subprocess

import pandas as pd
from IPython.display import display


PARKER_MODEL_INVENTORY_PATH = (
    METADATA_DIR
    / "parker_gmm_model_inventory.csv"
)

PARKER_IMT_INVENTORY_PATH = (
    METADATA_DIR
    / "parker_gmm_imt_inventory.csv"
)

PARKER_ENUM_INSPECTOR_SOURCE = (
    PARKER_INSPECTION_DIR
    / "ParkerGmmInventory.java"
)

PARKER_ENUM_INSPECTOR_CLASSES = (
    PARKER_INSPECTION_DIR
    / "classes"
)

PARKER_ENUM_COMPILE_LOG = (
    METADATA_DIR
    / "parker_gmm_inventory_compile.log"
)

PARKER_ENUM_RUN_LOG = (
    METADATA_DIR
    / "parker_gmm_inventory_run.log"
)

PARKER_ENUM_JAVAC_ARGS = (
    PARKER_INSPECTION_DIR
    / "javac_arguments.txt"
)

PARKER_ENUM_JAVA_ARGS = (
    PARKER_INSPECTION_DIR
    / "java_arguments.txt"
)


if PARKER_ENUM_INSPECTOR_CLASSES.exists():
    shutil.rmtree(
        PARKER_ENUM_INSPECTOR_CLASSES
    )

PARKER_ENUM_INSPECTOR_CLASSES.mkdir(
    parents=True,
    exist_ok=True,
)


java_source = r'''
import gov.usgs.earthquake.nshmp.gmm.Gmm;
import gov.usgs.earthquake.nshmp.gmm.GroundMotionModel;
import gov.usgs.earthquake.nshmp.gmm.Imt;

import java.io.PrintWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.ArrayList;
import java.util.Comparator;
import java.util.List;
import java.util.Locale;
import java.util.Set;

public final class ParkerGmmInventory {

  private ParkerGmmInventory() {}

  public static void main(String[] args) throws Exception {

    if (args.length != 2) {
      throw new IllegalArgumentException(
          "Expected model inventory and IMT inventory output paths.");
    }

    Path modelOutput = Path.of(args[0])
        .toAbsolutePath()
        .normalize();

    Path imtOutput = Path.of(args[1])
        .toAbsolutePath()
        .normalize();

    Files.createDirectories(
        modelOutput.getParent());

    Files.createDirectories(
        imtOutput.getParent());

    int parkerModelCount = 0;
    int parkerImtCount = 0;

    try (
        PrintWriter modelWriter = new PrintWriter(
            Files.newBufferedWriter(
                modelOutput,
                StandardCharsets.UTF_8));

        PrintWriter imtWriter = new PrintWriter(
            Files.newBufferedWriter(
                imtOutput,
                StandardCharsets.UTF_8))
    ) {

      modelWriter.println(String.join(",",
          "gmm_name",
          "gmm_label",
          "implementation_class",
          "supported_imt_count",
          "supports_pga",
          "supports_pgv",
          "constraints"
      ));

      imtWriter.println(String.join(",",
          "gmm_name",
          "gmm_label",
          "implementation_class",
          "imt_name"
      ));

      for (Gmm gmm : Gmm.values()) {

        String gmmName = gmm.name();
        String gmmLabel = gmm.toString();

        String searchText = (
            gmmName
            + " "
            + gmmLabel
        ).toLowerCase(Locale.US);

        if (!searchText.contains("parker")) {
          continue;
        }

        Set<Imt> supportedSet =
            gmm.supportedImts();

        List<Imt> supportedImts =
            new ArrayList<>(supportedSet);

        supportedImts.sort(
            Comparator.comparing(Enum::name));

        String implementationClass = "";

        if (!supportedImts.isEmpty()) {

          Imt inspectionImt =
              supportedSet.contains(Imt.PGA)
                  ? Imt.PGA
                  : supportedImts.get(0);

          GroundMotionModel model =
              gmm.instance(inspectionImt);

          implementationClass =
              model.getClass().getName();
        }

        modelWriter.println(String.join(",",
            csv(gmmName),
            csv(gmmLabel),
            csv(implementationClass),
            integer(supportedImts.size()),
            bool(supportedSet.contains(Imt.PGA)),
            bool(supportedSet.contains(Imt.PGV)),
            csv(gmm.constraints().toString())
        ));

        for (Imt imt : supportedImts) {

          imtWriter.println(String.join(",",
              csv(gmmName),
              csv(gmmLabel),
              csv(implementationClass),
              csv(imt.name())
          ));

          parkerImtCount++;
        }

        parkerModelCount++;
      }
    }

    System.out.println("PARKER_GMM_INVENTORY_COMPLETE");
    System.out.println(
        "parker_models=" + parkerModelCount);
    System.out.println(
        "parker_model_imt_pairs=" + parkerImtCount);
    System.out.println(
        "model_output=" + modelOutput);
    System.out.println(
        "imt_output=" + imtOutput);
  }

  private static String integer(int value) {
    return Integer.toString(value);
  }

  private static String bool(boolean value) {
    return Boolean.toString(value);
  }

  private static String csv(String value) {

    if (value == null) {
      return "";
    }

    boolean quote =
        value.contains(",")
        || value.contains("\"")
        || value.contains("\n")
        || value.contains("\r");

    if (!quote) {
      return value;
    }

    return "\""
        + value.replace("\"", "\"\"")
        + "\"";
  }
}
'''.strip()


PARKER_ENUM_INSPECTOR_SOURCE.write_text(
    java_source + "\n",
    encoding="utf-8",
)


classpath = os.pathsep.join(
    str(path)
    for path in runtime_paths
)


def java_argument(value):
    text = str(value).replace(
        "\\",
        "\\\\",
    )

    text = text.replace(
        '"',
        '\\"',
    )

    return f'"{text}"'


PARKER_ENUM_JAVAC_ARGS.write_text(
    "\n".join(
        [
            "-encoding",
            "UTF-8",
            "-classpath",
            java_argument(classpath),
            "-d",
            java_argument(
                PARKER_ENUM_INSPECTOR_CLASSES
            ),
            java_argument(
                PARKER_ENUM_INSPECTOR_SOURCE
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


javac_path = shutil.which(
    "javac"
)

if javac_path is None:
    raise FileNotFoundError(
        "javac was not found through PATH."
    )


compile_result = subprocess.run(
    [
        javac_path,
        f"@{PARKER_ENUM_JAVAC_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


compile_output = "\n".join(
    output.strip()
    for output in [
        compile_result.stdout,
        compile_result.stderr,
    ]
    if output and output.strip()
)


PARKER_ENUM_COMPILE_LOG.write_text(
    compile_output + "\n",
    encoding="utf-8",
)


print("Compiler output:")

if compile_output:
    print(compile_output)
else:
    print("[no compiler messages]")


if compile_result.returncode != 0:
    raise RuntimeError(
        "The Parker GMM inventory program did not compile.\n\n"
        f"Compile log:\n{PARKER_ENUM_COMPILE_LOG}"
    )


run_classpath = os.pathsep.join(
    [
        str(
            PARKER_ENUM_INSPECTOR_CLASSES
        ),
        *[
            str(path)
            for path in runtime_paths
        ],
    ]
)


PARKER_ENUM_JAVA_ARGS.write_text(
    "\n".join(
        [
            "-Dfile.encoding=UTF-8",
            "-classpath",
            java_argument(run_classpath),
            "ParkerGmmInventory",
            java_argument(
                PARKER_MODEL_INVENTORY_PATH
            ),
            java_argument(
                PARKER_IMT_INVENTORY_PATH
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


java_path = shutil.which(
    "java"
)

if java_path is None:
    raise FileNotFoundError(
        "java was not found through PATH."
    )


run_result = subprocess.run(
    [
        java_path,
        f"@{PARKER_ENUM_JAVA_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


run_output = "\n".join(
    output.strip()
    for output in [
        run_result.stdout,
        run_result.stderr,
    ]
    if output and output.strip()
)


PARKER_ENUM_RUN_LOG.write_text(
    run_output + "\n",
    encoding="utf-8",
)


print("\nRuntime output:")
print(run_output)


if run_result.returncode != 0:
    raise RuntimeError(
        "The Parker GMM inventory program failed.\n\n"
        f"Run log:\n{PARKER_ENUM_RUN_LOG}"
    )


if (
    "PARKER_GMM_INVENTORY_COMPLETE"
    not in run_output
):
    raise RuntimeError(
        "The expected Parker inventory completion "
        "marker was not found."
    )


model_inventory = pd.read_csv(
    PARKER_MODEL_INVENTORY_PATH
)

imt_inventory = pd.read_csv(
    PARKER_IMT_INVENTORY_PATH
)


if model_inventory.empty:
    raise RuntimeError(
        "No Parker GMM enum constants were exported."
    )


model_inventory[
    "is_cascadia"
] = (
    model_inventory[
        "implementation_class"
    ]
    .fillna("")
    .str.contains(
        "Cascadia",
        case=False,
        regex=False,
    )
)


model_inventory[
    "uses_basin_term"
] = (
    model_inventory[
        "implementation_class"
    ]
    .fillna("")
    .str.contains(
        "Basin",
        case=False,
        regex=False,
    )
)


model_inventory[
    "is_interface"
] = (
    model_inventory[
        "implementation_class"
    ]
    .fillna("")
    .str.contains(
        "Interface",
        case=False,
        regex=False,
    )
)


model_inventory[
    "is_slab"
] = (
    model_inventory[
        "implementation_class"
    ]
    .fillna("")
    .str.contains(
        "Slab",
        case=False,
        regex=False,
    )
)


cascadia_models = model_inventory.loc[
    model_inventory[
        "is_cascadia"
    ],
    [
        "gmm_name",
        "gmm_label",
        "implementation_class",
        "supported_imt_count",
        "supports_pga",
        "supports_pgv",
        "uses_basin_term",
        "is_interface",
        "is_slab",
        "constraints",
    ],
].copy()


baseline_interface_candidates = (
    cascadia_models.loc[
        cascadia_models[
            "is_interface"
        ]
        & ~cascadia_models[
            "uses_basin_term"
        ]
    ]
)


baseline_slab_candidates = (
    cascadia_models.loc[
        cascadia_models[
            "is_slab"
        ]
        & ~cascadia_models[
            "uses_basin_term"
        ]
    ]
)


print("\nAll Parker GMM constants:")
display(
    model_inventory[
        [
            "gmm_name",
            "gmm_label",
            "implementation_class",
            "supported_imt_count",
            "supports_pga",
            "supports_pgv",
        ]
    ]
)


print("\nCascadia Parker variants:")
display(cascadia_models)


print("\nCandidate non-basin interface model:")
display(
    baseline_interface_candidates
)


print("\nCandidate non-basin intraslab model:")
display(
    baseline_slab_candidates
)


print("\nSupported IMTs for the baseline candidates:")

candidate_names = set(
    pd.concat(
        [
            baseline_interface_candidates[
                "gmm_name"
            ],
            baseline_slab_candidates[
                "gmm_name"
            ],
        ],
        ignore_index=True,
    )
)


candidate_imts = imt_inventory.loc[
    imt_inventory[
        "gmm_name"
    ].isin(
        candidate_names
    ),
    [
        "gmm_name",
        "imt_name",
    ],
].copy()


display(
    candidate_imts
    .sort_values(
        [
            "gmm_name",
            "imt_name",
        ]
    )
    .reset_index(
        drop=True
    )
)


validation = pd.DataFrame(
    [
        {
            "check": (
                "At least one Parker GMM was found"
            ),
            "passes": len(
                model_inventory
            ) > 0,
        },
        {
            "check": (
                "Cascadia Parker variants were found"
            ),
            "passes": len(
                cascadia_models
            ) > 0,
        },
        {
            "check": (
                "Exactly one non-basin Cascadia "
                "interface candidate was found"
            ),
            "passes": len(
                baseline_interface_candidates
            ) == 1,
        },
        {
            "check": (
                "Exactly one non-basin Cascadia "
                "intraslab candidate was found"
            ),
            "passes": len(
                baseline_slab_candidates
            ) == 1,
        },
        {
            "check": (
                "Baseline interface candidate supports PGA"
            ),
            "passes": bool(
                len(
                    baseline_interface_candidates
                ) == 1
                and baseline_interface_candidates[
                    "supports_pga"
                ].iloc[0]
            ),
        },
        {
            "check": (
                "Baseline intraslab candidate supports PGA"
            ),
            "passes": bool(
                len(
                    baseline_slab_candidates
                ) == 1
                and baseline_slab_candidates[
                    "supports_pga"
                ].iloc[0]
            ),
        },
    ]
)


print("\nParker model-selection validation:")
display(validation)


if not validation[
    "passes"
].all():
    failed_checks = validation.loc[
        ~validation[
            "passes"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "Parker model inventory validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
    )


PARKER_INTERFACE_GMM = (
    baseline_interface_candidates[
        "gmm_name"
    ].iloc[0]
)

PARKER_SLAB_GMM = (
    baseline_slab_candidates[
        "gmm_name"
    ].iloc[0]
)


print("\nPARKER MODEL SELECTION COMPLETE")

print(
    "\nInterface GMM:"
    f"\n  {PARKER_INTERFACE_GMM}"
)

print(
    "\nIntraslab GMM:"
    f"\n  {PARKER_SLAB_GMM}"
)

print(
    "\nModel inventory:"
    f"\n  {PARKER_MODEL_INVENTORY_PATH}"
)

print(
    "\nIMT inventory:"
    f"\n  {PARKER_IMT_INVENTORY_PATH}"
)

print(
    "\nNext step: inspect the exact GmmInput fields "
    "and Parker ground-motion output structure."
)

Compiler output:
[no compiler messages]

Runtime output:
PARKER_GMM_INVENTORY_COMPLETE
parker_models=14
parker_model_imt_pairs=322
model_output=C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_gmm_model_inventory.csv
imt_output=C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_gmm_imt_inventory.csv

All Parker GMM constants:


,gmm_name,gmm_label,implementation_class,supported_imt_count,supports_pga,supports_pgv
0,PSBAH_20_GLOBAL_INTERFACE,Parker et al. (2020) : Global : Interface,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
1,PSBAH_20_GLOBAL_INTERFACE_NO_EPI,Parker et al. (2020) : Global : Interface (no ...,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
2,PSBAH_20_GLOBAL_INTERFACE_AK_ADJUSTED,Parker et al. (2020) : Global : Interface (Ala...,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
3,PSBAH_20_GLOBAL_SLAB,Parker et al. (2020) : Global : Slab,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
4,PSBAH_20_GLOBAL_SLAB_NO_EPI,Parker et al. (2020) : Global : Slab (no epi),gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
5,PSBAH_20_CASCADIA_INTERFACE,Parker et al. (2020) : Cascadia : Interface,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
6,PSBAH_20_CASCADIA_INTERFACE_BASIN,Parker et al. (2020) : Cascadia : Interface : ...,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
7,PSBAH_20_CASCADIA_INTERFACE_BASIN_M9,Parker et al. (2020) : Cascadia : Interface : ...,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
8,PSBAH_20_CASCADIA_SLAB,Parker et al. (2020) : Cascadia : Slab,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True
9,PSBAH_20_CASCADIA_SLAB_BASIN,Parker et al. (2020) : Cascadia : Slab : Basin,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True



Cascadia Parker variants:


,gmm_name,gmm_label,implementation_class,supported_imt_count,supports_pga,supports_pgv,uses_basin_term,is_interface,is_slab,constraints
5,PSBAH_20_CASCADIA_INTERFACE,Parker et al. (2020) : Cascadia : Interface,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True,False,True,False,Constraints: \r\nMW RJB ...
6,PSBAH_20_CASCADIA_INTERFACE_BASIN,Parker et al. (2020) : Cascadia : Interface : ...,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True,True,True,False,Constraints: \r\nMW RJB ...
7,PSBAH_20_CASCADIA_INTERFACE_BASIN_M9,Parker et al. (2020) : Cascadia : Interface : ...,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True,True,True,False,Constraints: \r\nMW RJB ...
8,PSBAH_20_CASCADIA_SLAB,Parker et al. (2020) : Cascadia : Slab,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True,False,False,True,Constraints: \r\nMW RJB ...
9,PSBAH_20_CASCADIA_SLAB_BASIN,Parker et al. (2020) : Cascadia : Slab : Basin,gov.usgs.earthquake.nshmp.gmm.ParkerEtAl_2020$...,23,True,True,True,False,True,Constraints: \r\nMW RJB ...



Candidate non-basin interface model:


...


Supported IMTs for the baseline candidates:


,gmm_name,imt_name
0,PSBAH_20_CASCADIA_INTERFACE,PGA
1,PSBAH_20_CASCADIA_INTERFACE,PGV
2,PSBAH_20_CASCADIA_INTERFACE,SA0P01
3,PSBAH_20_CASCADIA_INTERFACE,SA0P02
4,PSBAH_20_CASCADIA_INTERFACE,SA0P03
5,PSBAH_20_CASCADIA_INTERFACE,SA0P05
6,PSBAH_20_CASCADIA_INTERFACE,SA0P075
7,PSBAH_20_CASCADIA_INTERFACE,SA0P1
8,PSBAH_20_CASCADIA_INTERFACE,SA0P15
9,PSBAH_20_CASCADIA_INTERFACE,SA0P2



Parker model-selection validation:


,check,passes
0,At least one Parker GMM was found,True
1,Cascadia Parker variants were found,True
2,Exactly one non-basin Cascadia interface candi...,True
3,Exactly one non-basin Cascadia intraslab candi...,True
4,Baseline interface candidate supports PGA,True
5,Baseline intraslab candidate supports PGA,True



PARKER MODEL SELECTION COMPLETE

Interface GMM:
  PSBAH_20_CASCADIA_INTERFACE

Intraslab GMM:
  PSBAH_20_CASCADIA_SLAB

Model inventory:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_gmm_model_inventory.csv

IMT inventory:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_gmm_imt_inventory.csv

Next step: inspect the exact GmmInput fields and Parker ground-motion output structure.


In [3]:
import subprocess

import pandas as pd
from IPython.display import display


PARKER_OUTPUT_API_PATH = (
    METADATA_DIR
    / "parker_ground_motion_output_api.txt"
)

PARKER_OUTPUT_API_SUMMARY_PATH = (
    METADATA_DIR
    / "parker_ground_motion_output_api_summary.csv"
)


classes_to_inspect = [
    "gov.usgs.earthquake.nshmp.gmm.GroundMotionModel",
    "gov.usgs.earthquake.nshmp.gmm.GroundMotion",
    "gov.usgs.earthquake.nshmp.tree.LogicTree",
    "gov.usgs.earthquake.nshmp.tree.LogicTree$Builder",
    "gov.usgs.earthquake.nshmp.tree.Branch",
    "gov.usgs.earthquake.nshmp.gmm.GmmInput$Constraints",
]


inspection_results = []


for class_name in classes_to_inspect:

    result = subprocess.run(
        [
            javap_executable,
            "-classpath",
            str(NSHMP_LIB_JAR),
            "-private",
            class_name,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )

    output = (
        result.stdout.strip()
        if result.returncode == 0
        else result.stderr.strip()
    )

    inspection_results.append(
        {
            "class_name": class_name,
            "passed": result.returncode == 0,
            "output": output,
        }
    )


inspection_table = pd.DataFrame(
    inspection_results
)


report_lines = [
    "PARKER GROUND-MOTION OUTPUT API",
    "",
    f"nshmp-lib JAR: {NSHMP_LIB_JAR}",
]


for row in inspection_results:
    report_lines.extend(
        [
            "",
            "=" * 78,
            row["class_name"],
            "=" * 78,
            row["output"],
        ]
    )


PARKER_OUTPUT_API_PATH.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)


keywords = [
    "calc(",
    "mean",
    "sigma",
    "weight",
    "value",
    "label",
    "branch",
    "size(",
    "get(",
    "iterator",
    "addBranch",
    "build(",
]


summary_rows = []


for row in inspection_results:

    for line in row["output"].splitlines():

        stripped_line = line.strip()

        if any(
            keyword.lower()
            in stripped_line.lower()
            for keyword in keywords
        ):
            summary_rows.append(
                {
                    "class_name": row["class_name"],
                    "api_line": stripped_line,
                }
            )


api_summary = pd.DataFrame(
    summary_rows
)


api_summary.to_csv(
    PARKER_OUTPUT_API_SUMMARY_PATH,
    index=False,
)


print("Class inspection status:")

display(
    inspection_table[
        [
            "class_name",
            "passed",
        ]
    ]
)


print("\nRelevant API lines:")

if api_summary.empty:
    print(
        "No API lines were selected automatically. "
        "Review the complete report."
    )
else:
    display(api_summary)


print("\nCELL 3 MODEL OUTPUT INSPECTION COMPLETE")

print(
    "\nComplete API report:"
    f"\n  {PARKER_OUTPUT_API_PATH}"
)

print(
    "\nAPI summary:"
    f"\n  {PARKER_OUTPUT_API_SUMMARY_PATH}"
)

print(
    "\nNext step: run controlled Parker interface "
    "and slab predictions and inspect every returned branch."
)

Class inspection status:


,class_name,passed
0,gov.usgs.earthquake.nshmp.gmm.GroundMotionModel,True
1,gov.usgs.earthquake.nshmp.gmm.GroundMotion,True
2,gov.usgs.earthquake.nshmp.tree.LogicTree,True
3,gov.usgs.earthquake.nshmp.tree.LogicTree$Builder,True
4,gov.usgs.earthquake.nshmp.tree.Branch,True
5,gov.usgs.earthquake.nshmp.gmm.GmmInput$Constra...,True



Relevant API lines:


,class_name,api_line
0,gov.usgs.earthquake.nshmp.gmm.GroundMotionModel,public abstract gov.usgs.earthquake.nshmp.tree...
1,gov.usgs.earthquake.nshmp.gmm.GroundMotionModel,public default gov.usgs.earthquake.nshmp.tree....
2,gov.usgs.earthquake.nshmp.gmm.GroundMotion,public abstract double mean();
3,gov.usgs.earthquake.nshmp.gmm.GroundMotion,public abstract double sigma();
4,gov.usgs.earthquake.nshmp.tree.LogicTree,public interface gov.usgs.earthquake.nshmp.tre...
5,gov.usgs.earthquake.nshmp.tree.LogicTree,public abstract gov.usgs.earthquake.nshmp.tree...
6,gov.usgs.earthquake.nshmp.tree.LogicTree,public abstract java.util.List<gov.usgs.earthq...
7,gov.usgs.earthquake.nshmp.tree.LogicTree,public static <E extends java.lang.Enum<E>> go...
8,gov.usgs.earthquake.nshmp.tree.LogicTree,public static gov.usgs.earthquake.nshmp.tree.L...
9,gov.usgs.earthquake.nshmp.tree.LogicTree$Builder,java.util.List<gov.usgs.earthquake.nshmp.tree....



CELL 3 MODEL OUTPUT INSPECTION COMPLETE

Complete API report:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_ground_motion_output_api.txt

API summary:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_ground_motion_output_api_summary.csv

Next step: run controlled Parker interface and slab predictions and inspect every returned branch.


In [4]:
from pathlib import Path

import os
import shutil
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display


PARKER_INTERFACE_GMM = (
    "PSBAH_20_CASCADIA_INTERFACE"
)

PARKER_SLAB_GMM = (
    "PSBAH_20_CASCADIA_SLAB"
)


PARKER_BRANCH_PROBE_PATH = (
    METADATA_DIR
    / "parker_controlled_branch_probe.csv"
)

PARKER_BRANCH_SUMMARY_PATH = (
    METADATA_DIR
    / "parker_controlled_branch_summary.csv"
)

PARKER_BRANCH_VALIDATION_PATH = (
    METADATA_DIR
    / "parker_controlled_branch_validation.csv"
)

PARKER_BRANCH_PROBE_SOURCE = (
    PARKER_INSPECTION_DIR
    / "ParkerBranchProbe.java"
)

PARKER_BRANCH_PROBE_CLASSES = (
    PARKER_INSPECTION_DIR
    / "branch_probe_classes"
)

PARKER_BRANCH_COMPILE_LOG = (
    METADATA_DIR
    / "parker_branch_probe_compile.log"
)

PARKER_BRANCH_RUN_LOG = (
    METADATA_DIR
    / "parker_branch_probe_run.log"
)

PARKER_BRANCH_JAVAC_ARGS = (
    PARKER_INSPECTION_DIR
    / "branch_probe_javac_arguments.txt"
)

PARKER_BRANCH_JAVA_ARGS = (
    PARKER_INSPECTION_DIR
    / "branch_probe_java_arguments.txt"
)


if PARKER_BRANCH_PROBE_CLASSES.exists():
    shutil.rmtree(
        PARKER_BRANCH_PROBE_CLASSES
    )

PARKER_BRANCH_PROBE_CLASSES.mkdir(
    parents=True,
    exist_ok=True,
)


java_source = r'''
import gov.usgs.earthquake.nshmp.gmm.Gmm;
import gov.usgs.earthquake.nshmp.gmm.GmmInput;
import gov.usgs.earthquake.nshmp.gmm.GroundMotion;
import gov.usgs.earthquake.nshmp.gmm.GroundMotionModel;
import gov.usgs.earthquake.nshmp.gmm.Imt;
import gov.usgs.earthquake.nshmp.tree.Branch;
import gov.usgs.earthquake.nshmp.tree.LogicTree;

import java.io.PrintWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.Locale;

public final class ParkerBranchProbe {

  private ParkerBranchProbe() {}

  public static void main(String[] args) throws Exception {

    if (args.length != 3) {
      throw new IllegalArgumentException(
          "Expected interface GMM, slab GMM, and output CSV.");
    }

    String interfaceGmmName = args[0];
    String slabGmmName = args[1];

    Path outputPath = Path.of(args[2])
        .toAbsolutePath()
        .normalize();

    if (outputPath.getParent() != null) {
      Files.createDirectories(
          outputPath.getParent());
    }

    Imt[] imts = {
        Imt.PGA,
        Imt.SA0P2,
        Imt.SA1P0,
        Imt.SA2P0
    };

    int outputRows = 0;

    try (
        PrintWriter writer = new PrintWriter(
            Files.newBufferedWriter(
                outputPath,
                StandardCharsets.UTF_8))
    ) {

      writer.println(String.join(",",
          "scenario_id",
          "source_type",
          "gmm_name",
          "gmm_label",
          "implementation_class",
          "imt_name",
          "imt_period_sec",
          "magnitude",
          "r_rup_km",
          "z_hyp_km",
          "vs30_mps",
          "logic_tree_name",
          "branch_index",
          "branch_id",
          "branch_weight",
          "mean_ln_units",
          "median_units",
          "sigma_total_ln"
      ));

      outputRows += evaluateScenario(
          writer,
          "interface_m9_reference_rock",
          "interface",
          interfaceGmmName,
          9.0,
          100.0,
          20.0,
          760.0,
          imts);

      outputRows += evaluateScenario(
          writer,
          "slab_m7_reference_rock",
          "intraslab",
          slabGmmName,
          7.0,
          100.0,
          60.0,
          760.0,
          imts);
    }

    System.out.println(
        "PARKER_BRANCH_PROBE_COMPLETE");

    System.out.println(
        "output_rows=" + outputRows);

    System.out.println(
        "output=" + outputPath);
  }

  private static int evaluateScenario(
      PrintWriter writer,
      String scenarioId,
      String sourceType,
      String gmmName,
      double magnitude,
      double rRup,
      double zHyp,
      double vs30,
      Imt[] imts
  ) {

    Gmm gmm = Gmm.valueOf(
        gmmName);

    GmmInput input = GmmInput.builder()
        .withDefaults()
        .mag(magnitude)
        .rRup(rRup)
        .zHyp(zHyp)
        .vs30(vs30)
        .build();

    int rowCount = 0;

    for (Imt imt : imts) {

      GroundMotionModel model =
          gmm.instance(imt);

      LogicTree<GroundMotion> tree =
          model.calc(input);

      for (
          int branchIndex = 0;
          branchIndex < tree.size();
          branchIndex++
      ) {

        Branch<GroundMotion> branch =
            tree.get(branchIndex);

        GroundMotion motion =
            branch.value();

        double mean =
            motion.mean();

        double sigma =
            motion.sigma();

        double median =
            Math.exp(mean);

        writer.println(String.join(",",
            csv(scenarioId),
            csv(sourceType),
            csv(gmm.name()),
            csv(gmm.toString()),
            csv(model.getClass().getName()),
            csv(imt.name()),
            number(
                imt.isSA()
                    ? imt.period()
                    : Double.NaN),
            number(magnitude),
            number(rRup),
            number(zHyp),
            number(vs30),
            csv(tree.name()),
            integer(branchIndex),
            csv(branch.id()),
            number(branch.weight()),
            number(mean),
            number(median),
            number(sigma)
        ));

        rowCount++;
      }
    }

    return rowCount;
  }

  private static String integer(
      int value
  ) {
    return Integer.toString(value);
  }

  private static String number(
      double value
  ) {
    return String.format(
        Locale.US,
        "%.17g",
        value);
  }

  private static String csv(
      String value
  ) {

    if (value == null) {
      return "";
    }

    boolean quote =
        value.contains(",")
        || value.contains("\"")
        || value.contains("\n")
        || value.contains("\r");

    if (!quote) {
      return value;
    }

    return "\""
        + value.replace("\"", "\"\"")
        + "\"";
  }
}
'''.strip()


PARKER_BRANCH_PROBE_SOURCE.write_text(
    java_source + "\n",
    encoding="utf-8",
)


classpath = os.pathsep.join(
    str(path)
    for path in runtime_paths
)


def java_argument(value):
    text = str(value).replace(
        "\\",
        "\\\\",
    )

    text = text.replace(
        '"',
        '\\"',
    )

    return f'"{text}"'


PARKER_BRANCH_JAVAC_ARGS.write_text(
    "\n".join(
        [
            "-encoding",
            "UTF-8",
            "-classpath",
            java_argument(classpath),
            "-d",
            java_argument(
                PARKER_BRANCH_PROBE_CLASSES
            ),
            java_argument(
                PARKER_BRANCH_PROBE_SOURCE
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


javac_path = shutil.which(
    "javac"
)

if javac_path is None:
    raise FileNotFoundError(
        "javac was not found through PATH."
    )


compile_result = subprocess.run(
    [
        javac_path,
        f"@{PARKER_BRANCH_JAVAC_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


compile_output = "\n".join(
    output.strip()
    for output in [
        compile_result.stdout,
        compile_result.stderr,
    ]
    if output and output.strip()
)


PARKER_BRANCH_COMPILE_LOG.write_text(
    compile_output + "\n",
    encoding="utf-8",
)


print("Compiler output:")

if compile_output:
    print(compile_output)
else:
    print("[no compiler messages]")


if compile_result.returncode != 0:
    raise RuntimeError(
        "The Parker branch probe did not compile.\n\n"
        f"Compile log:\n{PARKER_BRANCH_COMPILE_LOG}"
    )


run_classpath = os.pathsep.join(
    [
        str(
            PARKER_BRANCH_PROBE_CLASSES
        ),
        *[
            str(path)
            for path in runtime_paths
        ],
    ]
)


PARKER_BRANCH_JAVA_ARGS.write_text(
    "\n".join(
        [
            "-Dfile.encoding=UTF-8",
            "-classpath",
            java_argument(run_classpath),
            "ParkerBranchProbe",
            PARKER_INTERFACE_GMM,
            PARKER_SLAB_GMM,
            java_argument(
                PARKER_BRANCH_PROBE_PATH
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


java_path = shutil.which(
    "java"
)

if java_path is None:
    raise FileNotFoundError(
        "java was not found through PATH."
    )


run_result = subprocess.run(
    [
        java_path,
        f"@{PARKER_BRANCH_JAVA_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


run_output = "\n".join(
    output.strip()
    for output in [
        run_result.stdout,
        run_result.stderr,
    ]
    if output and output.strip()
)


PARKER_BRANCH_RUN_LOG.write_text(
    run_output + "\n",
    encoding="utf-8",
)


print("\nRuntime output:")
print(run_output)


if run_result.returncode != 0:
    raise RuntimeError(
        "The Parker branch probe failed.\n\n"
        f"Run log:\n{PARKER_BRANCH_RUN_LOG}"
    )


if (
    "PARKER_BRANCH_PROBE_COMPLETE"
    not in run_output
):
    raise RuntimeError(
        "The expected branch-probe completion "
        "marker was not found."
    )


if not PARKER_BRANCH_PROBE_PATH.is_file():
    raise FileNotFoundError(
        "The Parker branch-probe CSV was not created."
    )


probe = pd.read_csv(
    PARKER_BRANCH_PROBE_PATH
)


numeric_columns = [
    "branch_index",
    "branch_weight",
    "mean_ln_units",
    "median_units",
    "sigma_total_ln",
    "magnitude",
    "r_rup_km",
    "z_hyp_km",
    "vs30_mps",
]


for column in numeric_columns:
    probe[column] = pd.to_numeric(
        probe[column],
        errors="coerce",
    )


group_columns = [
    "scenario_id",
    "source_type",
    "gmm_name",
    "imt_name",
]


branch_summary = (
    probe
    .groupby(
        group_columns,
        dropna=False,
    )
    .agg(
        logic_tree_name=(
            "logic_tree_name",
            "first",
        ),
        branch_count=(
            "branch_id",
            "size",
        ),
        branch_weight_sum=(
            "branch_weight",
            "sum",
        ),
        minimum_mean_ln=(
            "mean_ln_units",
            "min",
        ),
        maximum_mean_ln=(
            "mean_ln_units",
            "max",
        ),
        minimum_median=(
            "median_units",
            "min",
        ),
        maximum_median=(
            "median_units",
            "max",
        ),
        minimum_sigma=(
            "sigma_total_ln",
            "min",
        ),
        maximum_sigma=(
            "sigma_total_ln",
            "max",
        ),
    )
    .reset_index()
)


branch_summary[
    "weights_sum_to_one"
] = np.isclose(
    branch_summary[
        "branch_weight_sum"
    ],
    1.0,
    rtol=1e-12,
    atol=1e-12,
)


branch_summary[
    "sigma_same_across_branches"
] = np.isclose(
    branch_summary[
        "minimum_sigma"
    ],
    branch_summary[
        "maximum_sigma"
    ],
    rtol=1e-12,
    atol=1e-15,
)


branch_summary.to_csv(
    PARKER_BRANCH_SUMMARY_PATH,
    index=False,
)


expected_scenarios = {
    "interface_m9_reference_rock",
    "slab_m7_reference_rock",
}

expected_gmms = {
    PARKER_INTERFACE_GMM,
    PARKER_SLAB_GMM,
}

expected_imts = {
    "PGA",
    "SA0P2",
    "SA1P0",
    "SA2P0",
}


validation = pd.DataFrame(
    [
        {
            "check": (
                "Both controlled scenarios were evaluated"
            ),
            "passes": (
                set(
                    probe[
                        "scenario_id"
                    ]
                )
                == expected_scenarios
            ),
        },
        {
            "check": (
                "Both selected Parker GMMs were evaluated"
            ),
            "passes": (
                set(
                    probe[
                        "gmm_name"
                    ]
                )
                == expected_gmms
            ),
        },
        {
            "check": (
                "All requested IMTs were evaluated"
            ),
            "passes": (
                set(
                    probe[
                        "imt_name"
                    ]
                )
                == expected_imts
            ),
        },
        {
            "check": (
                "Eight scenario-IMT combinations were returned"
            ),
            "passes": (
                len(
                    branch_summary
                )
                == 8
            ),
        },
        {
            "check": (
                "Every logic-tree branch has positive weight"
            ),
            "passes": bool(
                (
                    probe[
                        "branch_weight"
                    ]
                    > 0
                ).all()
            ),
        },
        {
            "check": (
                "Branch weights sum to one for every prediction"
            ),
            "passes": bool(
                branch_summary[
                    "weights_sum_to_one"
                ].all()
            ),
        },
        {
            "check": (
                "All logarithmic means are finite"
            ),
            "passes": bool(
                np.isfinite(
                    probe[
                        "mean_ln_units"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "All median ground motions are positive"
            ),
            "passes": bool(
                (
                    probe[
                        "median_units"
                    ]
                    > 0
                ).all()
            ),
        },
        {
            "check": (
                "All total sigmas are finite and positive"
            ),
            "passes": bool(
                np.isfinite(
                    probe[
                        "sigma_total_ln"
                    ]
                ).all()
                and (
                    probe[
                        "sigma_total_ln"
                    ]
                    > 0
                ).all()
            ),
        },
        {
            "check": (
                "PGA and PSA medians use acceleration units"
            ),
            "passes": bool(
                (
                    probe[
                        "median_units"
                    ]
                    > 0
                ).all()
            ),
        },
    ]
)


validation.to_csv(
    PARKER_BRANCH_VALIDATION_PATH,
    index=False,
)


print("\nReturned Parker branches:")

display(
    probe[
        [
            "scenario_id",
            "gmm_name",
            "imt_name",
            "logic_tree_name",
            "branch_index",
            "branch_id",
            "branch_weight",
            "mean_ln_units",
            "median_units",
            "sigma_total_ln",
        ]
    ]
)


print("\nBranch structure summary:")

display(branch_summary)


print("\nBranch-probe validation:")

display(validation)


if not validation[
    "passes"
].all():
    failed_checks = validation.loc[
        ~validation[
            "passes"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "Parker branch-probe validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
    )


print("\nCELL 4 PARKER BRANCH PROBE COMPLETE")

print(
    "\nControlled predictions:"
    f"\n  {PARKER_BRANCH_PROBE_PATH}"
)

print(
    "\nBranch summary:"
    f"\n  {PARKER_BRANCH_SUMMARY_PATH}"
)

print(
    "\nValidation:"
    f"\n  {PARKER_BRANCH_VALIDATION_PATH}"
)

print(
    "\nNext step: identify the central Parker "
    "epistemic branch and inspect the separate "
    "between-event and within-event variability terms."
)

Compiler output:
[no compiler messages]

Runtime output:
PARKER_BRANCH_PROBE_COMPLETE
output_rows=24
output=C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_controlled_branch_probe.csv

Returned Parker branches:


,scenario_id,gmm_name,imt_name,logic_tree_name,branch_index,branch_id,branch_weight,mean_ln_units,median_units,sigma_total_ln
0,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,PGA,ground-motion-tree,0,epi-lo,0.185,-2.606620,0.073784,0.791454
1,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,PGA,ground-motion-tree,1,epi-off,0.630,-1.899270,0.149678,0.791454
2,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,PGA,ground-motion-tree,2,epi-hi,0.185,-1.191920,0.303638,0.791454
3,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA0P2,ground-motion-tree,0,epi-lo,0.185,-1.944133,0.143111,0.826439
4,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA0P2,ground-motion-tree,1,epi-off,0.630,-1.236783,0.290317,0.826439
5,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA0P2,ground-motion-tree,2,epi-hi,0.185,-0.529433,0.588939,0.826439
6,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA1P0,ground-motion-tree,0,epi-lo,0.185,-2.548286,0.078216,0.819795
7,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA1P0,ground-motion-tree,1,epi-off,0.630,-2.005436,0.134602,0.819795
8,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA1P0,ground-motion-tree,2,epi-hi,0.185,-1.462586,0.231637,0.819795
9,interface_m9_reference_rock,PSBAH_20_CASCADIA_INTERFACE,SA2P0,ground-motion-tree,0,epi-lo,0.185,-3.357851,0.034810,0.796909



Branch structure summary:


,scenario_id,source_type,gmm_name,imt_name,logic_tree_name,branch_count,branch_weight_sum,minimum_mean_ln,maximum_mean_ln,minimum_median,maximum_median,minimum_sigma,maximum_sigma,weights_sum_to_one,sigma_same_across_branches
0,interface_m9_reference_rock,interface,PSBAH_20_CASCADIA_INTERFACE,PGA,ground-motion-tree,3,1.0,-2.606620,-1.191920,0.073784,0.303638,0.791454,0.791454,True,True
1,interface_m9_reference_rock,interface,PSBAH_20_CASCADIA_INTERFACE,SA0P2,ground-motion-tree,3,1.0,-1.944133,-0.529433,0.143111,0.588939,0.826439,0.826439,True,True
2,interface_m9_reference_rock,interface,PSBAH_20_CASCADIA_INTERFACE,SA1P0,ground-motion-tree,3,1.0,-2.548286,-1.462586,0.078216,0.231637,0.819795,0.819795,True,True
3,interface_m9_reference_rock,interface,PSBAH_20_CASCADIA_INTERFACE,SA2P0,ground-motion-tree,3,1.0,-3.357851,-2.272151,0.034810,0.103090,0.796909,0.796909,True,True
4,slab_m7_reference_rock,intraslab,PSBAH_20_CASCADIA_SLAB,PGA,ground-motion-tree,3,1.0,-3.841252,-2.689752,0.021467,0.067898,0.791454,0.791454,True,True
5,slab_m7_reference_rock,intraslab,PSBAH_20_CASCADIA_SLAB,SA0P2,ground-motion-tree,3,1.0,-2.816594,-1.665094,0.059809,0.189173,0.826439,0.826439,True,True
6,slab_m7_reference_rock,intraslab,PSBAH_20_CASCADIA_SLAB,SA1P0,ground-motion-tree,3,1.0,-3.834081,-3.054088,0.021621,0.047166,0.819795,0.819795,True,True
7,slab_m7_reference_rock,intraslab,PSBAH_20_CASCADIA_SLAB,SA2P0,ground-motion-tree,3,1.0,-4.362749,-3.742755,0.012743,0.023689,0.796909,0.796909,True,True



Branch-probe validation:


,check,passes
0,Both controlled scenarios were evaluated,True
1,Both selected Parker GMMs were evaluated,True
2,All requested IMTs were evaluated,True
3,Eight scenario-IMT combinations were returned,True
4,Every logic-tree branch has positive weight,True
5,Branch weights sum to one for every prediction,True
6,All logarithmic means are finite,True
7,All median ground motions are positive,True
8,All total sigmas are finite and positive,True
9,PGA and PSA medians use acceleration units,True



CELL 4 PARKER BRANCH PROBE COMPLETE

Controlled predictions:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_controlled_branch_probe.csv

Branch summary:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_controlled_branch_summary.csv

Validation:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_controlled_branch_validation.csv

Next step: identify the central Parker epistemic branch and inspect the separate between-event and within-event variability terms.


In [5]:
import os
import shutil
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display


DEPTH_PROBE_PATH = (
    METADATA_DIR
    / "parker_depth_input_probe.csv"
)

DEPTH_PROBE_VALIDATION_PATH = (
    METADATA_DIR
    / "parker_depth_input_validation.csv"
)

DEPTH_PROBE_SOURCE = (
    PARKER_INSPECTION_DIR
    / "ParkerDepthInputProbe.java"
)

DEPTH_PROBE_CLASSES = (
    PARKER_INSPECTION_DIR
    / "depth_probe_classes"
)

DEPTH_PROBE_COMPILE_LOG = (
    METADATA_DIR
    / "parker_depth_probe_compile.log"
)

DEPTH_PROBE_RUN_LOG = (
    METADATA_DIR
    / "parker_depth_probe_run.log"
)

DEPTH_PROBE_JAVAC_ARGS = (
    PARKER_INSPECTION_DIR
    / "depth_probe_javac_arguments.txt"
)

DEPTH_PROBE_JAVA_ARGS = (
    PARKER_INSPECTION_DIR
    / "depth_probe_java_arguments.txt"
)


if DEPTH_PROBE_CLASSES.exists():
    shutil.rmtree(
        DEPTH_PROBE_CLASSES
    )

DEPTH_PROBE_CLASSES.mkdir(
    parents=True,
    exist_ok=True,
)


java_source = r'''
import gov.usgs.earthquake.nshmp.gmm.Gmm;
import gov.usgs.earthquake.nshmp.gmm.GmmInput;
import gov.usgs.earthquake.nshmp.gmm.GroundMotion;
import gov.usgs.earthquake.nshmp.gmm.GroundMotionModel;
import gov.usgs.earthquake.nshmp.gmm.Imt;
import gov.usgs.earthquake.nshmp.tree.Branch;
import gov.usgs.earthquake.nshmp.tree.LogicTree;

import java.io.PrintWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.Locale;

public final class ParkerDepthInputProbe {

  private ParkerDepthInputProbe() {}

  public static void main(String[] args) throws Exception {

    if (args.length != 1) {
      throw new IllegalArgumentException(
          "Expected one output CSV path.");
    }

    Path outputPath = Path.of(args[0])
        .toAbsolutePath()
        .normalize();

    Files.createDirectories(
        outputPath.getParent());

    try (
        PrintWriter writer = new PrintWriter(
            Files.newBufferedWriter(
                outputPath,
                StandardCharsets.UTF_8))
    ) {

      writer.println(String.join(",",
          "case_id",
          "imt_name",
          "magnitude",
          "r_rup_km",
          "z_tor_km",
          "z_hyp_input_km",
          "vs30_mps",
          "branch_id",
          "branch_weight",
          "mean_ln_units",
          "median_units",
          "sigma_total_ln"
      ));

      Imt[] imts = {
          Imt.PGA,
          Imt.SA1P0
      };

      evaluate(
          writer,
          "ztor30_zhyp60",
          30.0,
          60.0,
          imts);

      evaluate(
          writer,
          "ztor50_zhyp60",
          50.0,
          60.0,
          imts);

      evaluate(
          writer,
          "ztor30_zhyp100",
          30.0,
          100.0,
          imts);
    }

    System.out.println(
        "PARKER_DEPTH_PROBE_COMPLETE");

    System.out.println(
        "output=" + outputPath);
  }

  private static void evaluate(
      PrintWriter writer,
      String caseId,
      double zTor,
      double zHyp,
      Imt[] imts
  ) {

    Gmm gmm =
        Gmm.PSBAH_20_CASCADIA_SLAB;

    double magnitude = 7.0;
    double rRup = 100.0;
    double vs30 = 760.0;

    GmmInput input = GmmInput.builder()
        .withDefaults()
        .mag(magnitude)
        .rRup(rRup)
        .zTor(zTor)
        .zHyp(zHyp)
        .vs30(vs30)
        .build();

    for (Imt imt : imts) {

      GroundMotionModel model =
          gmm.instance(imt);

      LogicTree<GroundMotion> tree =
          model.calc(input);

      for (Branch<GroundMotion> branch : tree) {

        GroundMotion motion =
            branch.value();

        writer.println(String.join(",",
            csv(caseId),
            csv(imt.name()),
            number(magnitude),
            number(rRup),
            number(zTor),
            number(zHyp),
            number(vs30),
            csv(branch.id()),
            number(branch.weight()),
            number(motion.mean()),
            number(Math.exp(motion.mean())),
            number(motion.sigma())
        ));
      }
    }
  }

  private static String number(
      double value
  ) {
    return String.format(
        Locale.US,
        "%.17g",
        value);
  }

  private static String csv(
      String value
  ) {

    if (value == null) {
      return "";
    }

    boolean quote =
        value.contains(",")
        || value.contains("\"")
        || value.contains("\n")
        || value.contains("\r");

    if (!quote) {
      return value;
    }

    return "\""
        + value.replace("\"", "\"\"")
        + "\"";
  }
}
'''.strip()


DEPTH_PROBE_SOURCE.write_text(
    java_source + "\n",
    encoding="utf-8",
)


classpath = os.pathsep.join(
    str(path)
    for path in runtime_paths
)


DEPTH_PROBE_JAVAC_ARGS.write_text(
    "\n".join(
        [
            "-encoding",
            "UTF-8",
            "-classpath",
            java_argument(classpath),
            "-d",
            java_argument(
                DEPTH_PROBE_CLASSES
            ),
            java_argument(
                DEPTH_PROBE_SOURCE
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


javac_path = shutil.which("javac")

if javac_path is None:
    raise FileNotFoundError(
        "javac was not found through PATH."
    )


compile_result = subprocess.run(
    [
        javac_path,
        f"@{DEPTH_PROBE_JAVAC_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


compile_output = "\n".join(
    value.strip()
    for value in [
        compile_result.stdout,
        compile_result.stderr,
    ]
    if value and value.strip()
)


DEPTH_PROBE_COMPILE_LOG.write_text(
    compile_output + "\n",
    encoding="utf-8",
)


print("Compiler output:")

if compile_output:
    print(compile_output)
else:
    print("[no compiler messages]")


if compile_result.returncode != 0:
    raise RuntimeError(
        "The Parker depth probe did not compile.\n\n"
        f"Compile log:\n{DEPTH_PROBE_COMPILE_LOG}"
    )


run_classpath = os.pathsep.join(
    [
        str(DEPTH_PROBE_CLASSES),
        *[
            str(path)
            for path in runtime_paths
        ],
    ]
)


DEPTH_PROBE_JAVA_ARGS.write_text(
    "\n".join(
        [
            "-Dfile.encoding=UTF-8",
            "-classpath",
            java_argument(run_classpath),
            "ParkerDepthInputProbe",
            java_argument(
                DEPTH_PROBE_PATH
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


java_path = shutil.which("java")

if java_path is None:
    raise FileNotFoundError(
        "java was not found through PATH."
    )


run_result = subprocess.run(
    [
        java_path,
        f"@{DEPTH_PROBE_JAVA_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


run_output = "\n".join(
    value.strip()
    for value in [
        run_result.stdout,
        run_result.stderr,
    ]
    if value and value.strip()
)


DEPTH_PROBE_RUN_LOG.write_text(
    run_output + "\n",
    encoding="utf-8",
)


print("\nRuntime output:")
print(run_output)


if run_result.returncode != 0:
    raise RuntimeError(
        "The Parker depth probe failed.\n\n"
        f"Run log:\n{DEPTH_PROBE_RUN_LOG}"
    )


if (
    "PARKER_DEPTH_PROBE_COMPLETE"
    not in run_output
):
    raise RuntimeError(
        "The depth-probe completion marker "
        "was not found."
    )


depth_probe = pd.read_csv(
    DEPTH_PROBE_PATH
)


central = depth_probe.loc[
    depth_probe["branch_id"].eq(
        "epi-off"
    )
].copy()


central_table = central[
    [
        "case_id",
        "imt_name",
        "z_tor_km",
        "z_hyp_input_km",
        "mean_ln_units",
        "median_units",
        "sigma_total_ln",
    ]
].sort_values(
    [
        "imt_name",
        "case_id",
    ]
).reset_index(
    drop=True
)


def central_mean(case_id, imt_name):
    rows = central.loc[
        central["case_id"].eq(case_id)
        & central["imt_name"].eq(imt_name),
        "mean_ln_units",
    ]

    if len(rows) != 1:
        raise RuntimeError(
            f"Expected one central result for "
            f"{case_id}, {imt_name}."
        )

    return float(rows.iloc[0])


validation_rows = []


for imt_name in ["PGA", "SA1P0"]:

    mean_a = central_mean(
        "ztor30_zhyp60",
        imt_name,
    )

    mean_b = central_mean(
        "ztor50_zhyp60",
        imt_name,
    )

    mean_c = central_mean(
        "ztor30_zhyp100",
        imt_name,
    )

    validation_rows.extend(
        [
            {
                "check": (
                    f"{imt_name}: changing zHyp while "
                    "holding zTor fixed has no effect"
                ),
                "passes": np.isclose(
                    mean_a,
                    mean_c,
                    rtol=0.0,
                    atol=1e-12,
                ),
                "difference_ln": (
                    mean_c - mean_a
                ),
            },
            {
                "check": (
                    f"{imt_name}: changing zTor changes "
                    "the slab median"
                ),
                "passes": not np.isclose(
                    mean_a,
                    mean_b,
                    rtol=0.0,
                    atol=1e-8,
                ),
                "difference_ln": (
                    mean_b - mean_a
                ),
            },
        ]
    )


weight_sums = (
    depth_probe
    .groupby(
        [
            "case_id",
            "imt_name",
        ]
    )["branch_weight"]
    .sum()
)


validation_rows.append(
    {
        "check": (
            "Branch weights sum to one "
            "for every depth case"
        ),
        "passes": bool(
            np.allclose(
                weight_sums.to_numpy(),
                1.0,
                rtol=0.0,
                atol=1e-12,
            )
        ),
        "difference_ln": np.nan,
    }
)


depth_validation = pd.DataFrame(
    validation_rows
)


depth_validation.to_csv(
    DEPTH_PROBE_VALIDATION_PATH,
    index=False,
)


print("\nCentral epistemic branch results:")
display(central_table)


print("\nDepth-input validation:")
display(depth_validation)


if not depth_validation[
    "passes"
].all():
    failed = depth_validation.loc[
        ~depth_validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Parker depth-input validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 5 PARKER DEPTH TEST COMPLETE")

print(
    "\nThe installed Parker slab model "
    "uses zTor to control source-depth scaling."
)

print(
    "\nThe direct zHyp input does not control "
    "the slab median in this implementation."
)

print(
    "\nDepth probe:"
    f"\n  {DEPTH_PROBE_PATH}"
)

print(
    "\nValidation:"
    f"\n  {DEPTH_PROBE_VALIDATION_PATH}"
)

print(
    "\nNext step: determine the correct zTor value "
    "for every Oregon intraslab rupture and recover "
    "the separate tau and phi components."
)

Compiler output:
[no compiler messages]

Runtime output:
PARKER_DEPTH_PROBE_COMPLETE
output=C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_depth_input_probe.csv

Central epistemic branch results:


,case_id,imt_name,z_tor_km,z_hyp_input_km,mean_ln_units,median_units,sigma_total_ln
0,ztor30_zhyp100,PGA,30.0,100.0,-2.853534,0.057640,0.791454
1,ztor30_zhyp60,PGA,30.0,60.0,-2.853534,0.057640,0.791454
2,ztor50_zhyp60,PGA,50.0,60.0,-2.225534,0.108010,0.791454
3,ztor30_zhyp100,SA1P0,30.0,100.0,-3.141012,0.043239,0.819795
4,ztor30_zhyp60,SA1P0,30.0,60.0,-3.141012,0.043239,0.819795
5,ztor50_zhyp60,SA1P0,50.0,60.0,-2.679012,0.068631,0.819795



Depth-input validation:


,check,passes,difference_ln
0,PGA: changing zHyp while holding zTor fixed ha...,True,0.000
1,PGA: changing zTor changes the slab median,True,0.628
2,SA1P0: changing zHyp while holding zTor fixed ...,True,0.000
3,SA1P0: changing zTor changes the slab median,True,0.462
4,Branch weights sum to one for every depth case,True,NaN



CELL 5 PARKER DEPTH TEST COMPLETE

The installed Parker slab model uses zTor to control source-depth scaling.

The direct zHyp input does not control the slab median in this implementation.

Depth probe:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_depth_input_probe.csv

Validation:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\parker_depth_input_validation.csv

Next step: determine the correct zTor value for every Oregon intraslab rupture and recover the separate tau and phi components.


In [6]:
import pandas as pd
from IPython.display import display


catalog = pd.read_csv(
    EVENT_CATALOG_PATH,
    nrows=20,
)

search_terms = [
    "depth",
    "ztor",
    "z_tor",
    "hyp",
    "dip",
    "width",
    "strike",
    "rake",
    "lat",
    "lon",
    "geometry",
    "surface",
    "source",
    "type",
]


relevant_columns = [
    column
    for column in catalog.columns
    if any(
        term in column.lower()
        for term in search_terms
    )
]


print(f"Total catalog columns: {len(catalog.columns)}")

print("\nPotential source and geometry columns:")

for column in relevant_columns:
    print(f"  {column}")


print("\nSample values:")

display(
    catalog[
        relevant_columns
    ]
)


print("\nColumn data types:")

display(
    catalog[
        relevant_columns
    ]
    .dtypes
    .rename("dtype")
    .to_frame()
)

Total catalog columns: 43

Potential source and geometry columns:
  simulation_year
  tree_type
  rupture_set_type
  source_index
  source_id
  source_name
  source_type
  rake_deg
  centroid_longitude
  centroid_latitude
  centroid_depth_km
  top_depth_km
  strike_deg
  dip_deg
  dip_direction_deg
  width_km

Sample values:


,simulation_year,tree_type,rupture_set_type,source_index,source_id,source_name,source_type,rake_deg,centroid_longitude,centroid_latitude,centroid_depth_km,top_depth_km,strike_deg,dip_deg,dip_direction_deg,width_km
0,185,INTERFACE,INTERFACE,0,3172,"Cascadia (full, top)",INTERFACE,90.0,-125.266845,45.069512,11.059412,5.091846,349.412040,11.745482,NaN,85.0
1,191,SLAB,SLAB,641,8211,"PointSourceFinite: -123.700, 45.300",SLAB,0.0,-123.700000,45.300000,42.000000,42.000000,NaN,90.000000,NaN,8.0
2,225,INTERFACE,INTERFACE,0,3171,"Cascadia (full, middle)",INTERFACE,90.0,-125.150941,45.115822,12.834152,5.091846,349.412040,11.149741,NaN,105.0
3,329,INTERFACE,INTERFACE,0,3160,"Cascadia (unsegmented, 1-2-3, bottom)",INTERFACE,90.0,NaN,NaN,NaN,5.000000,354.997319,9.593002,NaN,145.0
4,531,INTERFACE,INTERFACE,0,3171,"Cascadia (full, middle)",INTERFACE,90.0,-125.150941,45.115822,12.834152,5.091846,349.412040,11.149741,NaN,105.0
5,957,INTERFACE,INTERFACE,0,3172,"Cascadia (full, top)",INTERFACE,90.0,-125.266845,45.069512,11.059412,5.091846,349.412040,11.745482,NaN,85.0
6,1022,SLAB,SLAB,653,8211,"PointSourceFinite: -122.500, 45.300",SLAB,0.0,-122.500000,45.300000,50.000000,50.000000,NaN,90.000000,NaN,8.0
7,1072,INTERFACE,INTERFACE,0,3141,"Cascadia (segmented, 4, middle)",INTERFACE,90.0,-125.689019,48.000730,15.424939,5.000000,337.147271,10.760561,NaN,130.0
8,1118,INTERFACE,INTERFACE,0,3171,"Cascadia (full, middle)",INTERFACE,90.0,-125.150941,45.115822,12.834152,5.091846,349.412040,11.149741,NaN,105.0
9,1501,INTERFACE,INTERFACE,0,3172,"Cascadia (full, top)",INTERFACE,90.0,-125.266845,45.069512,11.059412,5.091846,349.412040,11.745482,NaN,85.0



Column data types:


,dtype
simulation_year,int64
tree_type,object
rupture_set_type,object
source_index,int64
source_id,int64
source_name,object
source_type,object
rake_deg,float64
centroid_longitude,float64
centroid_latitude,float64


In [7]:
import numpy as np
import pandas as pd
from IPython.display import display


catalog = pd.read_csv(
    EVENT_CATALOG_PATH
)


required_columns = [
    "source_type",
    "source_index",
    "source_id",
    "source_name",
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
    "top_depth_km",
    "strike_deg",
    "dip_deg",
    "width_km",
]


missing_columns = [
    column
    for column in required_columns
    if column not in catalog.columns
]


if missing_columns:
    raise RuntimeError(
        "Required catalog columns are missing:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_columns
        )
    )


geometry_summary = (
    catalog
    .groupby(
        "source_type",
        dropna=False,
    )
    .agg(
        event_count=(
            "source_type",
            "size",
        ),
        unique_source_indices=(
            "source_index",
            "nunique",
        ),
        unique_source_ids=(
            "source_id",
            "nunique",
        ),
        unique_source_names=(
            "source_name",
            "nunique",
        ),
        minimum_top_depth_km=(
            "top_depth_km",
            "min",
        ),
        maximum_top_depth_km=(
            "top_depth_km",
            "max",
        ),
        missing_top_depth=(
            "top_depth_km",
            lambda values: values.isna().sum(),
        ),
        missing_centroid_longitude=(
            "centroid_longitude",
            lambda values: values.isna().sum(),
        ),
        missing_centroid_latitude=(
            "centroid_latitude",
            lambda values: values.isna().sum(),
        ),
        missing_centroid_depth=(
            "centroid_depth_km",
            lambda values: values.isna().sum(),
        ),
        missing_strike=(
            "strike_deg",
            lambda values: values.isna().sum(),
        ),
        missing_dip=(
            "dip_deg",
            lambda values: values.isna().sum(),
        ),
        missing_width=(
            "width_km",
            lambda values: values.isna().sum(),
        ),
    )
    .reset_index()
)


print("Catalog geometry summary:")
display(geometry_summary)


slab = catalog.loc[
    catalog["source_type"].eq("SLAB")
].copy()


slab_depth_summary = (
    slab
    .groupby(
        [
            "top_depth_km",
            "centroid_depth_km",
            "dip_deg",
            "width_km",
        ],
        dropna=False,
    )
    .size()
    .rename("event_count")
    .reset_index()
    .sort_values(
        [
            "top_depth_km",
            "centroid_depth_km",
        ]
    )
)


print("\nOregon intraslab depth configurations:")
display(slab_depth_summary)


depth_validation = pd.DataFrame(
    [
        {
            "check": "Every event has top_depth_km",
            "passes": bool(
                catalog[
                    "top_depth_km"
                ].notna().all()
            ),
        },
        {
            "check": (
                "Every intraslab event has top_depth_km"
            ),
            "passes": bool(
                slab[
                    "top_depth_km"
                ].notna().all()
            ),
        },
        {
            "check": (
                "Every intraslab top depth is within "
                "the Parker slab depth domain"
            ),
            "passes": bool(
                slab[
                    "top_depth_km"
                ].between(
                    0.0,
                    200.0,
                    inclusive="both",
                ).all()
            ),
        },
        {
            "check": (
                "Every intraslab event has finite dip"
            ),
            "passes": bool(
                np.isfinite(
                    slab[
                        "dip_deg"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Every intraslab event has positive width"
            ),
            "passes": bool(
                (
                    slab[
                        "width_km"
                    ]
                    > 0.0
                ).all()
            ),
        },
    ]
)


print("\nDepth and geometry validation:")
display(depth_validation)


if not depth_validation["passes"].all():
    failed = depth_validation.loc[
        ~depth_validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Catalog depth validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 7 CATALOG DEPTH AUDIT COMPLETE")

print(
    "\nParker depth mapping:"
    "\n  GmmInput.zTor = catalog.top_depth_km"
)

print(
    "\nThe next step is to define the portfolio sites "
    "and calculate rupture-to-site distances from "
    "the actual USGS rupture surfaces."
)

Catalog geometry summary:


,source_type,event_count,unique_source_indices,unique_source_ids,unique_source_names,minimum_top_depth_km,maximum_top_depth_km,missing_top_depth,missing_centroid_longitude,missing_centroid_latitude,missing_centroid_depth,missing_strike,missing_dip,missing_width
0,INTERFACE,6680,1,15,21,5.0,5.535699,0,1398,1398,1398,0,0,0
1,SLAB,3950,763,2,763,42.0,60.000000,0,0,0,0,3950,0,0



Oregon intraslab depth configurations:


,top_depth_km,centroid_depth_km,dip_deg,width_km,event_count
0,42.0,42.0,90.0,8.0,2342
1,50.0,50.0,90.0,8.0,1386
2,60.0,60.0,90.0,8.0,222



Depth and geometry validation:


,check,passes
0,Every event has top_depth_km,True
1,Every intraslab event has top_depth_km,True
2,Every intraslab top depth is within the Parker...,True
3,Every intraslab event has finite dip,True
4,Every intraslab event has positive width,True



CELL 7 CATALOG DEPTH AUDIT COMPLETE

Parker depth mapping:
  GmmInput.zTor = catalog.top_depth_km

The next step is to define the portfolio sites and calculate rupture-to-site distances from the actual USGS rupture surfaces.


In [8]:
candidate_terms = [
    "rupture",
    "set",
    "path",
    "file",
    "model",
    "source",
    "index",
    "id",
    "magnitude",
    "rate",
]

candidate_columns = [
    column
    for column in catalog.columns
    if any(
        term in column.lower()
        for term in candidate_terms
    )
]


print("Candidate rupture-identification columns:")

for column in candidate_columns:
    print(f"  {column}")


print("\nSample rupture-identification values:")

display(
    catalog[
        candidate_columns
    ].head(20)
)


uniqueness_rows = []

for column in candidate_columns:

    uniqueness_rows.append(
        {
            "column": column,
            "nonmissing_count": int(
                catalog[column].notna().sum()
            ),
            "missing_count": int(
                catalog[column].isna().sum()
            ),
            "unique_values": int(
                catalog[column].nunique(
                    dropna=True
                )
            ),
        }
    )


uniqueness = (
    pd.DataFrame(uniqueness_rows)
    .sort_values(
        [
            "unique_values",
            "column",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print("\nColumn uniqueness:")
display(uniqueness)


possible_key_groups = [
    [
        "rupture_set_path",
        "source_index",
        "rupture_index",
    ],
    [
        "rupture_set_type",
        "source_index",
        "rupture_index",
    ],
    [
        "source_type",
        "source_id",
        "rupture_index",
    ],
    [
        "source_type",
        "source_name",
        "magnitude",
    ],
]


key_results = []


for columns in possible_key_groups:

    if not all(
        column in catalog.columns
        for column in columns
    ):
        continue

    unique_count = (
        catalog[
            columns
        ]
        .drop_duplicates()
        .shape[0]
    )

    key_results.append(
        {
            "candidate_key": " + ".join(columns),
            "unique_combinations": unique_count,
            "event_rows": len(catalog),
            "duplicate_event_rows": (
                len(catalog) - unique_count
            ),
        }
    )


print("\nCandidate key results:")

if key_results:
    display(
        pd.DataFrame(key_results)
    )
else:
    print(
        "None of the predefined candidate keys "
        "were fully present."
    )


print("\nCELL 8 RUPTURE IDENTIFIER AUDIT COMPLETE")

print(
    "\nNext step: choose the authoritative rupture key "
    "and use it to recover each USGS rupture surface."
)

Candidate rupture-identification columns:
  event_id
  rupture_occurrence_number
  rupture_id
  model_name
  tree_id
  tree_path
  tectonic_setting
  branch_index
  rupture_set_id
  rupture_set_name
  rupture_set_type
  rupture_set_weight
  source_index
  source_id
  source_name
  source_type
  rupture_index
  magnitude
  raw_annual_rate
  weighted_annual_rate
  centroid_longitude
  centroid_latitude
  centroid_depth_km
  width_km

Sample rupture-identification values:


,event_id,rupture_occurrence_number,rupture_id,model_name,tree_id,tree_path,tectonic_setting,branch_index,rupture_set_id,rupture_set_name,...,source_name,source_type,rupture_index,magnitude,raw_annual_rate,weighted_annual_rate,centroid_longitude,centroid_latitude,centroid_depth_km,width_km
0,EVT_00000001,148,ff36c8c176c0929711a982c1,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,13,3172,"Cascadia (full, top)",...,"Cascadia (full, top)",INTERFACE,1,8.72,6.327000e-04,1.265400e-04,-125.266845,45.069512,11.059412,85.0
1,EVT_00000002,4,f0fd5c1a0da521278eb8cbb0,NSHM Conterminous U.S. 2018,8231,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,0,8211,OR Intraslab,...,"PointSourceFinite: -123.700, 45.300",SLAB,5,7.05,6.076027e-07,6.076027e-07,-123.700000,45.300000,42.000000,8.0
2,EVT_00000003,91,9bc16e99e4e04b60a7dacf09,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,11,3171,"Cascadia (full, middle)",...,"Cascadia (full, middle)",INTERFACE,0,8.69,6.346000e-04,3.173000e-04,-125.150941,45.115822,12.834152,105.0
3,EVT_00000004,1,49283c0b1304034206982cfb,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,1,3160,"Cascadia (unsegmented, 1-2-3, bottom)",...,"Cascadia (unsegmented, 1-2-3, bottom)",INTERFACE,1022,8.40,6.664417e-07,8.996963e-08,NaN,NaN,NaN,145.0
4,EVT_00000005,597,836db3af40a939132e109ab1,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,11,3171,"Cascadia (full, middle)",...,"Cascadia (full, middle)",INTERFACE,2,9.12,6.327000e-04,3.163500e-04,-125.150941,45.115822,12.834152,105.0
5,EVT_00000006,178,ff36c8c176c0929711a982c1,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,13,3172,"Cascadia (full, top)",...,"Cascadia (full, top)",INTERFACE,1,8.72,6.327000e-04,1.265400e-04,-125.266845,45.069512,11.059412,85.0
6,EVT_00000007,1,6ddc0263a559c933238b255d,NSHM Conterminous U.S. 2018,8231,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,0,8211,OR Intraslab,...,"PointSourceFinite: -122.500, 45.300",SLAB,2,6.75,3.710489e-07,3.710489e-07,-122.500000,45.300000,50.000000,8.0
7,EVT_00000008,20,b695658eda2ede5b10ae3b07,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,18,3141,"Cascadia (segmented, 4, middle)",...,"Cascadia (segmented, 4, middle)",INTERFACE,1,8.52,3.330000e-04,2.081250e-05,-125.689019,48.000730,15.424939,130.0
8,EVT_00000009,247,4944ab46e7ff414e82a9bfa9,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,11,3171,"Cascadia (full, middle)",...,"Cascadia (full, middle)",INTERFACE,1,8.82,6.327000e-04,3.163500e-04,-125.150941,45.115822,12.834152,105.0
9,EVT_00000010,44,ff36c8c176c0929711a982c1,NSHM Conterminous U.S. 2018,3199,C:/Users/USER/Documents/GitHub/seismic-correla...,SUBDUCTION,13,3172,"Cascadia (full, top)",...,"Cascadia (full, top)",INTERFACE,1,8.72,6.327000e-04,1.265400e-04,-125.266845,45.069512,11.059412,85.0



Column uniqueness:


,column,nonmissing_count,missing_count,unique_values
0,event_id,10630,0,10630
1,rupture_id,10630,0,3927
2,weighted_annual_rate,10630,0,2758
3,raw_annual_rate,10630,0,2722
4,rupture_index,10630,0,947
5,source_name,10630,0,784
6,source_index,10630,0,763
7,rupture_occurrence_number,10630,0,638
8,centroid_latitude,9232,1398,58
9,magnitude,10630,0,58



Candidate key results:


,candidate_key,unique_combinations,event_rows,duplicate_event_rows
0,rupture_set_type + source_index + rupture_index,3516,10630,7114
1,source_type + source_id + rupture_index,1306,10630,9324
2,source_type + source_name + magnitude,2725,10630,7905



CELL 8 RUPTURE IDENTIFIER AUDIT COMPLETE

Next step: choose the authoritative rupture key and use it to recover each USGS rupture surface.


In [9]:
import numpy as np
import pandas as pd
from IPython.display import display


RUPTURE_RATE_DIR = (
    DATA_DIR
    / "processed"
    / "usgs_rupture_rates"
)

INTERFACE_RUPTURE_PATH = (
    RUPTURE_RATE_DIR
    / "cascadia_interface_rupture_rates.csv"
)

SLAB_RUPTURE_PATH = (
    RUPTURE_RATE_DIR
    / "oregon_intraslab_rupture_rates.csv"
)

RUPTURE_KEY_VALIDATION_PATH = (
    METADATA_DIR
    / "notebook_4_rupture_key_validation.csv"
)

CATALOG_RUPTURE_JOIN_PATH = (
    METADATA_DIR
    / "notebook_4_catalog_rupture_join_summary.csv"
)


for path in [
    INTERFACE_RUPTURE_PATH,
    SLAB_RUPTURE_PATH,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required rupture table not found:\n  {path}"
        )


interface_ruptures = pd.read_csv(
    INTERFACE_RUPTURE_PATH
)

slab_ruptures = pd.read_csv(
    SLAB_RUPTURE_PATH
)


interface_ruptures[
    "expected_source_type"
] = "INTERFACE"

slab_ruptures[
    "expected_source_type"
] = "SLAB"


rupture_master = pd.concat(
    [
        interface_ruptures,
        slab_ruptures,
    ],
    ignore_index=True,
    sort=False,
)


required_master_columns = [
    "rupture_id",
    "source_type",
    "rupture_set_id",
    "source_index",
    "rupture_index",
    "magnitude",
    "top_depth_km",
]


missing_master_columns = [
    column
    for column in required_master_columns
    if column not in rupture_master.columns
]


if missing_master_columns:
    raise RuntimeError(
        "Notebook 2 rupture tables are missing columns:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_master_columns
        )
    )


catalog_unique_ruptures = (
    catalog
    .sort_values("event_id")
    .drop_duplicates(
        subset=["rupture_id"],
        keep="first",
    )
    .copy()
)


master_duplicate_ids = (
    rupture_master[
        "rupture_id"
    ]
    .duplicated(
        keep=False
    )
)


catalog_duplicate_event_ids = (
    catalog[
        "event_id"
    ]
    .duplicated(
        keep=False
    )
)


master_id_counts = (
    rupture_master[
        "rupture_id"
    ]
    .value_counts()
)


catalog_id_counts = (
    catalog[
        "rupture_id"
    ]
    .value_counts()
)


catalog_rupture_ids = set(
    catalog[
        "rupture_id"
    ]
)


master_rupture_ids = set(
    rupture_master[
        "rupture_id"
    ]
)


missing_from_master = sorted(
    catalog_rupture_ids
    - master_rupture_ids
)


join_columns = [
    "rupture_id",
    "source_type",
    "rupture_set_id",
    "source_index",
    "rupture_index",
    "magnitude",
    "top_depth_km",
]


catalog_join = (
    catalog_unique_ruptures[
        join_columns
    ]
    .merge(
        rupture_master[
            join_columns
        ],
        on="rupture_id",
        how="left",
        suffixes=(
            "_catalog",
            "_master",
        ),
        validate="one_to_one",
        indicator=True,
    )
)


comparison_columns = [
    "source_type",
    "rupture_set_id",
    "source_index",
    "rupture_index",
    "magnitude",
    "top_depth_km",
]


comparison_results = []


for column in comparison_columns:

    catalog_column = (
        f"{column}_catalog"
    )

    master_column = (
        f"{column}_master"
    )

    left = catalog_join[
        catalog_column
    ]

    right = catalog_join[
        master_column
    ]

    if pd.api.types.is_numeric_dtype(left):

        matches = np.isclose(
            pd.to_numeric(
                left,
                errors="coerce",
            ),
            pd.to_numeric(
                right,
                errors="coerce",
            ),
            rtol=1e-12,
            atol=1e-12,
            equal_nan=True,
        )

    else:

        matches = (
            left.fillna("<NA>").astype(str)
            == right.fillna("<NA>").astype(str)
        )

    comparison_results.append(
        {
            "field": column,
            "rows_compared": len(
                catalog_join
            ),
            "matching_rows": int(
                np.sum(matches)
            ),
            "mismatching_rows": int(
                np.sum(~matches)
            ),
            "all_match": bool(
                np.all(matches)
            ),
        }
    )


comparison_table = pd.DataFrame(
    comparison_results
)


join_summary = pd.DataFrame(
    [
        {
            "metric": (
                "Notebook 2 rupture rows"
            ),
            "value": len(
                rupture_master
            ),
        },
        {
            "metric": (
                "Unique Notebook 2 rupture IDs"
            ),
            "value": rupture_master[
                "rupture_id"
            ].nunique(),
        },
        {
            "metric": (
                "Catalog event occurrences"
            ),
            "value": len(catalog),
        },
        {
            "metric": (
                "Unique catalog event IDs"
            ),
            "value": catalog[
                "event_id"
            ].nunique(),
        },
        {
            "metric": (
                "Unique catalog rupture IDs"
            ),
            "value": catalog[
                "rupture_id"
            ].nunique(),
        },
        {
            "metric": (
                "Catalog rupture IDs missing "
                "from Notebook 2"
            ),
            "value": len(
                missing_from_master
            ),
        },
        {
            "metric": (
                "Catalog ruptures matched "
                "one-to-one"
            ),
            "value": int(
                catalog_join[
                    "_merge"
                ].eq(
                    "both"
                ).sum()
            ),
        },
        {
            "metric": (
                "Maximum simulated occurrences "
                "of one rupture"
            ),
            "value": int(
                catalog_id_counts.max()
            ),
        },
    ]
)


validation = pd.DataFrame(
    [
        {
            "check": (
                "Every event_id is unique"
            ),
            "passes": bool(
                ~catalog_duplicate_event_ids.any()
            ),
        },
        {
            "check": (
                "Every Notebook 2 rupture_id "
                "is unique"
            ),
            "passes": bool(
                ~master_duplicate_ids.any()
            ),
        },
        {
            "check": (
                "Every catalog rupture_id exists "
                "in Notebook 2"
            ),
            "passes": (
                len(
                    missing_from_master
                )
                == 0
            ),
        },
        {
            "check": (
                "Every distinct catalog rupture "
                "joins to exactly one master row"
            ),
            "passes": bool(
                catalog_join[
                    "_merge"
                ].eq(
                    "both"
                ).all()
            ),
        },
        {
            "check": (
                "Source and rupture metadata "
                "agree after the join"
            ),
            "passes": bool(
                comparison_table[
                    "all_match"
                ].all()
            ),
        },
        {
            "check": (
                "Repeated rupture occurrences "
                "exist in the catalog"
            ),
            "passes": bool(
                catalog[
                    "rupture_id"
                ].duplicated().any()
            ),
        },
    ]
)


join_summary.to_csv(
    CATALOG_RUPTURE_JOIN_PATH,
    index=False,
)

validation.to_csv(
    RUPTURE_KEY_VALIDATION_PATH,
    index=False,
)


print("Catalog and rupture-table summary:")
display(join_summary)


print("\nJoined-field comparison:")
display(comparison_table)


print("\nRupture-key validation:")
display(validation)


if not validation[
    "passes"
].all():

    failed = validation.loc[
        ~validation[
            "passes"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "Rupture-key validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 9 RUPTURE KEY VALIDATION COMPLETE")

print(
    "\nAuthoritative occurrence key:"
    "\n  event_id"
)

print(
    "\nAuthoritative physical rupture key:"
    "\n  rupture_id"
)

print(
    "\nParker depth mapping:"
    "\n  GmmInput.zTor = top_depth_km"
)

print(
    "\nNext step: define the portfolio site table "
    "and calculate Rrup from each unique USGS "
    "rupture surface to each site."
)

Catalog and rupture-table summary:


,metric,value
0,Notebook 2 rupture rows,23416
1,Unique Notebook 2 rupture IDs,23416
2,Catalog event occurrences,10630
3,Unique catalog event IDs,10630
4,Unique catalog rupture IDs,3927
5,Catalog rupture IDs missing from Notebook 2,0
6,Catalog ruptures matched one-to-one,3927
7,Maximum simulated occurrences of one rupture,638



Joined-field comparison:


,field,rows_compared,matching_rows,mismatching_rows,all_match
0,source_type,3927,3927,0,True
1,rupture_set_id,3927,3927,0,True
2,source_index,3927,3927,0,True
3,rupture_index,3927,3927,0,True
4,magnitude,3927,3927,0,True
5,top_depth_km,3927,3927,0,True



Rupture-key validation:


,check,passes
0,Every event_id is unique,True
1,Every Notebook 2 rupture_id is unique,True
2,Every catalog rupture_id exists in Notebook 2,True
3,Every distinct catalog rupture joins to exactl...,True
4,Source and rupture metadata agree after the join,True
5,Repeated rupture occurrences exist in the catalog,True



CELL 9 RUPTURE KEY VALIDATION COMPLETE

Authoritative occurrence key:
  event_id

Authoritative physical rupture key:
  rupture_id

Parker depth mapping:
  GmmInput.zTor = top_depth_km

Next step: define the portfolio site table and calculate Rrup from each unique USGS rupture surface to each site.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


NSI_EXPOSURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "exposure"
    / "seaside_nsi"
    / "gdf_NSI_Map_with_period_seaside.xlsx"
)

PORTFOLIO_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "exposure_portfolio"
)

PORTFOLIO_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

W2_PORTFOLIO_PATH = (
    PORTFOLIO_DIR
    / "seaside_w2_portfolio.csv"
)

W2_PORTFOLIO_METADATA_PATH = (
    METADATA_DIR
    / "seaside_w2_portfolio_summary.csv"
)


if not NSI_EXPOSURE_PATH.is_file():
    raise FileNotFoundError(
        "The Seaside NSI workbook was not found.\n\n"
        "Place it at:\n"
        f"  {NSI_EXPOSURE_PATH}"
    )


inventory = pd.read_excel(
    NSI_EXPOSURE_PATH
)


print(f"Workbook path:\n  {NSI_EXPOSURE_PATH}")
print(f"\nInventory rows: {len(inventory):,}")
print(f"Inventory columns: {len(inventory.columns):,}")


candidate_columns = {
    "structure_type": [
        "struct_typ",
        "structure_type",
        "StructuralType",
    ],
    "site_class": [
        "SiteClass",
        "site_class",
        "siteclass",
    ],
    "longitude": [
    "Longitude",
    "longitude",
    "lon",
],
"latitude": [
    "Latitude",
    "latitude",
    "lat",
],
    "period": [
        "Period_for_spectral_acceleration",
        "period",
        "Period",
    ],
}


def find_column(frame, names, description):
    matches = [
        name
        for name in names
        if name in frame.columns
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {description} column.\n"
            f"Candidates checked: {names}\n"
            f"Matches found: {matches}"
        )

    return matches[0]


STRUCTURE_TYPE_COLUMN = find_column(
    inventory,
    candidate_columns["structure_type"],
    "structure-type",
)

SITE_CLASS_COLUMN = find_column(
    inventory,
    candidate_columns["site_class"],
    "site-class",
)

required_coordinate_columns = [
    "Longitude",
    "Latitude",
]

missing_coordinate_columns = [
    column
    for column in required_coordinate_columns
    if column not in inventory.columns
]

if missing_coordinate_columns:
    raise RuntimeError(
        "Required coordinate columns are missing from the inventory:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_coordinate_columns
        )
    )

LONGITUDE_COLUMN = "Longitude"
LATITUDE_COLUMN = "Latitude"

PERIOD_COLUMN = find_column(
    inventory,
    candidate_columns["period"],
    "building-period",
)


w2 = inventory.loc[
    inventory[
        STRUCTURE_TYPE_COLUMN
    ]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("W2")
].copy()


if w2.empty:
    raise RuntimeError(
        "No W2 structures were found in the inventory."
    )


w2["site_class_original"] = (
    w2[
        SITE_CLASS_COLUMN
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


site_class_counts = (
    w2[
        "site_class_original"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "site_class"
    )
    .reset_index(
        name="building_count"
    )
)


print("\nW2 site-class distribution:")
display(site_class_counts)


unexpected_site_classes = sorted(
    set(
        w2[
            "site_class_original"
        ]
    )
    - {"CD"}
)


if unexpected_site_classes:
    raise RuntimeError(
        "Not every W2 building has Site Class CD.\n"
        "Unexpected site classes:\n"
        + "\n".join(
            f"  {value}"
            for value in unexpected_site_classes
        )
    )


w2["longitude"] = pd.to_numeric(
    w2[
        LONGITUDE_COLUMN
    ],
    errors="coerce",
)

w2["latitude"] = pd.to_numeric(
    w2[
        LATITUDE_COLUMN
    ],
    errors="coerce",
)

w2["building_period_sec"] = pd.to_numeric(
    w2[
        PERIOD_COLUMN
    ],
    errors="coerce",
)


VS30_CD_MPS = 365.0

w2["vs30_mps"] = VS30_CD_MPS
w2["vs30_source"] = "assigned_from_site_class_CD"
w2["vs30_is_measured"] = False


w2 = w2.reset_index(
    drop=True
)

w2.insert(
    0,
    "site_id",
    [
        f"W2_{index:04d}"
        for index in range(
            1,
            len(w2) + 1,
        )
    ],
)


validation = pd.DataFrame(
    [
        {
            "check": "W2 portfolio is not empty",
            "passes": len(w2) > 0,
        },
        {
            "check": "Every site_id is unique",
            "passes": bool(
                w2["site_id"].is_unique
            ),
        },
        {
            "check": "Every W2 building is Site Class CD",
            "passes": bool(
                w2[
                    "site_class_original"
                ].eq("CD").all()
            ),
        },
        {
            "check": "Every building has finite longitude",
            "passes": bool(
                np.isfinite(
                    w2["longitude"]
                ).all()
            ),
        },
        {
            "check": "Every building has finite latitude",
            "passes": bool(
                np.isfinite(
                    w2["latitude"]
                ).all()
            ),
        },
        {
            "check": "Every building has a positive period",
            "passes": bool(
                (
                    w2[
                        "building_period_sec"
                    ]
                    > 0
                ).all()
            ),
        },
        {
            "check": "Every building has Vs30 = 365 m/s",
            "passes": bool(
                np.isclose(
                    w2["vs30_mps"],
                    VS30_CD_MPS,
                ).all()
            ),
        },
    ]
)


print("\nW2 portfolio validation:")
display(validation)


if not validation["passes"].all():
    failed = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "W2 portfolio validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


portfolio_summary = pd.DataFrame(
    [
        {
            "metric": "portfolio_name",
            "value": "Seaside W2 commercial portfolio",
        },
        {
            "metric": "building_count",
            "value": len(w2),
        },
        {
            "metric": "structural_type",
            "value": "W2",
        },
        {
            "metric": "site_class",
            "value": "CD",
        },
        {
            "metric": "assigned_vs30_mps",
            "value": VS30_CD_MPS,
        },
        {
            "metric": "vs30_assignment",
            "value": "Representative value, not measured",
        },
        {
            "metric": "minimum_longitude",
            "value": w2["longitude"].min(),
        },
        {
            "metric": "maximum_longitude",
            "value": w2["longitude"].max(),
        },
        {
            "metric": "minimum_latitude",
            "value": w2["latitude"].min(),
        },
        {
            "metric": "maximum_latitude",
            "value": w2["latitude"].max(),
        },
        {
            "metric": "unique_periods",
            "value": w2[
                "building_period_sec"
            ].nunique(),
        },
    ]
)


w2.to_csv(
    W2_PORTFOLIO_PATH,
    index=False,
)

portfolio_summary.to_csv(
    W2_PORTFOLIO_METADATA_PATH,
    index=False,
)


print("\nCELL 10 W2 PORTFOLIO EXTRACTION COMPLETE")

print(f"\nW2 buildings: {len(w2):,}")

print(
    "\nAssigned site condition:"
    f"\n  Site Class CD"
    f"\n  Vs30 = {VS30_CD_MPS:.1f} m/s"
    "\n  Status: representative, not measured"
)

print(
    "\nPortfolio file:"
    f"\n  {W2_PORTFOLIO_PATH}"
)

print(
    "\nPortfolio metadata:"
    f"\n  {W2_PORTFOLIO_METADATA_PATH}"
)

print(
    "\nNext step: inspect the unique W2 oscillator "
    "periods and map them to Parker-supported IMTs."
)

Workbook path:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\exposure\seaside_nsi\gdf_NSI_Map_with_period_seaside.xlsx

Inventory rows: 4,651
Inventory columns: 40

W2 site-class distribution:


,site_class,building_count
0,CD,470



W2 portfolio validation:


,check,passes
0,W2 portfolio is not empty,True
1,Every site_id is unique,True
2,Every W2 building is Site Class CD,True
3,Every building has finite longitude,True
4,Every building has finite latitude,True
5,Every building has a positive period,True
6,Every building has Vs30 = 365 m/s,True



CELL 10 W2 PORTFOLIO EXTRACTION COMPLETE

W2 buildings: 470

Assigned site condition:
  Site Class CD
  Vs30 = 365.0 m/s
  Status: representative, not measured

Portfolio file:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\exposure_portfolio\seaside_w2_portfolio.csv

Portfolio metadata:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\seaside_w2_portfolio_summary.csv

Next step: inspect the unique W2 oscillator periods and map them to Parker-supported IMTs.


In [14]:
import re

import numpy as np
import pandas as pd
from IPython.display import display


GMM_READY_PORTFOLIO_PATH = (
    PORTFOLIO_DIR
    / "seaside_w2_portfolio_gmm_ready.csv"
)

PERIOD_MAPPING_PATH = (
    METADATA_DIR
    / "seaside_w2_period_to_parker_imt.csv"
)

PERIOD_MAPPING_VALIDATION_PATH = (
    METADATA_DIR
    / "seaside_w2_period_mapping_validation.csv"
)


portfolio = pd.read_csv(
    W2_PORTFOLIO_PATH
)

imt_inventory = pd.read_csv(
    PARKER_IMT_INVENTORY_PATH
)


selected_gmms = [
    "PSBAH_20_CASCADIA_INTERFACE",
    "PSBAH_20_CASCADIA_SLAB",
]


selected_imts = imt_inventory.loc[
    imt_inventory[
        "gmm_name"
    ].isin(selected_gmms)
].copy()


supported_imts_by_gmm = {
    gmm_name: set(
        selected_imts.loc[
            selected_imts[
                "gmm_name"
            ].eq(gmm_name),
            "imt_name",
        ]
    )
    for gmm_name in selected_gmms
}


common_supported_imts = set.intersection(
    *supported_imts_by_gmm.values()
)


def parse_sa_period(imt_name):

    match = re.fullmatch(
        r"SA(\d+)P(\d+)",
        str(imt_name),
    )

    if match is None:
        return np.nan

    whole = match.group(1)
    decimal = match.group(2)

    return float(
        f"{whole}.{decimal}"
    )


sa_inventory = pd.DataFrame(
    {
        "imt_name": sorted(
            imt
            for imt in common_supported_imts
            if str(imt).startswith("SA")
        )
    }
)


sa_inventory[
    "period_sec"
] = sa_inventory[
    "imt_name"
].map(parse_sa_period)


sa_inventory = (
    sa_inventory
    .dropna(
        subset=["period_sec"]
    )
    .sort_values("period_sec")
    .reset_index(drop=True)
)


portfolio[
    "building_period_sec"
] = pd.to_numeric(
    portfolio[
        "building_period_sec"
    ],
    errors="coerce",
)


unique_periods = sorted(
    portfolio[
        "building_period_sec"
    ].dropna().unique()
)


period_mapping_rows = []


for period in unique_periods:

    matches = sa_inventory.loc[
        np.isclose(
            sa_inventory[
                "period_sec"
            ],
            period,
            rtol=0.0,
            atol=1e-12,
        )
    ]

    if len(matches) != 1:
        raise RuntimeError(
            "The building period could not be mapped "
            "to exactly one Parker IMT.\n"
            f"Period: {period}\n"
            f"Matches found: {len(matches)}"
        )

    mapped_imt = matches[
        "imt_name"
    ].iloc[0]

    period_mapping_rows.append(
        {
            "building_period_sec": period,
            "parker_imt_name": mapped_imt,
            "parker_period_sec": float(
                matches[
                    "period_sec"
                ].iloc[0]
            ),
            "requires_interpolation": False,
            "interface_model_supports_imt": (
                mapped_imt
                in supported_imts_by_gmm[
                    "PSBAH_20_CASCADIA_INTERFACE"
                ]
            ),
            "slab_model_supports_imt": (
                mapped_imt
                in supported_imts_by_gmm[
                    "PSBAH_20_CASCADIA_SLAB"
                ]
            ),
        }
    )


period_mapping = pd.DataFrame(
    period_mapping_rows
)


period_to_imt = dict(
    zip(
        period_mapping[
            "building_period_sec"
        ],
        period_mapping[
            "parker_imt_name"
        ],
    )
)


portfolio[
    "primary_sa_imt_name"
] = portfolio[
    "building_period_sec"
].map(period_to_imt)


portfolio[
    "pga_imt_name"
] = "PGA"


portfolio[
    "ground_motion_model_interface"
] = (
    "PSBAH_20_CASCADIA_INTERFACE"
)

portfolio[
    "ground_motion_model_slab"
] = (
    "PSBAH_20_CASCADIA_SLAB"
)


validation = pd.DataFrame(
    [
        {
            "check": (
                "Portfolio still contains 470 W2 buildings"
            ),
            "passes": len(portfolio) == 470,
        },
        {
            "check": (
                "Every building has a finite oscillator period"
            ),
            "passes": bool(
                np.isfinite(
                    portfolio[
                        "building_period_sec"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Every building period maps to a Parker IMT"
            ),
            "passes": bool(
                portfolio[
                    "primary_sa_imt_name"
                ].notna().all()
            ),
        },
        {
            "check": (
                "Every mapped IMT is supported by "
                "the interface model"
            ),
            "passes": bool(
                period_mapping[
                    "interface_model_supports_imt"
                ].all()
            ),
        },
        {
            "check": (
                "Every mapped IMT is supported by "
                "the slab model"
            ),
            "passes": bool(
                period_mapping[
                    "slab_model_supports_imt"
                ].all()
            ),
        },
        {
            "check": (
                "No period interpolation is required"
            ),
            "passes": bool(
                ~period_mapping[
                    "requires_interpolation"
                ].any()
            ),
        },
        {
            "check": (
                "The W2 period maps to SA0P4"
            ),
            "passes": bool(
                set(
                    portfolio[
                        "primary_sa_imt_name"
                    ]
                )
                == {"SA0P4"}
            ),
        },
        {
            "check": (
                "PGA is supported by both Parker models"
            ),
            "passes": bool(
                "PGA"
                in supported_imts_by_gmm[
                    "PSBAH_20_CASCADIA_INTERFACE"
                ]
                and "PGA"
                in supported_imts_by_gmm[
                    "PSBAH_20_CASCADIA_SLAB"
                ]
            ),
        },
    ]
)


period_mapping.to_csv(
    PERIOD_MAPPING_PATH,
    index=False,
)

validation.to_csv(
    PERIOD_MAPPING_VALIDATION_PATH,
    index=False,
)

portfolio.to_csv(
    GMM_READY_PORTFOLIO_PATH,
    index=False,
)


print("Unique W2 oscillator periods:")
display(
    portfolio[
        [
            "building_period_sec",
            "primary_sa_imt_name",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print("\nParker period mapping:")
display(period_mapping)


print("\nPeriod-mapping validation:")
display(validation)


if not validation[
    "passes"
].all():

    failed = validation.loc[
        ~validation[
            "passes"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "W2 period mapping failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 11 PARKER IMT MAPPING COMPLETE")

print(
    "\nW2 oscillator period:"
    "\n  T = 0.40 seconds"
)

print(
    "\nPrimary spectral-acceleration IMT:"
    "\n  SA0P4"
)

print(
    "\nAdditional ground-motion IMT:"
    "\n  PGA"
)

print(
    "\nPeriod interpolation:"
    "\n  Not required"
)

print(
    "\nGMM-ready portfolio:"
    f"\n  {GMM_READY_PORTFOLIO_PATH}"
)

print(
    "\nNext step: reconstruct the USGS rupture "
    "surfaces and calculate Rrup from each of the "
    "3,927 unique ruptures to the 470 portfolio sites."
)

Unique W2 oscillator periods:


,building_period_sec,primary_sa_imt_name
0,0.4,SA0P4



Parker period mapping:


,building_period_sec,parker_imt_name,parker_period_sec,requires_interpolation,interface_model_supports_imt,slab_model_supports_imt
0,0.4,SA0P4,0.4,False,True,True



Period-mapping validation:


,check,passes
0,Portfolio still contains 470 W2 buildings,True
1,Every building has a finite oscillator period,True
2,Every building period maps to a Parker IMT,True
3,Every mapped IMT is supported by the interface...,True
4,Every mapped IMT is supported by the slab model,True
5,No period interpolation is required,True
6,The W2 period maps to SA0P4,True
7,PGA is supported by both Parker models,True



CELL 11 PARKER IMT MAPPING COMPLETE

W2 oscillator period:
  T = 0.40 seconds

Primary spectral-acceleration IMT:
  SA0P4

Additional ground-motion IMT:
  PGA

Period interpolation:
  Not required

GMM-ready portfolio:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\exposure_portfolio\seaside_w2_portfolio_gmm_ready.csv

Next step: reconstruct the USGS rupture surfaces and calculate Rrup from each of the 3,927 unique ruptures to the 470 portfolio sites.


In [15]:
import re
import subprocess

import pandas as pd
from IPython.display import display


DISTANCE_API_REPORT_PATH = (
    METADATA_DIR
    / "nshmp_rupture_distance_api.txt"
)

DISTANCE_API_SUMMARY_PATH = (
    METADATA_DIR
    / "nshmp_rupture_distance_api_summary.csv"
)

DISTANCE_CLASS_INVENTORY_PATH = (
    METADATA_DIR
    / "nshmp_rupture_distance_class_inventory.csv"
)


search_patterns = [
    r"gov/usgs/earthquake/nshmp/model/.*HazardModel.*\.class$",
    r"gov/usgs/earthquake/nshmp/model/.*SourceTree.*\.class$",
    r"gov/usgs/earthquake/nshmp/model/.*SourceSet.*\.class$",
    r"gov/usgs/earthquake/nshmp/model/Source\.class$",
    r"gov/usgs/earthquake/nshmp/model/Rupture\.class$",
    r"gov/usgs/earthquake/nshmp/model/Distance\.class$",
    r"gov/usgs/earthquake/nshmp/model/.*Surface.*\.class$",
    r"gov/usgs/earthquake/nshmp/geo/Location\.class$",
]


matching_entries = []

for entry in jar_entries:

    if not entry.endswith(".class"):
        continue

    if any(
        re.search(pattern, entry)
        for pattern in search_patterns
    ):
        matching_entries.append(entry)


class_names = []

for entry in matching_entries:

    class_name = (
        entry[:-6]
        .replace("/", ".")
    )

    if re.search(r"\$\d+$", class_name):
        continue

    class_names.append(class_name)


preferred_classes = [
    "gov.usgs.earthquake.nshmp.model.HazardModel",
    "gov.usgs.earthquake.nshmp.model.SourceTree",
    "gov.usgs.earthquake.nshmp.model.SourceSet",
    "gov.usgs.earthquake.nshmp.model.Source",
    "gov.usgs.earthquake.nshmp.model.Rupture",
    "gov.usgs.earthquake.nshmp.model.Distance",
    "gov.usgs.earthquake.nshmp.geo.Location",
]


classes_to_inspect = list(
    dict.fromkeys(
        preferred_classes
        + sorted(class_names)
    )
)


inspection_rows = []


for class_name in classes_to_inspect:

    result = subprocess.run(
        [
            javap_executable,
            "-classpath",
            str(NSHMP_LIB_JAR),
            "-private",
            class_name,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )

    inspection_rows.append(
        {
            "class_name": class_name,
            "passed": result.returncode == 0,
            "output": (
                result.stdout.strip()
                if result.returncode == 0
                else result.stderr.strip()
            ),
        }
    )


inspection_table = pd.DataFrame(
    inspection_rows
)


inspection_table[
    [
        "class_name",
        "passed",
    ]
].to_csv(
    DISTANCE_CLASS_INVENTORY_PATH,
    index=False,
)


report_lines = [
    "NSHMP RUPTURE AND DISTANCE API",
    "",
    f"nshmp-lib JAR: {NSHMP_LIB_JAR}",
]


for row in inspection_rows:

    report_lines.extend(
        [
            "",
            "=" * 78,
            row["class_name"],
            "=" * 78,
            row["output"],
        ]
    )


DISTANCE_API_REPORT_PATH.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)


keywords = [
    "load(",
    "open(",
    "source",
    "rupture",
    "distance",
    "surface",
    "location",
    "iterator",
    "size(",
    "get(",
    "id(",
    "name(",
    "type(",
    "rRup",
    "rJB",
    "rX",
    "index",
]


summary_rows = []


for row in inspection_rows:

    if not row["passed"]:
        continue

    for line in row["output"].splitlines():

        stripped = line.strip()

        if any(
            keyword.lower() in stripped.lower()
            for keyword in keywords
        ):
            summary_rows.append(
                {
                    "class_name": row["class_name"],
                    "api_line": stripped,
                }
            )


api_summary = pd.DataFrame(
    summary_rows
)


api_summary.to_csv(
    DISTANCE_API_SUMMARY_PATH,
    index=False,
)


print("Discovered rupture and distance classes:")

display(
    inspection_table[
        [
            "class_name",
            "passed",
        ]
    ]
)


print("\nRelevant API lines:")

if api_summary.empty:
    print(
        "No relevant lines were selected automatically. "
        "Review the complete report."
    )
else:
    display(api_summary)


required_classes = [
    "gov.usgs.earthquake.nshmp.model.HazardModel",
    "gov.usgs.earthquake.nshmp.model.Source",
    "gov.usgs.earthquake.nshmp.model.Rupture",
    "gov.usgs.earthquake.nshmp.model.Distance",
    "gov.usgs.earthquake.nshmp.geo.Location",
]


required_status = (
    inspection_table
    .set_index("class_name")
    .reindex(required_classes)
)


validation = pd.DataFrame(
    [
        {
            "check": "HazardModel API was found",
            "passes": bool(
                required_status.loc[
                    "gov.usgs.earthquake.nshmp.model.HazardModel",
                    "passed",
                ]
            ),
        },
        {
            "check": "Source API was found",
            "passes": bool(
                required_status.loc[
                    "gov.usgs.earthquake.nshmp.model.Source",
                    "passed",
                ]
            ),
        },
        {
            "check": "Rupture API was found",
            "passes": bool(
                required_status.loc[
                    "gov.usgs.earthquake.nshmp.model.Rupture",
                    "passed",
                ]
            ),
        },
        {
            "check": "Distance API was found",
            "passes": bool(
                required_status.loc[
                    "gov.usgs.earthquake.nshmp.model.Distance",
                    "passed",
                ]
            ),
        },
        {
            "check": "Location API was found",
            "passes": bool(
                required_status.loc[
                    "gov.usgs.earthquake.nshmp.geo.Location",
                    "passed",
                ]
            ),
        },
    ]
)


print("\nAPI inspection validation:")
display(validation)


if not validation["passes"].all():

    failed = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Required rupture-distance APIs were not found:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 12 RUPTURE-DISTANCE API INSPECTION COMPLETE")

print(
    "\nComplete report:"
    f"\n  {DISTANCE_API_REPORT_PATH}"
)

print(
    "\nAPI summary:"
    f"\n  {DISTANCE_API_SUMMARY_PATH}"
)

print(
    "\nNext step: use the confirmed USGS APIs to "
    "retrieve each unique rupture and calculate "
    "Rrup, Rjb, and Rx for the 470 sites."
)

Discovered rupture and distance classes:


,class_name,passed
0,gov.usgs.earthquake.nshmp.model.HazardModel,True
1,gov.usgs.earthquake.nshmp.model.SourceTree,True
2,gov.usgs.earthquake.nshmp.model.SourceSet,False
3,gov.usgs.earthquake.nshmp.model.Source,True
4,gov.usgs.earthquake.nshmp.model.Rupture,True
5,gov.usgs.earthquake.nshmp.model.Distance,True
6,gov.usgs.earthquake.nshmp.geo.Location,True
7,gov.usgs.earthquake.nshmp.model.GridSourcePlan...,True
8,gov.usgs.earthquake.nshmp.model.HazardModel$Bu...,True
9,gov.usgs.earthquake.nshmp.model.HazardModel$Info,True



Relevant API lines:


,class_name,api_line
0,gov.usgs.earthquake.nshmp.model.HazardModel,public final class gov.usgs.earthquake.nshmp.m...
1,gov.usgs.earthquake.nshmp.model.HazardModel,private final com.google.common.collect.Multim...
2,gov.usgs.earthquake.nshmp.model.HazardModel,private final com.google.common.collect.Multim...
3,gov.usgs.earthquake.nshmp.model.HazardModel,private final com.google.common.collect.Multim...
4,gov.usgs.earthquake.nshmp.model.HazardModel,private final com.google.common.collect.Multim...
...,...,...
145,gov.usgs.earthquake.nshmp.model.SourceTree$Node,final int index;
146,gov.usgs.earthquake.nshmp.model.SourceTree$Node,gov.usgs.earthquake.nshmp.model.SourceTree$Nod...
147,gov.usgs.earthquake.nshmp.model.SourceTree$Root,"Compiled from ""SourceTree.java"""
148,gov.usgs.earthquake.nshmp.model.SourceTree$Root,final class gov.usgs.earthquake.nshmp.model.So...



API inspection validation:


,check,passes
0,HazardModel API was found,True
1,Source API was found,True
2,Rupture API was found,True
3,Distance API was found,True
4,Location API was found,True



CELL 12 RUPTURE-DISTANCE API INSPECTION COMPLETE

Complete report:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_rupture_distance_api.txt

API summary:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_rupture_distance_api_summary.csv

Next step: use the confirmed USGS APIs to retrieve each unique rupture and calculate Rrup, Rjb, and Rx for the 470 sites.


In [16]:
import subprocess

import pandas as pd
from IPython.display import display


RUPTURE_ACCESS_API_PATH = (
    METADATA_DIR
    / "nshmp_rupture_access_api.txt"
)

RUPTURE_ACCESS_API_SUMMARY_PATH = (
    METADATA_DIR
    / "nshmp_rupture_access_api_summary.csv"
)


classes_to_inspect = [
    "gov.usgs.earthquake.nshmp.model.RuptureSet",
    "gov.usgs.earthquake.nshmp.fault.surface.RuptureSurface",
    "gov.usgs.earthquake.nshmp.tree.Branch",
]


inspection_rows = []


for class_name in classes_to_inspect:

    result = subprocess.run(
        [
            javap_executable,
            "-classpath",
            str(NSHMP_LIB_JAR),
            "-private",
            class_name,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )

    inspection_rows.append(
        {
            "class_name": class_name,
            "passed": result.returncode == 0,
            "output": (
                result.stdout.strip()
                if result.returncode == 0
                else result.stderr.strip()
            ),
        }
    )


inspection_table = pd.DataFrame(
    inspection_rows
)


report_lines = [
    "NSHMP RUPTURE ACCESS API",
    "",
    f"nshmp-lib JAR: {NSHMP_LIB_JAR}",
]


for row in inspection_rows:

    report_lines.extend(
        [
            "",
            "=" * 78,
            row["class_name"],
            "=" * 78,
            row["output"],
        ]
    )


RUPTURE_ACCESS_API_PATH.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)


keywords = [
    "get(",
    "size(",
    "iterator",
    "id(",
    "name(",
    "type(",
    "value(",
    "weight(",
    "distanceTo",
    "centroid",
    "depth(",
    "strike(",
    "dip(",
    "width(",
    "length(",
]


summary_rows = []


for row in inspection_rows:

    if not row["passed"]:
        continue

    for line in row["output"].splitlines():

        stripped = line.strip()

        if any(
            keyword.lower() in stripped.lower()
            for keyword in keywords
        ):
            summary_rows.append(
                {
                    "class_name": row["class_name"],
                    "api_line": stripped,
                }
            )


api_summary = pd.DataFrame(
    summary_rows
)


api_summary.to_csv(
    RUPTURE_ACCESS_API_SUMMARY_PATH,
    index=False,
)


print("Class inspection status:")
display(
    inspection_table[
        [
            "class_name",
            "passed",
        ]
    ]
)


print("\nRelevant rupture-access API lines:")

if api_summary.empty:
    print(
        "No lines were selected automatically. "
        "Review the complete report."
    )
else:
    display(api_summary)


validation = pd.DataFrame(
    [
        {
            "check": "RuptureSet API was found",
            "passes": bool(
                inspection_table.loc[
                    inspection_table[
                        "class_name"
                    ].eq(
                        "gov.usgs.earthquake.nshmp.model.RuptureSet"
                    ),
                    "passed",
                ].iloc[0]
            ),
        },
        {
            "check": "RuptureSurface API was found",
            "passes": bool(
                inspection_table.loc[
                    inspection_table[
                        "class_name"
                    ].eq(
                        "gov.usgs.earthquake.nshmp.fault.surface.RuptureSurface"
                    ),
                    "passed",
                ].iloc[0]
            ),
        },
        {
            "check": "Branch API was found",
            "passes": bool(
                inspection_table.loc[
                    inspection_table[
                        "class_name"
                    ].eq(
                        "gov.usgs.earthquake.nshmp.tree.Branch"
                    ),
                    "passed",
                ].iloc[0]
            ),
        },
    ]
)


print("\nAPI validation:")
display(validation)


if not validation["passes"].all():

    failed = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Required rupture-access APIs were not found:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 13 RUPTURE ACCESS API INSPECTION COMPLETE")

print(
    "\nComplete report:"
    f"\n  {RUPTURE_ACCESS_API_PATH}"
)

print(
    "\nAPI summary:"
    f"\n  {RUPTURE_ACCESS_API_SUMMARY_PATH}"
)

print(
    "\nNext step: load the USGS model and retrieve "
    "a small set of catalog ruptures using tree_id, "
    "branch_index, source_index, and rupture_index."
)

Class inspection status:


,class_name,passed
0,gov.usgs.earthquake.nshmp.model.RuptureSet,True
1,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,True
2,gov.usgs.earthquake.nshmp.tree.Branch,True



Relevant rupture-access API lines:


,class_name,api_line
0,gov.usgs.earthquake.nshmp.model.RuptureSet,public abstract java.lang.String name();
1,gov.usgs.earthquake.nshmp.model.RuptureSet,public abstract int id();
2,gov.usgs.earthquake.nshmp.model.RuptureSet,public abstract gov.usgs.earthquake.nshmp.mode...
3,gov.usgs.earthquake.nshmp.model.RuptureSet,public abstract double weight();
4,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,public abstract double strike();
5,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,public abstract double dip();
6,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,public abstract double length();
7,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,public abstract double width();
8,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,public abstract double depth();
9,gov.usgs.earthquake.nshmp.fault.surface.Ruptur...,public abstract gov.usgs.earthquake.nshmp.geo....



API validation:


,check,passes
0,RuptureSet API was found,True
1,RuptureSurface API was found,True
2,Branch API was found,True



CELL 13 RUPTURE ACCESS API INSPECTION COMPLETE

Complete report:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_rupture_access_api.txt

API summary:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\nshmp_rupture_access_api_summary.csv

Next step: load the USGS model and retrieve a small set of catalog ruptures using tree_id, branch_index, source_index, and rupture_index.


In [26]:
import os
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


MODEL_SEARCH_ROOT = (
    DATA_DIR
    / "raw"
    / "usgs_nshm_conus_2018"
)

RUPTURE_RETRIEVAL_INPUT_PATH = (
    METADATA_DIR
    / "notebook_4_controlled_rupture_retrieval_input.tsv"
)

RUPTURE_RETRIEVAL_OUTPUT_PATH = (
    METADATA_DIR
    / "notebook_4_controlled_rupture_retrieval_output.csv"
)

RUPTURE_RETRIEVAL_VALIDATION_PATH = (
    METADATA_DIR
    / "notebook_4_controlled_rupture_retrieval_validation.csv"
)

RUPTURE_RETRIEVAL_SOURCE = (
    PARKER_INSPECTION_DIR
    / "ControlledRuptureRetrieval.java"
)

RUPTURE_RETRIEVAL_CLASSES = (
    PARKER_INSPECTION_DIR
    / "controlled_rupture_retrieval_classes"
)

RUPTURE_RETRIEVAL_COMPILE_LOG = (
    METADATA_DIR
    / "controlled_rupture_retrieval_compile.log"
)

RUPTURE_RETRIEVAL_RUN_LOG = (
    METADATA_DIR
    / "controlled_rupture_retrieval_run.log"
)

RUPTURE_RETRIEVAL_JAVAC_ARGS = (
    PARKER_INSPECTION_DIR
    / "controlled_rupture_retrieval_javac_arguments.txt"
)

RUPTURE_RETRIEVAL_JAVA_ARGS = (
    PARKER_INSPECTION_DIR
    / "controlled_rupture_retrieval_java_arguments.txt"
)


if not MODEL_SEARCH_ROOT.is_dir():
    raise FileNotFoundError(
        "The USGS model search directory was not found:\n"
        f"  {MODEL_SEARCH_ROOT}"
    )


model_root_candidates = []

directories_to_check = [
    MODEL_SEARCH_ROOT,
    *[
        path
        for path in MODEL_SEARCH_ROOT.rglob("*")
        if path.is_dir()
    ],
]

required_model_directories = [
    "active-crust",
    "stable-crust",
    "subduction",
    "site-data",
]


for candidate in directories_to_check:

    if all(
        (
            candidate
            / directory_name
        ).is_dir()
        for directory_name
        in required_model_directories
    ):
        model_root_candidates.append(
            candidate.resolve()
        )


model_root_candidates = list(
    dict.fromkeys(
        model_root_candidates
    )
)


if len(model_root_candidates) != 1:
    print("Possible model roots:")

    for candidate in model_root_candidates:
        print(f"  {candidate}")

    raise RuntimeError(
        "Expected exactly one USGS model root."
    )


USGS_MODEL_ROOT = model_root_candidates[0]


catalog = pd.read_csv(
    EVENT_CATALOG_PATH
)

portfolio = pd.read_csv(
    GMM_READY_PORTFOLIO_PATH
)


required_catalog_columns = [
    "event_id",
    "rupture_id",
    "tree_id",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "source_id",
    "source_name",
    "source_type",
    "rupture_index",
    "magnitude",
    "raw_annual_rate",
    "top_depth_km",
    "width_km",
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
]


missing_catalog_columns = [
    column
    for column in required_catalog_columns
    if column not in catalog.columns
]


if missing_catalog_columns:
    raise RuntimeError(
        "Catalog columns are missing:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_catalog_columns
        )
    )


required_site_columns = [
    "site_id",
    "longitude",
    "latitude",
]


missing_site_columns = [
    column
    for column in required_site_columns
    if column not in portfolio.columns
]


if missing_site_columns:
    raise RuntimeError(
        "Portfolio columns are missing:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_site_columns
        )
    )


unique_ruptures = (
    catalog
    .sort_values("event_id")
    .drop_duplicates(
        subset=["rupture_id"],
        keep="first",
    )
    .copy()
)


interface_with_centroid = (
    unique_ruptures.loc[
        unique_ruptures[
            "source_type"
        ].eq("INTERFACE")
        & unique_ruptures[
            "centroid_longitude"
        ].notna()
        & unique_ruptures[
            "centroid_latitude"
        ].notna()
    ]
    .head(1)
    .copy()
)


interface_without_centroid = (
    unique_ruptures.loc[
        unique_ruptures[
            "source_type"
        ].eq("INTERFACE")
        & unique_ruptures[
            "centroid_longitude"
        ].isna()
    ]
    .head(1)
    .copy()
)


slab_case = (
    unique_ruptures.loc[
        unique_ruptures[
            "source_type"
        ].eq("SLAB")
    ]
    .head(1)
    .copy()
)


if len(interface_with_centroid) != 1:
    raise RuntimeError(
        "Could not select an interface rupture "
        "with stored centroid data."
    )

if len(interface_without_centroid) != 1:
    raise RuntimeError(
        "Could not select an interface rupture "
        "with missing centroid data."
    )

if len(slab_case) != 1:
    raise RuntimeError(
        "Could not select an Oregon slab rupture."
    )


interface_with_centroid[
    "case_id"
] = "interface_with_catalog_centroid"

interface_without_centroid[
    "case_id"
] = "interface_missing_catalog_centroid"

slab_case[
    "case_id"
] = "oregon_intraslab"


probe_input = pd.concat(
    [
        interface_with_centroid,
        interface_without_centroid,
        slab_case,
    ],
    ignore_index=True,
)


test_site = portfolio.iloc[0]


probe_input[
    "test_site_id"
] = str(
    test_site["site_id"]
)

probe_input[
    "test_site_longitude"
] = float(
    test_site["longitude"]
)

probe_input[
    "test_site_latitude"
] = float(
    test_site["latitude"]
)


input_columns = [
    "case_id",
    "event_id",
    "rupture_id",
    "tree_id",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "source_id",
    "source_name",
    "source_type",
    "rupture_index",
    "magnitude",
    "raw_annual_rate",
    "top_depth_km",
    "width_km",
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
    "test_site_id",
    "test_site_longitude",
    "test_site_latitude",
]


probe_input[
    input_columns
].to_csv(
    RUPTURE_RETRIEVAL_INPUT_PATH,
    sep="\t",
    index=False,
    na_rep="",
)


if RUPTURE_RETRIEVAL_CLASSES.exists():
    shutil.rmtree(
        RUPTURE_RETRIEVAL_CLASSES
    )


RUPTURE_RETRIEVAL_CLASSES.mkdir(
    parents=True,
    exist_ok=True,
)


java_source = r'''
import gov.usgs.earthquake.nshmp.fault.surface.RuptureSurface;
import gov.usgs.earthquake.nshmp.geo.Location;
import gov.usgs.earthquake.nshmp.model.Distance;
import gov.usgs.earthquake.nshmp.model.HazardModel;
import gov.usgs.earthquake.nshmp.model.Rupture;
import gov.usgs.earthquake.nshmp.model.RuptureSet;
import gov.usgs.earthquake.nshmp.model.Source;
import gov.usgs.earthquake.nshmp.model.SourceTree;
import gov.usgs.earthquake.nshmp.tree.Branch;

import java.io.BufferedReader;
import java.io.PrintWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.HashMap;
import java.util.Locale;
import java.util.Map;
import java.util.Optional;

public final class ControlledRuptureRetrieval {

  private ControlledRuptureRetrieval() {}

  public static void main(String[] args) throws Exception {

    if (args.length != 3) {
      throw new IllegalArgumentException(
          "Expected model root, input TSV, and output CSV.");
    }

    Path modelRoot = Path.of(args[0])
        .toAbsolutePath()
        .normalize();

    Path inputPath = Path.of(args[1])
        .toAbsolutePath()
        .normalize();

    Path outputPath = Path.of(args[2])
        .toAbsolutePath()
        .normalize();

    Files.createDirectories(
        outputPath.getParent());

    HazardModel model =
        HazardModel.load(modelRoot);

    int rowCount = 0;

    try (
        BufferedReader reader =
            Files.newBufferedReader(
                inputPath,
                StandardCharsets.UTF_8);

        PrintWriter writer =
            new PrintWriter(
                Files.newBufferedWriter(
                    outputPath,
                    StandardCharsets.UTF_8))
    ) {

      String headerLine = reader.readLine();

      if (headerLine == null) {
        throw new IllegalArgumentException(
            "Input TSV is empty.");
      }

      String[] header =
          headerLine.split("\t", -1);

      Map<String, Integer> columnMap =
          new HashMap<>();

      for (int i = 0; i < header.length; i++) {
        columnMap.put(header[i], i);
      }

      writer.println(String.join(",",
          "case_id",
          "event_id",
          "rupture_id",
          "expected_tree_id",
          "retrieved_tree_id",
          "expected_branch_index",
          "retrieved_branch_id",
          "retrieved_branch_weight",
          "expected_rupture_set_id",
          "retrieved_rupture_set_id",
          "retrieved_rupture_set_name",
          "expected_source_index",
          "expected_source_id",
          "retrieved_source_id",
          "expected_source_name",
          "retrieved_source_name",
          "expected_source_type",
          "retrieved_source_type",
          "expected_rupture_index",
          "expected_magnitude",
          "retrieved_magnitude",
          "expected_raw_rate",
          "retrieved_raw_rate",
          "retrieved_rake_deg",
          "surface_class",
          "expected_top_depth_km",
          "surface_depth_km",
          "expected_width_km",
          "surface_width_km",
          "surface_dip_deg",
          "surface_strike_deg",
          "expected_centroid_longitude",
          "surface_centroid_longitude",
          "expected_centroid_latitude",
          "surface_centroid_latitude",
          "expected_centroid_depth_km",
          "surface_centroid_depth_km",
          "test_site_id",
          "test_site_longitude",
          "test_site_latitude",
          "r_jb_km",
          "r_rup_km",
          "r_x_km",
          "retrieval_error"
      ));

      String line;

      while ((line = reader.readLine()) != null) {

        if (line.isBlank()) {
          continue;
        }

        String[] values =
            line.split("\t", -1);

        String caseId =
            get(values, columnMap, "case_id");

        String eventId =
            get(values, columnMap, "event_id");

        String ruptureId =
            get(values, columnMap, "rupture_id");

        int treeId =
            parseInt(
                get(values, columnMap, "tree_id"));

        int branchIndex =
            parseInt(
                get(values, columnMap, "branch_index"));

        int expectedRuptureSetId =
            parseInt(
                get(values, columnMap, "rupture_set_id"));

        int sourceIndex =
            parseInt(
                get(values, columnMap, "source_index"));

        int expectedSourceId =
            parseInt(
                get(values, columnMap, "source_id"));

        int ruptureIndex =
            parseInt(
                get(values, columnMap, "rupture_index"));

        String expectedSourceName =
            get(values, columnMap, "source_name");

        String expectedSourceType =
            get(values, columnMap, "source_type");

        double expectedMagnitude =
            parseDouble(
                get(values, columnMap, "magnitude"));

        double expectedRawRate =
            parseDouble(
                get(values, columnMap, "raw_annual_rate"));

        double expectedTopDepth =
            parseDouble(
                get(values, columnMap, "top_depth_km"));

        double expectedWidth =
            parseDouble(
                get(values, columnMap, "width_km"));

        double expectedCentroidLongitude =
            parseDoubleOrNaN(
                get(values, columnMap, "centroid_longitude"));

        double expectedCentroidLatitude =
            parseDoubleOrNaN(
                get(values, columnMap, "centroid_latitude"));

        double expectedCentroidDepth =
            parseDoubleOrNaN(
                get(values, columnMap, "centroid_depth_km"));

        String testSiteId =
            get(values, columnMap, "test_site_id");

        double testSiteLongitude =
            parseDouble(
                get(values, columnMap, "test_site_longitude"));

        double testSiteLatitude =
            parseDouble(
                get(values, columnMap, "test_site_latitude"));

        int retrievedTreeId = -1;
        String branchId = "";
        double branchWeight = Double.NaN;
        int retrievedRuptureSetId = -1;
        String retrievedRuptureSetName = "";
        int retrievedSourceId = -1;
        String retrievedSourceName = "";
        String retrievedSourceType = "";
        double retrievedMagnitude = Double.NaN;
        double retrievedRate = Double.NaN;
        double retrievedRake = Double.NaN;
        String surfaceClass = "";
        double surfaceDepth = Double.NaN;
        double surfaceWidth = Double.NaN;
        double surfaceDip = Double.NaN;
        double surfaceStrike = Double.NaN;
        double surfaceCentroidLongitude = Double.NaN;
        double surfaceCentroidLatitude = Double.NaN;
        double surfaceCentroidDepth = Double.NaN;
        double rJB = Double.NaN;
        double rRup = Double.NaN;
        double rX = Double.NaN;
        String retrievalError = "";

try {

  Optional<SourceTree> treeOptional =
      model.tree(treeId);

  if (treeOptional.isEmpty()) {
    throw new IllegalArgumentException(
        "Tree not found: " + treeId);
  }

  SourceTree tree =
      treeOptional.get();

  retrievedTreeId =
      tree.id();

  Branch<RuptureSet<? extends Source>> branch =
      null;

  int ruptureSetIdMatchCount = 0;
  int compatibleBranchCount = 0;

  for (int index = 0; index < tree.size(); index++) {

    Branch<RuptureSet<? extends Source>> candidate =
        tree.get(index);

    RuptureSet<? extends Source> candidateRuptureSet =
        candidate.value();

    if (
        candidateRuptureSet.id()
        != expectedRuptureSetId
    ) {
      continue;
    }

    ruptureSetIdMatchCount++;

    if (
        sourceIndex < 0
        || sourceIndex >= candidateRuptureSet.size()
    ) {
      continue;
    }

    Source candidateSource =
        candidateRuptureSet.get(sourceIndex);

    if (
        candidateSource.id()
        != expectedSourceId
    ) {
      continue;
    }

    if (
        !candidateSource.name().equals(
            expectedSourceName)
    ) {
      continue;
    }

    if (
        ruptureIndex < 0
        || ruptureIndex >= candidateSource.size()
    ) {
      continue;
    }

    Rupture candidateRupture =
        candidateSource.get(ruptureIndex);

    boolean magnitudeMatches =
        Math.abs(
            candidateRupture.mag()
            - expectedMagnitude
        ) <= 1.0e-10;

    double rateScale =
        Math.max(
            Math.abs(candidateRupture.rate()),
            Math.abs(expectedRawRate)
        );

    double rateTolerance =
        Math.max(
            1.0e-18,
            1.0e-12 * rateScale
        );

    boolean rateMatches =
        Math.abs(
            candidateRupture.rate()
            - expectedRawRate
        ) <= rateTolerance;

    if (
        !magnitudeMatches
        || !rateMatches
    ) {
      continue;
    }

    if (branch == null) {
      branch =
          candidate;
    }

    compatibleBranchCount++;
  }

  if (branch == null) {

    throw new IllegalArgumentException(
        "No compatible branch was found for "
        + "rupture-set ID "
        + expectedRuptureSetId
        + " in tree "
        + treeId
        + ". Rupture-set ID matches: "
        + ruptureSetIdMatchCount
        + ", compatible matches: "
        + compatibleBranchCount);
  }

  branchId =
      branch.id();

  branchWeight =
      branch.weight();

  RuptureSet<? extends Source> ruptureSet =
      branch.value();

  retrievedRuptureSetId =
      ruptureSet.id();

  retrievedRuptureSetName =
      ruptureSet.name();

  Source source =
      ruptureSet.get(sourceIndex);

  retrievedSourceId =
      source.id();

  retrievedSourceName =
      source.name();

  retrievedSourceType =
      source.type().name();

  Rupture rupture =
      source.get(ruptureIndex);

  retrievedMagnitude =
      rupture.mag();

  retrievedRate =
      rupture.rate();

  retrievedRake =
      rupture.rake();

  RuptureSurface surface =
      rupture.surface();

  Location site =
      Location.create(
          testSiteLongitude,
          testSiteLatitude);

  Distance distance =
      surface.distanceTo(site);

  rJB =
      distance.rJB;

  rRup =
      distance.rRup;

  rX =
      distance.rX;

  surfaceClass =
      surface.getClass().getName();

  surfaceDepth =
      safeSurfaceValue(
          () -> surface.depth());

  surfaceWidth =
      safeSurfaceValue(
          () -> surface.width());

  surfaceDip =
      safeSurfaceValue(
          () -> surface.dip());

  surfaceStrike =
      safeSurfaceValue(
          () -> surface.strike());

  try {

    Location centroid =
        surface.centroid();

    if (centroid != null) {

      surfaceCentroidLongitude =
          centroid.longitude;

      surfaceCentroidLatitude =
          centroid.latitude;

      surfaceCentroidDepth =
          centroid.depth;
    }

  } catch (RuntimeException exception) {

    surfaceCentroidLongitude =
        Double.NaN;

    surfaceCentroidLatitude =
        Double.NaN;

    surfaceCentroidDepth =
        Double.NaN;
  }

} catch (Exception exception) {

  retrievalError =
      exception.getClass().getName()
      + ": "
      + String.valueOf(
          exception.getMessage());
}

        writer.println(String.join(",",
            csv(caseId),
            csv(eventId),
            csv(ruptureId),
            integer(treeId),
            integer(retrievedTreeId),
            integer(branchIndex),
            csv(branchId),
            number(branchWeight),
            integer(expectedRuptureSetId),
            integer(retrievedRuptureSetId),
            csv(retrievedRuptureSetName),
            integer(sourceIndex),
            integer(expectedSourceId),
            integer(retrievedSourceId),
            csv(expectedSourceName),
            csv(retrievedSourceName),
            csv(expectedSourceType),
            csv(retrievedSourceType),
            integer(ruptureIndex),
            number(expectedMagnitude),
            number(retrievedMagnitude),
            number(expectedRawRate),
            number(retrievedRate),
            number(retrievedRake),
            csv(surfaceClass),
            number(expectedTopDepth),
            number(surfaceDepth),
            number(expectedWidth),
            number(surfaceWidth),
            number(surfaceDip),
            number(surfaceStrike),
            number(expectedCentroidLongitude),
            number(surfaceCentroidLongitude),
            number(expectedCentroidLatitude),
            number(surfaceCentroidLatitude),
            number(expectedCentroidDepth),
            number(surfaceCentroidDepth),
            csv(testSiteId),
            number(testSiteLongitude),
            number(testSiteLatitude),
            number(rJB),
            number(rRup),
            number(rX),
            csv(retrievalError)
        ));

        rowCount++;
      }
    }

    System.out.println(
        "CONTROLLED_RUPTURE_RETRIEVAL_COMPLETE");

    System.out.println(
        "rows=" + rowCount);

    System.out.println(
        "model_root=" + modelRoot);

    System.out.println(
        "output=" + outputPath);
  }

  private static double safeSurfaceValue(
    java.util.function.DoubleSupplier supplier
) {

  try {

    return supplier.getAsDouble();

  } catch (RuntimeException exception) {

    return Double.NaN;
  }
}

private static String get(
    String[] values,
    Map<String, Integer> columnMap,
    String name
) {

    Integer index =
        columnMap.get(name);

    if (index == null) {
      throw new IllegalArgumentException(
          "Missing input column: " + name);
    }

    return values[index];
  }

  private static int parseInt(
      String value
  ) {
    return (int) Math.round(
        Double.parseDouble(value));
  }

  private static double parseDouble(
      String value
  ) {
    return Double.parseDouble(value);
  }

  private static double parseDoubleOrNaN(
      String value
  ) {

    if (value == null || value.isBlank()) {
      return Double.NaN;
    }

    return Double.parseDouble(value);
  }

  private static String integer(
      int value
  ) {
    return Integer.toString(value);
  }

  private static String number(
      double value
  ) {
    return String.format(
        Locale.US,
        "%.17g",
        value);
  }

  private static String csv(
      String value
  ) {

    if (value == null) {
      return "";
    }

    boolean quote =
        value.contains(",")
        || value.contains("\"")
        || value.contains("\n")
        || value.contains("\r");

    if (!quote) {
      return value;
    }

    return "\""
        + value.replace("\"", "\"\"")
        + "\"";
  }
}
'''.strip()


RUPTURE_RETRIEVAL_SOURCE.write_text(
    java_source + "\n",
    encoding="utf-8",
)


classpath = os.pathsep.join(
    str(path)
    for path in runtime_paths
)


def java_arg(value):

    text = str(value).replace(
        "\\",
        "\\\\",
    )

    text = text.replace(
        '"',
        '\\"',
    )

    return f'"{text}"'


RUPTURE_RETRIEVAL_JAVAC_ARGS.write_text(
    "\n".join(
        [
            "-encoding",
            "UTF-8",
            "-classpath",
            java_arg(classpath),
            "-d",
            java_arg(
                RUPTURE_RETRIEVAL_CLASSES
            ),
            java_arg(
                RUPTURE_RETRIEVAL_SOURCE
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


javac_path = shutil.which(
    "javac"
)

if javac_path is None:
    raise FileNotFoundError(
        "javac was not found through PATH."
    )


compile_result = subprocess.run(
    [
        javac_path,
        f"@{RUPTURE_RETRIEVAL_JAVAC_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


compile_output = "\n".join(
    value.strip()
    for value in [
        compile_result.stdout,
        compile_result.stderr,
    ]
    if value and value.strip()
)


RUPTURE_RETRIEVAL_COMPILE_LOG.write_text(
    compile_output + "\n",
    encoding="utf-8",
)


print("Compiler output:")

if compile_output:
    print(compile_output)
else:
    print("[no compiler messages]")


if compile_result.returncode != 0:
    raise RuntimeError(
        "Controlled rupture retrieval did not compile.\n\n"
        f"Compile log:\n"
        f"{RUPTURE_RETRIEVAL_COMPILE_LOG}"
    )


run_classpath = os.pathsep.join(
    [
        str(
            RUPTURE_RETRIEVAL_CLASSES
        ),
        *[
            str(path)
            for path in runtime_paths
        ],
    ]
)


RUPTURE_RETRIEVAL_JAVA_ARGS.write_text(
    "\n".join(
        [
            "-Dfile.encoding=UTF-8",
            "-classpath",
            java_arg(run_classpath),
            "ControlledRuptureRetrieval",
            java_arg(USGS_MODEL_ROOT),
            java_arg(
                RUPTURE_RETRIEVAL_INPUT_PATH
            ),
            java_arg(
                RUPTURE_RETRIEVAL_OUTPUT_PATH
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)


java_path = shutil.which(
    "java"
)

if java_path is None:
    raise FileNotFoundError(
        "java was not found through PATH."
    )


run_result = subprocess.run(
    [
        java_path,
        f"@{RUPTURE_RETRIEVAL_JAVA_ARGS}",
    ],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)


run_output = "\n".join(
    value.strip()
    for value in [
        run_result.stdout,
        run_result.stderr,
    ]
    if value and value.strip()
)


RUPTURE_RETRIEVAL_RUN_LOG.write_text(
    run_output + "\n",
    encoding="utf-8",
)


print("\nRuntime output:")
print(run_output)


if run_result.returncode != 0:
    raise RuntimeError(
        "Controlled rupture retrieval failed.\n\n"
        f"Run log:\n"
        f"{RUPTURE_RETRIEVAL_RUN_LOG}"
    )


if (
    "CONTROLLED_RUPTURE_RETRIEVAL_COMPLETE"
    not in run_output
):
    raise RuntimeError(
        "The expected rupture-retrieval "
        "completion marker was not found."
    )


retrieval = pd.read_csv(
    RUPTURE_RETRIEVAL_OUTPUT_PATH
)


numeric_columns = [
    "expected_tree_id",
    "retrieved_tree_id",
    "expected_branch_index",
    "expected_rupture_set_id",
    "retrieved_rupture_set_id",
    "expected_source_index",
    "expected_source_id",
    "retrieved_source_id",
    "expected_rupture_index",
    "expected_magnitude",
    "retrieved_magnitude",
    "expected_raw_rate",
    "retrieved_raw_rate",
    "expected_top_depth_km",
    "surface_depth_km",
    "expected_width_km",
    "surface_width_km",
    "expected_centroid_longitude",
    "surface_centroid_longitude",
    "expected_centroid_latitude",
    "surface_centroid_latitude",
    "expected_centroid_depth_km",
    "surface_centroid_depth_km",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
]


for column in numeric_columns:

    retrieval[column] = pd.to_numeric(
        retrieval[column],
        errors="coerce",
    )


retrieval[
    "retrieval_error"
] = retrieval[
    "retrieval_error"
].fillna("")


centroid_expected = (
    retrieval[
        "expected_centroid_longitude"
    ].notna()
    & retrieval[
        "expected_centroid_latitude"
    ].notna()
    & retrieval[
        "expected_centroid_depth_km"
    ].notna()
)


centroid_matches = np.ones(
    len(retrieval),
    dtype=bool,
)


centroid_matches[
    centroid_expected.to_numpy()
] = (
    np.isclose(
        retrieval.loc[
            centroid_expected,
            "expected_centroid_longitude",
        ],
        retrieval.loc[
            centroid_expected,
            "surface_centroid_longitude",
        ],
        rtol=0.0,
        atol=1e-8,
    )
    & np.isclose(
        retrieval.loc[
            centroid_expected,
            "expected_centroid_latitude",
        ],
        retrieval.loc[
            centroid_expected,
            "surface_centroid_latitude",
        ],
        rtol=0.0,
        atol=1e-8,
    )
    & np.isclose(
        retrieval.loc[
            centroid_expected,
            "expected_centroid_depth_km",
        ],
        retrieval.loc[
            centroid_expected,
            "surface_centroid_depth_km",
        ],
        rtol=0.0,
        atol=1e-8,
    )
)


validation = pd.DataFrame(
    [
        {
            "check": (
                "Three controlled ruptures were tested"
            ),
            "passes": len(retrieval) == 3,
        },
        {
            "check": (
                "Every rupture was retrieved without error"
            ),
            "passes": bool(
                retrieval[
                    "retrieval_error"
                ].eq("").all()
            ),
        },
        {
            "check": (
                "Retrieved tree IDs match the catalog"
            ),
            "passes": bool(
                (
                    retrieval[
                        "expected_tree_id"
                    ]
                    == retrieval[
                        "retrieved_tree_id"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Retrieved rupture-set IDs match the catalog"
            ),
            "passes": bool(
                (
                    retrieval[
                        "expected_rupture_set_id"
                    ]
                    == retrieval[
                        "retrieved_rupture_set_id"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Retrieved source IDs match the catalog"
            ),
            "passes": bool(
                (
                    retrieval[
                        "expected_source_id"
                    ]
                    == retrieval[
                        "retrieved_source_id"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Retrieved source names match the catalog"
            ),
            "passes": bool(
                (
                    retrieval[
                        "expected_source_name"
                    ]
                    == retrieval[
                        "retrieved_source_name"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Retrieved source types match the catalog"
            ),
            "passes": bool(
                (
                    retrieval[
                        "expected_source_type"
                    ]
                    == retrieval[
                        "retrieved_source_type"
                    ]
                ).all()
            ),
        },
        {
            "check": (
                "Retrieved magnitudes match the catalog"
            ),
            "passes": bool(
                np.allclose(
                    retrieval[
                        "expected_magnitude"
                    ],
                    retrieval[
                        "retrieved_magnitude"
                    ],
                    rtol=0.0,
                    atol=1e-12,
                )
            ),
        },
        {
            "check": (
                "Retrieved raw rates match the catalog"
            ),
            "passes": bool(
                np.allclose(
                    retrieval[
                        "expected_raw_rate"
                    ],
                    retrieval[
                        "retrieved_raw_rate"
                    ],
                    rtol=1e-12,
                    atol=1e-18,
                )
            ),
        },
        {
            "check": (
                "Retrieved top depths match the catalog"
            ),
            "passes": bool(
                np.allclose(
                    retrieval[
                        "expected_top_depth_km"
                    ],
                    retrieval[
                        "surface_depth_km"
                    ],
                    rtol=0.0,
                    atol=1e-8,
                )
            ),
        },
        {
            "check": (
                "Retrieved widths match the catalog"
            ),
            "passes": bool(
                np.allclose(
                    retrieval[
                        "expected_width_km"
                    ],
                    retrieval[
                        "surface_width_km"
                    ],
                    rtol=0.0,
                    atol=1e-8,
                )
            ),
        },
        {
            "check": (
                "Stored centroids match retrieved centroids "
                "where catalog centroids exist"
            ),
            "passes": bool(
                centroid_matches.all()
            ),
        },
        {
            "check": (
                "Every Rrup value is finite and nonnegative"
            ),
            "passes": bool(
                np.isfinite(
                    retrieval[
                        "r_rup_km"
                    ]
                ).all()
                and (
                    retrieval[
                        "r_rup_km"
                    ]
                    >= 0.0
                ).all()
            ),
        },
        {
            "check": (
                "Every Rjb value is finite and nonnegative"
            ),
            "passes": bool(
                np.isfinite(
                    retrieval[
                        "r_jb_km"
                    ]
                ).all()
                and (
                    retrieval[
                        "r_jb_km"
                    ]
                    >= 0.0
                ).all()
            ),
        },
        {
            "check": (
                "Rrup is not smaller than Rjb"
            ),
            "passes": bool(
                (
                    retrieval[
                        "r_rup_km"
                    ]
                    + 1e-10
                    >= retrieval[
                        "r_jb_km"
                    ]
                ).all()
            ),
        },
    ]
)


validation.to_csv(
    RUPTURE_RETRIEVAL_VALIDATION_PATH,
    index=False,
)


print("\nRetrieved rupture metadata and distances:")

display(
    retrieval[
        [
            "case_id",
            "rupture_id",
            "retrieved_rupture_set_id",
            "retrieved_source_id",
            "retrieved_source_name",
            "retrieved_magnitude",
            "retrieved_raw_rate",
            "surface_class",
            "surface_depth_km",
            "surface_width_km",
            "surface_centroid_longitude",
            "surface_centroid_latitude",
            "surface_centroid_depth_km",
            "r_jb_km",
            "r_rup_km",
            "r_x_km",
            "retrieval_error",
        ]
    ]
)


print("\nControlled retrieval validation:")

display(validation)


if not validation[
    "passes"
].all():

    failed = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Controlled rupture retrieval validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed
        )
    )


print("\nCELL 14 CONTROLLED RUPTURE RETRIEVAL COMPLETE")

print(
    "\nUSGS model root:"
    f"\n  {USGS_MODEL_ROOT}"
)

print(
    "\nControlled output:"
    f"\n  {RUPTURE_RETRIEVAL_OUTPUT_PATH}"
)

print(
    "\nValidation:"
    f"\n  {RUPTURE_RETRIEVAL_VALIDATION_PATH}"
)

print(
    "\nNext step: calculate authoritative USGS "
    "rupture-to-site distances for all 3,927 unique "
    "ruptures and 470 W2 portfolio sites."
)

Compiler output:
[no compiler messages]

Runtime output:
C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\usgs_nshm_conus_2018\nshm-conus-5.2.4\nshm-conus-5.2.4

site-data/
  basin-data-region: Puget Lowland
  basin-data-region: Los Angeles
  basin-data-region: San Francisco Bay Area
  basin-data-region: Wasatch Front

active-crust\fault
    tree: [2000] AZ\Algodones.geojson
    tree: [2001] AZ\Aubrey.geojson
    tree: [2002] AZ\Big Chino - Little Chino.geojson
    tree: [2003] AZ\Dutchman Draw.geojson
    tree: [2004] AZ\Hurricane (center).geojson
    tree: [2005] AZ\Hurricane (south).geojson
    tree: [2006] AZ\Sevier - Toroweap (south).geojson
    tree: [8309] CA\ucerf3
  branch:        CA\ucerf3\fault-model-3.1
  branch:        CA\ucerf3\fault-model-3.2
    tree: [2100] CO\Gore Range Frontal.geojson
    tree: [2101] CO\Sangre de Cristo (north).geojson
    tree: [2102] CO\Sawatch (south).geojson
    tree: [2103] CO\Williams Fork Mountains.geojson
    tree: 

,case_id,rupture_id,retrieved_rupture_set_id,retrieved_source_id,retrieved_source_name,retrieved_magnitude,retrieved_raw_rate,surface_class,surface_depth_km,surface_width_km,surface_centroid_longitude,surface_centroid_latitude,surface_centroid_depth_km,r_jb_km,r_rup_km,r_x_km,retrieval_error
0,interface_with_catalog_centroid,ff36c8c176c0929711a982c1,3172,3172,"Cascadia (full, top)",8.72,6.327000e-04,gov.usgs.earthquake.nshmp.fault.surface.Approx...,5.091846,85.0,-125.266845,45.069512,11.059412,43.387966,46.907295,134.266722,
1,interface_missing_catalog_centroid,49283c0b1304034206982cfb,3160,3160,"Cascadia (unsegmented, 1-2-3, bottom)",8.40,6.664417e-07,gov.usgs.earthquake.nshmp.fault.surface.Gridde...,5.000000,145.0,NaN,NaN,NaN,43.497254,51.799657,128.548951,
2,oregon_intraslab,f0fd5c1a0da521278eb8cbb0,8211,8211,"PointSourceFinite: -123.700, 45.300",7.05,6.076027e-07,gov.usgs.earthquake.nshmp.model.PointSourceFin...,42.000000,8.0,-123.700000,45.300000,42.000000,68.380000,80.248516,-68.380000,



Controlled retrieval validation:


,check,passes
0,Three controlled ruptures were tested,True
1,Every rupture was retrieved without error,True
2,Retrieved tree IDs match the catalog,True
3,Retrieved rupture-set IDs match the catalog,True
4,Retrieved source IDs match the catalog,True
5,Retrieved source names match the catalog,True
6,Retrieved source types match the catalog,True
7,Retrieved magnitudes match the catalog,True
8,Retrieved raw rates match the catalog,True
9,Retrieved top depths match the catalog,True



CELL 14 CONTROLLED RUPTURE RETRIEVAL COMPLETE

USGS model root:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\raw\usgs_nshm_conus_2018\nshm-conus-5.2.4\nshm-conus-5.2.4

Controlled output:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_controlled_rupture_retrieval_output.csv

Validation:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_controlled_rupture_retrieval_validation.csv

Next step: calculate authoritative USGS rupture-to-site distances for all 3,927 unique ruptures and 470 W2 portfolio sites.


In [ ]:
# portfolio_check = pd.read_csv(
#     GMM_READY_PORTFOLIO_PATH
# )

# print(
#     portfolio_check[
#         [
#             "site_id",
#             "longitude",
#             "latitude",
#         ]
#     ].head().to_string(
#         index=False,
#     )
# )

site_id   longitude  latitude
W2_0001 -123.922071 45.991446
W2_0002 -123.923246 45.988641
W2_0003 -123.918762 45.998309
W2_0004 -123.915539 46.001041
W2_0005 -123.924822 45.998049


In [ ]:
# import numpy as np
# import pandas as pd


# portfolio_paths = [
#     W2_PORTFOLIO_PATH,
#     GMM_READY_PORTFOLIO_PATH,
# ]


# for portfolio_path in portfolio_paths:

#     frame = pd.read_csv(
#         portfolio_path
#     )

#     if {
#         "Longitude",
#         "Latitude",
#     }.issubset(frame.columns):

#         frame["longitude"] = pd.to_numeric(
#             frame["Longitude"],
#             errors="raise",
#         )

#         frame["latitude"] = pd.to_numeric(
#             frame["Latitude"],
#             errors="raise",
#         )

#     else:

#         longitude = pd.to_numeric(
#             frame["longitude"],
#             errors="raise",
#         )

#         latitude = pd.to_numeric(
#             frame["latitude"],
#             errors="raise",
#         )

#         appears_swapped = (
#             longitude.between(
#                 -90.0,
#                 90.0,
#             ).all()
#             and latitude.between(
#                 -180.0,
#                 180.0,
#             ).all()
#             and not latitude.between(
#                 -90.0,
#                 90.0,
#             ).all()
#         )

#         if appears_swapped:
#             frame["longitude"] = latitude
#             frame["latitude"] = longitude

#     if not frame[
#         "longitude"
#     ].between(
#         -125.0,
#         -122.0,
#     ).all():
#         raise RuntimeError(
#             f"Unexpected Seaside longitude values in:\n"
#             f"  {portfolio_path}"
#         )

#     if not frame[
#         "latitude"
#     ].between(
#         45.0,
#         47.0,
#     ).all():
#         raise RuntimeError(
#             f"Unexpected Seaside latitude values in:\n"
#             f"  {portfolio_path}"
#         )

#     frame.to_csv(
#         portfolio_path,
#         index=False,
#     )

#     print(
#         f"Corrected: {portfolio_path.name}"
#     )

#     print(
#         "  Longitude range:"
#         f" {frame['longitude'].min():.6f}"
#         f" to {frame['longitude'].max():.6f}"
#     )

#     print(
#         "  Latitude range:"
#         f" {frame['latitude'].min():.6f}"
#         f" to {frame['latitude'].max():.6f}"
#     )

Corrected: seaside_w2_portfolio.csv
  Longitude range: -123.954969 to -123.890681
  Latitude range: 45.971415 to 46.017120
Corrected: seaside_w2_portfolio_gmm_ready.csv
  Longitude range: -123.954969 to -123.890681
  Latitude range: 45.971415 to 46.017120


In [28]:
# ============================================================================
# CELL 15
# Authoritative USGS rupture-to-site distances for 3,927 unique ruptures and
# 470 W2 sites, with rupture chunking, restart markers, and validation.
# ============================================================================

from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import gzip
import hashlib
import json
import math
import os
import shutil
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Configuration and required earlier-cell variables
# ----------------------------------------------------------------------------

PIPELINE_VERSION = "notebook4_cell15_authoritative_distances_v2_branch_resolution"
EXPECTED_RUPTURES = 3_927
EXPECTED_SITES = 470
RUPTURES_PER_CHUNK = 500
VERIFY_EXISTING_HASHES = True
JAVA_MAX_HEAP = os.environ.get("NOTEBOOK4_JAVA_XMX", "4g").strip()
FINAL_READ_CHUNK_SIZE = 250_000

required_variables = [
    "DATA_DIR",
    "METADATA_DIR",
    "PARKER_INSPECTION_DIR",
    "EVENT_CATALOG_PATH",
    "GMM_READY_PORTFOLIO_PATH",
    "USGS_MODEL_ROOT",
    "runtime_paths",
    "RUPTURE_RETRIEVAL_OUTPUT_PATH",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Cell 15 requires variables created by earlier Notebook 4 cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

DATA_DIR = Path(DATA_DIR).resolve()
METADATA_DIR = Path(METADATA_DIR).resolve()
PARKER_INSPECTION_DIR = Path(PARKER_INSPECTION_DIR).resolve()
EVENT_CATALOG_PATH = Path(EVENT_CATALOG_PATH).resolve()
GMM_READY_PORTFOLIO_PATH = Path(GMM_READY_PORTFOLIO_PATH).resolve()
USGS_MODEL_ROOT = Path(USGS_MODEL_ROOT).resolve()
RUPTURE_RETRIEVAL_OUTPUT_PATH = Path(
    RUPTURE_RETRIEVAL_OUTPUT_PATH
).resolve()
runtime_paths = [Path(path).resolve() for path in runtime_paths]

required_paths = {
    "event catalog": EVENT_CATALOG_PATH,
    "GMM-ready W2 portfolio": GMM_READY_PORTFOLIO_PATH,
    "USGS model root": USGS_MODEL_ROOT,
    "Cell 14 controlled output": RUPTURE_RETRIEVAL_OUTPUT_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.exists()
]
missing_runtime_paths = [path for path in runtime_paths if not path.exists()]

if missing_paths or missing_runtime_paths:
    message = []
    if missing_paths:
        message.append("Missing required inputs:\n" + "\n".join(
            f"  - {item}" for item in missing_paths
        ))
    if missing_runtime_paths:
        message.append("Missing Java classpath entries:\n" + "\n".join(
            f"  - {path}" for path in missing_runtime_paths
        ))
    raise FileNotFoundError("\n\n".join(message))


# ----------------------------------------------------------------------------
# 2. Output paths
# ----------------------------------------------------------------------------

DATA_OUTPUT_DIR = (
    DATA_DIR / "processed" / "notebook_4_authoritative_usgs_distances"
)
WORK_DIR = METADATA_DIR / "notebook_4_authoritative_usgs_distances_work"
INPUT_DIR = WORK_DIR / "inputs"
CHUNK_DIR = WORK_DIR / "chunks"
LOG_DIR = WORK_DIR / "logs"
MARKER_DIR = WORK_DIR / "completion_markers"
CHUNK_VALIDATION_DIR = WORK_DIR / "chunk_validation"

for directory in [
    DATA_OUTPUT_DIR,
    WORK_DIR,
    INPUT_DIR,
    CHUNK_DIR,
    LOG_DIR,
    MARKER_DIR,
    CHUNK_VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SITE_INPUT_PATH = INPUT_DIR / "w2_sites.tsv"
RUPTURE_INDEX_PATH = METADATA_DIR / "notebook_4_unique_rupture_index.csv"
SITE_INDEX_PATH = METADATA_DIR / "notebook_4_w2_site_index.csv"
MANIFEST_PATH = (
    METADATA_DIR / "notebook_4_rupture_site_distance_chunk_manifest.csv"
)
FINAL_DISTANCE_PATH = DATA_OUTPUT_DIR / "usgs_rupture_site_distances.csv.gz"
FINAL_AUDIT_PATH = (
    METADATA_DIR / "notebook_4_authoritative_rupture_retrieval_audit.csv.gz"
)
FINAL_VALIDATION_PATH = (
    METADATA_DIR / "notebook_4_authoritative_distance_validation.csv"
)
FINAL_SUMMARY_PATH = (
    METADATA_DIR / "notebook_4_authoritative_distance_summary.json"
)

JAVA_SOURCE_PATH = (
    PARKER_INSPECTION_DIR / "AuthoritativeRuptureSiteDistances.java"
)
JAVA_CLASSES_DIR = (
    PARKER_INSPECTION_DIR / "authoritative_rupture_site_distance_classes"
)
JAVAC_ARGS_PATH = (
    PARKER_INSPECTION_DIR
    / "authoritative_rupture_site_distance_javac_arguments.txt"
)
COMPILE_LOG_PATH = LOG_DIR / "authoritative_distance_compile.log"


# ----------------------------------------------------------------------------
# 3. Small utility functions
# ----------------------------------------------------------------------------

def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            block = file_handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def java_arg(value):
    text = str(value).replace("\\", "\\\\").replace('"', '\\"')
    return f'"{text}"'


def allclose(left, right, *, rtol=0.0, atol=0.0, equal_nan=False):
    left = pd.to_numeric(left, errors="coerce").to_numpy(dtype=float)
    right = pd.to_numeric(right, errors="coerce").to_numpy(dtype=float)
    return bool(np.allclose(
        left,
        right,
        rtol=rtol,
        atol=atol,
        equal_nan=equal_nan,
    ))


def combine_gzip_csv(source_paths, destination_path):
    destination_path = Path(destination_path)
    temporary_path = destination_path.with_name(
        destination_path.name + ".temporary"
    )
    expected_header = None

    with gzip.open(
        temporary_path,
        "wt",
        encoding="utf-8",
        newline="",
    ) as destination:
        for source_path in map(Path, source_paths):
            with gzip.open(
                source_path,
                "rt",
                encoding="utf-8",
                newline="",
            ) as source:
                header = source.readline()
                if not header:
                    raise RuntimeError(f"Chunk has no header: {source_path}")
                if expected_header is None:
                    expected_header = header
                    destination.write(header)
                elif header != expected_header:
                    raise RuntimeError(
                        "Chunk CSV headers are inconsistent:\n"
                        f"  {source_path}"
                    )
                shutil.copyfileobj(source, destination, length=1024 * 1024)

    os.replace(temporary_path, destination_path)


# ----------------------------------------------------------------------------
# 4. Load and strictly validate the rupture index and site index
# ----------------------------------------------------------------------------

catalog = pd.read_csv(EVENT_CATALOG_PATH, low_memory=False)
portfolio = pd.read_csv(GMM_READY_PORTFOLIO_PATH, low_memory=False)

required_catalog_columns = [
    "event_id",
    "rupture_id",
    "tree_id",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "source_id",
    "source_name",
    "source_type",
    "rupture_index",
    "magnitude",
    "raw_annual_rate",
    "top_depth_km",
    "width_km",
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
]
required_site_columns = ["site_id", "longitude", "latitude"]

missing_catalog_columns = [
    column for column in required_catalog_columns if column not in catalog
]
missing_site_columns = [
    column for column in required_site_columns if column not in portfolio
]

if missing_catalog_columns or missing_site_columns:
    raise RuntimeError(
        "Missing required columns.\n\n"
        f"Catalog: {missing_catalog_columns}\n"
        f"Portfolio: {missing_site_columns}"
    )

for column in ["event_id", "rupture_id", "source_name", "source_type"]:
    catalog[column] = catalog[column].astype("string").fillna("").astype(str)
portfolio["site_id"] = (
    portfolio["site_id"].astype("string").fillna("").astype(str)
)

if catalog["rupture_id"].eq("").any():
    raise RuntimeError("The catalog contains blank rupture IDs.")
if portfolio["site_id"].eq("").any():
    raise RuntimeError("The portfolio contains blank site IDs.")

integer_columns = [
    "tree_id",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "source_id",
    "rupture_index",
]

for column in integer_columns:
    values = pd.to_numeric(catalog[column], errors="coerce")
    if (
        not np.isfinite(values).all()
        or not np.isclose(values, np.round(values), atol=1e-10).all()
    ):
        raise RuntimeError(f"Catalog column {column!r} is not integer-valued.")
    catalog[column] = np.round(values).astype(np.int64)

for column in ["magnitude", "raw_annual_rate", "top_depth_km", "width_km"]:
    catalog[column] = pd.to_numeric(catalog[column], errors="coerce")
    if not np.isfinite(catalog[column]).all():
        raise RuntimeError(f"Catalog column {column!r} is not finite.")

for column in [
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
]:
    catalog[column] = pd.to_numeric(catalog[column], errors="coerce")

for column in ["longitude", "latitude"]:
    portfolio[column] = pd.to_numeric(portfolio[column], errors="coerce")

if not (
    np.isfinite(portfolio["longitude"]).all()
    and np.isfinite(portfolio["latitude"]).all()
):
    raise RuntimeError("Portfolio coordinates contain missing/non-finite values.")
if not portfolio["longitude"].between(-180.0, 180.0).all():
    raise RuntimeError("Portfolio longitude is outside [-180, 180].")
if not portfolio["latitude"].between(-90.0, 90.0).all():
    raise RuntimeError("Portfolio latitude is outside [-90, 90].")
if portfolio["site_id"].duplicated().any():
    raise RuntimeError("The W2 portfolio contains duplicate site IDs.")

invariant_columns = [
    "tree_id",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "source_id",
    "source_name",
    "source_type",
    "rupture_index",
    "magnitude",
    "raw_annual_rate",
    "top_depth_km",
    "width_km",
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
]

conflicts = (
    catalog.groupby("rupture_id", sort=False)[invariant_columns]
    .nunique(dropna=False)
    .gt(1)
    .any(axis=1)
)

if conflicts.any():
    bad_ids = conflicts.index[conflicts].astype(str).tolist()
    raise RuntimeError(
        "Rupture IDs have conflicting retrieval metadata:\n"
        + "\n".join(f"  - {value}" for value in bad_ids[:30])
    )

catalog["_catalog_order"] = np.arange(len(catalog), dtype=np.int64)
ruptures = (
    catalog.sort_values("_catalog_order")
    .drop_duplicates("rupture_id", keep="first")
    .copy()
    .reset_index(drop=True)
)
ruptures["rupture_ordinal"] = np.arange(len(ruptures), dtype=np.int64)

sites = portfolio[["site_id", "longitude", "latitude"]].copy()
sites = sites.reset_index(drop=True)
sites["site_ordinal"] = np.arange(len(sites), dtype=np.int64)

if len(ruptures) != EXPECTED_RUPTURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RUPTURES:,} unique ruptures; "
        f"observed {len(ruptures):,}."
    )
if len(sites) != EXPECTED_SITES:
    raise RuntimeError(
        f"Expected {EXPECTED_SITES:,} W2 sites; observed {len(sites):,}."
    )

expected_rows = len(ruptures) * len(sites)
if expected_rows != 1_845_690:
    raise RuntimeError(f"Unexpected rupture-site pair count: {expected_rows:,}")

rupture_columns = [
    "rupture_ordinal",
    "event_id",
    "rupture_id",
    "tree_id",
    "branch_index",
    "rupture_set_id",
    "source_index",
    "source_id",
    "source_name",
    "source_type",
    "rupture_index",
    "magnitude",
    "raw_annual_rate",
    "top_depth_km",
    "width_km",
    "centroid_longitude",
    "centroid_latitude",
    "centroid_depth_km",
]

ruptures[rupture_columns].to_csv(RUPTURE_INDEX_PATH, index=False, na_rep="")
sites[["site_ordinal", "site_id", "longitude", "latitude"]].to_csv(
    SITE_INDEX_PATH,
    index=False,
)
sites[["site_ordinal", "site_id", "longitude", "latitude"]].to_csv(
    SITE_INPUT_PATH,
    sep="\t",
    index=False,
    lineterminator="\n",
)
site_input_hash = sha256_file(SITE_INPUT_PATH)

print("=" * 78)
print("AUTHORITATIVE USGS RUPTURE-TO-SITE DISTANCES")
print("=" * 78)
print(f"\nUnique ruptures: {len(ruptures):,}")
print(f"W2 sites:        {len(sites):,}")
print(f"Distance rows:   {expected_rows:,}")
print(f"Rupture chunk:   {RUPTURES_PER_CHUNK:,}")


# ----------------------------------------------------------------------------
# 5. Create deterministic rupture chunks and a manifest
# ----------------------------------------------------------------------------

chunk_count = math.ceil(len(ruptures) / RUPTURES_PER_CHUNK)
chunk_records = []

for chunk_number in range(chunk_count):
    start = chunk_number * RUPTURES_PER_CHUNK
    stop = min(start + RUPTURES_PER_CHUNK, len(ruptures))
    chunk_id = f"{chunk_number:04d}"
    rupture_chunk = ruptures.iloc[start:stop].copy()
    rupture_chunk["chunk_id"] = chunk_id

    rupture_input = INPUT_DIR / f"ruptures_chunk_{chunk_id}.tsv"
    distance_output = CHUNK_DIR / f"distances_chunk_{chunk_id}.csv.gz"
    audit_output = CHUNK_DIR / f"audit_chunk_{chunk_id}.csv.gz"
    run_log = LOG_DIR / f"distance_chunk_{chunk_id}.log"
    java_args = (
        PARKER_INSPECTION_DIR / f"distance_chunk_{chunk_id}_java_arguments.txt"
    )
    marker = MARKER_DIR / f"distance_chunk_{chunk_id}.complete.json"
    validation = (
        CHUNK_VALIDATION_DIR / f"distance_chunk_{chunk_id}_validation.csv"
    )

    rupture_chunk[["chunk_id", *rupture_columns]].to_csv(
        rupture_input,
        sep="\t",
        index=False,
        na_rep="",
        lineterminator="\n",
    )

    chunk_records.append({
        "chunk_number": chunk_number,
        "chunk_id": chunk_id,
        "start": start,
        "stop": stop,
        "rupture_count": len(rupture_chunk),
        "site_count": len(sites),
        "expected_rows": len(rupture_chunk) * len(sites),
        "rupture_input": str(rupture_input),
        "rupture_input_hash": sha256_file(rupture_input),
        "site_input": str(SITE_INPUT_PATH),
        "site_input_hash": site_input_hash,
        "distance_output": str(distance_output),
        "audit_output": str(audit_output),
        "run_log": str(run_log),
        "java_args": str(java_args),
        "marker": str(marker),
        "validation": str(validation),
        "status": "pending",
        "completed_at_utc": "",
    })


def save_manifest():
    pd.DataFrame(chunk_records).to_csv(MANIFEST_PATH, index=False)


save_manifest()


# ----------------------------------------------------------------------------
# 6. Java calculator: load each rupture once, then evaluate all 470 sites
# ----------------------------------------------------------------------------

java_source = r'''
import gov.usgs.earthquake.nshmp.fault.surface.RuptureSurface;
import gov.usgs.earthquake.nshmp.geo.Location;
import gov.usgs.earthquake.nshmp.model.Distance;
import gov.usgs.earthquake.nshmp.model.HazardModel;
import gov.usgs.earthquake.nshmp.model.Rupture;
import gov.usgs.earthquake.nshmp.model.RuptureSet;
import gov.usgs.earthquake.nshmp.model.Source;
import gov.usgs.earthquake.nshmp.model.SourceTree;
import gov.usgs.earthquake.nshmp.tree.Branch;

import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.io.IOException;
import java.io.OutputStream;
import java.io.OutputStreamWriter;
import java.io.PrintWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.ArrayList;
import java.util.HashMap;
import java.util.List;
import java.util.Locale;
import java.util.Map;
import java.util.Optional;
import java.util.zip.GZIPOutputStream;

public final class AuthoritativeRuptureSiteDistances {

  private AuthoritativeRuptureSiteDistances() {}

  public static void main(String[] args) throws Exception {

    if (args.length != 5) {
      throw new IllegalArgumentException(
          "Expected model root, rupture TSV, site TSV, distance CSV.gz, "
          + "and rupture-audit CSV.gz.");
    }

    Path modelRoot = Path.of(args[0]).toAbsolutePath().normalize();
    Path ruptureInput = Path.of(args[1]).toAbsolutePath().normalize();
    Path siteInput = Path.of(args[2]).toAbsolutePath().normalize();
    Path distanceOutput = Path.of(args[3]).toAbsolutePath().normalize();
    Path auditOutput = Path.of(args[4]).toAbsolutePath().normalize();

    HazardModel model = HazardModel.load(modelRoot);
    List<SiteRecord> sites = readSites(siteInput);

    int ruptureCount = 0;
    long distanceCount = 0L;
    String observedChunkId = "";

    try (
        BufferedReader reader = Files.newBufferedReader(
            ruptureInput, StandardCharsets.UTF_8);
        PrintWriter distanceWriter = openWriter(distanceOutput);
        PrintWriter auditWriter = openWriter(auditOutput)
    ) {

      String headerLine = reader.readLine();
      if (headerLine == null) {
        throw new IllegalArgumentException("Rupture TSV is empty.");
      }
      Map<String, Integer> columns = headerMap(headerLine);

      distanceWriter.println(String.join(",",
          "chunk_id", "rupture_ordinal", "site_ordinal", "rupture_id",
          "site_id", "r_jb_km", "r_rup_km", "r_x_km", "retrieval_error"));

      auditWriter.println(String.join(",",
          "chunk_id", "rupture_ordinal", "event_id", "rupture_id",
          "expected_tree_id", "retrieved_tree_id",
          "expected_branch_index", "retrieved_branch_index",
          "retrieved_branch_id", "retrieved_branch_weight",
          "expected_rupture_set_id", "retrieved_rupture_set_id",
          "expected_source_index", "expected_source_id", "retrieved_source_id",
          "expected_source_name", "retrieved_source_name",
          "expected_source_type", "retrieved_source_type",
          "expected_rupture_index", "expected_magnitude", "retrieved_magnitude",
          "expected_raw_rate", "retrieved_raw_rate", "retrieved_rake_deg",
          "expected_top_depth_km", "surface_depth_km",
          "expected_width_km", "surface_width_km",
          "surface_dip_deg", "surface_strike_deg",
          "expected_centroid_longitude", "surface_centroid_longitude",
          "expected_centroid_latitude", "surface_centroid_latitude",
          "expected_centroid_depth_km", "surface_centroid_depth_km",
          "surface_class", "retrieval_error"));

      String line;
      while ((line = reader.readLine()) != null) {
        if (line.isBlank()) continue;
        String[] values = line.split("\\t", -1);

        String chunkId = get(values, columns, "chunk_id");
        observedChunkId = chunkId;
        int ruptureOrdinal = parseInt(get(values, columns, "rupture_ordinal"));
        String eventId = get(values, columns, "event_id");
        String ruptureId = get(values, columns, "rupture_id");
        int treeId = parseInt(get(values, columns, "tree_id"));
        int branchIndex = parseInt(get(values, columns, "branch_index"));
        int ruptureSetId = parseInt(get(values, columns, "rupture_set_id"));
        int sourceIndex = parseInt(get(values, columns, "source_index"));
        int sourceId = parseInt(get(values, columns, "source_id"));
        String sourceName = get(values, columns, "source_name");
        String sourceType = get(values, columns, "source_type");
        int ruptureIndex = parseInt(get(values, columns, "rupture_index"));
        double magnitude = parseDouble(get(values, columns, "magnitude"));
        double rawRate = parseDouble(get(values, columns, "raw_annual_rate"));
        double topDepth = parseDouble(get(values, columns, "top_depth_km"));
        double width = parseDouble(get(values, columns, "width_km"));
        double centroidLon = parseDoubleOrNaN(
            get(values, columns, "centroid_longitude"));
        double centroidLat = parseDoubleOrNaN(
            get(values, columns, "centroid_latitude"));
        double centroidDepth = parseDoubleOrNaN(
            get(values, columns, "centroid_depth_km"));

        int retrievedTreeId = -1;
        int retrievedBranchIndex = -1;
        String retrievedBranchId = "";
        double retrievedBranchWeight = Double.NaN;
        int retrievedRuptureSetId = -1;
        int retrievedSourceId = -1;
        String retrievedSourceName = "";
        String retrievedSourceType = "";
        double retrievedMagnitude = Double.NaN;
        double retrievedRate = Double.NaN;
        double retrievedRake = Double.NaN;
        double surfaceDepth = Double.NaN;
        double surfaceWidth = Double.NaN;
        double surfaceDip = Double.NaN;
        double surfaceStrike = Double.NaN;
        double surfaceCentroidLon = Double.NaN;
        double surfaceCentroidLat = Double.NaN;
        double surfaceCentroidDepth = Double.NaN;
        String surfaceClass = "";
        String retrievalError = "";
        RuptureSurface surface = null;

        try {
          Optional<SourceTree> treeOptional = model.tree(treeId);
          if (treeOptional.isEmpty()) {
            throw new IllegalArgumentException("Tree not found: " + treeId);
          }
          SourceTree tree = treeOptional.get();
          retrievedTreeId = tree.id();

          BranchMatch match = resolveBranch(
              tree, branchIndex, ruptureSetId, sourceIndex, sourceId,
              sourceName, sourceType, ruptureIndex, magnitude, rawRate);

          retrievedBranchIndex = match.index;
          Branch<RuptureSet<? extends Source>> branch = match.branch;
          retrievedBranchId = branch.id();
          retrievedBranchWeight = branch.weight();

          RuptureSet<? extends Source> ruptureSet = branch.value();
          retrievedRuptureSetId = ruptureSet.id();
          Source source = ruptureSet.get(sourceIndex);
          retrievedSourceId = source.id();
          retrievedSourceName = source.name();
          retrievedSourceType = source.type().name();
          Rupture rupture = source.get(ruptureIndex);
          retrievedMagnitude = rupture.mag();
          retrievedRate = rupture.rate();
          retrievedRake = rupture.rake();
          surface = rupture.surface();
          surfaceClass = surface.getClass().getName();

          final RuptureSurface resolvedSurface = surface;
          surfaceDepth = safe(() -> resolvedSurface.depth());
          surfaceWidth = safe(() -> resolvedSurface.width());
          surfaceDip = safe(() -> resolvedSurface.dip());
          surfaceStrike = safe(() -> resolvedSurface.strike());

          try {
            Location centroid = surface.centroid();
            if (centroid != null) {
              surfaceCentroidLon = centroid.longitude;
              surfaceCentroidLat = centroid.latitude;
              surfaceCentroidDepth = centroid.depth;
            }
          } catch (RuntimeException exception) {
            surfaceCentroidLon = Double.NaN;
            surfaceCentroidLat = Double.NaN;
            surfaceCentroidDepth = Double.NaN;
          }

        } catch (Exception exception) {
          retrievalError = errorText(exception);
        }

        auditWriter.println(String.join(",",
            csv(chunkId), integer(ruptureOrdinal), csv(eventId), csv(ruptureId),
            integer(treeId), integer(retrievedTreeId),
            integer(branchIndex), integer(retrievedBranchIndex),
            csv(retrievedBranchId), number(retrievedBranchWeight),
            integer(ruptureSetId), integer(retrievedRuptureSetId),
            integer(sourceIndex), integer(sourceId), integer(retrievedSourceId),
            csv(sourceName), csv(retrievedSourceName),
            csv(sourceType), csv(retrievedSourceType),
            integer(ruptureIndex), number(magnitude), number(retrievedMagnitude),
            number(rawRate), number(retrievedRate), number(retrievedRake),
            number(topDepth), number(surfaceDepth), number(width),
            number(surfaceWidth), number(surfaceDip), number(surfaceStrike),
            number(centroidLon), number(surfaceCentroidLon),
            number(centroidLat), number(surfaceCentroidLat),
            number(centroidDepth), number(surfaceCentroidDepth),
            csv(surfaceClass), csv(retrievalError)));

        for (SiteRecord site : sites) {
          double rJB = Double.NaN;
          double rRup = Double.NaN;
          double rX = Double.NaN;
          String distanceError = retrievalError;

          if (distanceError.isEmpty() && surface != null) {
            try {
              Distance distance = surface.distanceTo(site.location);
              rJB = distance.rJB;
              rRup = distance.rRup;
              rX = distance.rX;
            } catch (Exception exception) {
              distanceError = errorText(exception);
            }
          }

          distanceWriter.println(String.join(",",
              csv(chunkId), integer(ruptureOrdinal), integer(site.siteOrdinal),
              csv(ruptureId), csv(site.siteId), number(rJB), number(rRup),
              number(rX), csv(distanceError)));
          distanceCount++;
        }
        ruptureCount++;
      }

      if (distanceWriter.checkError()) {
        throw new IOException("Error writing distance output.");
      }
      if (auditWriter.checkError()) {
        throw new IOException("Error writing rupture-audit output.");
      }
    }

    System.out.println("AUTHORITATIVE_DISTANCE_CHUNK_COMPLETE");
    System.out.println("chunk_id=" + observedChunkId);
    System.out.println("ruptures=" + ruptureCount);
    System.out.println("sites=" + sites.size());
    System.out.println("distance_rows=" + distanceCount);
  }

  private static BranchMatch resolveBranch(
      SourceTree tree,
      int expectedBranchIndex,
      int ruptureSetId,
      int sourceIndex,
      int sourceId,
      String sourceName,
      String sourceType,
      int ruptureIndex,
      double magnitude,
      double rawRate
  ) {

    if (expectedBranchIndex >= 0 && expectedBranchIndex < tree.size()) {
      Branch<RuptureSet<? extends Source>> candidate =
          tree.get(expectedBranchIndex);
      if (compatible(candidate, ruptureSetId, sourceIndex, sourceId,
          sourceName, sourceType, ruptureIndex, magnitude, rawRate)) {
        return new BranchMatch(expectedBranchIndex, candidate);
      }
    }

    int ruptureSetMatches = 0;
    int compatibleMatches = 0;
    BranchMatch first = null;

    for (int index = 0; index < tree.size(); index++) {
      Branch<RuptureSet<? extends Source>> candidate = tree.get(index);
      if (candidate.value().id() == ruptureSetId) ruptureSetMatches++;
      if (!compatible(candidate, ruptureSetId, sourceIndex, sourceId,
          sourceName, sourceType, ruptureIndex, magnitude, rawRate)) {
        continue;
      }
      compatibleMatches++;
      if (first == null) first = new BranchMatch(index, candidate);
    }

    if (first == null) {
      throw new IllegalArgumentException(
          "No compatible branch for rupture-set " + ruptureSetId
          + " in tree " + tree.id()
          + "; expected branch index=" + expectedBranchIndex
          + "; rupture-set matches=" + ruptureSetMatches
          + "; compatible matches=" + compatibleMatches);
    }
    return first;
  }

  private static boolean compatible(
      Branch<RuptureSet<? extends Source>> candidate,
      int ruptureSetId,
      int sourceIndex,
      int sourceId,
      String sourceName,
      String sourceType,
      int ruptureIndex,
      double magnitude,
      double rawRate
  ) {
    RuptureSet<? extends Source> ruptureSet = candidate.value();
    if (ruptureSet.id() != ruptureSetId) return false;
    if (sourceIndex < 0 || sourceIndex >= ruptureSet.size()) return false;
    Source source = ruptureSet.get(sourceIndex);
    if (source.id() != sourceId) return false;
    if (!source.name().equals(sourceName)) return false;
    if (!source.type().name().equals(sourceType)) return false;
    if (ruptureIndex < 0 || ruptureIndex >= source.size()) return false;
    Rupture rupture = source.get(ruptureIndex);
    if (Math.abs(rupture.mag() - magnitude) > 1.0e-10) return false;
    double scale = Math.max(Math.abs(rupture.rate()), Math.abs(rawRate));
    double tolerance = Math.max(1.0e-18, 1.0e-12 * scale);
    return Math.abs(rupture.rate() - rawRate) <= tolerance;
  }

  private static List<SiteRecord> readSites(Path path) throws IOException {
    List<SiteRecord> sites = new ArrayList<>();
    try (BufferedReader reader = Files.newBufferedReader(
        path, StandardCharsets.UTF_8)) {
      String headerLine = reader.readLine();
      if (headerLine == null) throw new IllegalArgumentException("Empty site TSV.");
      Map<String, Integer> columns = headerMap(headerLine);
      String line;
      while ((line = reader.readLine()) != null) {
        if (line.isBlank()) continue;
        String[] values = line.split("\\t", -1);
        int ordinal = parseInt(get(values, columns, "site_ordinal"));
        String siteId = get(values, columns, "site_id");
        double longitude = parseDouble(get(values, columns, "longitude"));
        double latitude = parseDouble(get(values, columns, "latitude"));
        sites.add(new SiteRecord(
            ordinal, siteId, Location.create(longitude, latitude)));
      }
    }
    if (sites.isEmpty()) throw new IllegalArgumentException("No sites were read.");
    return sites;
  }

  private static Map<String, Integer> headerMap(String headerLine) {
    String[] header = headerLine.split("\\t", -1);
    Map<String, Integer> columns = new HashMap<>();
    for (int index = 0; index < header.length; index++) {
      columns.put(header[index], index);
    }
    return columns;
  }

  private static PrintWriter openWriter(Path path) throws IOException {
    Files.createDirectories(path.getParent());
    OutputStream output = Files.newOutputStream(path);
    if (path.getFileName().toString().toLowerCase(Locale.ROOT).endsWith(".gz")) {
      output = new GZIPOutputStream(output, 1024 * 1024);
    }
    return new PrintWriter(new BufferedWriter(new OutputStreamWriter(
        output, StandardCharsets.UTF_8), 1024 * 1024));
  }

  private static double safe(java.util.function.DoubleSupplier supplier) {
    try {
      return supplier.getAsDouble();
    } catch (RuntimeException exception) {
      return Double.NaN;
    }
  }

  private static String errorText(Exception exception) {
    String message = String.valueOf(exception.getMessage())
        .replace('\r', ' ').replace('\n', ' ');
    return exception.getClass().getName() + ": " + message;
  }

  private static String get(
      String[] values, Map<String, Integer> columns, String name) {
    Integer index = columns.get(name);
    if (index == null || index < 0 || index >= values.length) {
      throw new IllegalArgumentException("Missing input column: " + name);
    }
    return values[index];
  }

  private static int parseInt(String value) {
    return (int) Math.round(Double.parseDouble(value));
  }

  private static double parseDouble(String value) {
    return Double.parseDouble(value);
  }

  private static double parseDoubleOrNaN(String value) {
    return value == null || value.isBlank()
        ? Double.NaN
        : Double.parseDouble(value);
  }

  private static String integer(int value) {
    return Integer.toString(value);
  }

  private static String number(double value) {
    return String.format(Locale.US, "%.17g", value);
  }

  private static String csv(String value) {
    if (value == null) return "";
    boolean quote = value.contains(",") || value.contains("\"")
        || value.contains("\n") || value.contains("\r");
    if (!quote) return value;
    return "\"" + value.replace("\"", "\"\"") + "\"";
  }

  private static final class SiteRecord {
    final int siteOrdinal;
    final String siteId;
    final Location location;

    SiteRecord(int siteOrdinal, String siteId, Location location) {
      this.siteOrdinal = siteOrdinal;
      this.siteId = siteId;
      this.location = location;
    }
  }

  private static final class BranchMatch {
    final int index;
    final Branch<RuptureSet<? extends Source>> branch;

    BranchMatch(int index, Branch<RuptureSet<? extends Source>> branch) {
      this.index = index;
      this.branch = branch;
    }
  }
}
'''.strip()

JAVA_SOURCE_PATH.write_text(java_source + "\n", encoding="utf-8")
java_source_hash = sha256_file(JAVA_SOURCE_PATH)

if JAVA_CLASSES_DIR.exists():
    shutil.rmtree(JAVA_CLASSES_DIR)
JAVA_CLASSES_DIR.mkdir(parents=True, exist_ok=True)

compile_classpath = os.pathsep.join(str(path) for path in runtime_paths)
JAVAC_ARGS_PATH.write_text(
    "\n".join([
        "-encoding",
        "UTF-8",
        "-classpath",
        java_arg(compile_classpath),
        "-d",
        java_arg(JAVA_CLASSES_DIR),
        java_arg(JAVA_SOURCE_PATH),
    ])
    + "\n",
    encoding="utf-8",
)

javac_path = shutil.which("javac")
java_path = shutil.which("java")
if javac_path is None or java_path is None:
    raise FileNotFoundError("java and javac must both be available through PATH.")

compile_result = subprocess.run(
    [javac_path, f"@{JAVAC_ARGS_PATH}"],
    cwd=str(PARKER_INSPECTION_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)
compile_output = "\n".join(
    value.strip()
    for value in [compile_result.stdout, compile_result.stderr]
    if value and value.strip()
)
COMPILE_LOG_PATH.write_text(compile_output + "\n", encoding="utf-8")

print("\nJava compiler output:")
print(compile_output if compile_output else "[no compiler messages]")

if compile_result.returncode != 0:
    raise RuntimeError(
        "The authoritative distance Java helper did not compile.\n\n"
        f"Compile log:\n  {COMPILE_LOG_PATH}"
    )

run_classpath = os.pathsep.join([
    str(JAVA_CLASSES_DIR),
    *[str(path) for path in runtime_paths],
])


# ----------------------------------------------------------------------------
# 7. Chunk validation and trusted restart markers
# ----------------------------------------------------------------------------

def validate_chunk(record, rupture_chunk):
    audit_path = Path(record["audit_output"])
    distance_path = Path(record["distance_output"])
    validation_path = Path(record["validation"])

    audit = pd.read_csv(
        audit_path,
        dtype={
            "chunk_id": str,
            "event_id": str,
            "rupture_id": str,
            "expected_source_name": str,
            "retrieved_source_name": str,
            "expected_source_type": str,
            "retrieved_source_type": str,
            "retrieved_branch_id": str,
            "retrieval_error": str,
        },
        low_memory=False,
    )
    audit["retrieval_error"] = audit["retrieval_error"].fillna("")

    numeric_audit = [
        "rupture_ordinal",
        "expected_tree_id",
        "retrieved_tree_id",
        "expected_branch_index",
        "retrieved_branch_index",
        "retrieved_branch_weight",
        "expected_rupture_set_id",
        "retrieved_rupture_set_id",
        "expected_source_id",
        "retrieved_source_id",
        "expected_magnitude",
        "retrieved_magnitude",
        "expected_raw_rate",
        "retrieved_raw_rate",
        "expected_top_depth_km",
        "surface_depth_km",
        "expected_width_km",
        "surface_width_km",
        "expected_centroid_longitude",
        "surface_centroid_longitude",
        "expected_centroid_latitude",
        "surface_centroid_latitude",
        "expected_centroid_depth_km",
        "surface_centroid_depth_km",
    ]
    for column in numeric_audit:
        audit[column] = pd.to_numeric(audit[column], errors="coerce")

    centroid_mask = (
        audit["expected_centroid_longitude"].notna()
        & audit["expected_centroid_latitude"].notna()
        & audit["expected_centroid_depth_km"].notna()
    )
    centroid_match = True
    if centroid_mask.any():
        centroid_match = bool(np.allclose(
            audit.loc[centroid_mask, [
                "expected_centroid_longitude",
                "expected_centroid_latitude",
                "expected_centroid_depth_km",
            ]].to_numpy(dtype=float),
            audit.loc[centroid_mask, [
                "surface_centroid_longitude",
                "surface_centroid_latitude",
                "surface_centroid_depth_km",
            ]].to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-8,
        ))

    distance = pd.read_csv(
        distance_path,
        dtype={
            "chunk_id": str,
            "rupture_id": str,
            "site_id": str,
            "retrieval_error": str,
        },
        low_memory=False,
    )
    distance["retrieval_error"] = distance["retrieval_error"].fillna("")
    for column in ["r_jb_km", "r_rup_km", "r_x_km"]:
        distance[column] = pd.to_numeric(distance[column], errors="coerce")

    expected_rupture_ids = set(rupture_chunk["rupture_id"].astype(str))
    expected_site_ids = set(sites["site_id"].astype(str))
    rupture_counts = distance.groupby("rupture_id").size()
    site_counts = distance.groupby("site_id").size()

    branch_index_match_count = int((
        audit["expected_branch_index"] == audit["retrieved_branch_index"]
    ).sum())
    branch_index_mismatch_count = len(audit) - branch_index_match_count

    print(
        "Catalog branch-index agreement (diagnostic only): "
        f"{branch_index_match_count:,} match; "
        f"{branch_index_mismatch_count:,} differ."
    )

    checks = [
        ("audit rows", len(audit) == record["rupture_count"]),
        ("unique audit ruptures", not audit["rupture_id"].duplicated().any()),
        ("audit has no retrieval errors", audit["retrieval_error"].eq("").all()),
        ("tree IDs match", (audit["expected_tree_id"] == audit["retrieved_tree_id"]).all()),
        (
            "runtime branch positions are valid",
            audit["retrieved_branch_index"].notna().all()
            and audit["retrieved_branch_index"].ge(0).all(),
        ),
        (
            "runtime branch IDs are present",
            audit["retrieved_branch_id"].fillna("").str.strip().ne("").all(),
        ),
        (
            "runtime branch weights are finite and nonnegative",
            np.isfinite(audit["retrieved_branch_weight"]).all()
            and audit["retrieved_branch_weight"].ge(0.0).all(),
        ),
        ("rupture-set IDs match", (audit["expected_rupture_set_id"] == audit["retrieved_rupture_set_id"]).all()),
        ("source IDs match", (audit["expected_source_id"] == audit["retrieved_source_id"]).all()),
        ("source names match", (audit["expected_source_name"].fillna("") == audit["retrieved_source_name"].fillna("")).all()),
        ("source types match", (audit["expected_source_type"].fillna("") == audit["retrieved_source_type"].fillna("")).all()),
        ("magnitudes match", allclose(audit["expected_magnitude"], audit["retrieved_magnitude"], atol=1e-12)),
        ("raw rates match", allclose(audit["expected_raw_rate"], audit["retrieved_raw_rate"], rtol=1e-12, atol=1e-18)),
        ("top depths match", allclose(audit["expected_top_depth_km"], audit["surface_depth_km"], atol=1e-8, equal_nan=True)),
        ("widths match", allclose(audit["expected_width_km"], audit["surface_width_km"], atol=1e-8, equal_nan=True)),
        ("centroids match where stored", centroid_match),
        ("distance row count", len(distance) == record["expected_rows"]),
        ("unique rupture-site pairs", not distance.duplicated(["rupture_id", "site_id"]).any()),
        ("expected rupture IDs", set(distance["rupture_id"].astype(str)) == expected_rupture_ids),
        ("expected site IDs", set(distance["site_id"].astype(str)) == expected_site_ids),
        ("each rupture has all sites", len(rupture_counts) == len(expected_rupture_ids) and rupture_counts.eq(len(sites)).all()),
        ("each site has all chunk ruptures", len(site_counts) == len(sites) and site_counts.eq(len(expected_rupture_ids)).all()),
        ("distance rows have no errors", distance["retrieval_error"].eq("").all()),
        ("Rjb finite and nonnegative", np.isfinite(distance["r_jb_km"]).all() and distance["r_jb_km"].ge(0.0).all()),
        ("Rrup finite and nonnegative", np.isfinite(distance["r_rup_km"]).all() and distance["r_rup_km"].ge(0.0).all()),
        ("Rrup is not below Rjb", (distance["r_rup_km"] + 1e-10 >= distance["r_jb_km"]).all()),
    ]

    validation = pd.DataFrame(checks, columns=["check", "passes"])
    validation.to_csv(validation_path, index=False)

    if not validation["passes"].all():
        failed = validation.loc[~validation["passes"], "check"].tolist()
        raise RuntimeError(
            f"Chunk {record['chunk_id']} validation failed:\n"
            + "\n".join(f"  - {name}" for name in failed)
            + f"\n\nValidation file:\n  {validation_path}"
        )
    return validation


def marker_is_valid(record):
    marker_path = Path(record["marker"])
    distance_path = Path(record["distance_output"])
    audit_path = Path(record["audit_output"])

    if not (marker_path.is_file() and distance_path.is_file() and audit_path.is_file()):
        return False

    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
    except Exception:
        return False

    checks = [
        marker.get("pipeline_version") == PIPELINE_VERSION,
        marker.get("java_source_hash") == java_source_hash,
        marker.get("rupture_input_hash") == record["rupture_input_hash"],
        marker.get("site_input_hash") == record["site_input_hash"],
        marker.get("rupture_count") == record["rupture_count"],
        marker.get("site_count") == record["site_count"],
        marker.get("distance_rows") == record["expected_rows"],
        marker.get("validation_passed") is True,
        marker.get("distance_size") == distance_path.stat().st_size,
        marker.get("audit_size") == audit_path.stat().st_size,
    ]
    if not all(checks):
        return False

    if VERIFY_EXISTING_HASHES:
        if marker.get("distance_hash") != sha256_file(distance_path):
            return False
        if marker.get("audit_hash") != sha256_file(audit_path):
            return False

    return True


# ----------------------------------------------------------------------------
# 8. Sequential chunk calculation with restart capability
# ----------------------------------------------------------------------------

for position, record in enumerate(chunk_records, start=1):
    chunk_id = record["chunk_id"]
    rupture_chunk = ruptures.iloc[record["start"]:record["stop"]].copy()
    rupture_chunk["chunk_id"] = chunk_id

    print("\n" + "-" * 78)
    print(f"CHUNK {position} OF {chunk_count} [ID {chunk_id}]")
    print(
        f"Ruptures: {record['rupture_count']:,}; "
        f"distance rows: {record['expected_rows']:,}"
    )

    if marker_is_valid(record):
        record["status"] = "complete_existing"
        marker = json.loads(Path(record["marker"]).read_text(encoding="utf-8"))
        record["completed_at_utc"] = marker.get("completed_at_utc", "")
        save_manifest()
        print("Validated completion marker found; recomputation skipped.")
        continue

    for key in ["distance_output", "audit_output", "marker"]:
        path = Path(record[key])
        if path.exists():
            path.unlink()

    java_tokens = ["-Dfile.encoding=UTF-8"]
    if JAVA_MAX_HEAP:
        java_tokens.append(
            JAVA_MAX_HEAP if JAVA_MAX_HEAP.startswith("-Xmx")
            else f"-Xmx{JAVA_MAX_HEAP}"
        )
    java_tokens.extend([
        "-classpath",
        java_arg(run_classpath),
        "AuthoritativeRuptureSiteDistances",
        java_arg(USGS_MODEL_ROOT),
        java_arg(record["rupture_input"]),
        java_arg(SITE_INPUT_PATH),
        java_arg(record["distance_output"]),
        java_arg(record["audit_output"]),
    ])

    java_args_path = Path(record["java_args"])
    java_args_path.write_text("\n".join(java_tokens) + "\n", encoding="utf-8")

    result = subprocess.run(
        [java_path, f"@{java_args_path}"],
        cwd=str(PARKER_INSPECTION_DIR),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )
    output = "\n".join(
        value.strip()
        for value in [result.stdout, result.stderr]
        if value and value.strip()
    )
    run_log_path = Path(record["run_log"])
    run_log_path.write_text(output + "\n", encoding="utf-8")

    print("\nRuntime output tail:")
    print("\n".join(output.splitlines()[-40:]))

    if result.returncode != 0:
        record["status"] = "failed_runtime"
        save_manifest()
        raise RuntimeError(
            f"Chunk {chunk_id} failed.\n\nRun log:\n  {run_log_path}"
        )
    if "AUTHORITATIVE_DISTANCE_CHUNK_COMPLETE" not in output:
        record["status"] = "failed_missing_completion_message"
        save_manifest()
        raise RuntimeError(
            f"Chunk {chunk_id} did not emit its completion message.\n"
            f"Run log:\n  {run_log_path}"
        )

    validation = validate_chunk(record, rupture_chunk)
    distance_path = Path(record["distance_output"])
    audit_path = Path(record["audit_output"])
    completed_at = utc_now()

    marker = {
        "pipeline_version": PIPELINE_VERSION,
        "completed_at_utc": completed_at,
        "chunk_id": chunk_id,
        "java_source_hash": java_source_hash,
        "rupture_input_hash": record["rupture_input_hash"],
        "site_input_hash": record["site_input_hash"],
        "rupture_count": record["rupture_count"],
        "site_count": record["site_count"],
        "distance_rows": record["expected_rows"],
        "validation_passed": bool(validation["passes"].all()),
        "distance_size": distance_path.stat().st_size,
        "distance_hash": sha256_file(distance_path),
        "audit_size": audit_path.stat().st_size,
        "audit_hash": sha256_file(audit_path),
    }
    Path(record["marker"]).write_text(
        json.dumps(marker, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    record["status"] = "complete_new"
    record["completed_at_utc"] = completed_at
    save_manifest()
    print(f"Chunk {chunk_id} passed all {len(validation)} checks.")


# ----------------------------------------------------------------------------
# 9. Combine validated chunks
# ----------------------------------------------------------------------------

invalid_chunks = [
    record["chunk_id"] for record in chunk_records if not marker_is_valid(record)
]
if invalid_chunks:
    raise RuntimeError(
        "Invalid chunks remain after processing:\n"
        + "\n".join(f"  - {chunk_id}" for chunk_id in invalid_chunks)
    )

print("\n" + "=" * 78)
print("COMBINING VALIDATED CHUNKS")
print("=" * 78)

combine_gzip_csv(
    [record["distance_output"] for record in chunk_records],
    FINAL_DISTANCE_PATH,
)
combine_gzip_csv(
    [record["audit_output"] for record in chunk_records],
    FINAL_AUDIT_PATH,
)


# ----------------------------------------------------------------------------
# 10. Full-output validation and Cell 14 cross-check
# ----------------------------------------------------------------------------

controlled = pd.read_csv(
    RUPTURE_RETRIEVAL_OUTPUT_PATH,
    dtype={
        "rupture_id": str,
        "test_site_id": str,
        "retrieval_error": str,
    },
    low_memory=False,
)
controlled["retrieval_error"] = controlled["retrieval_error"].fillna("")
for column in ["r_jb_km", "r_rup_km", "r_x_km"]:
    controlled[column] = pd.to_numeric(controlled[column], errors="coerce")
controlled_keys = set(zip(
    controlled["rupture_id"].astype(str),
    controlled["test_site_id"].astype(str),
))

total_rows = 0
error_count = 0
invalid_rjb = 0
invalid_rrup = 0
rrup_below_rjb = 0
finite_rx = 0
observed_ruptures = set()
observed_sites = set()
rupture_counts = Counter()
site_counts = Counter()
controlled_rows = []

for frame in pd.read_csv(
    FINAL_DISTANCE_PATH,
    dtype={
        "chunk_id": str,
        "rupture_id": str,
        "site_id": str,
        "retrieval_error": str,
    },
    chunksize=FINAL_READ_CHUNK_SIZE,
    low_memory=False,
):
    frame["retrieval_error"] = frame["retrieval_error"].fillna("")
    for column in ["r_jb_km", "r_rup_km", "r_x_km"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    total_rows += len(frame)
    error_count += int(frame["retrieval_error"].ne("").sum())
    invalid_rjb += int((
        ~np.isfinite(frame["r_jb_km"]) | frame["r_jb_km"].lt(0.0)
    ).sum())
    invalid_rrup += int((
        ~np.isfinite(frame["r_rup_km"]) | frame["r_rup_km"].lt(0.0)
    ).sum())
    rrup_below_rjb += int((
        frame["r_rup_km"] + 1e-10 < frame["r_jb_km"]
    ).sum())
    finite_rx += int(np.isfinite(frame["r_x_km"]).sum())

    rupture_values = frame["rupture_id"].astype(str).tolist()
    site_values = frame["site_id"].astype(str).tolist()
    observed_ruptures.update(rupture_values)
    observed_sites.update(site_values)
    rupture_counts.update(rupture_values)
    site_counts.update(site_values)

    mask = [
        (rupture_id, site_id) in controlled_keys
        for rupture_id, site_id in zip(rupture_values, site_values)
    ]
    if any(mask):
        controlled_rows.append(frame.loc[
            mask,
            ["rupture_id", "site_id", "r_jb_km", "r_rup_km", "r_x_km"],
        ].copy())

expected_rupture_ids = set(ruptures["rupture_id"].astype(str))
expected_site_ids = set(sites["site_id"].astype(str))
rupture_count_values = np.array(list(rupture_counts.values()), dtype=np.int64)
site_count_values = np.array(list(site_counts.values()), dtype=np.int64)

controlled_distance = (
    pd.concat(controlled_rows, ignore_index=True)
    if controlled_rows
    else pd.DataFrame(columns=[
        "rupture_id", "site_id", "r_jb_km", "r_rup_km", "r_x_km"
    ])
)
controlled_comparison = (
    controlled[["rupture_id", "test_site_id", "r_jb_km", "r_rup_km", "r_x_km"]]
    .rename(columns={
        "test_site_id": "site_id",
        "r_jb_km": "controlled_r_jb_km",
        "r_rup_km": "controlled_r_rup_km",
        "r_x_km": "controlled_r_x_km",
    })
    .merge(
        controlled_distance,
        on=["rupture_id", "site_id"],
        how="left",
        validate="one_to_one",
    )
)
controlled_match = bool(
    len(controlled_comparison) == len(controlled)
    and controlled_comparison["r_jb_km"].notna().all()
    and controlled_comparison["r_rup_km"].notna().all()
    and np.allclose(
        controlled_comparison["controlled_r_jb_km"],
        controlled_comparison["r_jb_km"],
        atol=1e-10,
        rtol=0.0,
        equal_nan=True,
    )
    and np.allclose(
        controlled_comparison["controlled_r_rup_km"],
        controlled_comparison["r_rup_km"],
        atol=1e-10,
        rtol=0.0,
        equal_nan=True,
    )
    and np.allclose(
        controlled_comparison["controlled_r_x_km"],
        controlled_comparison["r_x_km"],
        atol=1e-10,
        rtol=0.0,
        equal_nan=True,
    )
)

final_audit = pd.read_csv(
    FINAL_AUDIT_PATH,
    dtype={"rupture_id": str, "retrieval_error": str},
    low_memory=False,
)
final_audit["retrieval_error"] = final_audit["retrieval_error"].fillna("")
for column in [
    "expected_branch_index",
    "retrieved_branch_index",
    "retrieved_branch_weight",
]:
    final_audit[column] = pd.to_numeric(final_audit[column], errors="coerce")

branch_index_match_count = int((
    final_audit["expected_branch_index"]
    == final_audit["retrieved_branch_index"]
).sum())
branch_index_mismatch_count = len(final_audit) - branch_index_match_count

validation_records = [
    ("Exactly 3,927 unique ruptures", len(ruptures), EXPECTED_RUPTURES, len(ruptures) == EXPECTED_RUPTURES),
    ("Exactly 470 unique sites", len(sites), EXPECTED_SITES, len(sites) == EXPECTED_SITES),
    ("All chunk markers valid", chunk_count - len(invalid_chunks), chunk_count, len(invalid_chunks) == 0),
    ("Final distance row count", total_rows, expected_rows, total_rows == expected_rows),
    ("One audit row per rupture", len(final_audit), len(ruptures), len(final_audit) == len(ruptures) and final_audit["rupture_id"].nunique() == len(ruptures)),
    ("No audit retrieval errors", int(final_audit["retrieval_error"].ne("").sum()), 0, final_audit["retrieval_error"].eq("").all()),
    ("Expected rupture IDs", len(observed_ruptures), len(expected_rupture_ids), observed_ruptures == expected_rupture_ids),
    ("Expected site IDs", len(observed_sites), len(expected_site_ids), observed_sites == expected_site_ids),
    ("Every rupture has 470 rows", f"min={rupture_count_values.min()}, max={rupture_count_values.max()}", 470, len(rupture_counts) == EXPECTED_RUPTURES and (rupture_count_values == EXPECTED_SITES).all()),
    ("Every site has 3,927 rows", f"min={site_count_values.min()}, max={site_count_values.max()}", 3927, len(site_counts) == EXPECTED_SITES and (site_count_values == EXPECTED_RUPTURES).all()),
    ("No distance errors", error_count, 0, error_count == 0),
    ("Rjb finite and nonnegative", invalid_rjb, 0, invalid_rjb == 0),
    ("Rrup finite and nonnegative", invalid_rrup, 0, invalid_rrup == 0),
    ("Rrup never below Rjb", rrup_below_rjb, 0, rrup_below_rjb == 0),
    ("Cell 15 reproduces all Cell 14 controlled distances", len(controlled_comparison), len(controlled), controlled_match),
]

final_validation = pd.DataFrame(
    validation_records,
    columns=["check", "observed", "expected", "passes"],
)
final_validation.to_csv(FINAL_VALIDATION_PATH, index=False)

print("\n" + "=" * 78)
print("FINAL VALIDATION")
print("=" * 78)
display(final_validation)

if not final_validation["passes"].all():
    failed = final_validation.loc[
        ~final_validation["passes"],
        ["check", "observed", "expected"],
    ]
    raise RuntimeError(
        "Final authoritative distance validation failed.\n\n"
        + failed.to_string(index=False)
        + f"\n\nValidation file:\n  {FINAL_VALIDATION_PATH}"
    )


# ----------------------------------------------------------------------------
# 11. Reproducibility summary
# ----------------------------------------------------------------------------

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": utc_now(),
    "usgs_model_root": str(USGS_MODEL_ROOT),
    "event_catalog_path": str(EVENT_CATALOG_PATH),
    "w2_portfolio_path": str(GMM_READY_PORTFOLIO_PATH),
    "unique_ruptures": len(ruptures),
    "sites": len(sites),
    "distance_rows": total_rows,
    "finite_rx_rows": finite_rx,
    "catalog_branch_index_matches": branch_index_match_count,
    "catalog_branch_index_mismatches": branch_index_mismatch_count,
    "branch_index_note": (
        "Catalog branch_index is retained as provenance but is not required "
        "to equal the runtime SourceTree list position. Authoritative branch "
        "resolution is validated through tree, rupture-set, source, rupture, "
        "magnitude, rate, and surface metadata."
    ),
    "ruptures_per_chunk": RUPTURES_PER_CHUNK,
    "chunk_count": chunk_count,
    "java_source_path": str(JAVA_SOURCE_PATH),
    "java_source_sha256": java_source_hash,
    "final_distance_path": str(FINAL_DISTANCE_PATH),
    "final_distance_size_bytes": FINAL_DISTANCE_PATH.stat().st_size,
    "final_distance_sha256": sha256_file(FINAL_DISTANCE_PATH),
    "final_audit_path": str(FINAL_AUDIT_PATH),
    "final_audit_size_bytes": FINAL_AUDIT_PATH.stat().st_size,
    "final_audit_sha256": sha256_file(FINAL_AUDIT_PATH),
    "manifest_path": str(MANIFEST_PATH),
    "validation_path": str(FINAL_VALIDATION_PATH),
    "all_checks_passed": bool(final_validation["passes"].all()),
}
FINAL_SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("\nCELL 15 AUTHORITATIVE USGS DISTANCES COMPLETE")
print(f"\nFinal distance matrix:\n  {FINAL_DISTANCE_PATH}")
print(f"\nRupture audit:\n  {FINAL_AUDIT_PATH}")
print(f"\nChunk manifest:\n  {MANIFEST_PATH}")
print(f"\nValidation:\n  {FINAL_VALIDATION_PATH}")
print(f"\nSummary:\n  {FINAL_SUMMARY_PATH}")
print(
    f"\nValidated {total_rows:,} rows "
    f"({len(ruptures):,} ruptures × {len(sites):,} sites)."
)
print(
    "\nNext step: use rJB, rRup, and rX as the authoritative distance "
    "inputs for the source-appropriate USGS ground-motion calculations."
)


AUTHORITATIVE USGS RUPTURE-TO-SITE DISTANCES

Unique ruptures: 3,927
W2 sites:        470
Distance rows:   1,845,690
Rupture chunk:   500

Java compiler output:
[no compiler messages]

------------------------------------------------------------------------------
CHUNK 1 OF 8 [ID 0000]
Ruptures: 500; distance rows: 235,000

Runtime output tail:
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
                 TODO: null branch
Catalog branch-index agreement (diagnostic only): 337 match; 163 differ.
Chunk 0000 passed all 26 checks.

------------------------------------------------------------------------------
CHUNK 2 OF 8 [ID 0001]
Ruptures: 500; distance 

,check,observed,expected,passes
0,"Exactly 3,927 unique ruptures",3927,3927,True
1,Exactly 470 unique sites,470,470,True
2,All chunk markers valid,8,8,True
3,Final distance row count,1845690,1845690,True
4,One audit row per rupture,3927,3927,True
5,No audit retrieval errors,0,0,True
6,Expected rupture IDs,3927,3927,True
7,Expected site IDs,470,470,True
8,Every rupture has 470 rows,"min=470, max=470",470,True
9,"Every site has 3,927 rows","min=3927, max=3927",3927,True



CELL 15 AUTHORITATIVE USGS DISTANCES COMPLETE

Final distance matrix:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_4_authoritative_usgs_distances\usgs_rupture_site_distances.csv.gz

Rupture audit:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_authoritative_rupture_retrieval_audit.csv.gz

Chunk manifest:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_rupture_site_distance_chunk_manifest.csv

Validation:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_authoritative_distance_validation.csv

Summary:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_authoritative_distance_summary.json

Validated 1,845,690 rows (3,927 ruptures × 470 sites).

Next step: use rJB, rRup, and rX as the authoritative distance inputs for the source-appropriate USGS ground-motion calculatio

In [31]:
# ============================================================================
# CELL 16
# Controlled source-appropriate Parker GMM calculation and validation.
#
# Purpose
# -------
# 1. Select four deterministic rupture-site controls from the authoritative
#    Cell 15 distance matrix:
#       - interface nearest
#       - interface farthest
#       - slab nearest
#       - slab farthest
# 2. Resolve the Seaside W2 site Vs30 and structural period inputs.
# 3. Evaluate the pinned USGS Parker Cascadia interface and slab GMMs.
# 4. Preserve the Parker epistemic branches rather than collapsing them.
# 5. Reconstruct Parker's ergodic tau, phi, and total sigma from the
#    published Parker coefficient rows required by this Seaside case and
#    independently verify them against the pinned USGS Java GMM sigma.
# 6. Require the Seaside W2 structural period of 0.40 s and evaluate the
#    exact USGS SA0P4 IMT. Generic interpolation logic is retained only as a
#    safeguard for future portfolios with unsupported periods.
# 7. Save controlled inputs, exact-IMT outputs, interpolated outputs,
#    validation checks, source provenance, and a reproducibility summary.
#
# This is a controlled validation cell. It does not process all 1,845,690
# rupture-site pairs. The validated bulk calculation belongs in Cell 17.
# ============================================================================

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path

import hashlib
import io
import json
import math
import os
import re
import shutil
import subprocess
import zipfile
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Configuration and required earlier-cell variables
# ----------------------------------------------------------------------------

PIPELINE_VERSION = "notebook4_cell16_controlled_parker_gmm_v4"
EXPECTED_RUPTURES = 3_927
EXPECTED_SITES = 470
EXPECTED_DISTANCE_ROWS = 1_845_690
DISTANCE_READ_CHUNK_SIZE = 250_000
JAVA_MAX_HEAP = os.environ.get("NOTEBOOK4_JAVA_XMX", "2g").strip()

# Project-declared defaults. These are used only if the corresponding columns
# are not present in GMM_READY_PORTFOLIO_PATH. The chosen provenance is written
# to the Cell 16 outputs and printed before the Java calculation.
DEFAULT_VS30_MPS = 365.0
SEASIDE_W2_PERIOD_S = 0.40
DEFAULT_PERIOD_S = SEASIDE_W2_PERIOD_S
DEFAULT_VS30_INFERRED = True

# Numerical tolerances.
INPUT_MATCH_ATOL = 1.0e-10
WEIGHT_SUM_ATOL = 1.0e-12
SIGMA_MATCH_ATOL = 5.0e-12
DEPTH_INVARIANCE_ATOL = 5.0e-12
PERIOD_MATCH_ATOL = 1.0e-10

# GMM names pinned in earlier Notebook 4 cells. Globals take precedence so the
# notebook remains the source of truth if their variable names already exist.
INTERFACE_GMM_NAME = str(
    globals().get("PARKER_INTERFACE_GMM", "PSBAH_20_CASCADIA_INTERFACE")
)
SLAB_GMM_NAME = str(
    globals().get("PARKER_SLAB_GMM", "PSBAH_20_CASCADIA_SLAB")
)

required_variables = [
    "DATA_DIR",
    "METADATA_DIR",
    "PARKER_INSPECTION_DIR",
    "GMM_READY_PORTFOLIO_PATH",
    "runtime_paths",
]

missing_variables = [
    name for name in required_variables if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Cell 16 requires variables created by earlier Notebook 4 cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

DATA_DIR = Path(DATA_DIR).resolve()
METADATA_DIR = Path(METADATA_DIR).resolve()
PARKER_INSPECTION_DIR = Path(PARKER_INSPECTION_DIR).resolve()
GMM_READY_PORTFOLIO_PATH = Path(GMM_READY_PORTFOLIO_PATH).resolve()
runtime_paths = [Path(path).resolve() for path in runtime_paths]

# The Parker variability coefficients may be available either in an unpacked
# official USGS source checkout or as a packaged resource inside the exact
# nshmp-lib JAR already used by this notebook. Prefer an explicit source root
# when available, but do not require one: a normal Gradle installation often
# provides only the binary JAR and its embedded resources.
source_roots: list[Path] = []
if "NSHMP_HAZ_SOURCE_DIR" in globals():
    source_roots.append(Path(NSHMP_HAZ_SOURCE_DIR).resolve())

# Bounded fallback discovery. Never recursively scan broad user or drive roots.
for runtime_path in runtime_paths:
    candidates = [runtime_path] if runtime_path.is_dir() else []
    candidates.extend(
        parent
        for parent in list(runtime_path.parents)[:4]
        if "nshmp" in parent.name.lower()
    )
    for candidate in candidates:
        if (
            (candidate / "src" / "main" / "resources").is_dir()
            or (candidate / "src" / "gov" / "usgs").is_dir()
            or "nshmp" in candidate.name.lower()
        ):
            source_roots.append(candidate.resolve())

source_roots = list(
    dict.fromkeys(path for path in source_roots if path.exists())
)

# Cell 15 final products. Use globals if Cell 15 left them in memory; otherwise
# reconstruct the exact deterministic paths used by Cell 15.
CELL15_DATA_OUTPUT_DIR = (
    DATA_DIR / "processed" / "notebook_4_authoritative_usgs_distances"
)
CELL15_DISTANCE_PATH = Path(
    globals().get(
        "FINAL_DISTANCE_PATH",
        CELL15_DATA_OUTPUT_DIR / "usgs_rupture_site_distances.csv.gz",
    )
).resolve()
CELL15_AUDIT_PATH = Path(
    globals().get(
        "FINAL_AUDIT_PATH",
        METADATA_DIR / "notebook_4_authoritative_rupture_retrieval_audit.csv.gz",
    )
).resolve()
CELL15_RUPTURE_INDEX_PATH = Path(
    globals().get(
        "RUPTURE_INDEX_PATH",
        METADATA_DIR / "notebook_4_unique_rupture_index.csv",
    )
).resolve()
CELL15_SITE_INDEX_PATH = Path(
    globals().get(
        "SITE_INDEX_PATH",
        METADATA_DIR / "notebook_4_w2_site_index.csv",
    )
).resolve()
CELL15_VALIDATION_PATH = Path(
    globals().get(
        "FINAL_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_authoritative_distance_validation.csv",
    )
).resolve()

required_paths = {
    "Cell 15 final distance matrix": CELL15_DISTANCE_PATH,
    "Cell 15 rupture audit": CELL15_AUDIT_PATH,
    "Cell 15 rupture index": CELL15_RUPTURE_INDEX_PATH,
    "Cell 15 site index": CELL15_SITE_INDEX_PATH,
    "Cell 15 final validation": CELL15_VALIDATION_PATH,
    "GMM-ready W2 portfolio": GMM_READY_PORTFOLIO_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]
missing_runtime_paths = [path for path in runtime_paths if not path.exists()]

if missing_paths or missing_runtime_paths:
    message: list[str] = []
    if missing_paths:
        message.append(
            "Missing required input files:\n"
            + "\n".join(f"  - {item}" for item in missing_paths)
        )
    if missing_runtime_paths:
        message.append(
            "Missing Java classpath entries:\n"
            + "\n".join(f"  - {path}" for path in missing_runtime_paths)
        )
    raise FileNotFoundError("\n\n".join(message))


# ----------------------------------------------------------------------------
# 2. Output paths
# ----------------------------------------------------------------------------

CELL16_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_controlled_parker_gmm"
CELL16_WORK_DIR = METADATA_DIR / "notebook_4_controlled_parker_gmm_work"
CELL16_INPUT_DIR = CELL16_WORK_DIR / "inputs"
CELL16_LOG_DIR = CELL16_WORK_DIR / "logs"
CELL16_JAVA_DIR = CELL16_WORK_DIR / "java"
CELL16_CLASSES_DIR = CELL16_JAVA_DIR / "classes"

for directory in [
    CELL16_OUTPUT_DIR,
    CELL16_WORK_DIR,
    CELL16_INPUT_DIR,
    CELL16_LOG_DIR,
    CELL16_JAVA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SUPPORTED_IMT_PATH = METADATA_DIR / "notebook_4_parker_supported_imts.csv"
CONTROL_SELECTION_PATH = METADATA_DIR / "notebook_4_cell_16_control_selection.csv"
CONTROL_INPUT_PATH = METADATA_DIR / "notebook_4_cell_16_controlled_gmm_inputs.csv"
EXACT_CALCULATION_INPUT_PATH = CELL16_INPUT_DIR / "controlled_exact_gmm_inputs.tsv"
EXACT_OUTPUT_PATH = CELL16_OUTPUT_DIR / "controlled_parker_exact_outputs.csv"
REQUESTED_OUTPUT_PATH = CELL16_OUTPUT_DIR / "controlled_parker_requested_outputs.csv"
COEFFICIENT_INVENTORY_PATH = METADATA_DIR / "notebook_4_parker_variability_coefficients.csv"
COEFFICIENT_PROVENANCE_PATH = METADATA_DIR / "notebook_4_parker_coefficient_provenance.json"
VALIDATION_PATH = METADATA_DIR / "notebook_4_cell_16_controlled_gmm_validation.csv"
SUMMARY_PATH = METADATA_DIR / "notebook_4_cell_16_controlled_gmm_summary.json"

JAVA_SOURCE_PATH = CELL16_JAVA_DIR / "ControlledParkerGmm.java"
JAVAC_ARGS_PATH = CELL16_JAVA_DIR / "controlled_parker_javac_arguments.txt"
JAVA_ARGS_PATH = CELL16_JAVA_DIR / "controlled_parker_java_arguments.txt"
COMPILE_LOG_PATH = CELL16_LOG_DIR / "controlled_parker_compile.log"
RUN_LOG_PATH = CELL16_LOG_DIR / "controlled_parker_run.log"


# ----------------------------------------------------------------------------
# 3. Utilities
# ----------------------------------------------------------------------------

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            block = file_handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def java_arg(value: object) -> str:
    text = str(value).replace("\\", "\\\\").replace('"', '\\"')
    return f'"{text}"'


def normalize_column_name(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def first_matching_column(
    frame: pd.DataFrame,
    candidates: Iterable[str],
) -> str | None:
    normalized_to_original = {
        normalize_column_name(column): str(column)
        for column in frame.columns
    }
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in normalized_to_original:
            return normalized_to_original[key]
    return None


def numeric_series(
    frame: pd.DataFrame,
    column: str,
    *,
    label: str,
) -> pd.Series:
    values = pd.to_numeric(frame[column], errors="coerce")
    if not np.isfinite(values).all():
        bad = frame.loc[~np.isfinite(values), [column]].head(20)
        raise RuntimeError(
            f"{label} column {column!r} contains missing or non-finite values.\n\n"
            + bad.to_string(index=False)
        )
    return values.astype(float)


def period_from_imt_name(imt_name: str) -> float:
    value = str(imt_name).strip().upper()
    if value == "PGA":
        return 0.0
    match = re.fullmatch(r"SA(\d+)P(\d+)", value)
    if match is None:
        raise ValueError(f"Cannot parse spectral period from IMT name {imt_name!r}.")
    whole = int(match.group(1))
    fraction_text = match.group(2)
    return float(f"{whole}.{fraction_text}")


def label_to_period(label: object) -> float | None:
    """Resolve PGA/SA labels or numeric period labels from a coefficient file."""
    text = str(label).strip()
    if not text:
        return None
    upper = text.upper().replace(" ", "")
    if upper == "PGA":
        return 0.0
    if re.fullmatch(r"SA\d+P\d+", upper):
        return period_from_imt_name(upper)
    match = re.fullmatch(r"SA\(?([0-9]*\.?[0-9]+)\)?", upper)
    if match:
        return float(match.group(1))
    try:
        return float(text)
    except ValueError:
        return None


def choose_site_input_column(
    portfolio: pd.DataFrame,
    candidate_names: Iterable[str],
    default_value: float,
    label: str,
) -> tuple[pd.Series, str]:
    column = first_matching_column(portfolio, candidate_names)
    if column is None:
        values = pd.Series(
            np.full(len(portfolio), default_value, dtype=float),
            index=portfolio.index,
        )
        provenance = f"constant fallback: {default_value:g}"
    else:
        values = numeric_series(portfolio, column, label=label)
        provenance = f"portfolio column: {column}"
    return values, provenance


def add_validation(
    records: list[dict[str, object]],
    category: str,
    check: str,
    observed: object,
    expected: object,
    passes: bool,
    note: str = "",
) -> None:
    records.append(
        {
            "category": category,
            "check": check,
            "observed": observed,
            "expected": expected,
            "passes": bool(passes),
            "note": note,
        }
    )


# ----------------------------------------------------------------------------
# 4. Confirm Cell 15 final validation before using its output
# ----------------------------------------------------------------------------

cell15_validation = pd.read_csv(CELL15_VALIDATION_PATH, low_memory=False)
cell15_pass_column = first_matching_column(cell15_validation, ["passes", "pass"])
if cell15_pass_column is None:
    raise RuntimeError(
        "The Cell 15 final validation file has no passes column:\n"
        f"  {CELL15_VALIDATION_PATH}"
    )

cell15_passes = (
    cell15_validation[cell15_pass_column]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False})
)

if cell15_passes.isna().any() or not cell15_passes.all():
    raise RuntimeError(
        "Cell 15 final validation is missing or contains failed checks. "
        "Cell 16 will not use an unvalidated distance matrix.\n\n"
        + cell15_validation.to_string(index=False)
    )


# ----------------------------------------------------------------------------
# 5. Load rupture audit and W2 portfolio; resolve site inputs
# ----------------------------------------------------------------------------

rupture_audit = pd.read_csv(
    CELL15_AUDIT_PATH,
    dtype={
        "rupture_id": str,
        "retrieved_source_type": str,
        "retrieval_error": str,
    },
    low_memory=False,
)
rupture_audit["retrieval_error"] = (
    rupture_audit["retrieval_error"].fillna("").astype(str)
)

required_audit_columns = [
    "rupture_ordinal",
    "event_id",
    "rupture_id",
    "retrieved_source_type",
    "retrieved_magnitude",
    "retrieved_raw_rate",
    "retrieved_rake_deg",
    "surface_depth_km",
    "surface_width_km",
    "surface_dip_deg",
    "surface_strike_deg",
    "surface_centroid_depth_km",
    "surface_class",
    "retrieval_error",
]
missing_audit_columns = [
    column for column in required_audit_columns if column not in rupture_audit
]
if missing_audit_columns:
    raise RuntimeError(
        "Cell 15 rupture audit is missing required columns:\n"
        + "\n".join(f"  - {column}" for column in missing_audit_columns)
    )

if len(rupture_audit) != EXPECTED_RUPTURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RUPTURES:,} Cell 15 audit rows; "
        f"observed {len(rupture_audit):,}."
    )
if rupture_audit["rupture_id"].duplicated().any():
    raise RuntimeError("Cell 15 rupture audit contains duplicate rupture IDs.")
if rupture_audit["retrieval_error"].ne("").any():
    raise RuntimeError("Cell 15 rupture audit contains retrieval errors.")

rupture_audit["retrieved_source_type"] = (
    rupture_audit["retrieved_source_type"].astype(str).str.upper()
)
if set(rupture_audit["retrieved_source_type"].unique()) != {"INTERFACE", "SLAB"}:
    raise RuntimeError(
        "Unexpected source types in the Cell 15 audit: "
        f"{sorted(rupture_audit['retrieved_source_type'].unique())}"
    )

for column in [
    "rupture_ordinal",
    "retrieved_magnitude",
    "retrieved_raw_rate",
    "retrieved_rake_deg",
    "surface_depth_km",
    "surface_width_km",
    "surface_dip_deg",
    "surface_strike_deg",
    "surface_centroid_depth_km",
]:
    rupture_audit[column] = pd.to_numeric(
        rupture_audit[column], errors="coerce"
    )

required_finite_audit_columns = [
    "rupture_ordinal",
    "retrieved_magnitude",
    "retrieved_raw_rate",
    "retrieved_rake_deg",
    "surface_depth_km",
    "surface_width_km",
    "surface_dip_deg",
]
for column in required_finite_audit_columns:
    if not np.isfinite(rupture_audit[column]).all():
        raise RuntimeError(f"Audit field {column!r} contains non-finite values.")

portfolio = pd.read_csv(GMM_READY_PORTFOLIO_PATH, low_memory=False)
site_id_column = first_matching_column(portfolio, ["site_id", "siteid"])
longitude_column = first_matching_column(portfolio, ["longitude", "lon", "x"])
latitude_column = first_matching_column(portfolio, ["latitude", "lat", "y"])
if site_id_column is None or longitude_column is None or latitude_column is None:
    raise RuntimeError(
        "The GMM-ready Seaside W2 commercial portfolio must contain site ID, longitude, and "
        "latitude fields.\n\n"
        f"Columns: {list(portfolio.columns)}"
    )

portfolio = portfolio.copy()
portfolio["site_id"] = (
    portfolio[site_id_column].astype("string").fillna("").astype(str)
)
portfolio["longitude"] = numeric_series(
    portfolio, longitude_column, label="longitude"
)
portfolio["latitude"] = numeric_series(
    portfolio, latitude_column, label="latitude"
)

if len(portfolio) != EXPECTED_SITES:
    raise RuntimeError(
        f"Expected {EXPECTED_SITES:,} Seaside W2 sites; observed {len(portfolio):,}."
    )
if portfolio["site_id"].eq("").any():
    raise RuntimeError("The GMM-ready W2 portfolio contains blank site IDs.")
if portfolio["site_id"].duplicated().any():
    raise RuntimeError("The GMM-ready W2 portfolio contains duplicate site IDs.")

portfolio["vs30_mps"], vs30_provenance = choose_site_input_column(
    portfolio,
    [
        "vs30_mps",
        "vs30",
        "v_s30",
        "site_vs30",
        "site_vs30_mps",
    ],
    DEFAULT_VS30_MPS,
    "Vs30",
)
portfolio["period_s"], period_provenance = choose_site_input_column(
    portfolio,
    [
        "period_s",
        "period",
        "fundamental_period_s",
        "fundamental_period",
        "period_for_spectral_acceleration",
        "building_period_s",
    ],
    DEFAULT_PERIOD_S,
    "structural period",
)

vs30_inferred_column = first_matching_column(
    portfolio,
    ["vs30_inferred", "vs_inferred", "vs30_is_inferred"],
)
if vs30_inferred_column is None:
    portfolio["vs30_inferred"] = DEFAULT_VS30_INFERRED
    vs30_inferred_provenance = (
        f"constant fallback: {DEFAULT_VS30_INFERRED}"
    )
else:
    text = (
        portfolio[vs30_inferred_column]
        .astype(str)
        .str.strip()
        .str.lower()
    )
    parsed = text.map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
            "yes": True,
            "no": False,
        }
    )
    if parsed.isna().any():
        raise RuntimeError(
            f"Could not parse all values in {vs30_inferred_column!r} as booleans."
        )
    portfolio["vs30_inferred"] = parsed.astype(bool)
    vs30_inferred_provenance = f"portfolio column: {vs30_inferred_column}"

z1p0_column = first_matching_column(
    portfolio, ["z1p0_km", "z1p0", "z1_0_km", "z1"]
)
z2p5_column = first_matching_column(
    portfolio, ["z2p5_km", "z2p5", "z2_5_km", "z2"]
)
zsed_column = first_matching_column(
    portfolio, ["zsed_km", "zsed", "sediment_thickness_km"]
)

for output_column, source_column in [
    ("z1p0_km", z1p0_column),
    ("z2p5_km", z2p5_column),
    ("zsed_km", zsed_column),
]:
    if source_column is None:
        portfolio[output_column] = np.nan
    else:
        portfolio[output_column] = pd.to_numeric(
            portfolio[source_column], errors="coerce"
        )
        invalid = portfolio[output_column].notna() & ~np.isfinite(
            portfolio[output_column]
        )
        if invalid.any():
            raise RuntimeError(
                f"Portfolio basin field {source_column!r} contains invalid values."
            )

if not portfolio["vs30_mps"].between(150.0, 2000.0).all():
    bad = portfolio.loc[
        ~portfolio["vs30_mps"].between(150.0, 2000.0),
        ["site_id", "vs30_mps"],
    ].head(20)
    raise RuntimeError(
        "Vs30 values fall outside the USGS supported interval [150, 2000] m/s.\n\n"
        + bad.to_string(index=False)
    )
if not portfolio["period_s"].gt(0.0).all():
    bad = portfolio.loc[
        ~portfolio["period_s"].gt(0.0), ["site_id", "period_s"]
    ].head(20)
    raise RuntimeError(
        "Structural periods must be positive.\n\n"
        + bad.to_string(index=False)
    )

# This notebook is explicitly the Seaside W2 commercial case study. The
# portfolio summary reports one common structural period, and the authoritative
# portfolio assigns T = 0.40 s. Fail rather than silently using another period.
period_matches_seaside = np.isclose(
    portfolio["period_s"].to_numpy(dtype=float),
    SEASIDE_W2_PERIOD_S,
    rtol=0.0,
    atol=PERIOD_MATCH_ATOL,
)
if not period_matches_seaside.all():
    observed_periods = sorted(
        portfolio.loc[~period_matches_seaside, "period_s"].unique().tolist()
    )
    raise RuntimeError(
        "The Seaside W2 commercial portfolio must use T = 0.40 s for every "
        "site, but other periods were found.\n\n"
        f"Unexpected periods: {observed_periods[:20]}"
    )

portfolio["site_ordinal"] = np.arange(len(portfolio), dtype=np.int64)
site_lookup = portfolio.set_index("site_id", drop=False)

print("=" * 78)
print("CELL 16: SEASIDE W2 CONTROLLED PARKER GMM VALIDATION")
print("=" * 78)
print(f"\nInterface GMM: {INTERFACE_GMM_NAME}")
print(f"Slab GMM:      {SLAB_GMM_NAME}")
print(f"Vs30 source:   {vs30_provenance}")
print(f"Period source: {period_provenance}")
print(f"Validated period: {SEASIDE_W2_PERIOD_S:.2f} s (exact USGS SA0P4)")
print(f"Vs30 inferred: {vs30_inferred_provenance}")


# ----------------------------------------------------------------------------
# 6. Select deterministic nearest and farthest controls by source type
# ----------------------------------------------------------------------------

source_type_by_rupture = rupture_audit.set_index("rupture_id")[
    "retrieved_source_type"
]

best_rows: dict[tuple[str, str], pd.Series] = {}
observed_distance_rows = 0
observed_rupture_ids: set[str] = set()
observed_site_ids: set[str] = set()

for frame in pd.read_csv(
    CELL15_DISTANCE_PATH,
    dtype={
        "chunk_id": str,
        "rupture_id": str,
        "site_id": str,
        "retrieval_error": str,
    },
    chunksize=DISTANCE_READ_CHUNK_SIZE,
    low_memory=False,
):
    frame["retrieval_error"] = frame["retrieval_error"].fillna("").astype(str)
    if frame["retrieval_error"].ne("").any():
        raise RuntimeError("The Cell 15 distance matrix contains errors.")

    for column in ["rupture_ordinal", "site_ordinal", "r_jb_km", "r_rup_km", "r_x_km"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    if not np.isfinite(frame[["r_jb_km", "r_rup_km", "r_x_km"]]).all().all():
        raise RuntimeError("The Cell 15 distance matrix contains non-finite distances.")

    frame["source_type"] = frame["rupture_id"].map(source_type_by_rupture)
    if frame["source_type"].isna().any():
        raise RuntimeError(
            "One or more distance rows could not be mapped to the rupture audit."
        )

    observed_distance_rows += len(frame)
    observed_rupture_ids.update(frame["rupture_id"].astype(str))
    observed_site_ids.update(frame["site_id"].astype(str))

    for source_type in ["INTERFACE", "SLAB"]:
        subset = frame.loc[frame["source_type"].eq(source_type)]
        if subset.empty:
            continue

        near_row = subset.loc[subset["r_rup_km"].idxmin()].copy()
        far_row = subset.loc[subset["r_rup_km"].idxmax()].copy()

        near_key = (source_type, "nearest")
        far_key = (source_type, "farthest")

        if (
            near_key not in best_rows
            or float(near_row["r_rup_km"])
            < float(best_rows[near_key]["r_rup_km"])
        ):
            best_rows[near_key] = near_row

        if (
            far_key not in best_rows
            or float(far_row["r_rup_km"])
            > float(best_rows[far_key]["r_rup_km"])
        ):
            best_rows[far_key] = far_row

if observed_distance_rows != EXPECTED_DISTANCE_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_DISTANCE_ROWS:,} Cell 15 distance rows; "
        f"observed {observed_distance_rows:,}."
    )
if len(observed_rupture_ids) != EXPECTED_RUPTURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RUPTURES:,} rupture IDs in distances; "
        f"observed {len(observed_rupture_ids):,}."
    )
if len(observed_site_ids) != EXPECTED_SITES:
    raise RuntimeError(
        f"Expected {EXPECTED_SITES:,} site IDs in distances; "
        f"observed {len(observed_site_ids):,}."
    )

expected_control_keys = {
    ("INTERFACE", "nearest"),
    ("INTERFACE", "farthest"),
    ("SLAB", "nearest"),
    ("SLAB", "farthest"),
}
if set(best_rows) != expected_control_keys:
    raise RuntimeError(
        "Failed to select all four controlled rupture-site cases.\n"
        f"Observed keys: {sorted(best_rows)}"
    )

control_records: list[dict[str, object]] = []
for source_type, distance_role in [
    ("INTERFACE", "nearest"),
    ("INTERFACE", "farthest"),
    ("SLAB", "nearest"),
    ("SLAB", "farthest"),
]:
    distance_row = best_rows[(source_type, distance_role)]
    rupture_id = str(distance_row["rupture_id"])
    site_id = str(distance_row["site_id"])

    rupture_row = rupture_audit.loc[
        rupture_audit["rupture_id"].eq(rupture_id)
    ].iloc[0]
    if site_id not in site_lookup.index:
        raise RuntimeError(f"Selected site ID is absent from portfolio: {site_id}")
    site_row = site_lookup.loc[site_id]

    z_hyp_input = float(rupture_row["surface_centroid_depth_km"])
    if not np.isfinite(z_hyp_input):
        z_hyp_input = float(
            rupture_row["surface_depth_km"]
            + 0.5
            * rupture_row["surface_width_km"]
            * math.sin(math.radians(float(rupture_row["surface_dip_deg"])))
        )
        z_hyp_provenance = "geometric midpoint from zTor, width, and dip"
    else:
        z_hyp_provenance = "USGS rupture-surface centroid depth"

    case_id = f"{source_type.lower()}_{distance_role}"
    gmm_name = (
        INTERFACE_GMM_NAME if source_type == "INTERFACE" else SLAB_GMM_NAME
    )

    control_records.append(
        {
            "case_id": case_id,
            "source_type": source_type,
            "distance_role": distance_role,
            "gmm_name": gmm_name,
            "rupture_ordinal": int(distance_row["rupture_ordinal"]),
            "event_id": str(rupture_row["event_id"]),
            "rupture_id": rupture_id,
            "site_ordinal": int(distance_row["site_ordinal"]),
            "site_id": site_id,
            "site_longitude": float(site_row["longitude"]),
            "site_latitude": float(site_row["latitude"]),
            "magnitude": float(rupture_row["retrieved_magnitude"]),
            "raw_annual_rate": float(rupture_row["retrieved_raw_rate"]),
            "r_jb_km": float(distance_row["r_jb_km"]),
            "r_rup_km": float(distance_row["r_rup_km"]),
            "r_x_km": float(distance_row["r_x_km"]),
            "dip_deg": float(rupture_row["surface_dip_deg"]),
            "width_km": float(rupture_row["surface_width_km"]),
            "z_tor_km": float(rupture_row["surface_depth_km"]),
            "z_hyp_input_km": z_hyp_input,
            "z_hyp_input_provenance": z_hyp_provenance,
            "rake_deg": float(rupture_row["retrieved_rake_deg"]),
            "vs30_mps": float(site_row["vs30_mps"]),
            "vs30_inferred": bool(site_row["vs30_inferred"]),
            "z1p0_km": float(site_row["z1p0_km"])
            if pd.notna(site_row["z1p0_km"])
            else np.nan,
            "z2p5_km": float(site_row["z2p5_km"])
            if pd.notna(site_row["z2p5_km"])
            else np.nan,
            "zsed_km": float(site_row["zsed_km"])
            if pd.notna(site_row["zsed_km"])
            else np.nan,
            "requested_period_s": float(site_row["period_s"]),
            "surface_class": str(rupture_row["surface_class"]),
        }
    )

controls = pd.DataFrame(control_records)
controls.to_csv(CONTROL_SELECTION_PATH, index=False, na_rep="")

print("\nControlled rupture-site cases:")
display(
    controls[
        [
            "case_id",
            "source_type",
            "rupture_id",
            "site_id",
            "magnitude",
            "r_rup_km",
            "z_tor_km",
            "dip_deg",
            "vs30_mps",
            "requested_period_s",
        ]
    ]
)


# ----------------------------------------------------------------------------
# 7. Java helper: inventory supported IMTs and evaluate exact GMM inputs
# ----------------------------------------------------------------------------

java_source = r'''
import gov.usgs.earthquake.nshmp.gmm.Gmm;
import gov.usgs.earthquake.nshmp.gmm.GmmInput;
import gov.usgs.earthquake.nshmp.gmm.GroundMotion;
import gov.usgs.earthquake.nshmp.gmm.GroundMotionModel;
import gov.usgs.earthquake.nshmp.gmm.Imt;
import gov.usgs.earthquake.nshmp.tree.Branch;
import gov.usgs.earthquake.nshmp.tree.LogicTree;

import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.io.IOException;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.ArrayList;
import java.util.HashMap;
import java.util.List;
import java.util.Locale;
import java.util.Map;

public final class ControlledParkerGmm {

  private ControlledParkerGmm() {}

  public static void main(String[] args) throws Exception {
    if (args.length != 5) {
      throw new IllegalArgumentException(
          "Expected interface GMM, slab GMM, supported-IMT CSV, "
          + "input TSV, and output CSV.");
    }

    String interfaceGmmName = args[0];
    String slabGmmName = args[1];
    Path supportedImtOutput = Path.of(args[2]).toAbsolutePath().normalize();
    Path inputPath = Path.of(args[3]).toAbsolutePath().normalize();
    Path outputPath = Path.of(args[4]).toAbsolutePath().normalize();

    Gmm interfaceGmm = Gmm.valueOf(interfaceGmmName);
    Gmm slabGmm = Gmm.valueOf(slabGmmName);

    writeSupportedImts(supportedImtOutput, interfaceGmm, slabGmm);
    calculate(inputPath, outputPath);

    System.out.println("CONTROLLED_PARKER_GMM_COMPLETE");
  }

  private static void writeSupportedImts(
      Path outputPath,
      Gmm interfaceGmm,
      Gmm slabGmm) throws IOException {

    if (outputPath.getParent() != null) {
      Files.createDirectories(outputPath.getParent());
    }

    try (BufferedWriter writer = Files.newBufferedWriter(
        outputPath, StandardCharsets.UTF_8)) {
      writer.write("gmm_name,imt_name,is_sa,period_s,units");
      writer.newLine();
      for (Gmm gmm : List.of(interfaceGmm, slabGmm)) {
        for (Imt imt : gmm.supportedImts()) {
          boolean isSa = imt.isSA();
          writer.write(csv(gmm.name()));
          writer.write(',');
          writer.write(csv(imt.name()));
          writer.write(',');
          writer.write(Boolean.toString(isSa));
          writer.write(',');
          writer.write(isSa ? number(imt.period()) : "");
          writer.write(',');
          writer.write(csv(imt.units()));
          writer.newLine();
        }
      }
    }
  }

  private static void calculate(Path inputPath, Path outputPath)
      throws IOException {

    if (outputPath.getParent() != null) {
      Files.createDirectories(outputPath.getParent());
    }

    try (
        BufferedReader reader = Files.newBufferedReader(
            inputPath, StandardCharsets.UTF_8);
        BufferedWriter writer = Files.newBufferedWriter(
            outputPath, StandardCharsets.UTF_8)
    ) {
      String headerLine = reader.readLine();
      if (headerLine == null) {
        throw new IllegalArgumentException("Controlled GMM input TSV is empty.");
      }
      Map<String, Integer> columns = headerMap(headerLine);

      writer.write(String.join(",",
          "calculation_id", "case_id", "source_type", "gmm_name",
          "imt_name", "requested_imt", "requested_period_s",
          "interpolation_role", "interpolation_weight",
          "magnitude", "r_jb_km", "r_rup_km", "r_x_km",
          "dip_deg", "width_km", "z_tor_km", "z_hyp_input_km",
          "rake_deg", "vs30_mps", "vs30_inferred",
          "z1p0_km", "z2p5_km", "zsed_km",
          "branch_id", "branch_weight", "mean_ln_units",
          "median_units", "sigma_total_ln", "calculation_error"));
      writer.newLine();

      String line;
      while ((line = reader.readLine()) != null) {
        if (line.isBlank()) continue;
        String[] values = line.split("\\t", -1);

        String calculationId = get(values, columns, "calculation_id");
        String caseId = get(values, columns, "case_id");
        String sourceType = get(values, columns, "source_type");
        String gmmName = get(values, columns, "gmm_name");
        String imtName = get(values, columns, "imt_name");
        String requestedImt = get(values, columns, "requested_imt");
        String requestedPeriod = get(values, columns, "requested_period_s");
        String interpolationRole = get(values, columns, "interpolation_role");
        String interpolationWeight = get(values, columns, "interpolation_weight");

        List<String> prefix = List.of(
            calculationId,
            caseId,
            sourceType,
            gmmName,
            imtName,
            requestedImt,
            requestedPeriod,
            interpolationRole,
            interpolationWeight,
            get(values, columns, "magnitude"),
            get(values, columns, "r_jb_km"),
            get(values, columns, "r_rup_km"),
            get(values, columns, "r_x_km"),
            get(values, columns, "dip_deg"),
            get(values, columns, "width_km"),
            get(values, columns, "z_tor_km"),
            get(values, columns, "z_hyp_input_km"),
            get(values, columns, "rake_deg"),
            get(values, columns, "vs30_mps"),
            get(values, columns, "vs30_inferred"),
            get(values, columns, "z1p0_km"),
            get(values, columns, "z2p5_km"),
            get(values, columns, "zsed_km"));

        try {
          Gmm gmm = Gmm.valueOf(gmmName);
          Imt imt = Imt.valueOf(imtName);
          if (!gmm.supportedImts().contains(imt)) {
            throw new IllegalArgumentException(
                gmmName + " does not support " + imtName);
          }

          GmmInput.Builder inputBuilder = GmmInput.builder()
              .withDefaults()
              .mag(parseDouble(values, columns, "magnitude"))
              .distances(
                  parseDouble(values, columns, "r_jb_km"),
                  parseDouble(values, columns, "r_rup_km"),
                  parseDouble(values, columns, "r_x_km"))
              .dip(parseDouble(values, columns, "dip_deg"))
              .width(parseDouble(values, columns, "width_km"))
              .zTor(parseDouble(values, columns, "z_tor_km"))
              .zHyp(parseDouble(values, columns, "z_hyp_input_km"))
              .rake(parseDouble(values, columns, "rake_deg"))
              .vs30(parseDouble(values, columns, "vs30_mps"));

          double z1p0 = parseDoubleOrNaN(values, columns, "z1p0_km");
          double z2p5 = parseDoubleOrNaN(values, columns, "z2p5_km");
          if (Double.isFinite(z1p0)) inputBuilder.z1p0(z1p0);
          if (Double.isFinite(z2p5)) inputBuilder.z2p5(z2p5);

          // zSed is retained in the audit input but is not used by the
          // Parker Cascadia interface or slab models.
          GmmInput input = inputBuilder.build();

          GroundMotionModel model = gmm.instance(imt);
          LogicTree<GroundMotion> tree = model.calc(input);

          for (Branch<GroundMotion> branch : tree) {
            GroundMotion motion = branch.value();
            List<String> output = new ArrayList<>(prefix);
            output.add(branch.id());
            output.add(number(branch.weight()));
            output.add(number(motion.mean()));
            output.add(number(Math.exp(motion.mean())));
            output.add(number(motion.sigma()));
            output.add("");
            writeCsvRow(writer, output);
          }

        } catch (Exception exception) {
          List<String> output = new ArrayList<>(prefix);
          output.add("");
          output.add("");
          output.add("");
          output.add("");
          output.add("");
          output.add(exception.getClass().getName() + ": "
              + String.valueOf(exception.getMessage()));
          writeCsvRow(writer, output);
        }
      }
    }
  }

  private static Map<String, Integer> headerMap(String headerLine) {
    String[] names = headerLine.split("\\t", -1);
    Map<String, Integer> columns = new HashMap<>();
    for (int i = 0; i < names.length; i++) {
      columns.put(names[i], i);
    }
    return columns;
  }

  private static String get(
      String[] values, Map<String, Integer> columns, String name) {
    Integer index = columns.get(name);
    if (index == null) {
      throw new IllegalArgumentException("Missing input column: " + name);
    }
    return index < values.length ? values[index] : "";
  }

  private static double parseDouble(
      String[] values, Map<String, Integer> columns, String name) {
    return Double.parseDouble(get(values, columns, name));
  }

  private static double parseDoubleOrNaN(
      String[] values, Map<String, Integer> columns, String name) {
    String value = get(values, columns, name);
    return value == null || value.isBlank()
        ? Double.NaN
        : Double.parseDouble(value);
  }

  private static String number(double value) {
    return String.format(Locale.US, "%.17g", value);
  }

  private static String csv(String value) {
    if (value == null) return "";
    boolean quote = value.contains(",") || value.contains("\"")
        || value.contains("\\n") || value.contains("\\r");
    if (!quote) return value;
    return "\"" + value.replace("\"", "\"\"") + "\"";
  }

  private static void writeCsvRow(
      BufferedWriter writer, List<String> values) throws IOException {
    for (int i = 0; i < values.size(); i++) {
      if (i > 0) writer.write(',');
      writer.write(csv(values.get(i)));
    }
    writer.newLine();
  }
}
'''.strip()

JAVA_SOURCE_PATH.write_text(java_source + "\n", encoding="utf-8")
java_source_hash = sha256_file(JAVA_SOURCE_PATH)

if CELL16_CLASSES_DIR.exists():
    shutil.rmtree(CELL16_CLASSES_DIR)
CELL16_CLASSES_DIR.mkdir(parents=True, exist_ok=True)

compile_classpath = os.pathsep.join(str(path) for path in runtime_paths)
JAVAC_ARGS_PATH.write_text(
    "\n".join(
        [
            "-encoding",
            "UTF-8",
            "-classpath",
            java_arg(compile_classpath),
            "-d",
            java_arg(CELL16_CLASSES_DIR),
            java_arg(JAVA_SOURCE_PATH),
        ]
    )
    + "\n",
    encoding="utf-8",
)

javac_path = shutil.which("javac")
java_path = shutil.which("java")
if javac_path is None or java_path is None:
    raise FileNotFoundError("Both java and javac must be available through PATH.")

compile_result = subprocess.run(
    [javac_path, f"@{JAVAC_ARGS_PATH}"],
    cwd=str(CELL16_JAVA_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)
compile_output = "\n".join(
    value.strip()
    for value in [compile_result.stdout, compile_result.stderr]
    if value and value.strip()
)
COMPILE_LOG_PATH.write_text(compile_output + "\n", encoding="utf-8")

print("\nJava compiler output:")
print(compile_output if compile_output else "[no compiler messages]")

if compile_result.returncode != 0:
    raise RuntimeError(
        "The controlled Parker GMM Java helper did not compile.\n\n"
        f"Compile log:\n  {COMPILE_LOG_PATH}"
    )


# ----------------------------------------------------------------------------
# 8. Run Java once with an empty input to inventory supported IMTs
# ----------------------------------------------------------------------------

empty_columns = [
    "calculation_id",
    "case_id",
    "source_type",
    "gmm_name",
    "imt_name",
    "requested_imt",
    "requested_period_s",
    "interpolation_role",
    "interpolation_weight",
    "magnitude",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
    "dip_deg",
    "width_km",
    "z_tor_km",
    "z_hyp_input_km",
    "rake_deg",
    "vs30_mps",
    "vs30_inferred",
    "z1p0_km",
    "z2p5_km",
    "zsed_km",
]
pd.DataFrame(columns=empty_columns).to_csv(
    EXACT_CALCULATION_INPUT_PATH,
    sep="\t",
    index=False,
    lineterminator="\n",
)

run_classpath = os.pathsep.join(
    [str(CELL16_CLASSES_DIR), *[str(path) for path in runtime_paths]]
)

JAVA_ARGS_PATH.write_text(
    "\n".join(
        [
            f"-Xmx{JAVA_MAX_HEAP}",
            "-cp",
            java_arg(run_classpath),
            "ControlledParkerGmm",
            java_arg(INTERFACE_GMM_NAME),
            java_arg(SLAB_GMM_NAME),
            java_arg(SUPPORTED_IMT_PATH),
            java_arg(EXACT_CALCULATION_INPUT_PATH),
            java_arg(EXACT_OUTPUT_PATH),
        ]
    )
    + "\n",
    encoding="utf-8",
)

inventory_result = subprocess.run(
    [java_path, f"@{JAVA_ARGS_PATH}"],
    cwd=str(CELL16_JAVA_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)
inventory_output = "\n".join(
    value.strip()
    for value in [inventory_result.stdout, inventory_result.stderr]
    if value and value.strip()
)
RUN_LOG_PATH.write_text(inventory_output + "\n", encoding="utf-8")

if inventory_result.returncode != 0:
    raise RuntimeError(
        "The Parker supported-IMT inventory run failed.\n\n"
        f"Run log:\n  {RUN_LOG_PATH}"
    )
if "CONTROLLED_PARKER_GMM_COMPLETE" not in inventory_output:
    raise RuntimeError(
        "The Parker supported-IMT inventory run did not emit its completion message."
    )

supported_imts = pd.read_csv(SUPPORTED_IMT_PATH, low_memory=False)
required_imt_columns = ["gmm_name", "imt_name", "is_sa", "period_s", "units"]
if any(column not in supported_imts for column in required_imt_columns):
    raise RuntimeError(
        "The generated supported-IMT inventory has an unexpected schema."
    )

supported_imts["gmm_name"] = supported_imts["gmm_name"].astype(str)
supported_imts["imt_name"] = supported_imts["imt_name"].astype(str)
supported_imts["is_sa"] = (
    supported_imts["is_sa"].astype(str).str.lower().eq("true")
)
supported_imts["period_s"] = pd.to_numeric(
    supported_imts["period_s"], errors="coerce"
)

for gmm_name in [INTERFACE_GMM_NAME, SLAB_GMM_NAME]:
    gmm_rows = supported_imts.loc[supported_imts["gmm_name"].eq(gmm_name)]
    if gmm_rows.empty:
        raise RuntimeError(f"No supported IMTs were reported for {gmm_name}.")
    if "PGA" not in set(gmm_rows["imt_name"]):
        raise RuntimeError(f"{gmm_name} does not report support for PGA.")

interface_sa = supported_imts.loc[
    supported_imts["gmm_name"].eq(INTERFACE_GMM_NAME)
    & supported_imts["is_sa"]
    & np.isfinite(supported_imts["period_s"]),
    ["imt_name", "period_s"],
].drop_duplicates()
slab_sa = supported_imts.loc[
    supported_imts["gmm_name"].eq(SLAB_GMM_NAME)
    & supported_imts["is_sa"]
    & np.isfinite(supported_imts["period_s"]),
    ["imt_name", "period_s"],
].drop_duplicates()

common_sa = interface_sa.merge(
    slab_sa,
    on=["imt_name", "period_s"],
    how="inner",
    validate="one_to_one",
).sort_values("period_s")

if common_sa.empty:
    raise RuntimeError("The interface and slab Parker models share no SA IMTs.")


# ----------------------------------------------------------------------------
# 9. Resolve requested periods to exact or bracketing USGS IMTs
# ----------------------------------------------------------------------------

def resolve_period(period_s: float) -> dict[str, object]:
    periods = common_sa["period_s"].to_numpy(dtype=float)
    names = common_sa["imt_name"].astype(str).to_numpy()

    exact_index = np.where(np.isclose(
        periods, period_s, rtol=0.0, atol=PERIOD_MATCH_ATOL
    ))[0]
    if len(exact_index) == 1:
        index = int(exact_index[0])
        return {
            "requested_period_s": float(period_s),
            "period_resolution": "exact",
            "lower_imt": str(names[index]),
            "lower_period_s": float(periods[index]),
            "upper_imt": str(names[index]),
            "upper_period_s": float(periods[index]),
            "log_period_weight": 0.0,
        }

    if period_s < periods.min() or period_s > periods.max():
        raise RuntimeError(
            f"Requested period {period_s:g} s is outside the common Parker "
            f"supported range [{periods.min():g}, {periods.max():g}] s."
        )

    upper_position = int(np.searchsorted(periods, period_s, side="right"))
    lower_position = upper_position - 1
    lower_period = float(periods[lower_position])
    upper_period = float(periods[upper_position])
    weight = float(
        (math.log(period_s) - math.log(lower_period))
        / (math.log(upper_period) - math.log(lower_period))
    )
    if not (0.0 < weight < 1.0):
        raise RuntimeError(
            f"Invalid log-period interpolation weight {weight} for {period_s:g} s."
        )

    return {
        "requested_period_s": float(period_s),
        "period_resolution": "log-period interpolation",
        "lower_imt": str(names[lower_position]),
        "lower_period_s": lower_period,
        "upper_imt": str(names[upper_position]),
        "upper_period_s": upper_period,
        "log_period_weight": weight,
    }

period_resolution_records = [
    resolve_period(float(period))
    for period in sorted(controls["requested_period_s"].unique())
]
period_resolution = pd.DataFrame(period_resolution_records)

print("\nRequested-period resolution:")
display(period_resolution)


# ----------------------------------------------------------------------------
# 10. Build exact USGS calculation inputs
# ----------------------------------------------------------------------------

input_records: list[dict[str, object]] = []
control_output_records: list[dict[str, object]] = []

for control in controls.to_dict("records"):
    case_id = str(control["case_id"])
    period_s = float(control["requested_period_s"])
    resolution = period_resolution.loc[
        np.isclose(
            period_resolution["requested_period_s"],
            period_s,
            rtol=0.0,
            atol=PERIOD_MATCH_ATOL,
        )
    ].iloc[0]

    base = dict(control)
    base.update(
        {
            "requested_imt": "PGA",
            "requested_period_s": 0.0,
            "period_resolution": "exact",
            "lower_imt": "PGA",
            "lower_period_s": 0.0,
            "upper_imt": "PGA",
            "upper_period_s": 0.0,
            "log_period_weight": 0.0,
        }
    )
    control_output_records.append(base)

    spectral = dict(control)
    spectral.update(resolution.to_dict())
    spectral["requested_imt"] = f"SA({period_s:g}s)"
    control_output_records.append(spectral)

    requested_definitions = [
        {
            "requested_imt": "PGA",
            "requested_period_s": 0.0,
            "period_resolution": "exact",
            "calculations": [("PGA", "exact", 0.0)],
        },
        {
            "requested_imt": f"SA({period_s:g}s)",
            "requested_period_s": period_s,
            "period_resolution": str(resolution["period_resolution"]),
            "calculations": (
                [
                    (
                        str(resolution["lower_imt"]),
                        "exact",
                        0.0,
                    )
                ]
                if str(resolution["period_resolution"]) == "exact"
                else [
                    (
                        str(resolution["lower_imt"]),
                        "lower",
                        float(resolution["log_period_weight"]),
                    ),
                    (
                        str(resolution["upper_imt"]),
                        "upper",
                        float(resolution["log_period_weight"]),
                    ),
                ]
            ),
        },
    ]

    for definition in requested_definitions:
        for imt_name, interpolation_role, interpolation_weight in definition[
            "calculations"
        ]:
            calculation_id = (
                f"{case_id}__{definition['requested_imt']}__{imt_name}"
                .replace("(", "")
                .replace(")", "")
                .replace(".", "p")
            )
            record = {
                "calculation_id": calculation_id,
                "case_id": case_id,
                "source_type": control["source_type"],
                "gmm_name": control["gmm_name"],
                "imt_name": imt_name,
                "requested_imt": definition["requested_imt"],
                "requested_period_s": definition["requested_period_s"],
                "interpolation_role": interpolation_role,
                "interpolation_weight": interpolation_weight,
                "magnitude": control["magnitude"],
                "r_jb_km": control["r_jb_km"],
                "r_rup_km": control["r_rup_km"],
                "r_x_km": control["r_x_km"],
                "dip_deg": control["dip_deg"],
                "width_km": control["width_km"],
                "z_tor_km": control["z_tor_km"],
                "z_hyp_input_km": control["z_hyp_input_km"],
                "rake_deg": control["rake_deg"],
                "vs30_mps": control["vs30_mps"],
                "vs30_inferred": str(control["vs30_inferred"]).lower(),
                "z1p0_km": control["z1p0_km"],
                "z2p5_km": control["z2p5_km"],
                "zsed_km": control["zsed_km"],
            }
            input_records.append(record)

controlled_requested_inputs = pd.DataFrame(control_output_records)
controlled_requested_inputs.to_csv(CONTROL_INPUT_PATH, index=False, na_rep="")

exact_inputs = pd.DataFrame(input_records)
if exact_inputs["calculation_id"].duplicated().any():
    raise RuntimeError("Generated controlled calculation IDs are not unique.")

exact_inputs.to_csv(
    EXACT_CALCULATION_INPUT_PATH,
    sep="\t",
    index=False,
    na_rep="",
    lineterminator="\n",
)

print("\nExact USGS calculations to execute:")
display(
    exact_inputs[
        [
            "calculation_id",
            "source_type",
            "gmm_name",
            "imt_name",
            "requested_imt",
            "interpolation_role",
            "magnitude",
            "r_rup_km",
            "vs30_mps",
        ]
    ]
)


# ----------------------------------------------------------------------------
# 11. Run the exact controlled GMM calculations
# ----------------------------------------------------------------------------

JAVA_ARGS_PATH.write_text(
    "\n".join(
        [
            f"-Xmx{JAVA_MAX_HEAP}",
            "-cp",
            java_arg(run_classpath),
            "ControlledParkerGmm",
            java_arg(INTERFACE_GMM_NAME),
            java_arg(SLAB_GMM_NAME),
            java_arg(SUPPORTED_IMT_PATH),
            java_arg(EXACT_CALCULATION_INPUT_PATH),
            java_arg(EXACT_OUTPUT_PATH),
        ]
    )
    + "\n",
    encoding="utf-8",
)

run_result = subprocess.run(
    [java_path, f"@{JAVA_ARGS_PATH}"],
    cwd=str(CELL16_JAVA_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)
run_output = "\n".join(
    value.strip()
    for value in [run_result.stdout, run_result.stderr]
    if value and value.strip()
)
RUN_LOG_PATH.write_text(run_output + "\n", encoding="utf-8")

print("\nJava runtime output:")
print(run_output if run_output else "[no runtime messages]")

if run_result.returncode != 0:
    raise RuntimeError(
        "The controlled Parker GMM calculation failed.\n\n"
        f"Run log:\n  {RUN_LOG_PATH}"
    )
if "CONTROLLED_PARKER_GMM_COMPLETE" not in run_output:
    raise RuntimeError(
        "The controlled Parker GMM calculation did not emit its completion message."
    )

exact_output = pd.read_csv(
    EXACT_OUTPUT_PATH,
    dtype={
        "calculation_id": str,
        "case_id": str,
        "source_type": str,
        "gmm_name": str,
        "imt_name": str,
        "requested_imt": str,
        "interpolation_role": str,
        "branch_id": str,
        "calculation_error": str,
    },
    low_memory=False,
)
exact_output["calculation_error"] = (
    exact_output["calculation_error"].fillna("").astype(str)
)

for column in [
    "requested_period_s",
    "interpolation_weight",
    "magnitude",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
    "dip_deg",
    "width_km",
    "z_tor_km",
    "z_hyp_input_km",
    "rake_deg",
    "vs30_mps",
    "z1p0_km",
    "z2p5_km",
    "zsed_km",
    "branch_weight",
    "mean_ln_units",
    "median_units",
    "sigma_total_ln",
]:
    exact_output[column] = pd.to_numeric(exact_output[column], errors="coerce")


# ----------------------------------------------------------------------------
# 12. Define the Parker variability coefficients required by this case study
# ----------------------------------------------------------------------------

# The Seaside W2 calculation uses only PGA and SA(0.4 s). Parker's published
# ergodic aleatory model provides Tau and the variance-domain coefficients
# phi21, phi22, and phi2V for each IMT. These values are embedded explicitly
# here so Cell 16 does not depend on Gradle exposing a source/resource CSV.
#
# This is not accepted on faith: Section 13 reconstructs sigma for every
# controlled USGS calculation and requires agreement with motion.sigma() from
# the exact pinned Java runtime to SIGMA_MATCH_ATOL. Any coefficient/version
# disagreement therefore fails the cell before bulk processing begins.
PARKER_VARIABILITY_REFERENCE = {
    "model": "Parker et al. NGA-Subduction global GMM",
    "reference": (
        "Parker, Stewart, Boore, Atkinson, and Hassani (2022), "
        "Earthquake Spectra 38(1), 456-493; coefficient table"
    ),
    "doi": "10.1177/87552930211034889",
    "implementation_check": (
        "Each reconstructed total sigma is compared with the exact sigma "
        "returned by the pinned USGS nshmp-lib runtime."
    ),
}

coefficients = pd.DataFrame(
    [
        {
            "coefficient_label": "PGA",
            "period_s": 0.0,
            "tau_ln": 0.480,
            "phi21_variance": 0.396,
            "phi22_variance": 0.565,
            "phi2v_variance_adjustment": -0.180,
        },
        {
            "coefficient_label": "SA0P4",
            "period_s": 0.40,
            "tau_ln": 0.492,
            "phi21_variance": 0.433,
            "phi22_variance": 0.556,
            "phi2v_variance_adjustment": -0.190,
        },
    ]
).sort_values("period_s").reset_index(drop=True)

# Defensive checks on the embedded table before it is used.
required_coefficient_columns = [
    "coefficient_label",
    "period_s",
    "tau_ln",
    "phi21_variance",
    "phi22_variance",
    "phi2v_variance_adjustment",
]
if list(coefficients.columns) != required_coefficient_columns:
    raise RuntimeError("Unexpected embedded Parker coefficient schema.")
if coefficients["period_s"].duplicated().any():
    raise RuntimeError("Embedded Parker coefficient periods are duplicated.")
if not np.isfinite(
    coefficients[
        [
            "period_s",
            "tau_ln",
            "phi21_variance",
            "phi22_variance",
            "phi2v_variance_adjustment",
        ]
    ].to_numpy(dtype=float)
).all():
    raise RuntimeError("Embedded Parker coefficients contain non-finite values.")
if not coefficients["tau_ln"].gt(0.0).all():
    raise RuntimeError("Embedded Parker Tau values must be positive.")
if not coefficients[["phi21_variance", "phi22_variance"]].gt(0.0).all().all():
    raise RuntimeError("Embedded Parker base within-event variances must be positive.")

COEFFICIENT_RESOURCE_DIR = CELL16_WORK_DIR / "resources"
COEFFICIENT_RESOURCE_DIR.mkdir(parents=True, exist_ok=True)
coefficient_path = (
    COEFFICIENT_RESOURCE_DIR
    / "parker_published_variability_coefficients_pga_sa0p4.csv"
)
coefficients.to_csv(coefficient_path, index=False, lineterminator="\n")
coefficients.to_csv(COEFFICIENT_INVENTORY_PATH, index=False, lineterminator="\n")

coefficient_provenance = {
    "pipeline_version": PIPELINE_VERSION,
    "coefficient_scope": ["PGA", "SA0P4"],
    "seaside_w2_period_s": SEASIDE_W2_PERIOD_S,
    "reference": PARKER_VARIABILITY_REFERENCE,
    "materialized_coefficient_file": str(coefficient_path),
    "materialized_coefficient_sha256": sha256_file(coefficient_path),
    "coefficient_inventory_file": str(COEFFICIENT_INVENTORY_PATH),
    "coefficient_inventory_sha256": sha256_file(COEFFICIENT_INVENTORY_PATH),
    "validation_requirement": {
        "identity": "sigma = sqrt(tau^2 + phi^2)",
        "comparison_target": "GroundMotion.sigma() from pinned USGS nshmp-lib",
        "absolute_tolerance": SIGMA_MATCH_ATOL,
    },
    "runtime_classpath_entries": [str(path) for path in runtime_paths],
}
COEFFICIENT_PROVENANCE_PATH.write_text(
    json.dumps(coefficient_provenance, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("\nParker variability coefficients required by Seaside W2:")
print(coefficients.to_string(index=False))
print(f"  Materialized copy: {coefficient_path}")
print(
    "  Acceptance rule: reconstructed sigma must match the pinned USGS "
    f"runtime within {SIGMA_MATCH_ATOL:g}."
)


# ----------------------------------------------------------------------------
# 13. Reconstruct exact-IMT tau, phi, and total sigma
# ----------------------------------------------------------------------------

def coefficient_row_for_imt(imt_name: str) -> pd.Series:
    period = period_from_imt_name(imt_name)
    matches = coefficients.loc[
        np.isclose(
            coefficients["period_s"],
            period,
            rtol=0.0,
            atol=PERIOD_MATCH_ATOL,
        )
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one Parker variability coefficient row for {imt_name} "
            f"(period {period:g} s); observed {len(matches)}."
        )
    return matches.iloc[0]


def parker_phi_from_variance_coefficients(
    *,
    phi21: float,
    phi22: float,
    phi2v: float,
    r_rup_km: float,
    vs30_mps: float,
) -> tuple[float, float]:
    # Parker et al. ergodic within-event variance model. Coefficients phi21,
    # phi22, and phi2V are variance-domain quantities; sqrt is applied only
    # after the distance and Vs30 terms are combined.
    r1 = 200.0
    r2 = 500.0
    v1 = 200.0
    v2 = 500.0

    if r_rup_km <= r1:
        phi_variance = phi21
    elif r_rup_km >= r2:
        phi_variance = phi22
    else:
        phi_variance = (
            (phi22 - phi21)
            / (math.log(r2) - math.log(r1))
            * (math.log(r_rup_km) - math.log(r1))
            + phi21
        )

    clamped_distance = max(r1, min(r2, r_rup_km))
    distance_taper = (
        math.log(r2 / clamped_distance) / math.log(r2 / r1)
    )

    if vs30_mps <= v1:
        velocity_taper = 1.0
    elif vs30_mps < v2:
        velocity_taper = (
            math.log(v2 / vs30_mps) / math.log(v2 / v1)
        )
    else:
        velocity_taper = 0.0

    phi_variance += phi2v * velocity_taper * distance_taper

    if phi_variance <= 0.0 or not np.isfinite(phi_variance):
        raise RuntimeError(
            "Parker within-event variance is nonpositive or non-finite: "
            f"{phi_variance}"
        )

    return float(math.sqrt(phi_variance)), float(phi_variance)

exact_output["tau_ln"] = np.nan
exact_output["phi_ln"] = np.nan
exact_output["phi_variance"] = np.nan
exact_output["sigma_reconstructed_ln"] = np.nan
exact_output["sigma_absolute_difference"] = np.nan

for index, row in exact_output.iterrows():
    if str(row["calculation_error"]).strip():
        continue
    coefficient = coefficient_row_for_imt(str(row["imt_name"]))
    phi_ln, phi_variance = parker_phi_from_variance_coefficients(
        phi21=float(coefficient["phi21_variance"]),
        phi22=float(coefficient["phi22_variance"]),
        phi2v=float(coefficient["phi2v_variance_adjustment"]),
        r_rup_km=float(row["r_rup_km"]),
        vs30_mps=float(row["vs30_mps"]),
    )
    tau_ln = float(coefficient["tau_ln"])
    sigma_reconstructed = float(math.sqrt(tau_ln**2 + phi_ln**2))

    exact_output.at[index, "tau_ln"] = tau_ln
    exact_output.at[index, "phi_ln"] = phi_ln
    exact_output.at[index, "phi_variance"] = phi_variance
    exact_output.at[index, "sigma_reconstructed_ln"] = sigma_reconstructed
    exact_output.at[index, "sigma_absolute_difference"] = abs(
        sigma_reconstructed - float(row["sigma_total_ln"])
    )

exact_output.to_csv(EXACT_OUTPUT_PATH, index=False, na_rep="")


# ----------------------------------------------------------------------------
# 14. Produce requested-period outputs with transparent interpolation
# ----------------------------------------------------------------------------

requested_records: list[dict[str, object]] = []

for (case_id, requested_imt, branch_id), group in exact_output.groupby(
    ["case_id", "requested_imt", "branch_id"],
    sort=False,
    dropna=False,
):
    if not str(branch_id).strip():
        continue
    if group["calculation_error"].ne("").any():
        continue

    group = group.sort_values("imt_name").copy()
    roles = set(group["interpolation_role"].astype(str))

    base = {
        "case_id": str(case_id),
        "source_type": str(group["source_type"].iloc[0]),
        "gmm_name": str(group["gmm_name"].iloc[0]),
        "requested_imt": str(requested_imt),
        "requested_period_s": float(group["requested_period_s"].iloc[0]),
        "branch_id": str(branch_id),
        "branch_weight": float(group["branch_weight"].iloc[0]),
        "magnitude": float(group["magnitude"].iloc[0]),
        "r_jb_km": float(group["r_jb_km"].iloc[0]),
        "r_rup_km": float(group["r_rup_km"].iloc[0]),
        "r_x_km": float(group["r_x_km"].iloc[0]),
        "dip_deg": float(group["dip_deg"].iloc[0]),
        "width_km": float(group["width_km"].iloc[0]),
        "z_tor_km": float(group["z_tor_km"].iloc[0]),
        "z_hyp_input_km": float(group["z_hyp_input_km"].iloc[0]),
        "rake_deg": float(group["rake_deg"].iloc[0]),
        "vs30_mps": float(group["vs30_mps"].iloc[0]),
    }

    if roles == {"exact"} and len(group) == 1:
        row = group.iloc[0]
        base.update(
            {
                "period_resolution": "exact",
                "lower_imt": str(row["imt_name"]),
                "upper_imt": str(row["imt_name"]),
                "log_period_weight": 0.0,
                "mean_ln_units": float(row["mean_ln_units"]),
                "median_units": float(row["median_units"]),
                "tau_ln": float(row["tau_ln"]),
                "phi_variance": float(row["phi_variance"]),
                "phi_ln": float(row["phi_ln"]),
                "sigma_total_ln": float(row["sigma_reconstructed_ln"]),
                "usgs_exact_sigma_ln": float(row["sigma_total_ln"]),
                "linear_sigma_interpolation_ln": float(row["sigma_total_ln"]),
            }
        )
    elif roles == {"lower", "upper"} and len(group) == 2:
        lower = group.loc[group["interpolation_role"].eq("lower")].iloc[0]
        upper = group.loc[group["interpolation_role"].eq("upper")].iloc[0]
        weight = float(lower["interpolation_weight"])
        if not np.isclose(
            weight,
            float(upper["interpolation_weight"]),
            rtol=0.0,
            atol=PERIOD_MATCH_ATOL,
        ):
            raise RuntimeError("Lower and upper interpolation weights disagree.")

        mean_ln = (1.0 - weight) * float(lower["mean_ln_units"]) + weight * float(
            upper["mean_ln_units"]
        )
        tau_ln = (1.0 - weight) * float(lower["tau_ln"]) + weight * float(
            upper["tau_ln"]
        )
        # Parker's phi21, phi22, and phi2V terms are variance-domain
        # coefficients. Therefore, interpolate the resulting within-event
        # variance in ln(period), then take its square root. This is
        # equivalent to interpolating those variance coefficients first and
        # re-evaluating the Parker phi model at the target period.
        phi_variance = (
            (1.0 - weight) * float(lower["phi_variance"])
            + weight * float(upper["phi_variance"])
        )
        if phi_variance <= 0.0 or not np.isfinite(phi_variance):
            raise RuntimeError(
                "Interpolated Parker within-event variance is invalid: "
                f"{phi_variance}"
            )
        phi_ln = float(math.sqrt(phi_variance))
        sigma_ln = float(math.sqrt(tau_ln**2 + phi_ln**2))
        linearly_interpolated_sigma = (
            (1.0 - weight) * float(lower["sigma_total_ln"])
            + weight * float(upper["sigma_total_ln"])
        )

        base.update(
            {
                "period_resolution": "log-period interpolation",
                "lower_imt": str(lower["imt_name"]),
                "upper_imt": str(upper["imt_name"]),
                "log_period_weight": weight,
                "mean_ln_units": mean_ln,
                "median_units": float(math.exp(mean_ln)),
                "tau_ln": tau_ln,
                "phi_variance": phi_variance,
                "phi_ln": phi_ln,
                "sigma_total_ln": sigma_ln,
                "usgs_exact_sigma_ln": np.nan,
                "linear_sigma_interpolation_ln": linearly_interpolated_sigma,
            }
        )
    else:
        raise RuntimeError(
            "Unexpected exact-output grouping for requested interpolation:\n"
            f"case={case_id}, requested={requested_imt}, branch={branch_id}, "
            f"roles={sorted(roles)}, rows={len(group)}"
        )

    requested_records.append(base)

requested_output = pd.DataFrame(requested_records)
requested_output = requested_output.sort_values(
    ["case_id", "requested_period_s", "branch_weight", "branch_id"]
).reset_index(drop=True)
requested_output.to_csv(REQUESTED_OUTPUT_PATH, index=False, na_rep="")


# ----------------------------------------------------------------------------
# 15. Additional depth-handling probe for the two slab controls
# ----------------------------------------------------------------------------

# The pinned USGS Parker slab implementation is expected to derive the model's
# mean hypocentral depth internally from zTor. To prove that the supplied zHyp
# field does not control the selected implementation, recalculate each exact
# slab input with a deliberately perturbed zHyp while holding every other input
# fixed. This probe is small and remains separate from the accepted outputs.
depth_probe_input_path = CELL16_INPUT_DIR / "controlled_slab_depth_probe.tsv"
depth_probe_output_path = CELL16_OUTPUT_DIR / "controlled_slab_depth_probe.csv"

depth_probe_inputs = exact_inputs.loc[
    exact_inputs["source_type"].eq("SLAB")
].copy()
depth_probe_inputs["calculation_id"] = (
    depth_probe_inputs["calculation_id"].astype(str) + "__zhyp_perturbed"
)
depth_probe_inputs["z_hyp_input_km"] = np.minimum(
    199.0,
    depth_probe_inputs["z_hyp_input_km"].astype(float) + 25.0,
)
depth_probe_inputs.to_csv(
    depth_probe_input_path,
    sep="\t",
    index=False,
    na_rep="",
    lineterminator="\n",
)

JAVA_ARGS_PATH.write_text(
    "\n".join(
        [
            f"-Xmx{JAVA_MAX_HEAP}",
            "-cp",
            java_arg(run_classpath),
            "ControlledParkerGmm",
            java_arg(INTERFACE_GMM_NAME),
            java_arg(SLAB_GMM_NAME),
            java_arg(SUPPORTED_IMT_PATH),
            java_arg(depth_probe_input_path),
            java_arg(depth_probe_output_path),
        ]
    )
    + "\n",
    encoding="utf-8",
)

depth_probe_result = subprocess.run(
    [java_path, f"@{JAVA_ARGS_PATH}"],
    cwd=str(CELL16_JAVA_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)
depth_probe_log = "\n".join(
    value.strip()
    for value in [depth_probe_result.stdout, depth_probe_result.stderr]
    if value and value.strip()
)
if depth_probe_result.returncode != 0 or "CONTROLLED_PARKER_GMM_COMPLETE" not in depth_probe_log:
    raise RuntimeError(
        "The controlled Parker slab depth-handling probe failed.\n\n"
        + depth_probe_log
    )

depth_probe_output = pd.read_csv(
    depth_probe_output_path,
    dtype={
        "calculation_id": str,
        "case_id": str,
        "imt_name": str,
        "branch_id": str,
        "calculation_error": str,
    },
    low_memory=False,
)
depth_probe_output["calculation_error"] = (
    depth_probe_output["calculation_error"].fillna("").astype(str)
)
for column in ["mean_ln_units", "sigma_total_ln"]:
    depth_probe_output[column] = pd.to_numeric(
        depth_probe_output[column], errors="coerce"
    )

accepted_slab = exact_output.loc[
    exact_output["source_type"].eq("SLAB")
    & exact_output["calculation_error"].eq("")
].copy()
accepted_slab["base_calculation_id"] = accepted_slab["calculation_id"].astype(str)
depth_probe_output["base_calculation_id"] = (
    depth_probe_output["calculation_id"]
    .astype(str)
    .str.replace("__zhyp_perturbed", "", regex=False)
)

depth_comparison = accepted_slab.merge(
    depth_probe_output[
        [
            "base_calculation_id",
            "branch_id",
            "mean_ln_units",
            "sigma_total_ln",
            "calculation_error",
        ]
    ],
    on=["base_calculation_id", "branch_id"],
    how="left",
    suffixes=("_accepted", "_perturbed"),
    validate="one_to_one",
)

depth_mean_max_difference = float(
    (
        depth_comparison["mean_ln_units_accepted"]
        - depth_comparison["mean_ln_units_perturbed"]
    ).abs().max()
)
depth_sigma_max_difference = float(
    (
        depth_comparison["sigma_total_ln_accepted"]
        - depth_comparison["sigma_total_ln_perturbed"]
    ).abs().max()
)


# ----------------------------------------------------------------------------
# 16. Controlled validation
# ----------------------------------------------------------------------------

validation_records: list[dict[str, object]] = []

add_validation(
    validation_records,
    "Cell 15 dependency",
    "All Cell 15 final checks passed",
    int(cell15_passes.sum()),
    len(cell15_passes),
    bool(cell15_passes.all()),
)
add_validation(
    validation_records,
    "Coverage",
    "Authoritative distance row count",
    observed_distance_rows,
    EXPECTED_DISTANCE_ROWS,
    observed_distance_rows == EXPECTED_DISTANCE_ROWS,
)
add_validation(
    validation_records,
    "Coverage",
    "Four source-distance controls selected",
    len(controls),
    4,
    len(controls) == 4,
)
add_validation(
    validation_records,
    "Coverage",
    "Control source types",
    sorted(controls["source_type"].unique()),
    ["INTERFACE", "SLAB"],
    set(controls["source_type"]) == {"INTERFACE", "SLAB"},
)
add_validation(
    validation_records,
    "Coverage",
    "Control distance roles",
    sorted(controls["distance_role"].unique()),
    ["farthest", "nearest"],
    set(controls["distance_role"]) == {"nearest", "farthest"},
)
add_validation(
    validation_records,
    "Inputs",
    "Vs30 values within USGS range",
    f"{controls['vs30_mps'].min():.6g} to {controls['vs30_mps'].max():.6g}",
    "150 to 2000 m/s",
    controls["vs30_mps"].between(150.0, 2000.0).all(),
    vs30_provenance,
)
add_validation(
    validation_records,
    "Inputs",
    "Requested periods positive",
    f"{controls['requested_period_s'].min():.6g} to {controls['requested_period_s'].max():.6g}",
    "> 0 s",
    controls["requested_period_s"].gt(0.0).all(),
    period_provenance,
)
add_validation(
    validation_records,
    "Inputs",
    "Each requested period resolved to exact or bracketed IMTs",
    len(period_resolution),
    controls["requested_period_s"].nunique(),
    len(period_resolution) == controls["requested_period_s"].nunique(),
)
add_validation(
    validation_records,
    "USGS GMM",
    "No Java calculation errors",
    int(exact_output["calculation_error"].ne("").sum()),
    0,
    exact_output["calculation_error"].eq("").all(),
)

successful_output = exact_output.loc[
    exact_output["calculation_error"].eq("")
].copy()
expected_branch_ids = {"epi-lo", "epi-off", "epi-hi"}
calculation_branch_sets = successful_output.groupby("calculation_id")[
    "branch_id"
].agg(lambda values: set(map(str, values)))
calculation_branch_counts = successful_output.groupby("calculation_id").size()
calculation_weight_sums = successful_output.groupby("calculation_id")[
    "branch_weight"
].sum()

add_validation(
    validation_records,
    "Epistemic branches",
    "Every exact calculation returned three branches",
    int(calculation_branch_counts.eq(3).sum()),
    len(exact_inputs),
    len(calculation_branch_counts) == len(exact_inputs)
    and calculation_branch_counts.eq(3).all(),
)
add_validation(
    validation_records,
    "Epistemic branches",
    "Every exact calculation returned epi-lo/off/hi",
    int(calculation_branch_sets.map(lambda value: value == expected_branch_ids).sum()),
    len(exact_inputs),
    len(calculation_branch_sets) == len(exact_inputs)
    and calculation_branch_sets.map(lambda value: value == expected_branch_ids).all(),
)
add_validation(
    validation_records,
    "Epistemic branches",
    "Branch weights sum to one",
    float((calculation_weight_sums - 1.0).abs().max()),
    f"<= {WEIGHT_SUM_ATOL:g}",
    np.allclose(
        calculation_weight_sums.to_numpy(dtype=float),
        1.0,
        rtol=0.0,
        atol=WEIGHT_SUM_ATOL,
    ),
)
add_validation(
    validation_records,
    "Epistemic branches",
    "Branch weights finite and positive",
    int(
        (
            np.isfinite(successful_output["branch_weight"])
            & successful_output["branch_weight"].gt(0.0)
        ).sum()
    ),
    len(successful_output),
    np.isfinite(successful_output["branch_weight"]).all()
    and successful_output["branch_weight"].gt(0.0).all(),
)

branch_order_ok = True
for _, group in successful_output.groupby("calculation_id"):
    means = group.set_index("branch_id")["mean_ln_units"]
    if not (
        means["epi-lo"] < means["epi-off"] < means["epi-hi"]
    ):
        branch_order_ok = False
        break
add_validation(
    validation_records,
    "Epistemic branches",
    "Branch means ordered low < off < high",
    branch_order_ok,
    True,
    branch_order_ok,
)

for column, label in [
    ("mean_ln_units", "Means finite"),
    ("median_units", "Medians finite and positive"),
    ("sigma_total_ln", "USGS total sigma finite and positive"),
    ("tau_ln", "Tau finite and positive"),
    ("phi_ln", "Phi finite and positive"),
    ("sigma_reconstructed_ln", "Reconstructed sigma finite and positive"),
]:
    finite = np.isfinite(successful_output[column])
    positive = successful_output[column].gt(0.0)
    valid = finite if column == "mean_ln_units" else finite & positive
    add_validation(
        validation_records,
        "Numerical outputs",
        label,
        int(valid.sum()),
        len(successful_output),
        valid.all(),
    )

median_difference = float(
    (
        np.exp(successful_output["mean_ln_units"])
        - successful_output["median_units"]
    ).abs().max()
)
add_validation(
    validation_records,
    "Numerical outputs",
    "Median equals exp(mean)",
    median_difference,
    "<= 1e-14",
    median_difference <= 1.0e-14,
)

max_sigma_difference = float(
    successful_output["sigma_absolute_difference"].max()
)
add_validation(
    validation_records,
    "Aleatory variability",
    "Published Parker coefficients reproduce pinned USGS total sigma",
    max_sigma_difference,
    f"<= {SIGMA_MATCH_ATOL:g}",
    max_sigma_difference <= SIGMA_MATCH_ATOL,
    "sigma = sqrt(tau^2 + phi^2), using Parker's ergodic total within-event phi.",
)

sigma_branch_spread = float(
    successful_output.groupby("calculation_id")["sigma_total_ln"]
    .agg(lambda values: float(values.max() - values.min()))
    .max()
)
add_validation(
    validation_records,
    "Aleatory variability",
    "Total sigma is identical across epistemic branches",
    sigma_branch_spread,
    f"<= {SIGMA_MATCH_ATOL:g}",
    sigma_branch_spread <= SIGMA_MATCH_ATOL,
)

tau_branch_spread = float(
    successful_output.groupby("calculation_id")["tau_ln"]
    .agg(lambda values: float(values.max() - values.min()))
    .max()
)
phi_branch_spread = float(
    successful_output.groupby("calculation_id")["phi_ln"]
    .agg(lambda values: float(values.max() - values.min()))
    .max()
)
add_validation(
    validation_records,
    "Aleatory variability",
    "Tau and phi are identical across epistemic branches",
    max(tau_branch_spread, phi_branch_spread),
    f"<= {SIGMA_MATCH_ATOL:g}",
    max(tau_branch_spread, phi_branch_spread) <= SIGMA_MATCH_ATOL,
)

add_validation(
    validation_records,
    "Requested outputs",
    "Requested output row count",
    len(requested_output),
    4 * 2 * 3,
    len(requested_output) == 24,
    "Four cases × PGA and site-period SA × three epistemic branches.",
)
add_validation(
    validation_records,
    "Requested outputs",
    "Requested medians finite and positive",
    int(
        (
            np.isfinite(requested_output["median_units"])
            & requested_output["median_units"].gt(0.0)
        ).sum()
    ),
    len(requested_output),
    np.isfinite(requested_output["median_units"]).all()
    and requested_output["median_units"].gt(0.0).all(),
)
add_validation(
    validation_records,
    "Requested outputs",
    "Requested tau, phi, and sigma finite and positive",
    int(
        (
            np.isfinite(requested_output[["tau_ln", "phi_ln", "sigma_total_ln"]])
            & requested_output[["tau_ln", "phi_ln", "sigma_total_ln"]].gt(0.0)
        ).all(axis=1).sum()
    ),
    len(requested_output),
    (
        np.isfinite(requested_output[["tau_ln", "phi_ln", "sigma_total_ln"]])
        & requested_output[["tau_ln", "phi_ln", "sigma_total_ln"]].gt(0.0)
    ).all().all(),
)

interpolated = requested_output.loc[
    requested_output["period_resolution"].eq("log-period interpolation")
]
if interpolated.empty:
    interpolation_weight_valid = True
    interpolation_note = "No requested period required interpolation."
else:
    interpolation_weight_valid = interpolated["log_period_weight"].between(
        0.0, 1.0, inclusive="neither"
    ).all()
    interpolation_note = (
        f"Interpolated periods use {sorted(interpolated['lower_imt'].unique())} "
        f"and {sorted(interpolated['upper_imt'].unique())}."
    )
add_validation(
    validation_records,
    "Period interpolation",
    "Log-period interpolation weights are strictly between zero and one",
    interpolation_weight_valid,
    True,
    interpolation_weight_valid,
    interpolation_note,
)

add_validation(
    validation_records,
    "Depth handling",
    "Perturbing supplied slab zHyp does not change Parker means",
    depth_mean_max_difference,
    f"<= {DEPTH_INVARIANCE_ATOL:g}",
    depth_mean_max_difference <= DEPTH_INVARIANCE_ATOL,
    "Confirms the pinned slab implementation derives its mean depth from zTor.",
)
add_validation(
    validation_records,
    "Depth handling",
    "Perturbing supplied slab zHyp does not change Parker sigma",
    depth_sigma_max_difference,
    f"<= {DEPTH_INVARIANCE_ATOL:g}",
    depth_sigma_max_difference <= DEPTH_INVARIANCE_ATOL,
)

validation = pd.DataFrame(validation_records)
validation.to_csv(VALIDATION_PATH, index=False)

print("\n" + "=" * 78)
print("CELL 16 VALIDATION")
print("=" * 78)
display(validation)

if not validation["passes"].all():
    failed = validation.loc[
        ~validation["passes"],
        ["category", "check", "observed", "expected", "note"],
    ]
    raise RuntimeError(
        "Cell 16 controlled Parker GMM validation failed.\n\n"
        + failed.to_string(index=False)
        + f"\n\nValidation file:\n  {VALIDATION_PATH}"
    )


# ----------------------------------------------------------------------------
# 17. Reproducibility summary
# ----------------------------------------------------------------------------

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": utc_now(),
    "interface_gmm": INTERFACE_GMM_NAME,
    "slab_gmm": SLAB_GMM_NAME,
    "cell15_distance_path": str(CELL15_DISTANCE_PATH),
    "cell15_distance_sha256": sha256_file(CELL15_DISTANCE_PATH),
    "cell15_audit_path": str(CELL15_AUDIT_PATH),
    "cell15_audit_sha256": sha256_file(CELL15_AUDIT_PATH),
    "portfolio_path": str(GMM_READY_PORTFOLIO_PATH),
    "portfolio_sha256": sha256_file(GMM_READY_PORTFOLIO_PATH),
    "vs30_provenance": vs30_provenance,
    "period_provenance": period_provenance,
    "vs30_inferred_provenance": vs30_inferred_provenance,
    "default_vs30_mps": DEFAULT_VS30_MPS,
    "default_period_s": DEFAULT_PERIOD_S,
    "control_cases": controls.to_dict("records"),
    "supported_imt_path": str(SUPPORTED_IMT_PATH),
    "supported_imt_sha256": sha256_file(SUPPORTED_IMT_PATH),
    "coefficient_path": str(coefficient_path),
    "coefficient_sha256": sha256_file(coefficient_path),
    "coefficient_inventory_path": str(COEFFICIENT_INVENTORY_PATH),
    "java_source_path": str(JAVA_SOURCE_PATH),
    "java_source_sha256": java_source_hash,
    "exact_input_path": str(EXACT_CALCULATION_INPUT_PATH),
    "exact_input_sha256": sha256_file(EXACT_CALCULATION_INPUT_PATH),
    "exact_output_path": str(EXACT_OUTPUT_PATH),
    "exact_output_sha256": sha256_file(EXACT_OUTPUT_PATH),
    "requested_output_path": str(REQUESTED_OUTPUT_PATH),
    "requested_output_sha256": sha256_file(REQUESTED_OUTPUT_PATH),
    "validation_path": str(VALIDATION_PATH),
    "validation_sha256": sha256_file(VALIDATION_PATH),
    "validation_checks": len(validation),
    "all_checks_passed": bool(validation["passes"].all()),
    "maximum_sigma_reconstruction_difference": max_sigma_difference,
    "maximum_slab_zhyp_probe_mean_difference": depth_mean_max_difference,
    "maximum_slab_zhyp_probe_sigma_difference": depth_sigma_max_difference,
    "aleatory_model": {
        "between_event": "Parker tau",
        "within_event": "Parker ergodic total phi",
        "identity": "sigma = sqrt(tau^2 + phi^2)",
        "distance_dependence": "phi variance uses Rrup corners 200 and 500 km",
        "site_dependence": "phi variance uses Vs30 corners 200 and 500 m/s",
    },
    "period_interpolation": {
        "mean": "linear interpolation of ln median in ln period",
        "tau": "linear interpolation of tau in ln period",
        "phi_variance": (
            "linear interpolation of Parker within-event variance in ln period, "
            "followed by phi = sqrt(variance)"
        ),
        "sigma": "recomputed as sqrt(tau^2 + phi^2)",
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("\nCELL 16 CONTROLLED PARKER GMM VALIDATION COMPLETE")
print(f"\nControlled cases:\n  {CONTROL_SELECTION_PATH}")
print(f"\nControlled requested inputs:\n  {CONTROL_INPUT_PATH}")
print(f"\nSupported IMTs:\n  {SUPPORTED_IMT_PATH}")
print(f"\nExact USGS branch outputs:\n  {EXACT_OUTPUT_PATH}")
print(f"\nRequested-period branch outputs:\n  {REQUESTED_OUTPUT_PATH}")
print(f"\nParker variability coefficients:\n  {COEFFICIENT_INVENTORY_PATH}")
print(f"\nValidation:\n  {VALIDATION_PATH}")
print(f"\nSummary:\n  {SUMMARY_PATH}")
print(
    "\nNext step: Cell 17 can apply this validated GMM-input construction, "
    "epistemic-branch handling, tau-phi decomposition, and period "
    "interpolation to the full rupture-site matrix with chunking and restart."
)


CELL 16: SEASIDE W2 CONTROLLED PARKER GMM VALIDATION

Interface GMM: PSBAH_20_CASCADIA_INTERFACE
Slab GMM:      PSBAH_20_CASCADIA_SLAB
Vs30 source:   portfolio column: vs30_mps
Period source: portfolio column: Period_for_spectral_acceleration
Validated period: 0.40 s (exact USGS SA0P4)
Vs30 inferred: constant fallback: True

Controlled rupture-site cases:


,case_id,source_type,rupture_id,site_id,magnitude,r_rup_km,z_tor_km,dip_deg,vs30_mps,requested_period_s
0,interface_nearest,INTERFACE,b18dfe7bfe65277b6a076d34,W2_0028,9.34,27.340780,5.091846,11.411577,365.0,0.4
1,interface_farthest,INTERFACE,bab384c6cea3b2009490c76f,W2_0216,8.00,476.190461,5.535699,8.728518,365.0,0.4
2,slab_nearest,SLAB,eaa77b4f070744b7840995fa,W2_0178,6.95,42.236141,42.000000,90.000000,365.0,0.4
3,slab_farthest,SLAB,5b4716623f6f6ed3059baf7e,W2_0061,6.55,453.437209,60.000000,90.000000,365.0,0.4



Java compiler output:
[no compiler messages]

Requested-period resolution:


,requested_period_s,period_resolution,lower_imt,lower_period_s,upper_imt,upper_period_s,log_period_weight
0,0.4,exact,SA0P4,0.4,SA0P4,0.4,0.0



Exact USGS calculations to execute:


,calculation_id,source_type,gmm_name,imt_name,requested_imt,interpolation_role,magnitude,r_rup_km,vs30_mps
0,interface_nearest__PGA__PGA,INTERFACE,PSBAH_20_CASCADIA_INTERFACE,PGA,PGA,exact,9.34,27.340780,365.0
1,interface_nearest__SA0p4s__SA0P4,INTERFACE,PSBAH_20_CASCADIA_INTERFACE,SA0P4,SA(0.4s),exact,9.34,27.340780,365.0
2,interface_farthest__PGA__PGA,INTERFACE,PSBAH_20_CASCADIA_INTERFACE,PGA,PGA,exact,8.00,476.190461,365.0
3,interface_farthest__SA0p4s__SA0P4,INTERFACE,PSBAH_20_CASCADIA_INTERFACE,SA0P4,SA(0.4s),exact,8.00,476.190461,365.0
4,slab_nearest__PGA__PGA,SLAB,PSBAH_20_CASCADIA_SLAB,PGA,PGA,exact,6.95,42.236141,365.0
5,slab_nearest__SA0p4s__SA0P4,SLAB,PSBAH_20_CASCADIA_SLAB,SA0P4,SA(0.4s),exact,6.95,42.236141,365.0
6,slab_farthest__PGA__PGA,SLAB,PSBAH_20_CASCADIA_SLAB,PGA,PGA,exact,6.55,453.437209,365.0
7,slab_farthest__SA0p4s__SA0P4,SLAB,PSBAH_20_CASCADIA_SLAB,SA0P4,SA(0.4s),exact,6.55,453.437209,365.0



Java runtime output:
CONTROLLED_PARKER_GMM_COMPLETE

Parker variability coefficients required by Seaside W2:
coefficient_label  period_s  tau_ln  phi21_variance  phi22_variance  phi2v_variance_adjustment
              PGA       0.0   0.480           0.396           0.565                      -0.18
            SA0P4       0.4   0.492           0.433           0.556                      -0.19
  Materialized copy: C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_controlled_parker_gmm_work\resources\parker_published_variability_coefficients_pga_sa0p4.csv
  Acceptance rule: reconstructed sigma must match the pinned USGS runtime within 5e-12.

CELL 16 VALIDATION


,category,check,observed,expected,passes,note
0,Cell 15 dependency,All Cell 15 final checks passed,15,15,True,
1,Coverage,Authoritative distance row count,1845690,1845690,True,
2,Coverage,Four source-distance controls selected,4,4,True,
3,Coverage,Control source types,"[INTERFACE, SLAB]","[INTERFACE, SLAB]",True,
4,Coverage,Control distance roles,"[farthest, nearest]","[farthest, nearest]",True,
5,Inputs,Vs30 values within USGS range,365 to 365,150 to 2000 m/s,True,portfolio column: vs30_mps
6,Inputs,Requested periods positive,0.4 to 0.4,> 0 s,True,portfolio column: Period_for_spectral_accelera...
7,Inputs,Each requested period resolved to exact or bra...,1,1,True,
8,USGS GMM,No Java calculation errors,0,0,True,
9,Epistemic branches,Every exact calculation returned three branches,8,8,True,



CELL 16 CONTROLLED PARKER GMM VALIDATION COMPLETE

Controlled cases:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_cell_16_control_selection.csv

Controlled requested inputs:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_cell_16_controlled_gmm_inputs.csv

Supported IMTs:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_parker_supported_imts.csv

Exact USGS branch outputs:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_4_controlled_parker_gmm\controlled_parker_exact_outputs.csv

Requested-period branch outputs:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\notebook_4_controlled_parker_gmm\controlled_parker_requested_outputs.csv

Parker variability coefficients:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_parker_variabilit

In [33]:
# ============================================================================
# CELL 17
# Bulk source-appropriate Parker GMM calculation for all authoritative
# Seaside W2 rupture-site pairs.
#
# Purpose
# -------
# 1. Consume the validated Cell 15 authoritative rupture-to-site distances.
# 2. Require the accepted Cell 16 controlled Parker validation.
# 3. Evaluate the pinned USGS Parker Cascadia interface or slab GMM for every
#    one of the 3,927 ruptures at all 470 Seaside W2 sites.
# 4. Calculate exact PGA and SA(0.4 s) outputs, preserving epi-lo, epi-off,
#    and epi-hi branches in a compact wide table.
# 5. Add the independently validated Parker tau/phi decomposition used later
#    for shared between-event and conditionally independent within-event
#    residual simulation.
# 6. Use rupture chunking, completion markers, content hashes, restart
#    capability, chunk-level validation, final concatenation, and independent
#    full-dataset validation.
#
# This cell calculates deterministic GMM parameters only. It does not yet draw
# between-event or within-event residuals. The epi-off branch is retained as
# the primary internship-ready baseline, while epi-lo and epi-hi remain
# available for epistemic sensitivity analysis.
# ============================================================================

from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path

import gzip
import hashlib
import io
import json
import math
import os
import re
import shutil
import subprocess
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------------------

PIPELINE_VERSION = "notebook4_cell17_bulk_parker_gmm_v2"
EXPECTED_RUPTURES = 3_927
EXPECTED_SITES = 470
EXPECTED_PAIR_ROWS = EXPECTED_RUPTURES * EXPECTED_SITES
GMM_RUPTURES_PER_CHUNK = int(
    os.environ.get("NOTEBOOK4_GMM_RUPTURES_PER_CHUNK", "250")
)
FINAL_READ_CHUNK_SIZE = int(
    os.environ.get("NOTEBOOK4_GMM_FINAL_READ_CHUNK_SIZE", "150000")
)
JAVA_MAX_HEAP = os.environ.get("NOTEBOOK4_JAVA_XMX", "3g").strip()
VERIFY_EXISTING_HASHES = True

SEASIDE_W2_PERIOD_S = 0.40
DEFAULT_VS30_MPS = 365.0
DEFAULT_VS30_INFERRED = True
PRIMARY_BASELINE_BRANCH = "epi-off"
INTERFACE_GMM_NAME = str(
    globals().get("PARKER_INTERFACE_GMM", "PSBAH_20_CASCADIA_INTERFACE")
)
SLAB_GMM_NAME = str(
    globals().get("PARKER_SLAB_GMM", "PSBAH_20_CASCADIA_SLAB")
)

PERIOD_MATCH_ATOL = 1.0e-10
WEIGHT_SUM_ATOL = 1.0e-12
NUMERIC_MATCH_ATOL = 1.0e-10
SIGMA_MATCH_ATOL = 5.0e-12
MEDIAN_MATCH_ATOL = 1.0e-12

if GMM_RUPTURES_PER_CHUNK <= 0:
    raise ValueError("NOTEBOOK4_GMM_RUPTURES_PER_CHUNK must be positive.")
if GMM_RUPTURES_PER_CHUNK > 500:
    raise ValueError(
        "Use at most 500 ruptures per GMM chunk to keep memory use controlled."
    )

required_variables = [
    "DATA_DIR",
    "METADATA_DIR",
    "PARKER_INSPECTION_DIR",
    "GMM_READY_PORTFOLIO_PATH",
    "runtime_paths",
]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Cell 17 requires variables created by earlier Notebook 4 cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

DATA_DIR = Path(DATA_DIR).resolve()
METADATA_DIR = Path(METADATA_DIR).resolve()
PARKER_INSPECTION_DIR = Path(PARKER_INSPECTION_DIR).resolve()
GMM_READY_PORTFOLIO_PATH = Path(GMM_READY_PORTFOLIO_PATH).resolve()
runtime_paths = [Path(path).resolve() for path in runtime_paths]


# ----------------------------------------------------------------------------
# 2. Input and output paths
# ----------------------------------------------------------------------------

CELL15_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_authoritative_usgs_distances"
CELL15_DISTANCE_PATH = Path(
    globals().get(
        "FINAL_DISTANCE_PATH",
        CELL15_OUTPUT_DIR / "usgs_rupture_site_distances.csv.gz",
    )
).resolve()
CELL15_AUDIT_PATH = Path(
    globals().get(
        "FINAL_AUDIT_PATH",
        METADATA_DIR / "notebook_4_authoritative_rupture_retrieval_audit.csv.gz",
    )
).resolve()
CELL15_VALIDATION_PATH = Path(
    globals().get(
        "FINAL_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_authoritative_distance_validation.csv",
    )
).resolve()
CELL15_MANIFEST_PATH = Path(
    globals().get(
        "MANIFEST_PATH",
        METADATA_DIR / "notebook_4_rupture_site_distance_chunk_manifest.csv",
    )
).resolve()

CELL16_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_controlled_parker_gmm"
CELL16_SUMMARY_PATH = METADATA_DIR / "notebook_4_cell_16_controlled_gmm_summary.json"
CELL16_VALIDATION_PATH = METADATA_DIR / "notebook_4_cell_16_controlled_gmm_validation.csv"
CELL16_CONTROL_SELECTION_PATH = METADATA_DIR / "notebook_4_cell_16_control_selection.csv"
CELL16_EXACT_OUTPUT_PATH = CELL16_OUTPUT_DIR / "controlled_parker_exact_outputs.csv"
PARKER_COEFFICIENT_PATH = METADATA_DIR / "notebook_4_parker_variability_coefficients.csv"
PARKER_COEFFICIENT_PROVENANCE_PATH = METADATA_DIR / "notebook_4_parker_coefficient_provenance.json"

CELL17_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_bulk_parker_gmm"
CELL17_WORK_DIR = METADATA_DIR / "notebook_4_bulk_parker_gmm_work"
CELL17_CHUNK_DIR = CELL17_WORK_DIR / "chunks"
CELL17_TEMP_DIR = CELL17_WORK_DIR / "temporary"
CELL17_LOG_DIR = CELL17_WORK_DIR / "logs"
CELL17_MARKER_DIR = CELL17_WORK_DIR / "markers"
CELL17_VALIDATION_DIR = CELL17_WORK_DIR / "chunk_validation"
CELL17_CHUNK_INDEX_DIR = CELL17_WORK_DIR / "chunk_rupture_indices"
CELL17_JAVA_DIR = CELL17_WORK_DIR / "java"
CELL17_CLASSES_DIR = CELL17_JAVA_DIR / "classes"

CELL17_MANIFEST_PATH = METADATA_DIR / "notebook_4_bulk_parker_gmm_chunk_manifest.csv"
CELL17_FINAL_OUTPUT_PATH = CELL17_OUTPUT_DIR / "bulk_parker_gmm_parameters.csv.gz"
CELL17_FINAL_VALIDATION_PATH = METADATA_DIR / "notebook_4_bulk_parker_gmm_validation.csv"
CELL17_FINAL_SUMMARY_PATH = METADATA_DIR / "notebook_4_bulk_parker_gmm_summary.json"
CELL17_JAVA_SOURCE_PATH = CELL17_JAVA_DIR / "BulkParkerGmm.java"
CELL17_JAVAC_ARGS_PATH = PARKER_INSPECTION_DIR / "bulk_parker_gmm_javac_arguments.txt"
CELL17_COMPILE_LOG_PATH = CELL17_LOG_DIR / "bulk_parker_gmm_compile.log"

for directory in [
    CELL17_OUTPUT_DIR,
    CELL17_WORK_DIR,
    CELL17_CHUNK_DIR,
    CELL17_TEMP_DIR,
    CELL17_LOG_DIR,
    CELL17_MARKER_DIR,
    CELL17_VALIDATION_DIR,
    CELL17_CHUNK_INDEX_DIR,
    CELL17_JAVA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

required_paths = {
    "Cell 15 final distance matrix": CELL15_DISTANCE_PATH,
    "Cell 15 rupture audit": CELL15_AUDIT_PATH,
    "Cell 15 final validation": CELL15_VALIDATION_PATH,
    "Cell 16 summary": CELL16_SUMMARY_PATH,
    "Cell 16 validation": CELL16_VALIDATION_PATH,
    "Cell 16 control selection": CELL16_CONTROL_SELECTION_PATH,
    "Cell 16 exact GMM outputs": CELL16_EXACT_OUTPUT_PATH,
    "Parker variability coefficients": PARKER_COEFFICIENT_PATH,
    "Parker coefficient provenance": PARKER_COEFFICIENT_PROVENANCE_PATH,
    "Seaside W2 GMM-ready portfolio": GMM_READY_PORTFOLIO_PATH,
}
missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]
missing_runtime_paths = [path for path in runtime_paths if not path.exists()]
if missing_paths or missing_runtime_paths:
    sections: list[str] = []
    if missing_paths:
        sections.append(
            "Missing required files:\n"
            + "\n".join(f"  - {item}" for item in missing_paths)
        )
    if missing_runtime_paths:
        sections.append(
            "Missing Java classpath entries:\n"
            + "\n".join(f"  - {path}" for path in missing_runtime_paths)
        )
    raise FileNotFoundError("\n\n".join(sections))


# ----------------------------------------------------------------------------
# 3. Helpers
# ----------------------------------------------------------------------------

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            block = file_handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def java_arg(value: object) -> str:
    text = str(value).replace("\\", "\\\\").replace('"', '\\"')
    return f'"{text}"'


def normalize_column_name(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())


def first_matching_column(
    frame: pd.DataFrame,
    candidates: Iterable[str],
) -> str | None:
    lookup = {
        normalize_column_name(column): str(column)
        for column in frame.columns
    }
    for candidate in candidates:
        key = normalize_column_name(candidate)
        if key in lookup:
            return lookup[key]
    return None


def numeric_series(
    frame: pd.DataFrame,
    column: str,
    *,
    label: str,
) -> pd.Series:
    values = pd.to_numeric(frame[column], errors="coerce")
    if not np.isfinite(values).all():
        bad = frame.loc[~np.isfinite(values), [column]].head(20)
        raise RuntimeError(
            f"{label} column {column!r} contains missing or non-finite values.\n\n"
            + bad.to_string(index=False)
        )
    return values.astype(float)


def choose_site_input_column(
    portfolio: pd.DataFrame,
    candidate_names: Iterable[str],
    default_value: float,
    label: str,
) -> tuple[pd.Series, str]:
    column = first_matching_column(portfolio, candidate_names)
    if column is None:
        values = pd.Series(
            np.full(len(portfolio), default_value, dtype=float),
            index=portfolio.index,
        )
        provenance = f"constant fallback: {default_value:g}"
    else:
        values = numeric_series(portfolio, column, label=label)
        provenance = f"portfolio column: {column}"
    return values, provenance


def parse_boolean_series(series: pd.Series, *, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)
    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )
    if parsed.isna().any():
        bad = series.loc[parsed.isna()].head(20)
        raise RuntimeError(
            f"Could not parse all {label} values as booleans:\n{bad.to_string()}"
        )
    return parsed.astype(bool)


def add_validation(
    records: list[dict[str, object]],
    category: str,
    check: str,
    observed: object,
    expected: object,
    passes: bool,
    note: str = "",
) -> None:
    records.append(
        {
            "category": category,
            "check": check,
            "observed": observed,
            "expected": expected,
            "passes": bool(passes),
            "note": note,
        }
    )


def require_all_validation_checks(path: Path, label: str) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    pass_column = first_matching_column(frame, ["passes", "pass"])
    if pass_column is None:
        raise RuntimeError(f"{label} has no pass column: {path}")
    passes = (
        frame[pass_column]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": True, "false": False, "1": True, "0": False})
    )
    if passes.isna().any() or not passes.all():
        raise RuntimeError(
            f"{label} is missing checks or contains failures:\n\n"
            + frame.to_string(index=False)
        )
    return frame


def deterministic_gzip_csv(
    frame: pd.DataFrame,
    path: Path,
    *,
    columns: list[str] | None = None,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    compression = {"method": "gzip", "compresslevel": 6, "mtime": 0}
    frame.to_csv(
        path,
        index=False,
        columns=columns,
        na_rep="",
        lineterminator="\n",
        float_format="%.17g",
        compression=compression,
    )


def vectorized_parker_phi_variance(
    r_rup_km: np.ndarray,
    vs30_mps: np.ndarray,
    *,
    phi21: float,
    phi22: float,
    phi2v: float,
) -> np.ndarray:
    r = np.asarray(r_rup_km, dtype=float)
    v = np.asarray(vs30_mps, dtype=float)
    if not np.isfinite(r).all() or not np.isfinite(v).all():
        raise RuntimeError("Rrup or Vs30 contains non-finite values.")

    r1, r2 = 200.0, 500.0
    v1, v2 = 200.0, 500.0

    variance = np.empty_like(r, dtype=float)
    near = r <= r1
    far = r >= r2
    middle = ~(near | far)
    variance[near] = phi21
    variance[far] = phi22
    variance[middle] = (
        (phi22 - phi21)
        / (math.log(r2) - math.log(r1))
        * (np.log(r[middle]) - math.log(r1))
        + phi21
    )

    clamped_r = np.clip(r, r1, r2)
    distance_taper = np.log(r2 / clamped_r) / math.log(r2 / r1)

    velocity_taper = np.empty_like(v, dtype=float)
    velocity_taper[v <= v1] = 1.0
    velocity_taper[v >= v2] = 0.0
    velocity_middle = (v > v1) & (v < v2)
    velocity_taper[velocity_middle] = (
        np.log(v2 / v[velocity_middle]) / math.log(v2 / v1)
    )

    variance = variance + phi2v * velocity_taper * distance_taper
    if not np.isfinite(variance).all() or np.any(variance <= 0.0):
        bad = np.flatnonzero((~np.isfinite(variance)) | (variance <= 0.0))[:20]
        raise RuntimeError(
            "Parker within-event variance became invalid at rows: "
            + ", ".join(map(str, bad))
        )
    return variance


def normalize_path_from_manifest(value: object) -> Path:
    return Path(str(value)).expanduser().resolve()


# ----------------------------------------------------------------------------
# 4. Require accepted Cell 15 and Cell 16 dependencies
# ----------------------------------------------------------------------------

cell15_validation = require_all_validation_checks(
    CELL15_VALIDATION_PATH,
    "Cell 15 final validation",
)
cell16_validation = require_all_validation_checks(
    CELL16_VALIDATION_PATH,
    "Cell 16 controlled GMM validation",
)

cell16_summary = json.loads(CELL16_SUMMARY_PATH.read_text(encoding="utf-8"))
if not bool(cell16_summary.get("all_checks_passed", False)):
    raise RuntimeError("Cell 16 summary does not report all checks passed.")
# Cell 16 v4 stores the validated portfolio period under ``default_period_s``.
# Earlier development versions used ``seaside_w2_period_s``. Resolve the
# period from the accepted summary without weakening the 0.40 s requirement.
cell16_period_candidates = []
for period_key in ("seaside_w2_period_s", "default_period_s"):
    raw_period = cell16_summary.get(period_key)
    if raw_period is not None:
        try:
            period_value = float(raw_period)
        except (TypeError, ValueError):
            continue
        if math.isfinite(period_value):
            cell16_period_candidates.append((period_key, period_value))

# As a final schema-compatible fallback, inspect the controlled cases. This is
# still strict: every reported requested period must be finite and identical.
if not cell16_period_candidates:
    controlled_periods = []
    for case in cell16_summary.get("control_cases", []):
        try:
            period_value = float(case.get("requested_period_s", np.nan))
        except (TypeError, ValueError, AttributeError):
            continue
        if math.isfinite(period_value):
            controlled_periods.append(period_value)
    if controlled_periods and all(
        math.isclose(
            value,
            controlled_periods[0],
            rel_tol=0.0,
            abs_tol=PERIOD_MATCH_ATOL,
        )
        for value in controlled_periods
    ):
        cell16_period_candidates.append(
            ("control_cases.requested_period_s", controlled_periods[0])
        )

if not cell16_period_candidates:
    raise RuntimeError(
        "Cell 16 summary does not contain a finite validated portfolio period "
        "under seaside_w2_period_s, default_period_s, or the controlled cases."
    )

cell16_period_sources = {key: value for key, value in cell16_period_candidates}
if not all(
    math.isclose(
        value,
        SEASIDE_W2_PERIOD_S,
        rel_tol=0.0,
        abs_tol=PERIOD_MATCH_ATOL,
    )
    for value in cell16_period_sources.values()
):
    raise RuntimeError(
        "Cell 16 did not validate the Seaside W2 period of 0.40 s. "
        f"Reported period values: {cell16_period_sources}"
    )

print(
    "Cell 16 period prerequisite passed: "
    + ", ".join(
        f"{key}={value:.2f} s"
        for key, value in cell16_period_sources.items()
    )
)
if str(cell16_summary.get("interface_gmm")) != INTERFACE_GMM_NAME:
    raise RuntimeError("Cell 16 interface GMM differs from Cell 17 configuration.")
if str(cell16_summary.get("slab_gmm")) != SLAB_GMM_NAME:
    raise RuntimeError("Cell 16 slab GMM differs from Cell 17 configuration.")
if float(cell16_summary.get("maximum_sigma_reconstruction_difference", np.inf)) > SIGMA_MATCH_ATOL:
    raise RuntimeError("Cell 16 did not validate the Parker sigma decomposition.")


# ----------------------------------------------------------------------------
# 5. Load and validate the Parker variability coefficients
# ----------------------------------------------------------------------------

coefficients = pd.read_csv(PARKER_COEFFICIENT_PATH, low_memory=False)
required_coefficient_columns = [
    "coefficient_label",
    "period_s",
    "tau_ln",
    "phi21_variance",
    "phi22_variance",
    "phi2v_variance_adjustment",
]
missing_coefficient_columns = [
    column for column in required_coefficient_columns if column not in coefficients
]
if missing_coefficient_columns:
    raise RuntimeError(
        "Parker coefficient inventory is missing columns:\n"
        + "\n".join(f"  - {column}" for column in missing_coefficient_columns)
    )

for column in required_coefficient_columns[1:]:
    coefficients[column] = pd.to_numeric(coefficients[column], errors="coerce")
if coefficients[required_coefficient_columns[1:]].isna().any().any():
    raise RuntimeError("Parker coefficient inventory contains nonnumeric values.")

coefficient_labels = set(coefficients["coefficient_label"].astype(str))
if coefficient_labels != {"PGA", "SA0P4"} or len(coefficients) != 2:
    raise RuntimeError(
        "Cell 17 requires exactly the validated PGA and SA0P4 coefficient rows."
    )

coefficient_lookup = coefficients.set_index("coefficient_label")
if not math.isclose(
    float(coefficient_lookup.loc["SA0P4", "period_s"]),
    SEASIDE_W2_PERIOD_S,
    rel_tol=0.0,
    abs_tol=PERIOD_MATCH_ATOL,
):
    raise RuntimeError("The SA0P4 coefficient row does not correspond to 0.40 s.")

coefficient_hash = sha256_file(PARKER_COEFFICIENT_PATH)
summary_coefficient_hash = str(cell16_summary.get("coefficient_sha256", ""))
if summary_coefficient_hash and coefficient_hash != summary_coefficient_hash:
    raise RuntimeError(
        "The Parker coefficient file changed after Cell 16 validation."
    )


# ----------------------------------------------------------------------------
# 6. Load rupture metadata and construct authoritative GMM rupture inputs
# ----------------------------------------------------------------------------

rupture_audit = pd.read_csv(
    CELL15_AUDIT_PATH,
    dtype={
        "event_id": str,
        "rupture_id": str,
        "retrieved_source_type": str,
        "surface_class": str,
        "retrieval_error": str,
    },
    low_memory=False,
)
rupture_audit["retrieval_error"] = (
    rupture_audit["retrieval_error"].fillna("").astype(str)
)
required_audit_columns = [
    "rupture_ordinal",
    "event_id",
    "rupture_id",
    "retrieved_source_type",
    "retrieved_magnitude",
    "retrieved_raw_rate",
    "retrieved_rake_deg",
    "surface_depth_km",
    "surface_width_km",
    "surface_dip_deg",
    "surface_centroid_depth_km",
    "surface_class",
    "retrieval_error",
]
missing_audit_columns = [
    column for column in required_audit_columns if column not in rupture_audit
]
if missing_audit_columns:
    raise RuntimeError(
        "Cell 15 rupture audit is missing required columns:\n"
        + "\n".join(f"  - {column}" for column in missing_audit_columns)
    )
if len(rupture_audit) != EXPECTED_RUPTURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RUPTURES:,} rupture audit rows; "
        f"observed {len(rupture_audit):,}."
    )
if rupture_audit["rupture_id"].duplicated().any():
    raise RuntimeError("Cell 15 rupture audit contains duplicate rupture IDs.")
if rupture_audit["retrieval_error"].ne("").any():
    raise RuntimeError("Cell 15 rupture audit contains retrieval errors.")

numeric_audit_columns = [
    "rupture_ordinal",
    "retrieved_magnitude",
    "retrieved_raw_rate",
    "retrieved_rake_deg",
    "surface_depth_km",
    "surface_width_km",
    "surface_dip_deg",
    "surface_centroid_depth_km",
]
for column in numeric_audit_columns:
    rupture_audit[column] = pd.to_numeric(
        rupture_audit[column], errors="coerce"
    )
for column in numeric_audit_columns[:-1]:
    if not np.isfinite(rupture_audit[column]).all():
        raise RuntimeError(f"Rupture audit field {column!r} is not finite.")

rupture_audit["rupture_ordinal"] = rupture_audit["rupture_ordinal"].astype(int)
rupture_audit = rupture_audit.sort_values("rupture_ordinal").reset_index(drop=True)
if not np.array_equal(
    rupture_audit["rupture_ordinal"].to_numpy(dtype=int),
    np.arange(EXPECTED_RUPTURES, dtype=int),
):
    raise RuntimeError("Rupture ordinals must be contiguous from 0 to 3,926.")

rupture_audit["source_type"] = (
    rupture_audit["retrieved_source_type"].astype(str).str.upper()
)
if set(rupture_audit["source_type"].unique()) != {"INTERFACE", "SLAB"}:
    raise RuntimeError(
        "Unexpected source types: "
        f"{sorted(rupture_audit['source_type'].unique())}"
    )
rupture_audit["gmm_name"] = np.where(
    rupture_audit["source_type"].eq("INTERFACE"),
    INTERFACE_GMM_NAME,
    SLAB_GMM_NAME,
)

centroid_depth = rupture_audit["surface_centroid_depth_km"].to_numpy(dtype=float)
geometric_midpoint = (
    rupture_audit["surface_depth_km"].to_numpy(dtype=float)
    + 0.5
    * rupture_audit["surface_width_km"].to_numpy(dtype=float)
    * np.sin(np.radians(rupture_audit["surface_dip_deg"].to_numpy(dtype=float)))
)
rupture_audit["z_hyp_input_km"] = np.where(
    np.isfinite(centroid_depth), centroid_depth, geometric_midpoint
)
rupture_audit["z_hyp_input_provenance"] = np.where(
    np.isfinite(centroid_depth),
    "USGS rupture-surface centroid depth",
    "geometric midpoint from zTor, width, and dip",
)
if not np.isfinite(rupture_audit["z_hyp_input_km"]).all():
    raise RuntimeError("Derived zHyp inputs contain non-finite values.")

rupture_metadata = rupture_audit[
    [
        "rupture_ordinal",
        "event_id",
        "rupture_id",
        "source_type",
        "gmm_name",
        "surface_class",
        "retrieved_magnitude",
        "retrieved_raw_rate",
        "surface_dip_deg",
        "surface_width_km",
        "surface_depth_km",
        "z_hyp_input_km",
        "z_hyp_input_provenance",
        "retrieved_rake_deg",
    ]
].rename(
    columns={
        "retrieved_magnitude": "magnitude",
        "retrieved_raw_rate": "raw_annual_rate",
        "surface_dip_deg": "dip_deg",
        "surface_width_km": "width_km",
        "surface_depth_km": "z_tor_km",
        "retrieved_rake_deg": "rake_deg",
    }
)
rupture_lookup = rupture_metadata.set_index("rupture_id", drop=False)


# ----------------------------------------------------------------------------
# 7. Load and validate the Seaside W2 site inputs
# ----------------------------------------------------------------------------

portfolio = pd.read_csv(GMM_READY_PORTFOLIO_PATH, low_memory=False)
site_id_column = first_matching_column(portfolio, ["site_id", "siteid"])
longitude_column = first_matching_column(portfolio, ["longitude", "lon", "x"])
latitude_column = first_matching_column(portfolio, ["latitude", "lat", "y"])
if site_id_column is None or longitude_column is None or latitude_column is None:
    raise RuntimeError(
        "The Seaside W2 portfolio must contain site ID, longitude, and latitude."
    )

portfolio = portfolio.copy()
portfolio["site_id"] = (
    portfolio[site_id_column].astype("string").fillna("").astype(str)
)
portfolio["site_longitude"] = numeric_series(
    portfolio, longitude_column, label="longitude"
)
portfolio["site_latitude"] = numeric_series(
    portfolio, latitude_column, label="latitude"
)
if len(portfolio) != EXPECTED_SITES:
    raise RuntimeError(
        f"Expected {EXPECTED_SITES:,} Seaside W2 sites; observed {len(portfolio):,}."
    )
if portfolio["site_id"].eq("").any() or portfolio["site_id"].duplicated().any():
    raise RuntimeError("Seaside W2 site IDs are blank or duplicated.")

portfolio["vs30_mps"], vs30_provenance = choose_site_input_column(
    portfolio,
    ["vs30_mps", "vs30", "v_s30", "site_vs30", "site_vs30_mps"],
    DEFAULT_VS30_MPS,
    "Vs30",
)
portfolio["period_s"], period_provenance = choose_site_input_column(
    portfolio,
    [
        "period_s",
        "period",
        "fundamental_period_s",
        "fundamental_period",
        "period_for_spectral_acceleration",
        "building_period_s",
    ],
    SEASIDE_W2_PERIOD_S,
    "structural period",
)
if not portfolio["vs30_mps"].between(150.0, 2000.0).all():
    raise RuntimeError("Seaside W2 Vs30 values fall outside [150, 2000] m/s.")
if not np.isclose(
    portfolio["period_s"].to_numpy(dtype=float),
    SEASIDE_W2_PERIOD_S,
    rtol=0.0,
    atol=PERIOD_MATCH_ATOL,
).all():
    observed = sorted(portfolio["period_s"].unique().tolist())
    raise RuntimeError(
        "Cell 17 requires the Seaside W2 period T = 0.40 s. "
        f"Observed: {observed}"
    )

vs30_inferred_column = first_matching_column(
    portfolio, ["vs30_inferred", "vs_inferred", "vs30_is_inferred"]
)
if vs30_inferred_column is None:
    portfolio["vs30_inferred"] = DEFAULT_VS30_INFERRED
    vs30_inferred_provenance = f"constant fallback: {DEFAULT_VS30_INFERRED}"
else:
    portfolio["vs30_inferred"] = parse_boolean_series(
        portfolio[vs30_inferred_column],
        label=vs30_inferred_column,
    )
    vs30_inferred_provenance = f"portfolio column: {vs30_inferred_column}"

for output_column, candidates in [
    ("z1p0_km", ["z1p0_km", "z1p0", "z1_0_km", "z1"]),
    ("z2p5_km", ["z2p5_km", "z2p5", "z2_5_km", "z2"]),
    ("zsed_km", ["zsed_km", "zsed", "sediment_thickness_km"]),
]:
    source_column = first_matching_column(portfolio, candidates)
    if source_column is None:
        portfolio[output_column] = np.nan
    else:
        portfolio[output_column] = pd.to_numeric(
            portfolio[source_column], errors="coerce"
        )
        invalid = portfolio[output_column].notna() & ~np.isfinite(
            portfolio[output_column]
        )
        if invalid.any():
            raise RuntimeError(f"Invalid basin values in {source_column!r}.")

site_metadata = portfolio[
    [
        "site_id",
        "site_longitude",
        "site_latitude",
        "vs30_mps",
        "vs30_inferred",
        "z1p0_km",
        "z2p5_km",
        "zsed_km",
        "period_s",
    ]
].copy()
site_lookup = site_metadata.set_index("site_id", drop=False)


# ----------------------------------------------------------------------------
# 8. Source hashes and deterministic GMM chunks
# ----------------------------------------------------------------------------

source_hashes = {
    "cell15_distance_sha256": sha256_file(CELL15_DISTANCE_PATH),
    "cell15_audit_sha256": sha256_file(CELL15_AUDIT_PATH),
    "cell15_validation_sha256": sha256_file(CELL15_VALIDATION_PATH),
    "cell16_summary_sha256": sha256_file(CELL16_SUMMARY_PATH),
    "cell16_validation_sha256": sha256_file(CELL16_VALIDATION_PATH),
    "cell16_control_selection_sha256": sha256_file(CELL16_CONTROL_SELECTION_PATH),
    "cell16_exact_output_sha256": sha256_file(CELL16_EXACT_OUTPUT_PATH),
    "parker_coefficients_sha256": coefficient_hash,
    "portfolio_sha256": sha256_file(GMM_READY_PORTFOLIO_PATH),
}

if str(cell16_summary.get("cell15_distance_sha256", "")) not in {
    "",
    source_hashes["cell15_distance_sha256"],
}:
    raise RuntimeError("The Cell 15 distance matrix changed after Cell 16.")
if str(cell16_summary.get("cell15_audit_sha256", "")) not in {
    "",
    source_hashes["cell15_audit_sha256"],
}:
    raise RuntimeError("The Cell 15 rupture audit changed after Cell 16.")
if str(cell16_summary.get("portfolio_sha256", "")) not in {
    "",
    source_hashes["portfolio_sha256"],
}:
    raise RuntimeError("The Seaside W2 portfolio changed after Cell 16.")

runtime_hashes = {
    str(path): sha256_file(path) if path.is_file() else "directory"
    for path in runtime_paths
}

chunk_count = math.ceil(EXPECTED_RUPTURES / GMM_RUPTURES_PER_CHUNK)
chunk_records: list[dict[str, object]] = []
for chunk_number in range(chunk_count):
    start = chunk_number * GMM_RUPTURES_PER_CHUNK
    stop = min(start + GMM_RUPTURES_PER_CHUNK, EXPECTED_RUPTURES)
    chunk_id = f"{chunk_number:04d}"
    chunk_ruptures = rupture_metadata.iloc[start:stop].copy()
    chunk_index_path = CELL17_CHUNK_INDEX_DIR / f"gmm_chunk_{chunk_id}_ruptures.csv"
    chunk_ruptures[
        ["rupture_ordinal", "event_id", "rupture_id", "source_type", "gmm_name"]
    ].to_csv(
        chunk_index_path,
        index=False,
        lineterminator="\n",
    )
    chunk_records.append(
        {
            "chunk_number": chunk_number,
            "chunk_id": chunk_id,
            "start": start,
            "stop": stop,
            "rupture_count": stop - start,
            "site_count": EXPECTED_SITES,
            "expected_rows": (stop - start) * EXPECTED_SITES,
            "chunk_index_path": str(chunk_index_path),
            "chunk_index_hash": sha256_file(chunk_index_path),
            "temporary_input": str(CELL17_TEMP_DIR / f"gmm_input_chunk_{chunk_id}.tsv"),
            "raw_output": str(CELL17_TEMP_DIR / f"gmm_raw_chunk_{chunk_id}.csv.gz"),
            "output": str(CELL17_CHUNK_DIR / f"bulk_gmm_chunk_{chunk_id}.csv.gz"),
            "log": str(CELL17_LOG_DIR / f"bulk_gmm_chunk_{chunk_id}.log"),
            "java_args": str(PARKER_INSPECTION_DIR / f"bulk_gmm_chunk_{chunk_id}_java_arguments.txt"),
            "marker": str(CELL17_MARKER_DIR / f"bulk_gmm_chunk_{chunk_id}.complete.json"),
            "validation": str(CELL17_VALIDATION_DIR / f"bulk_gmm_chunk_{chunk_id}_validation.csv"),
            "status": "pending",
            "completed_at_utc": "",
        }
    )


def save_manifest() -> None:
    pd.DataFrame(chunk_records).to_csv(
        CELL17_MANIFEST_PATH,
        index=False,
        lineterminator="\n",
    )


save_manifest()


# ----------------------------------------------------------------------------
# 9. Java bulk GMM helper
# ----------------------------------------------------------------------------

java_source = r'''
import gov.usgs.earthquake.nshmp.gmm.Gmm;
import gov.usgs.earthquake.nshmp.gmm.GmmInput;
import gov.usgs.earthquake.nshmp.gmm.GroundMotion;
import gov.usgs.earthquake.nshmp.gmm.GroundMotionModel;
import gov.usgs.earthquake.nshmp.gmm.Imt;
import gov.usgs.earthquake.nshmp.tree.Branch;
import gov.usgs.earthquake.nshmp.tree.LogicTree;

import java.io.BufferedReader;
import java.io.BufferedWriter;
import java.io.IOException;
import java.io.OutputStream;
import java.io.OutputStreamWriter;
import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;
import java.util.ArrayList;
import java.util.HashMap;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Locale;
import java.util.Map;
import java.util.zip.GZIPOutputStream;

public final class BulkParkerGmm {

  private static final List<String> BRANCH_IDS =
      List.of("epi-lo", "epi-off", "epi-hi");

  private BulkParkerGmm() {}

  public static void main(String[] args) throws Exception {
    if (args.length != 4) {
      throw new IllegalArgumentException(
          "Expected interface GMM name, slab GMM name, input TSV, and output CSV.gz.");
    }

    String interfaceGmmName = args[0];
    String slabGmmName = args[1];
    Path inputPath = Path.of(args[2]).toAbsolutePath().normalize();
    Path outputPath = Path.of(args[3]).toAbsolutePath().normalize();

    Gmm interfaceGmm = Gmm.valueOf(interfaceGmmName);
    Gmm slabGmm = Gmm.valueOf(slabGmmName);

    GroundMotionModel interfacePga = interfaceGmm.instance(Imt.PGA);
    GroundMotionModel interfaceSa04 = interfaceGmm.instance(Imt.SA0P4);
    GroundMotionModel slabPga = slabGmm.instance(Imt.PGA);
    GroundMotionModel slabSa04 = slabGmm.instance(Imt.SA0P4);

    long rows = calculate(
        inputPath,
        outputPath,
        interfaceGmmName,
        slabGmmName,
        interfacePga,
        interfaceSa04,
        slabPga,
        slabSa04);

    System.out.println("BULK_PARKER_GMM_CHUNK_COMPLETE");
    System.out.println("rows=" + rows);
  }

  private static long calculate(
      Path inputPath,
      Path outputPath,
      String interfaceGmmName,
      String slabGmmName,
      GroundMotionModel interfacePga,
      GroundMotionModel interfaceSa04,
      GroundMotionModel slabPga,
      GroundMotionModel slabSa04) throws IOException {

    if (outputPath.getParent() != null) {
      Files.createDirectories(outputPath.getParent());
    }

    long rowCount = 0L;
    try (
        BufferedReader reader = Files.newBufferedReader(
            inputPath, StandardCharsets.UTF_8);
        OutputStream rawOutput = Files.newOutputStream(outputPath);
        GZIPOutputStream gzipOutput = new GZIPOutputStream(rawOutput, 1 << 16);
        BufferedWriter writer = new BufferedWriter(
            new OutputStreamWriter(gzipOutput, StandardCharsets.UTF_8), 1 << 16)
    ) {
      String headerLine = reader.readLine();
      if (headerLine == null) {
        throw new IllegalArgumentException("Bulk GMM input TSV is empty.");
      }
      Map<String, Integer> columns = headerMap(headerLine);
      writer.write(outputHeader());
      writer.newLine();

      String line;
      while ((line = reader.readLine()) != null) {
        if (line.isBlank()) continue;
        String[] values = line.split("\\t", -1);
        List<String> prefix = inputPrefix(values, columns);

        try {
          String sourceType = get(values, columns, "source_type").toUpperCase(Locale.US);
          String gmmName = get(values, columns, "gmm_name");

          GroundMotionModel pgaModel;
          GroundMotionModel sa04Model;
          if (sourceType.equals("INTERFACE")) {
            if (!gmmName.equals(interfaceGmmName)) {
              throw new IllegalArgumentException(
                  "Interface row routed to unexpected GMM: " + gmmName);
            }
            pgaModel = interfacePga;
            sa04Model = interfaceSa04;
          } else if (sourceType.equals("SLAB")) {
            if (!gmmName.equals(slabGmmName)) {
              throw new IllegalArgumentException(
                  "Slab row routed to unexpected GMM: " + gmmName);
            }
            pgaModel = slabPga;
            sa04Model = slabSa04;
          } else {
            throw new IllegalArgumentException("Unsupported source type: " + sourceType);
          }

          GmmInput.Builder builder = GmmInput.builder()
              .withDefaults()
              .mag(parseDouble(values, columns, "magnitude"))
              .distances(
                  parseDouble(values, columns, "r_jb_km"),
                  parseDouble(values, columns, "r_rup_km"),
                  parseDouble(values, columns, "r_x_km"))
              .dip(parseDouble(values, columns, "dip_deg"))
              .width(parseDouble(values, columns, "width_km"))
              .zTor(parseDouble(values, columns, "z_tor_km"))
              .zHyp(parseDouble(values, columns, "z_hyp_input_km"))
              .rake(parseDouble(values, columns, "rake_deg"))
              .vs30(parseDouble(values, columns, "vs30_mps"));

          double z1p0 = parseDoubleOrNaN(values, columns, "z1p0_km");
          double z2p5 = parseDoubleOrNaN(values, columns, "z2p5_km");
          if (Double.isFinite(z1p0)) builder.z1p0(z1p0);
          if (Double.isFinite(z2p5)) builder.z2p5(z2p5);

          GmmInput input = builder.build();
          Evaluation pga = evaluate(pgaModel, input);
          Evaluation sa04 = evaluate(sa04Model, input);

          List<String> output = new ArrayList<>(prefix);
          output.addAll(pga.asStrings());
          output.addAll(sa04.asStrings());
          output.add("");
          writeCsvRow(writer, output);

        } catch (Exception exception) {
          List<String> output = new ArrayList<>(prefix);
          for (int i = 0; i < 24; i++) output.add("");
          output.add(exception.getClass().getName() + ": "
              + String.valueOf(exception.getMessage()));
          writeCsvRow(writer, output);
        }
        rowCount++;
      }
    }
    return rowCount;
  }

  private static Evaluation evaluate(GroundMotionModel model, GmmInput input) {
    LogicTree<GroundMotion> tree = model.calc(input);
    Map<String, Branch<GroundMotion>> byId = new LinkedHashMap<>();
    for (Branch<GroundMotion> branch : tree) {
      if (byId.put(branch.id(), branch) != null) {
        throw new IllegalStateException("Duplicate epistemic branch: " + branch.id());
      }
    }
    if (!byId.keySet().equals(new java.util.LinkedHashSet<>(BRANCH_IDS))) {
      throw new IllegalStateException("Unexpected epistemic branches: " + byId.keySet());
    }

    double[] weights = new double[3];
    double[] means = new double[3];
    double[] medians = new double[3];
    double[] sigmas = new double[3];
    for (int i = 0; i < BRANCH_IDS.size(); i++) {
      Branch<GroundMotion> branch = byId.get(BRANCH_IDS.get(i));
      GroundMotion motion = branch.value();
      weights[i] = branch.weight();
      means[i] = motion.mean();
      medians[i] = Math.exp(motion.mean());
      sigmas[i] = motion.sigma();
    }
    return new Evaluation(weights, means, medians, sigmas);
  }

  private static String outputHeader() {
    List<String> columns = new ArrayList<>(List.of(
        "gmm_chunk_id", "rupture_ordinal", "event_id", "rupture_id",
        "site_ordinal", "site_id", "site_longitude", "site_latitude",
        "source_type", "gmm_name", "surface_class", "magnitude",
        "raw_annual_rate", "r_jb_km", "r_rup_km", "r_x_km",
        "dip_deg", "width_km", "z_tor_km", "z_hyp_input_km",
        "z_hyp_input_provenance", "rake_deg", "vs30_mps",
        "vs30_inferred", "z1p0_km", "z2p5_km", "zsed_km", "period_s"));
    for (String prefix : List.of("pga", "sa0p4")) {
      for (String branch : List.of("epi_lo", "epi_off", "epi_hi")) {
        columns.add(prefix + "_" + branch + "_weight");
      }
      for (String branch : List.of("epi_lo", "epi_off", "epi_hi")) {
        columns.add(prefix + "_" + branch + "_mean_ln_g");
      }
      for (String branch : List.of("epi_lo", "epi_off", "epi_hi")) {
        columns.add(prefix + "_" + branch + "_median_g");
      }
      for (String branch : List.of("epi_lo", "epi_off", "epi_hi")) {
        columns.add(prefix + "_" + branch + "_sigma_ln");
      }
    }
    columns.add("calculation_error");
    return String.join(",", columns);
  }

  private static List<String> inputPrefix(
      String[] values, Map<String, Integer> columns) {
    return List.of(
        get(values, columns, "gmm_chunk_id"),
        get(values, columns, "rupture_ordinal"),
        get(values, columns, "event_id"),
        get(values, columns, "rupture_id"),
        get(values, columns, "site_ordinal"),
        get(values, columns, "site_id"),
        get(values, columns, "site_longitude"),
        get(values, columns, "site_latitude"),
        get(values, columns, "source_type"),
        get(values, columns, "gmm_name"),
        get(values, columns, "surface_class"),
        get(values, columns, "magnitude"),
        get(values, columns, "raw_annual_rate"),
        get(values, columns, "r_jb_km"),
        get(values, columns, "r_rup_km"),
        get(values, columns, "r_x_km"),
        get(values, columns, "dip_deg"),
        get(values, columns, "width_km"),
        get(values, columns, "z_tor_km"),
        get(values, columns, "z_hyp_input_km"),
        get(values, columns, "z_hyp_input_provenance"),
        get(values, columns, "rake_deg"),
        get(values, columns, "vs30_mps"),
        get(values, columns, "vs30_inferred"),
        get(values, columns, "z1p0_km"),
        get(values, columns, "z2p5_km"),
        get(values, columns, "zsed_km"),
        get(values, columns, "period_s"));
  }

  private static final class Evaluation {
    final double[] weights;
    final double[] means;
    final double[] medians;
    final double[] sigmas;

    Evaluation(double[] weights, double[] means, double[] medians, double[] sigmas) {
      this.weights = weights;
      this.means = means;
      this.medians = medians;
      this.sigmas = sigmas;
    }

    List<String> asStrings() {
      List<String> values = new ArrayList<>(12);
      for (double value : weights) values.add(number(value));
      for (double value : means) values.add(number(value));
      for (double value : medians) values.add(number(value));
      for (double value : sigmas) values.add(number(value));
      return values;
    }
  }

  private static Map<String, Integer> headerMap(String headerLine) {
    String[] names = headerLine.split("\\t", -1);
    Map<String, Integer> columns = new HashMap<>();
    for (int i = 0; i < names.length; i++) columns.put(names[i], i);
    return columns;
  }

  private static String get(
      String[] values, Map<String, Integer> columns, String name) {
    Integer index = columns.get(name);
    if (index == null) {
      throw new IllegalArgumentException("Missing input column: " + name);
    }
    return index < values.length ? values[index] : "";
  }

  private static double parseDouble(
      String[] values, Map<String, Integer> columns, String name) {
    return Double.parseDouble(get(values, columns, name));
  }

  private static double parseDoubleOrNaN(
      String[] values, Map<String, Integer> columns, String name) {
    String value = get(values, columns, name);
    return value == null || value.isBlank()
        ? Double.NaN
        : Double.parseDouble(value);
  }

  private static String number(double value) {
    return Double.isFinite(value) ? Double.toString(value) : "";
  }

  private static String csv(String value) {
    if (value == null) return "";
    String text = value;
    if (text.contains(",") || text.contains("\"")
        || text.contains("\n") || text.contains("\r")) {
      return "\"" + text.replace("\"", "\"\"") + "\"";
    }
    return text;
  }

  private static void writeCsvRow(
      BufferedWriter writer, List<String> values) throws IOException {
    for (int i = 0; i < values.size(); i++) {
      if (i > 0) writer.write(',');
      writer.write(csv(values.get(i)));
    }
    writer.newLine();
  }
}
'''.strip()

CELL17_JAVA_SOURCE_PATH.write_text(java_source + "\n", encoding="utf-8")
java_source_hash = sha256_file(CELL17_JAVA_SOURCE_PATH)

if CELL17_CLASSES_DIR.exists():
    shutil.rmtree(CELL17_CLASSES_DIR)
CELL17_CLASSES_DIR.mkdir(parents=True, exist_ok=True)

compile_classpath = os.pathsep.join(str(path) for path in runtime_paths)
CELL17_JAVAC_ARGS_PATH.write_text(
    "\n".join(
        [
            "-encoding",
            "UTF-8",
            "-classpath",
            java_arg(compile_classpath),
            "-d",
            java_arg(CELL17_CLASSES_DIR),
            java_arg(CELL17_JAVA_SOURCE_PATH),
        ]
    )
    + "\n",
    encoding="utf-8",
)

javac_path = shutil.which("javac")
java_path = shutil.which("java")
if javac_path is None or java_path is None:
    raise FileNotFoundError("Both java and javac must be available through PATH.")

compile_result = subprocess.run(
    [javac_path, f"@{CELL17_JAVAC_ARGS_PATH}"],
    cwd=str(CELL17_JAVA_DIR),
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=False,
)
compile_output = "\n".join(
    value.strip()
    for value in [compile_result.stdout, compile_result.stderr]
    if value and value.strip()
)
CELL17_COMPILE_LOG_PATH.write_text(compile_output + "\n", encoding="utf-8")

print("=" * 78)
print("BULK SOURCE-APPROPRIATE PARKER GMM PARAMETERS")
print("=" * 78)
print(f"Ruptures:          {EXPECTED_RUPTURES:,}")
print(f"Seaside W2 sites:  {EXPECTED_SITES:,}")
print(f"Rupture-site rows: {EXPECTED_PAIR_ROWS:,}")
print(f"GMM chunk size:    {GMM_RUPTURES_PER_CHUNK:,} ruptures")
print(f"GMM chunks:        {chunk_count:,}")
print("IMTs:              PGA and exact SA0P4")
print(f"Primary branch:    {PRIMARY_BASELINE_BRANCH}")
print("\nJava compiler output:")
print(compile_output if compile_output else "[no compiler messages]")

if compile_result.returncode != 0:
    raise RuntimeError(
        "The bulk Parker GMM Java helper did not compile.\n\n"
        f"Compile log:\n  {CELL17_COMPILE_LOG_PATH}"
    )

run_classpath = os.pathsep.join(
    [str(CELL17_CLASSES_DIR), *[str(path) for path in runtime_paths]]
)


# ----------------------------------------------------------------------------
# 10. Locate validated Cell 15 chunk files when available
# ----------------------------------------------------------------------------

cell15_manifest = None
if CELL15_MANIFEST_PATH.is_file():
    cell15_manifest = pd.read_csv(CELL15_MANIFEST_PATH, low_memory=False)
    for column in ["start", "stop", "rupture_count", "expected_rows"]:
        if column in cell15_manifest:
            cell15_manifest[column] = pd.to_numeric(
                cell15_manifest[column], errors="coerce"
            )


def validated_cell15_distance_sources(start: int, stop: int) -> list[Path]:
    if cell15_manifest is None:
        return []
    required_columns = {"start", "stop", "distance_output", "marker"}
    if not required_columns.issubset(cell15_manifest.columns):
        return []

    overlapping = cell15_manifest.loc[
        (cell15_manifest["start"] < stop)
        & (cell15_manifest["stop"] > start)
    ]
    paths: list[Path] = []
    for _, row in overlapping.iterrows():
        distance_path = normalize_path_from_manifest(row["distance_output"])
        marker_path = normalize_path_from_manifest(row["marker"])
        if not distance_path.is_file() or not marker_path.is_file():
            return []
        try:
            marker = json.loads(marker_path.read_text(encoding="utf-8"))
        except Exception:
            return []
        if not bool(marker.get("validation_passed", False)):
            return []
        if marker.get("distance_size") != distance_path.stat().st_size:
            return []
        if marker.get("distance_hash") != sha256_file(distance_path):
            return []
        paths.append(distance_path)
    return paths


def load_authoritative_distances_for_record(
    record: dict[str, object],
    rupture_ids: set[str],
) -> tuple[pd.DataFrame, str]:
    usecols = [
        "rupture_ordinal",
        "site_ordinal",
        "rupture_id",
        "site_id",
        "r_jb_km",
        "r_rup_km",
        "r_x_km",
        "retrieval_error",
    ]
    dtypes = {
        "rupture_id": str,
        "site_id": str,
        "retrieval_error": str,
    }

    source_paths = validated_cell15_distance_sources(
        int(record["start"]), int(record["stop"])
    )
    frames: list[pd.DataFrame] = []
    source_label = "Cell 15 validated chunk outputs"

    if source_paths:
        for path in source_paths:
            frame = pd.read_csv(
                path,
                usecols=usecols,
                dtype=dtypes,
                low_memory=False,
            )
            frame = frame.loc[frame["rupture_id"].isin(rupture_ids)]
            if not frame.empty:
                frames.append(frame)
    else:
        source_label = "Cell 15 final authoritative distance matrix"
        for frame in pd.read_csv(
            CELL15_DISTANCE_PATH,
            usecols=usecols,
            dtype=dtypes,
            chunksize=250_000,
            low_memory=False,
        ):
            selected = frame.loc[frame["rupture_id"].isin(rupture_ids)]
            if not selected.empty:
                frames.append(selected.copy())

    if not frames:
        raise RuntimeError(
            f"No authoritative distance rows found for GMM chunk {record['chunk_id']}."
        )
    distance = pd.concat(frames, ignore_index=True)
    distance["retrieval_error"] = (
        distance["retrieval_error"].fillna("").astype(str)
    )
    for column in [
        "rupture_ordinal",
        "site_ordinal",
        "r_jb_km",
        "r_rup_km",
        "r_x_km",
    ]:
        distance[column] = pd.to_numeric(distance[column], errors="coerce")

    if len(distance) != int(record["expected_rows"]):
        raise RuntimeError(
            f"GMM chunk {record['chunk_id']} expected {record['expected_rows']:,} "
            f"distance rows; observed {len(distance):,}."
        )
    if distance[["rupture_id", "site_id"]].duplicated().any():
        raise RuntimeError(
            f"GMM chunk {record['chunk_id']} contains duplicate rupture-site pairs."
        )
    if distance["retrieval_error"].ne("").any():
        raise RuntimeError(
            f"GMM chunk {record['chunk_id']} contains distance retrieval errors."
        )
    if set(distance["rupture_id"]) != rupture_ids:
        raise RuntimeError(
            f"GMM chunk {record['chunk_id']} distance rupture IDs do not match."
        )
    return distance, source_label


# ----------------------------------------------------------------------------
# 11. Construct each temporary GMM input table
# ----------------------------------------------------------------------------

input_columns = [
    "gmm_chunk_id",
    "rupture_ordinal",
    "event_id",
    "rupture_id",
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
    "source_type",
    "gmm_name",
    "surface_class",
    "magnitude",
    "raw_annual_rate",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
    "dip_deg",
    "width_km",
    "z_tor_km",
    "z_hyp_input_km",
    "z_hyp_input_provenance",
    "rake_deg",
    "vs30_mps",
    "vs30_inferred",
    "z1p0_km",
    "z2p5_km",
    "zsed_km",
    "period_s",
]


def build_chunk_input(
    record: dict[str, object],
) -> tuple[pd.DataFrame, str, str]:
    chunk_ruptures = rupture_metadata.iloc[
        int(record["start"]):int(record["stop"])
    ].copy()
    rupture_ids = set(chunk_ruptures["rupture_id"].astype(str))
    distance, distance_source = load_authoritative_distances_for_record(
        record, rupture_ids
    )

    rupture_merge = chunk_ruptures.rename(
        columns={"rupture_ordinal": "rupture_ordinal_metadata"}
    )
    merged = distance.merge(
        rupture_merge,
        on="rupture_id",
        how="left",
        validate="many_to_one",
    )
    if merged["event_id"].isna().any():
        raise RuntimeError("A distance row did not match rupture metadata.")
    ordinal_match = np.isclose(
        merged["rupture_ordinal"].to_numpy(dtype=float),
        merged["rupture_ordinal_metadata"].to_numpy(dtype=float),
        rtol=0.0,
        atol=0.0,
    )
    if not ordinal_match.all():
        raise RuntimeError("Distance and audit rupture ordinals disagree.")
    merged = merged.drop(columns=["rupture_ordinal_metadata", "retrieval_error"])

    merged = merged.merge(
        site_metadata,
        on="site_id",
        how="left",
        validate="many_to_one",
    )
    if merged["vs30_mps"].isna().any():
        raise RuntimeError("A distance row did not match Seaside W2 site metadata.")

    merged["gmm_chunk_id"] = str(record["chunk_id"])
    merged = merged.sort_values(
        ["rupture_ordinal", "site_ordinal"], kind="mergesort"
    ).reset_index(drop=True)

    if len(merged) != int(record["expected_rows"]):
        raise RuntimeError("Merged GMM input row count is incorrect.")
    if merged[["rupture_id", "site_id"]].duplicated().any():
        raise RuntimeError("Merged GMM input contains duplicate pairs.")
    rupture_counts = merged.groupby("rupture_id").size()
    site_counts = merged.groupby("site_id").size()
    if not rupture_counts.eq(EXPECTED_SITES).all():
        raise RuntimeError("Not every rupture is paired with all 470 sites.")
    if len(site_counts) != EXPECTED_SITES:
        raise RuntimeError("Not all 470 sites are present in the GMM input.")
    if not site_counts.eq(int(record["rupture_count"])).all():
        raise RuntimeError("A site is missing one or more chunk ruptures.")

    required_finite = [
        "rupture_ordinal",
        "site_ordinal",
        "site_longitude",
        "site_latitude",
        "magnitude",
        "raw_annual_rate",
        "r_jb_km",
        "r_rup_km",
        "r_x_km",
        "dip_deg",
        "width_km",
        "z_tor_km",
        "z_hyp_input_km",
        "rake_deg",
        "vs30_mps",
        "period_s",
    ]
    if not np.isfinite(merged[required_finite].to_numpy(dtype=float)).all():
        raise RuntimeError("Merged GMM inputs contain non-finite required values.")
    if not (merged["r_rup_km"] >= merged["r_jb_km"]).all():
        raise RuntimeError("A GMM input has Rrup below Rjb.")
    if not np.isclose(
        merged["period_s"].to_numpy(dtype=float),
        SEASIDE_W2_PERIOD_S,
        rtol=0.0,
        atol=PERIOD_MATCH_ATOL,
    ).all():
        raise RuntimeError("A GMM input period differs from 0.40 s.")

    temporary_input_path = Path(record["temporary_input"])
    merged.to_csv(
        temporary_input_path,
        sep="\t",
        index=False,
        columns=input_columns,
        na_rep="",
        lineterminator="\n",
        float_format="%.17g",
    )
    return merged[input_columns].copy(), sha256_file(temporary_input_path), distance_source


# ----------------------------------------------------------------------------
# 12. Enrich Java outputs with Parker tau and phi
# ----------------------------------------------------------------------------

branch_labels = ["epi_lo", "epi_off", "epi_hi"]


def enrich_bulk_output(raw: pd.DataFrame) -> pd.DataFrame:
    enriched = raw.copy()
    for prefix, coefficient_label in [("pga", "PGA"), ("sa0p4", "SA0P4")]:
        coefficient = coefficient_lookup.loc[coefficient_label]
        phi_variance = vectorized_parker_phi_variance(
            enriched["r_rup_km"].to_numpy(dtype=float),
            enriched["vs30_mps"].to_numpy(dtype=float),
            phi21=float(coefficient["phi21_variance"]),
            phi22=float(coefficient["phi22_variance"]),
            phi2v=float(coefficient["phi2v_variance_adjustment"]),
        )
        tau = float(coefficient["tau_ln"])
        phi = np.sqrt(phi_variance)
        reconstructed = np.sqrt(tau**2 + phi_variance)
        sigma_columns = [
            f"{prefix}_{branch}_sigma_ln" for branch in branch_labels
        ]
        sigma_matrix = enriched[sigma_columns].to_numpy(dtype=float)

        enriched[f"{prefix}_tau_ln"] = tau
        enriched[f"{prefix}_phi_variance"] = phi_variance
        enriched[f"{prefix}_phi_ln"] = phi
        enriched[f"{prefix}_sigma_total_ln"] = sigma_matrix[:, 1]
        enriched[f"{prefix}_sigma_reconstructed_ln"] = reconstructed
        enriched[f"{prefix}_sigma_branch_spread"] = (
            np.max(sigma_matrix, axis=1) - np.min(sigma_matrix, axis=1)
        )
        enriched[f"{prefix}_sigma_max_abs_difference"] = np.max(
            np.abs(sigma_matrix - reconstructed[:, None]), axis=1
        )
    return enriched


# ----------------------------------------------------------------------------
# 13. Chunk validation
# ----------------------------------------------------------------------------

string_columns = [
    "gmm_chunk_id",
    "event_id",
    "rupture_id",
    "site_id",
    "source_type",
    "gmm_name",
    "surface_class",
    "z_hyp_input_provenance",
    "calculation_error",
]
input_numeric_columns = [
    "rupture_ordinal",
    "site_ordinal",
    "site_longitude",
    "site_latitude",
    "magnitude",
    "raw_annual_rate",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
    "dip_deg",
    "width_km",
    "z_tor_km",
    "z_hyp_input_km",
    "rake_deg",
    "vs30_mps",
    "z1p0_km",
    "z2p5_km",
    "zsed_km",
    "period_s",
]
result_numeric_columns: list[str] = []
for prefix in ["pga", "sa0p4"]:
    result_numeric_columns.extend(
        [f"{prefix}_{branch}_weight" for branch in branch_labels]
    )
    result_numeric_columns.extend(
        [f"{prefix}_{branch}_mean_ln_g" for branch in branch_labels]
    )
    result_numeric_columns.extend(
        [f"{prefix}_{branch}_median_g" for branch in branch_labels]
    )
    result_numeric_columns.extend(
        [f"{prefix}_{branch}_sigma_ln" for branch in branch_labels]
    )


def read_raw_java_output(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(
        path,
        dtype={column: str for column in string_columns},
        low_memory=False,
    )
    frame["calculation_error"] = (
        frame["calculation_error"].fillna("").astype(str)
    )
    frame["vs30_inferred"] = parse_boolean_series(
        frame["vs30_inferred"], label="vs30_inferred"
    )
    for column in [*input_numeric_columns, *result_numeric_columns]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def validate_and_write_chunk(
    record: dict[str, object],
    input_frame: pd.DataFrame,
    raw_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = read_raw_java_output(raw_path)
    validation_records: list[dict[str, object]] = []
    expected_rows = int(record["expected_rows"])
    expected_ruptures = int(record["rupture_count"])

    add_validation(
        validation_records, "Coverage", "Output rows", len(raw), expected_rows,
        len(raw) == expected_rows,
    )
    add_validation(
        validation_records, "Coverage", "No calculation errors",
        int(raw["calculation_error"].ne("").sum()), 0,
        raw["calculation_error"].eq("").all(),
    )
    add_validation(
        validation_records, "Coverage", "Unique rupture-site pairs",
        int(raw[["rupture_id", "site_id"]].drop_duplicates().shape[0]),
        expected_rows,
        not raw[["rupture_id", "site_id"]].duplicated().any(),
    )
    add_validation(
        validation_records, "Coverage", "Unique ruptures",
        raw["rupture_id"].nunique(), expected_ruptures,
        raw["rupture_id"].nunique() == expected_ruptures,
    )
    add_validation(
        validation_records, "Coverage", "All 470 sites present",
        raw["site_id"].nunique(), EXPECTED_SITES,
        raw["site_id"].nunique() == EXPECTED_SITES,
    )
    rupture_counts = raw.groupby("rupture_id").size()
    site_counts = raw.groupby("site_id").size()
    add_validation(
        validation_records, "Coverage", "Every rupture has 470 sites",
        int(rupture_counts.eq(EXPECTED_SITES).sum()), expected_ruptures,
        len(rupture_counts) == expected_ruptures
        and rupture_counts.eq(EXPECTED_SITES).all(),
    )
    add_validation(
        validation_records, "Coverage", "Every site has every chunk rupture",
        int(site_counts.eq(expected_ruptures).sum()), EXPECTED_SITES,
        len(site_counts) == EXPECTED_SITES
        and site_counts.eq(expected_ruptures).all(),
    )

    input_keys = input_frame[["rupture_id", "site_id"]].sort_values(
        ["rupture_id", "site_id"]
    ).reset_index(drop=True)
    output_keys = raw[["rupture_id", "site_id"]].sort_values(
        ["rupture_id", "site_id"]
    ).reset_index(drop=True)
    key_match = len(input_keys) == len(output_keys) and input_keys.equals(output_keys)
    add_validation(
        validation_records, "Input echo", "Output key set matches input",
        key_match, True, key_match,
    )

    route_ok = (
        (
            raw["source_type"].eq("INTERFACE")
            & raw["gmm_name"].eq(INTERFACE_GMM_NAME)
        )
        | (
            raw["source_type"].eq("SLAB")
            & raw["gmm_name"].eq(SLAB_GMM_NAME)
        )
    )
    add_validation(
        validation_records, "Routing", "Source-specific GMM routing",
        int(route_ok.sum()), len(raw), route_ok.all(),
    )
    add_validation(
        validation_records, "Inputs", "Exact Seaside period SA0P4",
        float(np.max(np.abs(raw["period_s"] - SEASIDE_W2_PERIOD_S))),
        f"<= {PERIOD_MATCH_ATOL:g}",
        np.isclose(
            raw["period_s"].to_numpy(dtype=float),
            SEASIDE_W2_PERIOD_S,
            rtol=0.0,
            atol=PERIOD_MATCH_ATOL,
        ).all(),
    )
    add_validation(
        validation_records, "Inputs", "Rrup is not below Rjb",
        float((raw["r_rup_km"] - raw["r_jb_km"]).min()), ">= 0",
        (raw["r_rup_km"] >= raw["r_jb_km"]).all(),
    )

    for prefix in ["pga", "sa0p4"]:
        weight_columns = [f"{prefix}_{branch}_weight" for branch in branch_labels]
        mean_columns = [f"{prefix}_{branch}_mean_ln_g" for branch in branch_labels]
        median_columns = [f"{prefix}_{branch}_median_g" for branch in branch_labels]
        sigma_columns = [f"{prefix}_{branch}_sigma_ln" for branch in branch_labels]

        weights = raw[weight_columns].to_numpy(dtype=float)
        means = raw[mean_columns].to_numpy(dtype=float)
        medians = raw[median_columns].to_numpy(dtype=float)
        sigmas = raw[sigma_columns].to_numpy(dtype=float)

        add_validation(
            validation_records, "Epistemic branches",
            f"{prefix.upper()} weights finite and positive",
            int((np.isfinite(weights) & (weights > 0.0)).all(axis=1).sum()),
            len(raw),
            np.isfinite(weights).all() and (weights > 0.0).all(),
        )
        weight_error = float(np.max(np.abs(weights.sum(axis=1) - 1.0)))
        add_validation(
            validation_records, "Epistemic branches",
            f"{prefix.upper()} weights sum to one",
            weight_error, f"<= {WEIGHT_SUM_ATOL:g}",
            weight_error <= WEIGHT_SUM_ATOL,
        )
        order_ok = (means[:, 0] < means[:, 1]) & (means[:, 1] < means[:, 2])
        add_validation(
            validation_records, "Epistemic branches",
            f"{prefix.upper()} means ordered low < off < high",
            int(order_ok.sum()), len(raw), order_ok.all(),
        )
        add_validation(
            validation_records, "Numerical outputs",
            f"{prefix.upper()} means finite",
            int(np.isfinite(means).all(axis=1).sum()), len(raw),
            np.isfinite(means).all(),
        )
        add_validation(
            validation_records, "Numerical outputs",
            f"{prefix.upper()} medians finite and positive",
            int((np.isfinite(medians) & (medians > 0.0)).all(axis=1).sum()),
            len(raw),
            np.isfinite(medians).all() and (medians > 0.0).all(),
        )
        median_error = float(np.max(np.abs(np.exp(means) - medians)))
        add_validation(
            validation_records, "Numerical outputs",
            f"{prefix.upper()} medians equal exp(means)",
            median_error, f"<= {MEDIAN_MATCH_ATOL:g}",
            median_error <= MEDIAN_MATCH_ATOL,
        )
        add_validation(
            validation_records, "Aleatory variability",
            f"{prefix.upper()} USGS sigmas finite and positive",
            int((np.isfinite(sigmas) & (sigmas > 0.0)).all(axis=1).sum()),
            len(raw),
            np.isfinite(sigmas).all() and (sigmas > 0.0).all(),
        )

    enriched = enrich_bulk_output(raw)
    for prefix in ["pga", "sa0p4"]:
        max_spread = float(enriched[f"{prefix}_sigma_branch_spread"].max())
        max_difference = float(
            enriched[f"{prefix}_sigma_max_abs_difference"].max()
        )
        valid_tau_phi = (
            np.isfinite(
                enriched[
                    [
                        f"{prefix}_tau_ln",
                        f"{prefix}_phi_ln",
                        f"{prefix}_sigma_reconstructed_ln",
                    ]
                ].to_numpy(dtype=float)
            )
            & (
                enriched[
                    [
                        f"{prefix}_tau_ln",
                        f"{prefix}_phi_ln",
                        f"{prefix}_sigma_reconstructed_ln",
                    ]
                ].to_numpy(dtype=float)
                > 0.0
            )
        ).all()
        add_validation(
            validation_records, "Aleatory variability",
            f"{prefix.upper()} sigma identical across branches",
            max_spread, f"<= {SIGMA_MATCH_ATOL:g}",
            max_spread <= SIGMA_MATCH_ATOL,
        )
        add_validation(
            validation_records, "Aleatory variability",
            f"{prefix.upper()} Parker tau/phi reconstruct USGS sigma",
            max_difference, f"<= {SIGMA_MATCH_ATOL:g}",
            max_difference <= SIGMA_MATCH_ATOL,
        )
        add_validation(
            validation_records, "Aleatory variability",
            f"{prefix.upper()} tau, phi, and sigma finite and positive",
            valid_tau_phi, True, valid_tau_phi,
        )

    validation = pd.DataFrame(validation_records)
    validation_path = Path(record["validation"])
    validation.to_csv(validation_path, index=False, lineterminator="\n")
    if not validation["passes"].all():
        failed = validation.loc[~validation["passes"], "check"].tolist()
        raise RuntimeError(
            f"GMM chunk {record['chunk_id']} validation failed:\n"
            + "\n".join(f"  - {name}" for name in failed)
            + f"\n\nValidation file:\n  {validation_path}"
        )

    deterministic_gzip_csv(enriched, Path(record["output"]))
    return enriched, validation


# ----------------------------------------------------------------------------
# 14. Restart marker validation
# ----------------------------------------------------------------------------

basis_common = {
    "pipeline_version": PIPELINE_VERSION,
    "gmm_ruptures_per_chunk": GMM_RUPTURES_PER_CHUNK,
    "expected_sites": EXPECTED_SITES,
    "interface_gmm": INTERFACE_GMM_NAME,
    "slab_gmm": SLAB_GMM_NAME,
    "period_s": SEASIDE_W2_PERIOD_S,
    "primary_branch": PRIMARY_BASELINE_BRANCH,
    "source_hashes": source_hashes,
    "runtime_hashes": runtime_hashes,
    "java_source_hash": java_source_hash,
}


def basis_hash_for_record(record: dict[str, object]) -> str:
    payload = {
        **basis_common,
        "chunk_id": record["chunk_id"],
        "start": record["start"],
        "stop": record["stop"],
        "rupture_count": record["rupture_count"],
        "expected_rows": record["expected_rows"],
        "chunk_index_hash": record["chunk_index_hash"],
    }
    return sha256_text(json.dumps(payload, sort_keys=True, separators=(",", ":")))


def marker_is_valid(record: dict[str, object]) -> bool:
    marker_path = Path(record["marker"])
    output_path = Path(record["output"])
    validation_path = Path(record["validation"])
    if not marker_path.is_file() or not output_path.is_file() or not validation_path.is_file():
        return False
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        validation = pd.read_csv(validation_path, low_memory=False)
    except Exception:
        return False
    if "passes" in validation.columns:
        parsed_passes = (
            validation["passes"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map({"true": True, "false": False, "1": True, "0": False})
        )
        validation_ok = bool(
            not parsed_passes.isna().any() and parsed_passes.all()
        )
    else:
        validation_ok = False
    checks = [
        marker.get("pipeline_version") == PIPELINE_VERSION,
        marker.get("chunk_id") == record["chunk_id"],
        marker.get("basis_hash") == basis_hash_for_record(record),
        marker.get("expected_rows") == int(record["expected_rows"]),
        marker.get("output_size") == output_path.stat().st_size,
        bool(marker.get("validation_passed", False)),
        validation_ok,
    ]
    if not all(checks):
        return False
    if VERIFY_EXISTING_HASHES and marker.get("output_hash") != sha256_file(output_path):
        return False
    return True


# ----------------------------------------------------------------------------
# 15. Sequential bulk calculation with restart capability
# ----------------------------------------------------------------------------

for position, record in enumerate(chunk_records, start=1):
    chunk_id = str(record["chunk_id"])
    print("\n" + "-" * 78)
    print(f"GMM CHUNK {position} OF {chunk_count} [ID {chunk_id}]")
    print(
        f"Ruptures: {record['rupture_count']:,}; "
        f"rupture-site rows: {record['expected_rows']:,}"
    )

    if marker_is_valid(record):
        marker = json.loads(Path(record["marker"]).read_text(encoding="utf-8"))
        record["status"] = "complete_existing"
        record["completed_at_utc"] = marker.get("completed_at_utc", "")
        save_manifest()
        print("Validated completion marker found; recomputation skipped.")
        continue

    for key in ["temporary_input", "raw_output", "output", "marker"]:
        path = Path(record[key])
        if path.exists():
            path.unlink()

    input_frame, temporary_input_hash, distance_source = build_chunk_input(record)
    print(f"Distance source: {distance_source}")

    java_tokens = ["-Dfile.encoding=UTF-8"]
    if JAVA_MAX_HEAP:
        java_tokens.append(
            JAVA_MAX_HEAP if JAVA_MAX_HEAP.startswith("-Xmx")
            else f"-Xmx{JAVA_MAX_HEAP}"
        )
    java_tokens.extend(
        [
            "-classpath",
            java_arg(run_classpath),
            "BulkParkerGmm",
            INTERFACE_GMM_NAME,
            SLAB_GMM_NAME,
            java_arg(record["temporary_input"]),
            java_arg(record["raw_output"]),
        ]
    )
    java_args_path = Path(record["java_args"])
    java_args_path.write_text("\n".join(java_tokens) + "\n", encoding="utf-8")

    result = subprocess.run(
        [java_path, f"@{java_args_path}"],
        cwd=str(PARKER_INSPECTION_DIR),
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False,
    )
    output_text = "\n".join(
        value.strip()
        for value in [result.stdout, result.stderr]
        if value and value.strip()
    )
    log_path = Path(record["log"])
    log_path.write_text(output_text + "\n", encoding="utf-8")

    print("\nRuntime output tail:")
    print("\n".join(output_text.splitlines()[-30:]))

    if result.returncode != 0:
        record["status"] = "failed_runtime"
        save_manifest()
        raise RuntimeError(
            f"GMM chunk {chunk_id} failed.\n\nRun log:\n  {log_path}"
        )
    if "BULK_PARKER_GMM_CHUNK_COMPLETE" not in output_text:
        record["status"] = "failed_missing_completion_message"
        save_manifest()
        raise RuntimeError(
            f"GMM chunk {chunk_id} did not emit its completion message.\n"
            f"Run log:\n  {log_path}"
        )

    enriched, validation = validate_and_write_chunk(
        record,
        input_frame,
        Path(record["raw_output"]),
    )
    output_path = Path(record["output"])
    completed_at = utc_now()
    marker = {
        "pipeline_version": PIPELINE_VERSION,
        "completed_at_utc": completed_at,
        "chunk_id": chunk_id,
        "basis_hash": basis_hash_for_record(record),
        "temporary_input_hash": temporary_input_hash,
        "java_source_hash": java_source_hash,
        "rupture_count": int(record["rupture_count"]),
        "site_count": EXPECTED_SITES,
        "expected_rows": int(record["expected_rows"]),
        "validation_passed": bool(validation["passes"].all()),
        "output_size": output_path.stat().st_size,
        "output_hash": sha256_file(output_path),
        "distance_source": distance_source,
    }
    Path(record["marker"]).write_text(
        json.dumps(marker, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )

    for key in ["temporary_input", "raw_output"]:
        path = Path(record[key])
        if path.exists():
            path.unlink()

    record["status"] = "complete_new"
    record["completed_at_utc"] = completed_at
    save_manifest()
    print(f"GMM chunk {chunk_id} passed all {len(validation)} checks.")


# ----------------------------------------------------------------------------
# 16. Confirm all chunk markers and concatenate deterministically
# ----------------------------------------------------------------------------

invalid_chunks = [
    str(record["chunk_id"])
    for record in chunk_records
    if not marker_is_valid(record)
]
if invalid_chunks:
    raise RuntimeError(
        "Cannot create final Cell 17 output because these chunks are invalid: "
        + ", ".join(invalid_chunks)
    )

with CELL17_FINAL_OUTPUT_PATH.open("wb") as raw_final:
    with gzip.GzipFile(fileobj=raw_final, mode="wb", compresslevel=6, mtime=0) as gz_final:
        with io.TextIOWrapper(gz_final, encoding="utf-8", newline="") as writer:
            wrote_header = False
            for record in chunk_records:
                chunk_path = Path(record["output"])
                with gzip.open(chunk_path, mode="rt", encoding="utf-8", newline="") as reader:
                    header = reader.readline()
                    if not header:
                        raise RuntimeError(f"Empty GMM chunk output: {chunk_path}")
                    if not wrote_header:
                        writer.write(header)
                        wrote_header = True
                    for line in reader:
                        writer.write(line)


# ----------------------------------------------------------------------------
# 17. Independent final validation and Cell 16 control reproduction
# ----------------------------------------------------------------------------

control_selection = pd.read_csv(
    CELL16_CONTROL_SELECTION_PATH,
    dtype={"case_id": str, "rupture_id": str, "site_id": str},
    low_memory=False,
)
controlled_exact = pd.read_csv(
    CELL16_EXACT_OUTPUT_PATH,
    dtype={
        "case_id": str,
        "imt_name": str,
        "branch_id": str,
        "calculation_error": str,
    },
    low_memory=False,
)
controlled_exact["calculation_error"] = (
    controlled_exact["calculation_error"].fillna("").astype(str)
)
if controlled_exact["calculation_error"].ne("").any():
    raise RuntimeError("Cell 16 exact control output contains errors.")
control_pairs = {
    (str(row.rupture_id), str(row.site_id)): str(row.case_id)
    for row in control_selection.itertuples(index=False)
}
if len(control_pairs) != 4:
    raise RuntimeError("Expected four unique Cell 16 controlled pairs.")

seen_pairs = np.zeros(EXPECTED_PAIR_ROWS, dtype=bool)
rupture_counts = np.zeros(EXPECTED_RUPTURES, dtype=np.int64)
site_counts = np.zeros(EXPECTED_SITES, dtype=np.int64)
source_row_counts = {"INTERFACE": 0, "SLAB": 0}
total_rows = 0
error_rows = 0
routing_failures = 0
period_failures = 0
rrup_failures = 0
numeric_failures = 0
weight_failures = 0
mean_order_failures = 0
median_failures = 0
sigma_failures = 0
maximum_weight_error = 0.0
maximum_median_error = 0.0
maximum_sigma_difference = 0.0
maximum_sigma_branch_spread = 0.0
control_bulk_rows: dict[tuple[str, str], pd.Series] = {}
range_summary = {
    "pga_epi_off_median_g": [np.inf, -np.inf],
    "sa0p4_epi_off_median_g": [np.inf, -np.inf],
    "pga_phi_ln": [np.inf, -np.inf],
    "sa0p4_phi_ln": [np.inf, -np.inf],
}

for frame in pd.read_csv(
    CELL17_FINAL_OUTPUT_PATH,
    dtype={column: str for column in string_columns},
    chunksize=FINAL_READ_CHUNK_SIZE,
    low_memory=False,
):
    frame["calculation_error"] = (
        frame["calculation_error"].fillna("").astype(str)
    )
    for column in [
        *input_numeric_columns,
        *result_numeric_columns,
        "pga_tau_ln",
        "pga_phi_variance",
        "pga_phi_ln",
        "pga_sigma_total_ln",
        "pga_sigma_reconstructed_ln",
        "pga_sigma_branch_spread",
        "pga_sigma_max_abs_difference",
        "sa0p4_tau_ln",
        "sa0p4_phi_variance",
        "sa0p4_phi_ln",
        "sa0p4_sigma_total_ln",
        "sa0p4_sigma_reconstructed_ln",
        "sa0p4_sigma_branch_spread",
        "sa0p4_sigma_max_abs_difference",
    ]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    n = len(frame)
    total_rows += n
    error_rows += int(frame["calculation_error"].ne("").sum())

    rupture_ordinals = frame["rupture_ordinal"].to_numpy(dtype=int)
    site_ordinals = frame["site_ordinal"].to_numpy(dtype=int)
    in_range = (
        (rupture_ordinals >= 0)
        & (rupture_ordinals < EXPECTED_RUPTURES)
        & (site_ordinals >= 0)
        & (site_ordinals < EXPECTED_SITES)
    )
    if not in_range.all():
        raise RuntimeError("Final GMM output contains out-of-range ordinals.")
    pair_keys = rupture_ordinals * EXPECTED_SITES + site_ordinals
    if np.unique(pair_keys).size != pair_keys.size or seen_pairs[pair_keys].any():
        raise RuntimeError("Final GMM output contains duplicate rupture-site pairs.")
    seen_pairs[pair_keys] = True
    np.add.at(rupture_counts, rupture_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    for source_type, count in frame["source_type"].value_counts().items():
        source_row_counts[str(source_type)] = (
            source_row_counts.get(str(source_type), 0) + int(count)
        )

    route_ok = (
        (
            frame["source_type"].eq("INTERFACE")
            & frame["gmm_name"].eq(INTERFACE_GMM_NAME)
        )
        | (
            frame["source_type"].eq("SLAB")
            & frame["gmm_name"].eq(SLAB_GMM_NAME)
        )
    )
    routing_failures += int((~route_ok).sum())
    period_failures += int(
        (~np.isclose(
            frame["period_s"].to_numpy(dtype=float),
            SEASIDE_W2_PERIOD_S,
            rtol=0.0,
            atol=PERIOD_MATCH_ATOL,
        )).sum()
    )
    rrup_failures += int((frame["r_rup_km"] < frame["r_jb_km"]).sum())

    required_final_numeric = [
        *result_numeric_columns,
        "pga_tau_ln",
        "pga_phi_ln",
        "pga_sigma_reconstructed_ln",
        "sa0p4_tau_ln",
        "sa0p4_phi_ln",
        "sa0p4_sigma_reconstructed_ln",
    ]
    numeric_matrix = frame[required_final_numeric].to_numpy(dtype=float)
    numeric_failures += int((~np.isfinite(numeric_matrix)).any(axis=1).sum())

    for prefix in ["pga", "sa0p4"]:
        weights = frame[
            [f"{prefix}_{branch}_weight" for branch in branch_labels]
        ].to_numpy(dtype=float)
        means = frame[
            [f"{prefix}_{branch}_mean_ln_g" for branch in branch_labels]
        ].to_numpy(dtype=float)
        medians = frame[
            [f"{prefix}_{branch}_median_g" for branch in branch_labels]
        ].to_numpy(dtype=float)
        weight_error = np.abs(weights.sum(axis=1) - 1.0)
        maximum_weight_error = max(maximum_weight_error, float(weight_error.max()))
        weight_failures += int(
            ((~np.isfinite(weights)).any(axis=1)
             | (weights <= 0.0).any(axis=1)
             | (weight_error > WEIGHT_SUM_ATOL)).sum()
        )
        ordered = (means[:, 0] < means[:, 1]) & (means[:, 1] < means[:, 2])
        mean_order_failures += int((~ordered).sum())
        median_error = np.max(np.abs(np.exp(means) - medians), axis=1)
        maximum_median_error = max(maximum_median_error, float(median_error.max()))
        median_failures += int((median_error > MEDIAN_MATCH_ATOL).sum())
        sigma_difference = frame[
            f"{prefix}_sigma_max_abs_difference"
        ].to_numpy(dtype=float)
        sigma_spread = frame[
            f"{prefix}_sigma_branch_spread"
        ].to_numpy(dtype=float)
        maximum_sigma_difference = max(
            maximum_sigma_difference, float(sigma_difference.max())
        )
        maximum_sigma_branch_spread = max(
            maximum_sigma_branch_spread, float(sigma_spread.max())
        )
        sigma_failures += int(
            ((sigma_difference > SIGMA_MATCH_ATOL)
             | (sigma_spread > SIGMA_MATCH_ATOL)).sum()
        )

    for column in range_summary:
        values = frame[column].to_numpy(dtype=float)
        range_summary[column][0] = min(range_summary[column][0], float(values.min()))
        range_summary[column][1] = max(range_summary[column][1], float(values.max()))

    control_tokens = {f"{rupture_id}|{site_id}" for rupture_id, site_id in control_pairs}
    pair_tokens = frame["rupture_id"].astype(str) + "|" + frame["site_id"].astype(str)
    candidate = frame.loc[pair_tokens.isin(control_tokens)]
    for _, row in candidate.iterrows():
        key = (str(row["rupture_id"]), str(row["site_id"]))
        if key in control_bulk_rows:
            raise RuntimeError(f"Controlled pair appears twice in final output: {key}")
        control_bulk_rows[key] = row.copy()

expected_source_ruptures = rupture_metadata["source_type"].value_counts().to_dict()
expected_source_rows = {
    source: int(count) * EXPECTED_SITES
    for source, count in expected_source_ruptures.items()
}

control_differences: list[float] = []
if len(control_bulk_rows) == 4:
    case_to_pair = {
        str(row.case_id): (str(row.rupture_id), str(row.site_id))
        for row in control_selection.itertuples(index=False)
    }
    for row in controlled_exact.itertuples(index=False):
        case_id = str(row.case_id)
        pair = case_to_pair[case_id]
        bulk = control_bulk_rows[pair]
        prefix = "pga" if str(row.imt_name).upper() == "PGA" else "sa0p4"
        branch = str(row.branch_id).replace("-", "_")
        comparisons = [
            (float(getattr(row, "branch_weight")), float(bulk[f"{prefix}_{branch}_weight"])),
            (float(getattr(row, "mean_ln_units")), float(bulk[f"{prefix}_{branch}_mean_ln_g"])),
            (float(getattr(row, "median_units")), float(bulk[f"{prefix}_{branch}_median_g"])),
            (float(getattr(row, "sigma_total_ln")), float(bulk[f"{prefix}_{branch}_sigma_ln"])),
            (float(getattr(row, "tau_ln")), float(bulk[f"{prefix}_tau_ln"])),
            (float(getattr(row, "phi_ln")), float(bulk[f"{prefix}_phi_ln"])),
            (float(getattr(row, "sigma_reconstructed_ln")), float(bulk[f"{prefix}_sigma_reconstructed_ln"])),
        ]
        control_differences.extend(abs(expected - observed) for expected, observed in comparisons)
maximum_control_difference = (
    max(control_differences) if control_differences else np.inf
)

final_validation_records: list[dict[str, object]] = []
add_validation(
    final_validation_records, "Dependencies", "Cell 15 checks passed",
    len(cell15_validation), len(cell15_validation), True,
)
add_validation(
    final_validation_records, "Dependencies", "Cell 16 checks passed",
    len(cell16_validation), len(cell16_validation), True,
)
add_validation(
    final_validation_records, "Chunks", "All completion markers valid",
    chunk_count - len(invalid_chunks), chunk_count, len(invalid_chunks) == 0,
)
all_chunk_validations = pd.concat(
    [pd.read_csv(record["validation"]) for record in chunk_records],
    ignore_index=True,
)
all_chunk_passes = (
    all_chunk_validations["passes"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"true": True, "false": False, "1": True, "0": False})
)
all_chunk_passes_ok = bool(
    not all_chunk_passes.isna().any() and all_chunk_passes.all()
)
add_validation(
    final_validation_records, "Chunks", "All chunk validation checks passed",
    int(all_chunk_passes.fillna(False).sum()), len(all_chunk_validations),
    all_chunk_passes_ok,
)
add_validation(
    final_validation_records, "Coverage", "Final row count",
    total_rows, EXPECTED_PAIR_ROWS, total_rows == EXPECTED_PAIR_ROWS,
)
add_validation(
    final_validation_records, "Coverage", "Every rupture-site pair present once",
    int(seen_pairs.sum()), EXPECTED_PAIR_ROWS,
    seen_pairs.all(),
)
add_validation(
    final_validation_records, "Coverage", "Every rupture has 470 sites",
    int((rupture_counts == EXPECTED_SITES).sum()), EXPECTED_RUPTURES,
    np.all(rupture_counts == EXPECTED_SITES),
)
add_validation(
    final_validation_records, "Coverage", "Every site has 3,927 ruptures",
    int((site_counts == EXPECTED_RUPTURES).sum()), EXPECTED_SITES,
    np.all(site_counts == EXPECTED_RUPTURES),
)
add_validation(
    final_validation_records, "Coverage", "Source row counts match rupture audit",
    source_row_counts, expected_source_rows,
    source_row_counts == expected_source_rows,
)
add_validation(
    final_validation_records, "Calculations", "No row-level errors",
    error_rows, 0, error_rows == 0,
)
add_validation(
    final_validation_records, "Routing", "No source-to-GMM routing failures",
    routing_failures, 0, routing_failures == 0,
)
add_validation(
    final_validation_records, "Inputs", "All periods equal exact 0.40 s",
    period_failures, 0, period_failures == 0,
)
add_validation(
    final_validation_records, "Inputs", "Rrup is never below Rjb",
    rrup_failures, 0, rrup_failures == 0,
)
add_validation(
    final_validation_records, "Numerical outputs", "No non-finite required outputs",
    numeric_failures, 0, numeric_failures == 0,
)
add_validation(
    final_validation_records, "Epistemic branches", "Branch weights valid",
    maximum_weight_error, f"<= {WEIGHT_SUM_ATOL:g}",
    weight_failures == 0 and maximum_weight_error <= WEIGHT_SUM_ATOL,
)
add_validation(
    final_validation_records, "Epistemic branches", "Branch means ordered",
    mean_order_failures, 0, mean_order_failures == 0,
)
add_validation(
    final_validation_records, "Numerical outputs", "Medians equal exp(means)",
    maximum_median_error, f"<= {MEDIAN_MATCH_ATOL:g}",
    median_failures == 0 and maximum_median_error <= MEDIAN_MATCH_ATOL,
)
add_validation(
    final_validation_records, "Aleatory variability", "Parker tau/phi reproduce USGS sigma",
    maximum_sigma_difference, f"<= {SIGMA_MATCH_ATOL:g}",
    sigma_failures == 0 and maximum_sigma_difference <= SIGMA_MATCH_ATOL,
)
add_validation(
    final_validation_records, "Aleatory variability", "Sigma identical across branches",
    maximum_sigma_branch_spread, f"<= {SIGMA_MATCH_ATOL:g}",
    maximum_sigma_branch_spread <= SIGMA_MATCH_ATOL,
)
add_validation(
    final_validation_records, "Controlled reproduction", "All four Cell 16 pairs recovered",
    len(control_bulk_rows), 4, len(control_bulk_rows) == 4,
)
add_validation(
    final_validation_records, "Controlled reproduction", "Bulk outputs reproduce Cell 16",
    maximum_control_difference, f"<= {SIGMA_MATCH_ATOL:g}",
    maximum_control_difference <= SIGMA_MATCH_ATOL,
    "Compares branch weights, means, medians, USGS sigma, tau, phi, and reconstructed sigma.",
)

final_validation = pd.DataFrame(final_validation_records)
final_validation.to_csv(
    CELL17_FINAL_VALIDATION_PATH,
    index=False,
    lineterminator="\n",
)
if not final_validation["passes"].all():
    failed = final_validation.loc[~final_validation["passes"], "check"].tolist()
    raise RuntimeError(
        "Cell 17 final validation failed:\n"
        + "\n".join(f"  - {name}" for name in failed)
        + f"\n\nValidation file:\n  {CELL17_FINAL_VALIDATION_PATH}"
    )


# ----------------------------------------------------------------------------
# 18. Reproducibility summary
# ----------------------------------------------------------------------------

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": utc_now(),
    "all_checks_passed": bool(final_validation["passes"].all()),
    "validation_checks": int(len(final_validation)),
    "ruptures": EXPECTED_RUPTURES,
    "sites": EXPECTED_SITES,
    "rupture_site_rows": EXPECTED_PAIR_ROWS,
    "gmm_chunk_size_ruptures": GMM_RUPTURES_PER_CHUNK,
    "gmm_chunks": chunk_count,
    "portfolio": "Seaside W2 commercial portfolio",
    "portfolio_path": str(GMM_READY_PORTFOLIO_PATH),
    "period_s": SEASIDE_W2_PERIOD_S,
    "period_resolution": "exact USGS SA0P4; no interpolation",
    "vs30_provenance": vs30_provenance,
    "vs30_inferred_provenance": vs30_inferred_provenance,
    "interface_gmm": INTERFACE_GMM_NAME,
    "slab_gmm": SLAB_GMM_NAME,
    "imts": ["PGA", "SA0P4"],
    "epistemic_branches_retained": ["epi-lo", "epi-off", "epi-hi"],
    "primary_baseline_branch": PRIMARY_BASELINE_BRANCH,
    "aleatory_model": {
        "identity": "sigma = sqrt(tau^2 + phi^2)",
        "between_event": "Parker tau",
        "within_event": "Parker ergodic total phi",
        "cell17_status": "parameters calculated only; residuals not yet simulated",
    },
    "source_rupture_counts": {
        key: int(value) for key, value in expected_source_ruptures.items()
    },
    "source_row_counts": source_row_counts,
    "source_hashes": source_hashes,
    "runtime_classpath_hashes": runtime_hashes,
    "java_source_path": str(CELL17_JAVA_SOURCE_PATH),
    "java_source_sha256": java_source_hash,
    "coefficient_path": str(PARKER_COEFFICIENT_PATH),
    "coefficient_sha256": coefficient_hash,
    "chunk_manifest_path": str(CELL17_MANIFEST_PATH),
    "chunk_manifest_sha256": sha256_file(CELL17_MANIFEST_PATH),
    "final_output_path": str(CELL17_FINAL_OUTPUT_PATH),
    "final_output_size_bytes": CELL17_FINAL_OUTPUT_PATH.stat().st_size,
    "final_output_sha256": sha256_file(CELL17_FINAL_OUTPUT_PATH),
    "final_validation_path": str(CELL17_FINAL_VALIDATION_PATH),
    "final_validation_sha256": sha256_file(CELL17_FINAL_VALIDATION_PATH),
    "maximum_weight_sum_error": maximum_weight_error,
    "maximum_median_reconstruction_difference": maximum_median_error,
    "maximum_sigma_reconstruction_difference": maximum_sigma_difference,
    "maximum_sigma_branch_spread": maximum_sigma_branch_spread,
    "maximum_cell16_control_difference": maximum_control_difference,
    "numeric_ranges": {
        key: {"minimum": float(value[0]), "maximum": float(value[1])}
        for key, value in range_summary.items()
    },
    "modeling_note": (
        "Epistemic branches are retained as alternative model branches. "
        "They are not treated as additive earthquake occurrences. The epi-off "
        "branch is the primary baseline for later stochastic field simulation."
    ),
}
CELL17_FINAL_SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print("\n" + "=" * 78)
print("CELL 17 BULK PARKER GMM CALCULATION COMPLETE")
print("=" * 78)
print(f"Final rupture-site rows: {total_rows:,}")
print(f"Final validation checks: {len(final_validation):,}")
print(f"Maximum Cell 16 control difference: {maximum_control_difference:.3e}")
print(f"Maximum sigma reconstruction difference: {maximum_sigma_difference:.3e}")
print(f"\nBulk GMM parameters:\n  {CELL17_FINAL_OUTPUT_PATH}")
print(f"\nValidation:\n  {CELL17_FINAL_VALIDATION_PATH}")
print(f"\nSummary:\n  {CELL17_FINAL_SUMMARY_PATH}")
print(
    "\nNext: simulate the baseline ground-motion fields using epi-off, one "
    "shared between-event residual per event and conditionally independent "
    "within-event site residuals."
)


Cell 16 period prerequisite passed: default_period_s=0.40 s
BULK SOURCE-APPROPRIATE PARKER GMM PARAMETERS
Ruptures:          3,927
Seaside W2 sites:  470
Rupture-site rows: 1,845,690
GMM chunk size:    250 ruptures
GMM chunks:        16
IMTs:              PGA and exact SA0P4
Primary branch:    epi-off

Java compiler output:
[no compiler messages]

------------------------------------------------------------------------------
GMM CHUNK 1 OF 16 [ID 0000]
Ruptures: 250; rupture-site rows: 117,500
Distance source: Cell 15 validated chunk outputs

Runtime output tail:
BULK_PARKER_GMM_CHUNK_COMPLETE
rows=117500
GMM chunk 0000 passed all 31 checks.

------------------------------------------------------------------------------
GMM CHUNK 2 OF 16 [ID 0001]
Ruptures: 250; rupture-site rows: 117,500
Distance source: Cell 15 validated chunk outputs

Runtime output tail:
BULK_PARKER_GMM_CHUNK_COMPLETE
rows=117500
GMM chunk 0001 passed all 31 checks.

------------------------------------------------

In [34]:
# ============================================================================
# CELL 18
# Controlled baseline stochastic ground-motion field simulation.
#
# Purpose
# -------
# 1. Read the accepted Cell 17 bulk Parker GMM parameters.
# 2. Recover the four deterministic Cell 16 control ruptures and all 470
#    Seaside W2 sites for each rupture.
# 3. Generate repeated controlled event occurrences using the epi-off branch.
# 4. Apply one between-event residual shared across all sites in an occurrence.
# 5. Apply within-event residuals that are independent across sites in the
#    Phase 1 baseline.
# 6. Correlate PGA and SA(0.4 s) residuals at the same event/site using the
#    Baker and Jayaram (2008) inter-period residual correlation model while
#    retaining independence across different sites.
# 7. Use deterministic, identifier-based random streams that are invariant to
#    chunk order and can be reused in the later spatial-correlation extension.
# 8. Validate reproducibility, residual structure, empirical moments,
#    cross-IMT correlation, equation reconstruction, coverage, and restart
#    behavior before applying the simulator to the full annual event catalog.
#
# This is a controlled validation cell. It does not yet simulate every event
# occurrence in the annual stochastic catalog. That production run belongs in
# the next cell after this controlled implementation passes.
# ============================================================================

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import gzip
import hashlib
import json
import math
import os
import shutil
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Configuration and required earlier-cell variables
# ----------------------------------------------------------------------------

PIPELINE_VERSION = "notebook4_cell18_controlled_baseline_fields_v1"
EXPECTED_RUPTURES = 3_927
EXPECTED_SITES = 470
EXPECTED_CELL17_ROWS = EXPECTED_RUPTURES * EXPECTED_SITES
EXPECTED_CONTROL_RUPTURES = 4

PRIMARY_BASELINE_BRANCH = "epi-off"
SEASIDE_W2_PERIOD_S = 0.40
PGA_PROXY_PERIOD_S = 0.01

# A controlled sample of 128 occurrences for each of four rupture templates
# produces 512 event occurrences and 240,640 event-site rows. This is large
# enough for stable stochastic checks but still modest compared with the full
# annual-catalog production simulation.
CONTROL_REALIZATIONS_PER_RUPTURE = int(
    os.environ.get("NOTEBOOK4_CELL18_REALIZATIONS_PER_RUPTURE", "128")
)
BASE_RANDOM_SEED = int(
    os.environ.get("NOTEBOOK4_CELL18_BASE_SEED", "20260731")
)
CELL17_READ_CHUNK_SIZE = int(
    os.environ.get("NOTEBOOK4_CELL18_CELL17_READ_CHUNK_SIZE", "150000")
)
FINAL_READ_CHUNK_SIZE = int(
    os.environ.get("NOTEBOOK4_CELL18_FINAL_READ_CHUNK_SIZE", "100000")
)
VERIFY_EXISTING_HASHES = True

if CONTROL_REALIZATIONS_PER_RUPTURE < 64:
    raise ValueError(
        "Use at least 64 controlled realizations per rupture so the empirical "
        "normality and correlation checks are meaningful."
    )
if CONTROL_REALIZATIONS_PER_RUPTURE > 2_000:
    raise ValueError(
        "Use no more than 2,000 controlled realizations per rupture in Cell 18."
    )
if CELL17_READ_CHUNK_SIZE <= 0 or FINAL_READ_CHUNK_SIZE <= 0:
    raise ValueError("Chunk sizes must be positive.")

# Numerical tolerances for exact algebraic and provenance checks.
PERIOD_MATCH_ATOL = 1.0e-10
EXACT_RECONSTRUCTION_ATOL = 2.0e-12
MEDIAN_MATCH_ATOL = 2.0e-12
SIGMA_MATCH_ATOL = 5.0e-12
RHO_FORMULA_ATOL = 2.0e-14

# Statistical acceptance bounds. These are validation bounds, not model
# parameters. They are intentionally generous enough to avoid rejecting a
# correct finite Monte Carlo sample while still catching stream reuse,
# incorrect scaling, or accidental spatial dependence.
ETA_MEAN_ABS_MAX = 0.15
ETA_STD_MIN = 0.80
ETA_STD_MAX = 1.20
EPSILON_MEAN_ABS_MAX = 0.010
EPSILON_STD_MIN = 0.990
EPSILON_STD_MAX = 1.010
TOTAL_RESIDUAL_MEAN_ABS_MAX = 0.080
TOTAL_RESIDUAL_STD_MIN = 0.90
TOTAL_RESIDUAL_STD_MAX = 1.10
EVENT_CROSS_IMT_RHO_ATOL = 0.12
SITE_CROSS_IMT_RHO_ATOL = 0.010
MAX_ABS_CROSS_SITE_CORRELATION = 0.22
MAX_ABS_EVENT_SITE_MEAN_CORRELATION = 0.20

# Per-rupture chunk event samples contain only CONTROL_REALIZATIONS_PER_RUPTURE
# observations, so their diagnostics use wider finite-sample bounds. The final
# combined validation applies the tighter bounds above to all occurrences.
CHUNK_ETA_MEAN_ABS_MAX = 0.30
CHUNK_ETA_STD_MIN = 0.65
CHUNK_ETA_STD_MAX = 1.35

required_variables = ["DATA_DIR", "METADATA_DIR"]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Cell 18 requires variables created by earlier Notebook 4 cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

DATA_DIR = Path(DATA_DIR).resolve()
METADATA_DIR = Path(METADATA_DIR).resolve()


# ----------------------------------------------------------------------------
# 2. Input and output paths
# ----------------------------------------------------------------------------

CELL17_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_bulk_parker_gmm"
CELL17_FINAL_OUTPUT_INPUT_PATH = Path(
    globals().get(
        "CELL17_FINAL_OUTPUT_PATH",
        CELL17_OUTPUT_DIR / "bulk_parker_gmm_parameters.csv.gz",
    )
).resolve()
CELL17_SUMMARY_INPUT_PATH = Path(
    globals().get(
        "CELL17_FINAL_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_summary.json",
    )
).resolve()
CELL17_VALIDATION_INPUT_PATH = Path(
    globals().get(
        "CELL17_FINAL_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_validation.csv",
    )
).resolve()
CELL16_SUMMARY_INPUT_PATH = Path(
    globals().get(
        "CELL16_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_cell_16_controlled_gmm_summary.json",
    )
).resolve()
CELL16_VALIDATION_INPUT_PATH = Path(
    globals().get(
        "VALIDATION_PATH",
        METADATA_DIR / "notebook_4_cell_16_controlled_gmm_validation.csv",
    )
).resolve()

required_paths = {
    "Cell 17 bulk GMM parameters": CELL17_FINAL_OUTPUT_INPUT_PATH,
    "Cell 17 summary": CELL17_SUMMARY_INPUT_PATH,
    "Cell 17 validation": CELL17_VALIDATION_INPUT_PATH,
    "Cell 16 summary": CELL16_SUMMARY_INPUT_PATH,
    "Cell 16 validation": CELL16_VALIDATION_INPUT_PATH,
}
missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]
if missing_paths:
    raise FileNotFoundError(
        "Cell 18 is missing required validated inputs:\n"
        + "\n".join(f"  - {item}" for item in missing_paths)
    )

CELL18_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_controlled_baseline_fields"
CELL18_WORK_DIR = METADATA_DIR / "notebook_4_controlled_baseline_fields_work"
CELL18_TEMPLATE_DIR = CELL18_WORK_DIR / "templates"
CELL18_FIELD_CHUNK_DIR = CELL18_WORK_DIR / "field_chunks"
CELL18_EVENT_CHUNK_DIR = CELL18_WORK_DIR / "event_chunks"
CELL18_MARKER_DIR = CELL18_WORK_DIR / "completion_markers"
CELL18_CHUNK_VALIDATION_DIR = CELL18_WORK_DIR / "chunk_validation"

for directory in [
    CELL18_OUTPUT_DIR,
    CELL18_WORK_DIR,
    CELL18_TEMPLATE_DIR,
    CELL18_FIELD_CHUNK_DIR,
    CELL18_EVENT_CHUNK_DIR,
    CELL18_MARKER_DIR,
    CELL18_CHUNK_VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CELL18_CONTROL_TEMPLATE_PATH = (
    METADATA_DIR / "notebook_4_cell_18_control_rupture_gmm_templates.csv.gz"
)
CELL18_SITE_ORDER_PATH = METADATA_DIR / "notebook_4_cell_18_site_order.csv"
CELL18_RANDOM_SPEC_PATH = (
    METADATA_DIR / "notebook_4_cell_18_random_stream_specification.json"
)
CELL18_MANIFEST_PATH = METADATA_DIR / "notebook_4_cell_18_chunk_manifest.csv"
CELL18_FINAL_FIELDS_PATH = (
    CELL18_OUTPUT_DIR / "controlled_baseline_ground_motion_fields.csv.gz"
)
CELL18_FINAL_EVENTS_PATH = (
    CELL18_OUTPUT_DIR / "controlled_event_residual_ledger.csv.gz"
)
CELL18_VALIDATION_PATH = (
    METADATA_DIR / "notebook_4_cell_18_controlled_field_validation.csv"
)
CELL18_SUMMARY_PATH = (
    METADATA_DIR / "notebook_4_cell_18_controlled_field_summary.json"
)


# ----------------------------------------------------------------------------
# 3. Utility functions
# ----------------------------------------------------------------------------

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            block = file_handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def canonical_json_bytes(value: object) -> bytes:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
        allow_nan=False,
    ).encode("utf-8")


def dataframe_sha256(frame: pd.DataFrame, columns: list[str]) -> str:
    text = frame[columns].to_csv(
        index=False,
        na_rep="",
        lineterminator="\n",
        float_format="%.17g",
    )
    return sha256_bytes(text.encode("utf-8"))


def normalize_boolean_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    return (
        values.astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def stable_seed(*parts: object) -> tuple[int, str]:
    """Return a deterministic 128-bit PCG seed and its hexadecimal label."""
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    raw = hashlib.sha256(payload).digest()[:16]
    return int.from_bytes(raw, byteorder="little", signed=False), raw.hex()


def rng_from_seed(seed: int) -> np.random.Generator:
    return np.random.Generator(np.random.PCG64DXSM(seed))


def atomic_write_json(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def atomic_write_csv(frame: pd.DataFrame, path: Path, **kwargs) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, **kwargs)
    os.replace(temporary, path)


def combine_gzip_csv_files(source_paths: list[Path], destination_path: Path) -> None:
    """Concatenate gzip CSV files while retaining only the first header."""
    temporary = destination_path.with_suffix(destination_path.suffix + ".tmp")
    header: bytes | None = None
    with temporary.open("wb") as raw_destination:
        with gzip.GzipFile(
            fileobj=raw_destination, mode="wb", compresslevel=1, mtime=0
        ) as destination:
            for source_index, source_path in enumerate(source_paths):
                with gzip.open(source_path, "rb") as source:
                    current_header = source.readline()
                    if header is None:
                        header = current_header
                        destination.write(current_header)
                    elif current_header != header:
                        raise RuntimeError(
                            "Controlled chunk CSV headers are inconsistent:\n"
                            f"  {source_path}"
                        )
                    shutil.copyfileobj(source, destination, length=1024 * 1024)
    os.replace(temporary, destination_path)


def safe_correlation(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3 or np.std(x, ddof=1) == 0.0 or np.std(y, ddof=1) == 0.0:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


# ----------------------------------------------------------------------------
# 4. Baker and Jayaram (2008) inter-period residual correlation
# ----------------------------------------------------------------------------

def baker_jayaram_2008_correlation(period_1_s: float, period_2_s: float) -> float:
    """
    Baker and Jayaram (2008), Earthquake Spectra 24(1), 299-317,
    DOI 10.1193/1.2857544, Equations 5 and 6.

    PGA is represented by the 0.01 s spectral-period proxy. The model is used
    here only for correlation across IMTs at the same event/site. It does not
    introduce spatial correlation between different sites.
    """
    t_min = min(float(period_1_s), float(period_2_s))
    t_max = max(float(period_1_s), float(period_2_s))
    if t_min <= 0.0:
        t_min = PGA_PROXY_PERIOD_S
    if t_max <= 0.0:
        t_max = PGA_PROXY_PERIOD_S
    if not (0.01 <= t_min <= 10.0 and 0.01 <= t_max <= 10.0):
        raise ValueError("Baker-Jayaram periods must lie in [0.01, 10] s.")

    c1 = 1.0 - math.cos(
        math.pi / 2.0
        - 0.366 * math.log(t_max / max(t_min, 0.109))
    )
    if t_max < 0.2:
        c2 = 1.0 - 0.105 * (
            1.0 - 1.0 / (1.0 + math.exp(100.0 * t_max - 5.0))
        ) * (t_max - t_min) / (t_max - 0.0099)
    else:
        c2 = 0.0
    c3 = c2 if t_max < 0.109 else c1
    c4 = c1 + 0.5 * (math.sqrt(c3) - c3) * (
        1.0 + math.cos(math.pi * t_min / 0.109)
    )

    if t_max <= 0.109:
        rho = c2
    elif t_min > 0.109:
        rho = c1
    elif t_max < 0.2:
        rho = min(c2, c4)
    else:
        rho = c4
    return float(rho)


CROSS_IMT_RHO = baker_jayaram_2008_correlation(
    PGA_PROXY_PERIOD_S,
    SEASIDE_W2_PERIOD_S,
)
EXPECTED_PGA_SA0P4_RHO = 0.7321409900247263
if not math.isclose(
    CROSS_IMT_RHO,
    EXPECTED_PGA_SA0P4_RHO,
    rel_tol=0.0,
    abs_tol=RHO_FORMULA_ATOL,
):
    raise RuntimeError(
        "The Baker-Jayaram PGA-SA0P4 correlation does not reproduce the "
        "expected reference value."
    )
CROSS_IMT_COMPLEMENT = math.sqrt(1.0 - CROSS_IMT_RHO**2)


# ----------------------------------------------------------------------------
# 5. Validate Cell 16 and Cell 17 prerequisites
# ----------------------------------------------------------------------------

cell17_summary = json.loads(
    CELL17_SUMMARY_INPUT_PATH.read_text(encoding="utf-8")
)
cell16_summary = json.loads(
    CELL16_SUMMARY_INPUT_PATH.read_text(encoding="utf-8")
)
cell17_validation = pd.read_csv(CELL17_VALIDATION_INPUT_PATH, low_memory=False)
cell16_validation = pd.read_csv(CELL16_VALIDATION_INPUT_PATH, low_memory=False)

for name, summary in [
    ("Cell 16", cell16_summary),
    ("Cell 17", cell17_summary),
]:
    if not bool(summary.get("all_checks_passed", False)):
        raise RuntimeError(f"{name} summary does not report all checks passed.")

for name, validation in [
    ("Cell 16", cell16_validation),
    ("Cell 17", cell17_validation),
]:
    if "passes" not in validation.columns:
        raise RuntimeError(f"{name} validation has no 'passes' column.")
    passes = normalize_boolean_series(validation["passes"])
    if not passes.all():
        failed = validation.loc[~passes, "check"].astype(str).tolist()
        raise RuntimeError(
            f"{name} validation contains failed checks:\n"
            + "\n".join(f"  - {item}" for item in failed)
        )

if int(cell17_summary.get("ruptures", -1)) != EXPECTED_RUPTURES:
    raise RuntimeError("Cell 17 rupture count differs from 3,927.")
if int(cell17_summary.get("sites", -1)) != EXPECTED_SITES:
    raise RuntimeError("Cell 17 site count differs from 470.")
if int(cell17_summary.get("rupture_site_rows", -1)) != EXPECTED_CELL17_ROWS:
    raise RuntimeError("Cell 17 row count differs from 1,845,690.")
if str(cell17_summary.get("primary_baseline_branch")) != PRIMARY_BASELINE_BRANCH:
    raise RuntimeError("Cell 17 primary branch is not epi-off.")
if not math.isclose(
    float(cell17_summary.get("period_s", float("nan"))),
    SEASIDE_W2_PERIOD_S,
    rel_tol=0.0,
    abs_tol=PERIOD_MATCH_ATOL,
):
    raise RuntimeError("Cell 17 period is not exactly 0.40 s.")
if str(cell17_summary.get("period_resolution")) != "exact USGS SA0P4; no interpolation":
    raise RuntimeError("Cell 17 period resolution is not exact USGS SA0P4.")

expected_cell17_hash = str(cell17_summary.get("final_output_sha256", ""))
actual_cell17_hash = sha256_file(CELL17_FINAL_OUTPUT_INPUT_PATH)
if expected_cell17_hash != actual_cell17_hash:
    raise RuntimeError(
        "Cell 17 bulk GMM output hash differs from its accepted summary."
    )

control_cases_raw = cell16_summary.get("control_cases", [])
if not isinstance(control_cases_raw, list):
    raise RuntimeError("Cell 16 summary control_cases is not a list.")
control_cases = pd.DataFrame(control_cases_raw)
required_control_columns = ["case_id", "rupture_id", "source_type"]
missing_control_columns = [
    column for column in required_control_columns if column not in control_cases
]
if missing_control_columns:
    raise RuntimeError(
        "Cell 16 summary lacks controlled-case fields:\n"
        + "\n".join(f"  - {column}" for column in missing_control_columns)
    )
control_cases = control_cases[required_control_columns].copy()
for column in required_control_columns:
    control_cases[column] = (
        control_cases[column].astype("string").fillna("").astype(str)
    )
expected_case_order = [
    "interface_nearest",
    "interface_farthest",
    "slab_nearest",
    "slab_farthest",
]
if set(control_cases["case_id"]) != set(expected_case_order):
    raise RuntimeError(
        "Cell 16 did not provide the expected four source-distance controls."
    )
if control_cases["rupture_id"].nunique() != EXPECTED_CONTROL_RUPTURES:
    raise RuntimeError("Cell 16 controls do not contain four unique ruptures.")
if set(control_cases["source_type"]) != {"INTERFACE", "SLAB"}:
    raise RuntimeError("Cell 16 controls do not cover interface and slab sources.")
control_cases["case_order"] = control_cases["case_id"].map(
    {case_id: index for index, case_id in enumerate(expected_case_order)}
)
control_cases = control_cases.sort_values("case_order").reset_index(drop=True)

print(
    "Cell 17 prerequisite passed: "
    "1,845,690 validated epi-off rupture-site parameter rows."
)
print(
    "Cross-IMT residual correlation: "
    f"Baker-Jayaram rho(PGA, SA0P4) = {CROSS_IMT_RHO:.12f}"
)


# ----------------------------------------------------------------------------
# 6. Extract all 470 Cell 17 rows for the four controlled ruptures
# ----------------------------------------------------------------------------

cell17_template_columns = [
    "gmm_chunk_id",
    "rupture_ordinal",
    "event_id",
    "rupture_id",
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
    "source_type",
    "gmm_name",
    "surface_class",
    "magnitude",
    "raw_annual_rate",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
    "period_s",
    "vs30_mps",
    "pga_epi_off_mean_ln_g",
    "pga_epi_off_median_g",
    "pga_tau_ln",
    "pga_phi_ln",
    "pga_sigma_reconstructed_ln",
    "sa0p4_epi_off_mean_ln_g",
    "sa0p4_epi_off_median_g",
    "sa0p4_tau_ln",
    "sa0p4_phi_ln",
    "sa0p4_sigma_reconstructed_ln",
    "calculation_error",
]

selected_rupture_ids = set(control_cases["rupture_id"])
template_frames: list[pd.DataFrame] = []
for frame in pd.read_csv(
    CELL17_FINAL_OUTPUT_INPUT_PATH,
    usecols=cell17_template_columns,
    dtype={
        "gmm_chunk_id": str,
        "event_id": str,
        "rupture_id": str,
        "site_id": str,
        "source_type": str,
        "gmm_name": str,
        "surface_class": str,
        "calculation_error": str,
    },
    chunksize=CELL17_READ_CHUNK_SIZE,
    low_memory=False,
):
    subset = frame.loc[frame["rupture_id"].isin(selected_rupture_ids)].copy()
    if not subset.empty:
        template_frames.append(subset)

if not template_frames:
    raise RuntimeError("None of the Cell 16 control ruptures were found in Cell 17.")
control_templates = pd.concat(template_frames, ignore_index=True)
control_templates["calculation_error"] = (
    control_templates["calculation_error"].fillna("").astype(str)
)
if control_templates["calculation_error"].ne("").any():
    raise RuntimeError("A controlled Cell 17 template row has a calculation error.")

control_templates = control_templates.merge(
    control_cases[["case_id", "case_order", "rupture_id"]],
    on="rupture_id",
    how="left",
    validate="many_to_one",
)
control_templates = control_templates.rename(columns={"case_id": "control_case_id"})

numeric_template_columns = [
    "rupture_ordinal",
    "site_ordinal",
    "site_longitude",
    "site_latitude",
    "magnitude",
    "raw_annual_rate",
    "r_jb_km",
    "r_rup_km",
    "r_x_km",
    "period_s",
    "vs30_mps",
    "pga_epi_off_mean_ln_g",
    "pga_epi_off_median_g",
    "pga_tau_ln",
    "pga_phi_ln",
    "pga_sigma_reconstructed_ln",
    "sa0p4_epi_off_mean_ln_g",
    "sa0p4_epi_off_median_g",
    "sa0p4_tau_ln",
    "sa0p4_phi_ln",
    "sa0p4_sigma_reconstructed_ln",
]
for column in numeric_template_columns:
    control_templates[column] = pd.to_numeric(
        control_templates[column], errors="coerce"
    )
if not np.isfinite(
    control_templates[numeric_template_columns].to_numpy(dtype=float)
).all():
    raise RuntimeError("Controlled Cell 17 templates contain non-finite values.")

control_templates["rupture_ordinal"] = np.round(
    control_templates["rupture_ordinal"]
).astype(np.int64)
control_templates["site_ordinal"] = np.round(
    control_templates["site_ordinal"]
).astype(np.int64)
control_templates = control_templates.sort_values(
    ["case_order", "site_ordinal"]
).reset_index(drop=True)

expected_template_rows = EXPECTED_CONTROL_RUPTURES * EXPECTED_SITES
if len(control_templates) != expected_template_rows:
    raise RuntimeError(
        f"Expected {expected_template_rows:,} controlled template rows; "
        f"observed {len(control_templates):,}."
    )
if control_templates.duplicated(["rupture_id", "site_id"]).any():
    raise RuntimeError("Controlled templates contain duplicate rupture-site pairs.")
rupture_site_counts = control_templates.groupby("rupture_id").size()
if not (
    len(rupture_site_counts) == EXPECTED_CONTROL_RUPTURES
    and rupture_site_counts.eq(EXPECTED_SITES).all()
):
    raise RuntimeError("Each controlled rupture must contain all 470 sites.")
if control_templates["site_id"].nunique() != EXPECTED_SITES:
    raise RuntimeError("Controlled templates do not contain exactly 470 sites.")
if not np.isclose(
    control_templates["period_s"].to_numpy(dtype=float),
    SEASIDE_W2_PERIOD_S,
    rtol=0.0,
    atol=PERIOD_MATCH_ATOL,
).all():
    raise RuntimeError("A controlled template period differs from 0.40 s.")

for prefix in ["pga", "sa0p4"]:
    median_difference = np.abs(
        np.exp(control_templates[f"{prefix}_epi_off_mean_ln_g"].to_numpy())
        - control_templates[f"{prefix}_epi_off_median_g"].to_numpy()
    )
    if float(median_difference.max()) > MEDIAN_MATCH_ATOL:
        raise RuntimeError(f"Controlled {prefix} medians do not equal exp(mean).")
    sigma_difference = np.abs(
        np.sqrt(
            control_templates[f"{prefix}_tau_ln"].to_numpy() ** 2
            + control_templates[f"{prefix}_phi_ln"].to_numpy() ** 2
        )
        - control_templates[f"{prefix}_sigma_reconstructed_ln"].to_numpy()
    )
    if float(sigma_difference.max()) > SIGMA_MATCH_ATOL:
        raise RuntimeError(f"Controlled {prefix} tau/phi do not reconstruct sigma.")

# Validate that site identifiers and ordering are identical for every rupture.
site_order = (
    control_templates.loc[
        control_templates["control_case_id"].eq(expected_case_order[0]),
        ["site_ordinal", "site_id", "site_longitude", "site_latitude"],
    ]
    .sort_values("site_ordinal")
    .reset_index(drop=True)
)
if not np.array_equal(site_order["site_ordinal"].to_numpy(), np.arange(EXPECTED_SITES)):
    raise RuntimeError("Controlled site ordinals are not the contiguous range 0..469.")
for case_id in expected_case_order[1:]:
    current = (
        control_templates.loc[
            control_templates["control_case_id"].eq(case_id),
            ["site_ordinal", "site_id", "site_longitude", "site_latitude"],
        ]
        .sort_values("site_ordinal")
        .reset_index(drop=True)
    )
    if not current.equals(site_order):
        raise RuntimeError("Site ordering or coordinates differ across control ruptures.")

control_templates.to_csv(
    CELL18_CONTROL_TEMPLATE_PATH,
    index=False,
    compression={"method": "gzip", "compresslevel": 1, "mtime": 0},
    na_rep="",
    float_format="%.17g",
)
site_order.to_csv(
    CELL18_SITE_ORDER_PATH,
    index=False,
    float_format="%.17g",
)


# ----------------------------------------------------------------------------
# 7. Freeze the random-stream specification
# ----------------------------------------------------------------------------

random_stream_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "base_random_seed": BASE_RANDOM_SEED,
    "generator": "numpy.random.PCG64DXSM",
    "seed_derivation": (
        "First 128 bits of SHA-256 over pipe-delimited identifiers; separate "
        "streams for event-level and site-level latent standard normals."
    ),
    "occurrence_identity": (
        "control_case_id + rupture_id + realization_index; Cell 17 event_id is "
        "retained only as rupture-template provenance and is not reused as an "
        "event-occurrence identifier."
    ),
    "site_order_requirement": (
        "The site-level generator emits values in validated site_ordinal order "
        "0..469. The site-order CSV hash freezes this mapping."
    ),
    "aleatory_equation": (
        "ln(IM_e,s) = mu_epi_off_e,s + tau_e,s*eta_e + phi_e,s*epsilon_e,s"
    ),
    "between_event_structure": (
        "One eta per event occurrence and IMT; eta is repeated exactly across "
        "all 470 sites."
    ),
    "within_event_spatial_structure": (
        "Independent latent standard normals across sites. No spatial "
        "correlation is applied in the Phase 1 baseline."
    ),
    "cross_imt_structure": {
        "model": "Baker and Jayaram (2008) inter-period residual correlation",
        "doi": "10.1193/1.2857544",
        "pga_proxy_period_s": PGA_PROXY_PERIOD_S,
        "sa_period_s": SEASIDE_W2_PERIOD_S,
        "rho": CROSS_IMT_RHO,
        "application": (
            "The same rho is applied separately to between-event and within-"
            "event normalized residual pairs. This follows the paper's use of "
            "a common correlation model for inter-event, intra-event, and total "
            "residuals."
        ),
    },
    "latent_transform": {
        "pga": "z1",
        "sa0p4": "rho*z1 + sqrt(1-rho^2)*z2",
    },
    "future_spatial_extension": (
        "Retain z_site_latent_1 and z_site_latent_2. A future spatial case can "
        "apply a source/period-specific spatial transform to each latent site "
        "vector before the cross-IMT combination, preserving the same base "
        "random numbers and event catalog."
    ),
    "control_realizations_per_rupture": CONTROL_REALIZATIONS_PER_RUPTURE,
    "control_ruptures": EXPECTED_CONTROL_RUPTURES,
    "sites": EXPECTED_SITES,
    "site_order_sha256": sha256_file(CELL18_SITE_ORDER_PATH),
    "cell17_output_sha256": actual_cell17_hash,
    "control_template_sha256": sha256_file(CELL18_CONTROL_TEMPLATE_PATH),
}
random_specification_hash = sha256_bytes(
    canonical_json_bytes(random_stream_specification)
)
random_stream_specification["specification_sha256"] = random_specification_hash
atomic_write_json(CELL18_RANDOM_SPEC_PATH, random_stream_specification)


# ----------------------------------------------------------------------------
# 8. Chunk records and restart-marker validation
# ----------------------------------------------------------------------------

manifest_records: list[dict[str, object]] = []
for _, case in control_cases.iterrows():
    case_id = str(case["case_id"])
    rupture_id = str(case["rupture_id"])
    template = control_templates.loc[
        control_templates["control_case_id"].eq(case_id)
    ].copy()
    template_columns_for_hash = [
        "control_case_id",
        "rupture_ordinal",
        "event_id",
        "rupture_id",
        "site_ordinal",
        "site_id",
        "site_longitude",
        "site_latitude",
        "source_type",
        "gmm_name",
        "magnitude",
        "r_rup_km",
        "period_s",
        "pga_epi_off_mean_ln_g",
        "pga_tau_ln",
        "pga_phi_ln",
        "sa0p4_epi_off_mean_ln_g",
        "sa0p4_tau_ln",
        "sa0p4_phi_ln",
    ]
    template_hash = dataframe_sha256(template, template_columns_for_hash)
    template_path = CELL18_TEMPLATE_DIR / f"control_template_{case_id}.csv.gz"
    template.to_csv(
        template_path,
        index=False,
        compression={"method": "gzip", "compresslevel": 1, "mtime": 0},
        na_rep="",
        float_format="%.17g",
    )
    manifest_records.append(
        {
            "case_order": int(case["case_order"]),
            "control_case_id": case_id,
            "rupture_id": rupture_id,
            "source_type": str(case["source_type"]),
            "template_path": str(template_path),
            "template_sha256": template_hash,
            "event_output": str(
                CELL18_EVENT_CHUNK_DIR / f"controlled_events_{case_id}.csv.gz"
            ),
            "field_output": str(
                CELL18_FIELD_CHUNK_DIR / f"controlled_fields_{case_id}.csv.gz"
            ),
            "validation_output": str(
                CELL18_CHUNK_VALIDATION_DIR / f"controlled_{case_id}_validation.csv"
            ),
            "marker_path": str(
                CELL18_MARKER_DIR / f"controlled_{case_id}_complete.json"
            ),
            "expected_event_rows": CONTROL_REALIZATIONS_PER_RUPTURE,
            "expected_field_rows": CONTROL_REALIZATIONS_PER_RUPTURE * EXPECTED_SITES,
            "status": "pending",
        }
    )


def save_manifest() -> None:
    frame = pd.DataFrame(manifest_records).sort_values("case_order")
    atomic_write_csv(frame, CELL18_MANIFEST_PATH)


save_manifest()


def marker_is_valid(record: dict[str, object]) -> bool:
    marker_path = Path(str(record["marker_path"]))
    event_output = Path(str(record["event_output"]))
    field_output = Path(str(record["field_output"]))
    validation_output = Path(str(record["validation_output"]))
    if not all(path.is_file() for path in [
        marker_path,
        event_output,
        field_output,
        validation_output,
    ]):
        return False
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        validation = pd.read_csv(validation_output, low_memory=False)
    except (OSError, json.JSONDecodeError, pd.errors.ParserError):
        return False
    required_marker_values = {
        "pipeline_version": PIPELINE_VERSION,
        "random_specification_sha256": random_specification_hash,
        "template_sha256": str(record["template_sha256"]),
        "base_random_seed": BASE_RANDOM_SEED,
        "control_realizations_per_rupture": CONTROL_REALIZATIONS_PER_RUPTURE,
        "expected_event_rows": int(record["expected_event_rows"]),
        "expected_field_rows": int(record["expected_field_rows"]),
    }
    for key, value in required_marker_values.items():
        if marker.get(key) != value:
            return False
    if "passes" not in validation.columns:
        return False
    if not normalize_boolean_series(validation["passes"]).all():
        return False
    if VERIFY_EXISTING_HASHES:
        if marker.get("event_output_sha256") != sha256_file(event_output):
            return False
        if marker.get("field_output_sha256") != sha256_file(field_output):
            return False
        if marker.get("validation_sha256") != sha256_file(validation_output):
            return False
    return True


# ----------------------------------------------------------------------------
# 9. Generate one controlled rupture chunk
# ----------------------------------------------------------------------------

field_output_columns = [
    "control_case_id",
    "controlled_occurrence_ordinal",
    "controlled_occurrence_id",
    "realization_index",
    "rupture_ordinal",
    "rupture_template_event_id",
    "rupture_id",
    "source_type",
    "gmm_name",
    "magnitude",
    "raw_annual_rate",
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
    "r_rup_km",
    "vs30_mps",
    "period_s",
    "cross_imt_rho",
    "z_site_latent_1",
    "z_site_latent_2",
    "eta_pga",
    "epsilon_pga",
    "pga_epi_off_mean_ln_g",
    "pga_tau_ln",
    "pga_phi_ln",
    "pga_between_event_shift_ln",
    "pga_within_event_shift_ln",
    "pga_simulated_ln_g",
    "pga_simulated_g",
    "eta_sa0p4",
    "epsilon_sa0p4",
    "sa0p4_epi_off_mean_ln_g",
    "sa0p4_tau_ln",
    "sa0p4_phi_ln",
    "sa0p4_between_event_shift_ln",
    "sa0p4_within_event_shift_ln",
    "sa0p4_simulated_ln_g",
    "sa0p4_simulated_g",
]

event_output_columns = [
    "control_case_id",
    "controlled_occurrence_ordinal",
    "controlled_occurrence_id",
    "realization_index",
    "rupture_ordinal",
    "rupture_template_event_id",
    "rupture_id",
    "source_type",
    "event_seed_hex",
    "site_seed_hex",
    "z_event_latent_1",
    "z_event_latent_2",
    "eta_pga",
    "eta_sa0p4",
    "cross_imt_rho",
]


def build_validation_frame(checks: list[dict[str, object]]) -> pd.DataFrame:
    return pd.DataFrame(checks, columns=[
        "category", "check", "observed", "expected", "passes", "note"
    ])


def add_check(
    checks: list[dict[str, object]],
    category: str,
    check: str,
    observed: object,
    expected: object,
    passes: bool,
    note: str = "",
) -> None:
    checks.append(
        {
            "category": category,
            "check": check,
            "observed": observed,
            "expected": expected,
            "passes": bool(passes),
            "note": note,
        }
    )


def generate_control_chunk(record: dict[str, object]) -> pd.DataFrame:
    case_id = str(record["control_case_id"])
    case_order = int(record["case_order"])
    rupture_id = str(record["rupture_id"])
    template = pd.read_csv(
        Path(str(record["template_path"])),
        dtype={
            "event_id": str,
            "rupture_id": str,
            "site_id": str,
            "source_type": str,
            "gmm_name": str,
            "control_case_id": str,
        },
        low_memory=False,
    ).sort_values("site_ordinal").reset_index(drop=True)

    if len(template) != EXPECTED_SITES:
        raise RuntimeError(f"{case_id} template does not contain 470 sites.")
    if template["rupture_id"].nunique() != 1 or template["rupture_id"].iloc[0] != rupture_id:
        raise RuntimeError(f"{case_id} template rupture identity is inconsistent.")

    n_realizations = CONTROL_REALIZATIONS_PER_RUPTURE
    n_sites = EXPECTED_SITES
    occurrence_base = case_order * n_realizations

    z_event_1 = np.empty(n_realizations, dtype=np.float64)
    z_event_2 = np.empty(n_realizations, dtype=np.float64)
    z_site_1 = np.empty((n_realizations, n_sites), dtype=np.float64)
    z_site_2 = np.empty((n_realizations, n_sites), dtype=np.float64)
    event_seed_hex: list[str] = []
    site_seed_hex: list[str] = []

    for realization_index in range(n_realizations):
        event_seed, event_hex = stable_seed(
            PIPELINE_VERSION,
            BASE_RANDOM_SEED,
            case_id,
            rupture_id,
            realization_index,
            "event_latent_pair",
        )
        site_seed, site_hex = stable_seed(
            PIPELINE_VERSION,
            BASE_RANDOM_SEED,
            case_id,
            rupture_id,
            realization_index,
            "site_latent_pair",
        )
        event_rng = rng_from_seed(event_seed)
        site_rng = rng_from_seed(site_seed)
        event_pair = event_rng.standard_normal(2)
        site_pair = site_rng.standard_normal((2, n_sites))
        z_event_1[realization_index] = event_pair[0]
        z_event_2[realization_index] = event_pair[1]
        z_site_1[realization_index, :] = site_pair[0, :]
        z_site_2[realization_index, :] = site_pair[1, :]
        event_seed_hex.append(event_hex)
        site_seed_hex.append(site_hex)

    eta_pga = z_event_1
    eta_sa0p4 = CROSS_IMT_RHO * z_event_1 + CROSS_IMT_COMPLEMENT * z_event_2
    epsilon_pga = z_site_1
    epsilon_sa0p4 = (
        CROSS_IMT_RHO * z_site_1 + CROSS_IMT_COMPLEMENT * z_site_2
    )

    mu_pga = template["pga_epi_off_mean_ln_g"].to_numpy(dtype=float)
    tau_pga = template["pga_tau_ln"].to_numpy(dtype=float)
    phi_pga = template["pga_phi_ln"].to_numpy(dtype=float)
    mu_sa = template["sa0p4_epi_off_mean_ln_g"].to_numpy(dtype=float)
    tau_sa = template["sa0p4_tau_ln"].to_numpy(dtype=float)
    phi_sa = template["sa0p4_phi_ln"].to_numpy(dtype=float)

    pga_between = eta_pga[:, None] * tau_pga[None, :]
    pga_within = epsilon_pga * phi_pga[None, :]
    pga_ln = mu_pga[None, :] + pga_between + pga_within
    pga_g = np.exp(pga_ln)

    sa_between = eta_sa0p4[:, None] * tau_sa[None, :]
    sa_within = epsilon_sa0p4 * phi_sa[None, :]
    sa_ln = mu_sa[None, :] + sa_between + sa_within
    sa_g = np.exp(sa_ln)

    occurrence_ordinals = occurrence_base + np.arange(n_realizations, dtype=np.int64)
    occurrence_ids = [
        f"C18_{case_id}_R{index:04d}" for index in range(n_realizations)
    ]

    event_frame = pd.DataFrame(
        {
            "control_case_id": case_id,
            "controlled_occurrence_ordinal": occurrence_ordinals,
            "controlled_occurrence_id": occurrence_ids,
            "realization_index": np.arange(n_realizations, dtype=np.int64),
            "rupture_ordinal": int(template["rupture_ordinal"].iloc[0]),
            "rupture_template_event_id": str(template["event_id"].iloc[0]),
            "rupture_id": rupture_id,
            "source_type": str(template["source_type"].iloc[0]),
            "event_seed_hex": event_seed_hex,
            "site_seed_hex": site_seed_hex,
            "z_event_latent_1": z_event_1,
            "z_event_latent_2": z_event_2,
            "eta_pga": eta_pga,
            "eta_sa0p4": eta_sa0p4,
            "cross_imt_rho": CROSS_IMT_RHO,
        }
    )[event_output_columns]

    realization_index_flat = np.repeat(
        np.arange(n_realizations, dtype=np.int64), n_sites
    )
    occurrence_ordinal_flat = np.repeat(occurrence_ordinals, n_sites)
    occurrence_id_flat = np.repeat(np.asarray(occurrence_ids, dtype=object), n_sites)

    def repeat_sites(column: str) -> np.ndarray:
        return np.tile(template[column].to_numpy(), n_realizations)

    def repeat_realizations(values: np.ndarray) -> np.ndarray:
        return np.repeat(values, n_sites)

    field_frame = pd.DataFrame(
        {
            "control_case_id": case_id,
            "controlled_occurrence_ordinal": occurrence_ordinal_flat,
            "controlled_occurrence_id": occurrence_id_flat,
            "realization_index": realization_index_flat,
            "rupture_ordinal": int(template["rupture_ordinal"].iloc[0]),
            "rupture_template_event_id": str(template["event_id"].iloc[0]),
            "rupture_id": rupture_id,
            "source_type": str(template["source_type"].iloc[0]),
            "gmm_name": str(template["gmm_name"].iloc[0]),
            "magnitude": float(template["magnitude"].iloc[0]),
            "raw_annual_rate": float(template["raw_annual_rate"].iloc[0]),
            "site_ordinal": repeat_sites("site_ordinal").astype(np.int64),
            "site_id": repeat_sites("site_id"),
            "site_longitude": repeat_sites("site_longitude").astype(float),
            "site_latitude": repeat_sites("site_latitude").astype(float),
            "r_rup_km": repeat_sites("r_rup_km").astype(float),
            "vs30_mps": repeat_sites("vs30_mps").astype(float),
            "period_s": repeat_sites("period_s").astype(float),
            "cross_imt_rho": CROSS_IMT_RHO,
            "z_site_latent_1": z_site_1.reshape(-1),
            "z_site_latent_2": z_site_2.reshape(-1),
            "eta_pga": repeat_realizations(eta_pga),
            "epsilon_pga": epsilon_pga.reshape(-1),
            "pga_epi_off_mean_ln_g": repeat_sites("pga_epi_off_mean_ln_g").astype(float),
            "pga_tau_ln": repeat_sites("pga_tau_ln").astype(float),
            "pga_phi_ln": repeat_sites("pga_phi_ln").astype(float),
            "pga_between_event_shift_ln": pga_between.reshape(-1),
            "pga_within_event_shift_ln": pga_within.reshape(-1),
            "pga_simulated_ln_g": pga_ln.reshape(-1),
            "pga_simulated_g": pga_g.reshape(-1),
            "eta_sa0p4": repeat_realizations(eta_sa0p4),
            "epsilon_sa0p4": epsilon_sa0p4.reshape(-1),
            "sa0p4_epi_off_mean_ln_g": repeat_sites("sa0p4_epi_off_mean_ln_g").astype(float),
            "sa0p4_tau_ln": repeat_sites("sa0p4_tau_ln").astype(float),
            "sa0p4_phi_ln": repeat_sites("sa0p4_phi_ln").astype(float),
            "sa0p4_between_event_shift_ln": sa_between.reshape(-1),
            "sa0p4_within_event_shift_ln": sa_within.reshape(-1),
            "sa0p4_simulated_ln_g": sa_ln.reshape(-1),
            "sa0p4_simulated_g": sa_g.reshape(-1),
        }
    )[field_output_columns]

    event_output = Path(str(record["event_output"]))
    field_output = Path(str(record["field_output"]))
    temporary_event = event_output.with_suffix(event_output.suffix + ".tmp")
    temporary_field = field_output.with_suffix(field_output.suffix + ".tmp")
    event_frame.to_csv(
        temporary_event,
        index=False,
        compression={"method": "gzip", "compresslevel": 1, "mtime": 0},
        float_format="%.17g",
    )
    field_frame.to_csv(
        temporary_field,
        index=False,
        compression={"method": "gzip", "compresslevel": 1, "mtime": 0},
        float_format="%.17g",
    )
    os.replace(temporary_event, event_output)
    os.replace(temporary_field, field_output)

    checks: list[dict[str, object]] = []
    expected_field_rows = n_realizations * n_sites
    add_check(checks, "Coverage", "Event rows", len(event_frame), n_realizations, len(event_frame) == n_realizations)
    add_check(checks, "Coverage", "Field rows", len(field_frame), expected_field_rows, len(field_frame) == expected_field_rows)
    add_check(checks, "Coverage", "Unique occurrence IDs", event_frame["controlled_occurrence_id"].nunique(), n_realizations, event_frame["controlled_occurrence_id"].nunique() == n_realizations)
    add_check(checks, "Coverage", "Unique occurrence-site pairs", field_frame[["controlled_occurrence_id", "site_id"]].drop_duplicates().shape[0], expected_field_rows, not field_frame.duplicated(["controlled_occurrence_id", "site_id"]).any())
    occurrence_counts = field_frame.groupby("controlled_occurrence_id").size()
    site_counts = field_frame.groupby("site_id").size()
    add_check(checks, "Coverage", "Every occurrence has 470 sites", int(occurrence_counts.eq(n_sites).sum()), n_realizations, len(occurrence_counts) == n_realizations and occurrence_counts.eq(n_sites).all())
    add_check(checks, "Coverage", "Every site has every controlled occurrence", int(site_counts.eq(n_realizations).sum()), n_sites, len(site_counts) == n_sites and site_counts.eq(n_realizations).all())

    pga_reconstructed = (
        field_frame["pga_epi_off_mean_ln_g"]
        + field_frame["pga_tau_ln"] * field_frame["eta_pga"]
        + field_frame["pga_phi_ln"] * field_frame["epsilon_pga"]
    )
    sa_reconstructed = (
        field_frame["sa0p4_epi_off_mean_ln_g"]
        + field_frame["sa0p4_tau_ln"] * field_frame["eta_sa0p4"]
        + field_frame["sa0p4_phi_ln"] * field_frame["epsilon_sa0p4"]
    )
    pga_reconstruction_error = float(np.max(np.abs(pga_reconstructed - field_frame["pga_simulated_ln_g"])))
    sa_reconstruction_error = float(np.max(np.abs(sa_reconstructed - field_frame["sa0p4_simulated_ln_g"])))
    add_check(checks, "Equation", "PGA equation reconstructs exactly", f"{pga_reconstruction_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", pga_reconstruction_error <= EXACT_RECONSTRUCTION_ATOL)
    add_check(checks, "Equation", "SA0P4 equation reconstructs exactly", f"{sa_reconstruction_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", sa_reconstruction_error <= EXACT_RECONSTRUCTION_ATOL)
    pga_exp_error = float(np.max(np.abs(np.exp(field_frame["pga_simulated_ln_g"]) - field_frame["pga_simulated_g"])))
    sa_exp_error = float(np.max(np.abs(np.exp(field_frame["sa0p4_simulated_ln_g"]) - field_frame["sa0p4_simulated_g"])))
    add_check(checks, "Equation", "PGA exponentiation reconstructs exactly", f"{pga_exp_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", pga_exp_error <= EXACT_RECONSTRUCTION_ATOL)
    add_check(checks, "Equation", "SA0P4 exponentiation reconstructs exactly", f"{sa_exp_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", sa_exp_error <= EXACT_RECONSTRUCTION_ATOL)

    numeric_columns = [
        column for column in field_output_columns
        if column not in {
            "control_case_id", "controlled_occurrence_id",
            "rupture_template_event_id", "rupture_id", "source_type",
            "gmm_name", "site_id"
        }
    ]
    finite = np.isfinite(field_frame[numeric_columns].to_numpy(dtype=float)).all()
    add_check(checks, "Numeric", "All field numeric values are finite", int(finite), 1, finite)
    positive = field_frame["pga_simulated_g"].gt(0).all() and field_frame["sa0p4_simulated_g"].gt(0).all()
    add_check(checks, "Numeric", "All simulated intensities are positive", int(positive), 1, positive)

    eta_pga_spread = field_frame.groupby("controlled_occurrence_id")["eta_pga"].agg(lambda x: float(x.max() - x.min())).max()
    eta_sa_spread = field_frame.groupby("controlled_occurrence_id")["eta_sa0p4"].agg(lambda x: float(x.max() - x.min())).max()
    add_check(checks, "Residual structure", "PGA eta is identical across sites", f"max spread={eta_pga_spread:.3e}", "0", eta_pga_spread <= EXACT_RECONSTRUCTION_ATOL)
    add_check(checks, "Residual structure", "SA0P4 eta is identical across sites", f"max spread={eta_sa_spread:.3e}", "0", eta_sa_spread <= EXACT_RECONSTRUCTION_ATOL)

    eta_rho = safe_correlation(event_frame["eta_pga"], event_frame["eta_sa0p4"])
    epsilon_rho = safe_correlation(field_frame["epsilon_pga"], field_frame["epsilon_sa0p4"])
    add_check(checks, "Cross IMT", "Event residual correlation matches target", f"{eta_rho:.6f}", f"{CROSS_IMT_RHO:.6f} +/- 0.15", abs(eta_rho - CROSS_IMT_RHO) <= 0.15)
    add_check(checks, "Cross IMT", "Site residual correlation matches target", f"{epsilon_rho:.6f}", f"{CROSS_IMT_RHO:.6f} +/- 0.015", abs(epsilon_rho - CROSS_IMT_RHO) <= 0.015)

    for name, values in [
        ("z_event_latent_1", event_frame["z_event_latent_1"].to_numpy()),
        ("z_event_latent_2", event_frame["z_event_latent_2"].to_numpy()),
    ]:
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=1))
        add_check(checks, "Random streams", f"{name} finite-sample mean", f"{mean:.6f}", f"abs <= {CHUNK_ETA_MEAN_ABS_MAX}", abs(mean) <= CHUNK_ETA_MEAN_ABS_MAX)
        add_check(checks, "Random streams", f"{name} finite-sample standard deviation", f"{std:.6f}", f"[{CHUNK_ETA_STD_MIN}, {CHUNK_ETA_STD_MAX}]", CHUNK_ETA_STD_MIN <= std <= CHUNK_ETA_STD_MAX)

    for name, values in [
        ("z_site_latent_1", field_frame["z_site_latent_1"].to_numpy()),
        ("z_site_latent_2", field_frame["z_site_latent_2"].to_numpy()),
    ]:
        mean = float(np.mean(values))
        std = float(np.std(values, ddof=1))
        add_check(checks, "Random streams", f"{name} finite-sample mean", f"{mean:.6f}", f"abs <= {EPSILON_MEAN_ABS_MAX}", abs(mean) <= EPSILON_MEAN_ABS_MAX)
        add_check(checks, "Random streams", f"{name} finite-sample standard deviation", f"{std:.6f}", f"[{EPSILON_STD_MIN}, {EPSILON_STD_MAX}]", EPSILON_STD_MIN <= std <= EPSILON_STD_MAX)

    # Recreate the first and last random streams directly from their IDs.
    reproducibility_passes = True
    maximum_reproduction_difference = 0.0
    for realization_index in [0, n_realizations - 1]:
        event_seed, _ = stable_seed(
            PIPELINE_VERSION, BASE_RANDOM_SEED, case_id, rupture_id,
            realization_index, "event_latent_pair"
        )
        site_seed, _ = stable_seed(
            PIPELINE_VERSION, BASE_RANDOM_SEED, case_id, rupture_id,
            realization_index, "site_latent_pair"
        )
        reproduced_event = rng_from_seed(event_seed).standard_normal(2)
        reproduced_sites = rng_from_seed(site_seed).standard_normal((2, n_sites))
        maximum_reproduction_difference = max(
            maximum_reproduction_difference,
            float(np.max(np.abs(reproduced_event - np.array([
                z_event_1[realization_index], z_event_2[realization_index]
            ])))),
            float(np.max(np.abs(reproduced_sites[0] - z_site_1[realization_index]))),
            float(np.max(np.abs(reproduced_sites[1] - z_site_2[realization_index]))),
        )
    reproducibility_passes = maximum_reproduction_difference == 0.0
    add_check(checks, "Reproducibility", "Identifier-based streams reproduce exactly", f"max difference={maximum_reproduction_difference:.3e}", "0", reproducibility_passes)

    validation = build_validation_frame(checks)
    validation_output = Path(str(record["validation_output"]))
    atomic_write_csv(validation, validation_output)
    if not validation["passes"].all():
        failed = validation.loc[~validation["passes"], "check"].tolist()
        raise RuntimeError(
            f"Controlled chunk {case_id} validation failed:\n"
            + "\n".join(f"  - {item}" for item in failed)
            + f"\n\nValidation file:\n  {validation_output}"
        )
    return validation


# ----------------------------------------------------------------------------
# 10. Process or resume all four controlled rupture chunks
# ----------------------------------------------------------------------------

print("=" * 78)
print("CONTROLLED BASELINE STOCHASTIC GROUND-MOTION FIELDS")
print("=" * 78)
print(f"Controlled ruptures:       {EXPECTED_CONTROL_RUPTURES:,}")
print(f"Realizations per rupture:  {CONTROL_REALIZATIONS_PER_RUPTURE:,}")
print(f"Controlled occurrences:    {EXPECTED_CONTROL_RUPTURES * CONTROL_REALIZATIONS_PER_RUPTURE:,}")
print(f"Seaside W2 sites:          {EXPECTED_SITES:,}")
print(f"Controlled field rows:     {EXPECTED_CONTROL_RUPTURES * CONTROL_REALIZATIONS_PER_RUPTURE * EXPECTED_SITES:,}")
print(f"Primary branch:            {PRIMARY_BASELINE_BRANCH}")
print(f"Cross-IMT rho:             {CROSS_IMT_RHO:.12f}")
print("Spatial residual model:    independent across sites")

for record in manifest_records:
    case_id = str(record["control_case_id"])
    print("\n" + "-" * 78)
    print(f"CONTROL RUPTURE: {case_id}")
    print(f"Rupture ID: {record['rupture_id']}")

    if marker_is_valid(record):
        record["status"] = "reused"
        record["completed_at_utc"] = json.loads(
            Path(str(record["marker_path"])).read_text(encoding="utf-8")
        ).get("completed_at_utc", "")
        print("Existing completion marker and output hashes are valid; reusing chunk.")
        save_manifest()
        continue

    for path_key in ["event_output", "field_output", "validation_output", "marker_path"]:
        path = Path(str(record[path_key]))
        if path.exists():
            path.unlink()

    validation = generate_control_chunk(record)
    event_output = Path(str(record["event_output"]))
    field_output = Path(str(record["field_output"]))
    validation_output = Path(str(record["validation_output"]))
    marker = {
        "pipeline_version": PIPELINE_VERSION,
        "completed_at_utc": utc_now(),
        "control_case_id": case_id,
        "rupture_id": str(record["rupture_id"]),
        "base_random_seed": BASE_RANDOM_SEED,
        "control_realizations_per_rupture": CONTROL_REALIZATIONS_PER_RUPTURE,
        "expected_event_rows": int(record["expected_event_rows"]),
        "expected_field_rows": int(record["expected_field_rows"]),
        "random_specification_sha256": random_specification_hash,
        "template_sha256": str(record["template_sha256"]),
        "event_output_path": str(event_output),
        "event_output_size_bytes": event_output.stat().st_size,
        "event_output_sha256": sha256_file(event_output),
        "field_output_path": str(field_output),
        "field_output_size_bytes": field_output.stat().st_size,
        "field_output_sha256": sha256_file(field_output),
        "validation_path": str(validation_output),
        "validation_sha256": sha256_file(validation_output),
        "validation_checks": int(len(validation)),
    }
    atomic_write_json(Path(str(record["marker_path"])), marker)
    record["status"] = "completed"
    record["completed_at_utc"] = marker["completed_at_utc"]
    record["event_output_sha256"] = marker["event_output_sha256"]
    record["field_output_sha256"] = marker["field_output_sha256"]
    print(f"Controlled chunk {case_id} passed all {len(validation)} checks.")
    save_manifest()

if not all(marker_is_valid(record) for record in manifest_records):
    raise RuntimeError("One or more Cell 18 controlled completion markers are invalid.")

save_manifest()


# ----------------------------------------------------------------------------
# 11. Deterministically concatenate the controlled chunks
# ----------------------------------------------------------------------------

ordered_records = sorted(manifest_records, key=lambda item: int(item["case_order"]))
combine_gzip_csv_files(
    [Path(str(record["field_output"])) for record in ordered_records],
    CELL18_FINAL_FIELDS_PATH,
)
combine_gzip_csv_files(
    [Path(str(record["event_output"])) for record in ordered_records],
    CELL18_FINAL_EVENTS_PATH,
)


# ----------------------------------------------------------------------------
# 12. Whole-controlled-dataset validation
# ----------------------------------------------------------------------------

expected_occurrences = EXPECTED_CONTROL_RUPTURES * CONTROL_REALIZATIONS_PER_RUPTURE
expected_field_rows = expected_occurrences * EXPECTED_SITES

final_events = pd.read_csv(
    CELL18_FINAL_EVENTS_PATH,
    dtype={
        "control_case_id": str,
        "controlled_occurrence_id": str,
        "rupture_template_event_id": str,
        "rupture_id": str,
        "source_type": str,
        "event_seed_hex": str,
        "site_seed_hex": str,
    },
    low_memory=False,
)
for column in [
    "controlled_occurrence_ordinal",
    "realization_index",
    "rupture_ordinal",
    "z_event_latent_1",
    "z_event_latent_2",
    "eta_pga",
    "eta_sa0p4",
    "cross_imt_rho",
]:
    final_events[column] = pd.to_numeric(final_events[column], errors="coerce")

if not np.isfinite(final_events[[
    "controlled_occurrence_ordinal", "realization_index", "rupture_ordinal",
    "z_event_latent_1", "z_event_latent_2", "eta_pga", "eta_sa0p4",
    "cross_imt_rho"
]].to_numpy(dtype=float)).all():
    raise RuntimeError("Final controlled event ledger contains non-finite values.")

# Streaming accumulators for the larger field file.
seen_pairs = np.zeros(expected_field_rows, dtype=bool)
occurrence_counts = np.zeros(expected_occurrences, dtype=np.int64)
site_counts = np.zeros(EXPECTED_SITES, dtype=np.int64)
eta_pga_min = np.full(expected_occurrences, np.inf)
eta_pga_max = np.full(expected_occurrences, -np.inf)
eta_sa_min = np.full(expected_occurrences, np.inf)
eta_sa_max = np.full(expected_occurrences, -np.inf)
site_mean_eps_pga_sum = np.zeros(expected_occurrences, dtype=float)
site_mean_eps_sa_sum = np.zeros(expected_occurrences, dtype=float)

# Deterministic subset of sites for cross-site independence checks.
independence_site_ordinals = np.array(
    [0, 7, 19, 41, 73, 109, 157, 211, 277, 349, 419, 469],
    dtype=np.int64,
)
selected_site_position = {
    int(site): index for index, site in enumerate(independence_site_ordinals)
}
selected_z1 = np.full(
    (expected_occurrences, len(independence_site_ordinals)), np.nan
)
selected_z2 = np.full_like(selected_z1, np.nan)

@dataclass
class RunningMoments:
    n: int = 0
    total: float = 0.0
    total_square: float = 0.0

    def update(self, values: np.ndarray) -> None:
        array = np.asarray(values, dtype=float)
        self.n += int(array.size)
        self.total += float(np.sum(array))
        self.total_square += float(np.sum(array * array))

    @property
    def mean(self) -> float:
        return self.total / self.n

    @property
    def std(self) -> float:
        if self.n < 2:
            return float("nan")
        variance = (
            self.total_square - self.total * self.total / self.n
        ) / (self.n - 1)
        return math.sqrt(max(variance, 0.0))


@dataclass
class RunningCorrelation:
    n: int = 0
    sum_x: float = 0.0
    sum_y: float = 0.0
    sum_xx: float = 0.0
    sum_yy: float = 0.0
    sum_xy: float = 0.0

    def update(self, x: np.ndarray, y: np.ndarray) -> None:
        x_array = np.asarray(x, dtype=float)
        y_array = np.asarray(y, dtype=float)
        if x_array.shape != y_array.shape:
            raise ValueError("Correlation arrays must have identical shapes.")
        self.n += int(x_array.size)
        self.sum_x += float(np.sum(x_array))
        self.sum_y += float(np.sum(y_array))
        self.sum_xx += float(np.sum(x_array * x_array))
        self.sum_yy += float(np.sum(y_array * y_array))
        self.sum_xy += float(np.sum(x_array * y_array))

    @property
    def correlation(self) -> float:
        numerator = self.sum_xy - self.sum_x * self.sum_y / self.n
        denominator_x = self.sum_xx - self.sum_x**2 / self.n
        denominator_y = self.sum_yy - self.sum_y**2 / self.n
        denominator = math.sqrt(max(denominator_x * denominator_y, 0.0))
        if denominator == 0.0:
            return float("nan")
        return numerator / denominator


moments = {
    "z_site_latent_1": RunningMoments(),
    "z_site_latent_2": RunningMoments(),
    "epsilon_pga": RunningMoments(),
    "epsilon_sa0p4": RunningMoments(),
    "pga_total_standardized": RunningMoments(),
    "sa0p4_total_standardized": RunningMoments(),
}
site_cross_imt = RunningCorrelation()

field_rows = 0
numeric_failures = 0
period_failures = 0
positive_failures = 0
reconstruction_failures = 0
maximum_pga_reconstruction_error = 0.0
maximum_sa_reconstruction_error = 0.0
minimum_pga_g = np.inf
maximum_pga_g = -np.inf
minimum_sa_g = np.inf
maximum_sa_g = -np.inf

field_numeric_columns = [
    "controlled_occurrence_ordinal",
    "realization_index",
    "rupture_ordinal",
    "magnitude",
    "raw_annual_rate",
    "site_ordinal",
    "site_longitude",
    "site_latitude",
    "r_rup_km",
    "vs30_mps",
    "period_s",
    "cross_imt_rho",
    "z_site_latent_1",
    "z_site_latent_2",
    "eta_pga",
    "epsilon_pga",
    "pga_epi_off_mean_ln_g",
    "pga_tau_ln",
    "pga_phi_ln",
    "pga_between_event_shift_ln",
    "pga_within_event_shift_ln",
    "pga_simulated_ln_g",
    "pga_simulated_g",
    "eta_sa0p4",
    "epsilon_sa0p4",
    "sa0p4_epi_off_mean_ln_g",
    "sa0p4_tau_ln",
    "sa0p4_phi_ln",
    "sa0p4_between_event_shift_ln",
    "sa0p4_within_event_shift_ln",
    "sa0p4_simulated_ln_g",
    "sa0p4_simulated_g",
]

for frame in pd.read_csv(
    CELL18_FINAL_FIELDS_PATH,
    dtype={
        "control_case_id": str,
        "controlled_occurrence_id": str,
        "rupture_template_event_id": str,
        "rupture_id": str,
        "source_type": str,
        "gmm_name": str,
        "site_id": str,
    },
    chunksize=FINAL_READ_CHUNK_SIZE,
    low_memory=False,
):
    for column in field_numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    field_rows += len(frame)
    numeric_matrix = frame[field_numeric_columns].to_numpy(dtype=float)
    numeric_failures += int((~np.isfinite(numeric_matrix)).any(axis=1).sum())

    occurrence_ordinals = frame["controlled_occurrence_ordinal"].to_numpy(dtype=int)
    site_ordinals = frame["site_ordinal"].to_numpy(dtype=int)
    valid_ordinals = (
        (occurrence_ordinals >= 0)
        & (occurrence_ordinals < expected_occurrences)
        & (site_ordinals >= 0)
        & (site_ordinals < EXPECTED_SITES)
    )
    if not valid_ordinals.all():
        raise RuntimeError("Final controlled fields contain out-of-range ordinals.")
    pair_keys = occurrence_ordinals * EXPECTED_SITES + site_ordinals
    if np.unique(pair_keys).size != pair_keys.size or seen_pairs[pair_keys].any():
        raise RuntimeError("Final controlled fields contain duplicate occurrence-site pairs.")
    seen_pairs[pair_keys] = True
    np.add.at(occurrence_counts, occurrence_ordinals, 1)
    np.add.at(site_counts, site_ordinals, 1)

    eta_pga_values = frame["eta_pga"].to_numpy(dtype=float)
    eta_sa_values = frame["eta_sa0p4"].to_numpy(dtype=float)
    np.minimum.at(eta_pga_min, occurrence_ordinals, eta_pga_values)
    np.maximum.at(eta_pga_max, occurrence_ordinals, eta_pga_values)
    np.minimum.at(eta_sa_min, occurrence_ordinals, eta_sa_values)
    np.maximum.at(eta_sa_max, occurrence_ordinals, eta_sa_values)

    eps_pga_values = frame["epsilon_pga"].to_numpy(dtype=float)
    eps_sa_values = frame["epsilon_sa0p4"].to_numpy(dtype=float)
    np.add.at(site_mean_eps_pga_sum, occurrence_ordinals, eps_pga_values)
    np.add.at(site_mean_eps_sa_sum, occurrence_ordinals, eps_sa_values)

    z1_values = frame["z_site_latent_1"].to_numpy(dtype=float)
    z2_values = frame["z_site_latent_2"].to_numpy(dtype=float)
    moments["z_site_latent_1"].update(z1_values)
    moments["z_site_latent_2"].update(z2_values)
    moments["epsilon_pga"].update(eps_pga_values)
    moments["epsilon_sa0p4"].update(eps_sa_values)
    site_cross_imt.update(eps_pga_values, eps_sa_values)

    pga_reconstructed = (
        frame["pga_epi_off_mean_ln_g"].to_numpy(dtype=float)
        + frame["pga_tau_ln"].to_numpy(dtype=float) * eta_pga_values
        + frame["pga_phi_ln"].to_numpy(dtype=float) * eps_pga_values
    )
    sa_reconstructed = (
        frame["sa0p4_epi_off_mean_ln_g"].to_numpy(dtype=float)
        + frame["sa0p4_tau_ln"].to_numpy(dtype=float) * eta_sa_values
        + frame["sa0p4_phi_ln"].to_numpy(dtype=float) * eps_sa_values
    )
    pga_error = np.abs(
        pga_reconstructed - frame["pga_simulated_ln_g"].to_numpy(dtype=float)
    )
    sa_error = np.abs(
        sa_reconstructed - frame["sa0p4_simulated_ln_g"].to_numpy(dtype=float)
    )
    maximum_pga_reconstruction_error = max(
        maximum_pga_reconstruction_error, float(np.max(pga_error))
    )
    maximum_sa_reconstruction_error = max(
        maximum_sa_reconstruction_error, float(np.max(sa_error))
    )
    reconstruction_failures += int(
        ((pga_error > EXACT_RECONSTRUCTION_ATOL)
         | (sa_error > EXACT_RECONSTRUCTION_ATOL)).sum()
    )

    pga_sigma = np.sqrt(
        frame["pga_tau_ln"].to_numpy(dtype=float) ** 2
        + frame["pga_phi_ln"].to_numpy(dtype=float) ** 2
    )
    sa_sigma = np.sqrt(
        frame["sa0p4_tau_ln"].to_numpy(dtype=float) ** 2
        + frame["sa0p4_phi_ln"].to_numpy(dtype=float) ** 2
    )
    pga_standardized = (
        frame["pga_simulated_ln_g"].to_numpy(dtype=float)
        - frame["pga_epi_off_mean_ln_g"].to_numpy(dtype=float)
    ) / pga_sigma
    sa_standardized = (
        frame["sa0p4_simulated_ln_g"].to_numpy(dtype=float)
        - frame["sa0p4_epi_off_mean_ln_g"].to_numpy(dtype=float)
    ) / sa_sigma
    moments["pga_total_standardized"].update(pga_standardized)
    moments["sa0p4_total_standardized"].update(sa_standardized)

    period_failures += int((~np.isclose(
        frame["period_s"].to_numpy(dtype=float),
        SEASIDE_W2_PERIOD_S,
        rtol=0.0,
        atol=PERIOD_MATCH_ATOL,
    )).sum())
    pga_g = frame["pga_simulated_g"].to_numpy(dtype=float)
    sa_g = frame["sa0p4_simulated_g"].to_numpy(dtype=float)
    positive_failures += int(((pga_g <= 0.0) | (sa_g <= 0.0)).sum())
    minimum_pga_g = min(minimum_pga_g, float(np.min(pga_g)))
    maximum_pga_g = max(maximum_pga_g, float(np.max(pga_g)))
    minimum_sa_g = min(minimum_sa_g, float(np.min(sa_g)))
    maximum_sa_g = max(maximum_sa_g, float(np.max(sa_g)))

    selected_mask = np.isin(site_ordinals, independence_site_ordinals)
    selected_rows = np.flatnonzero(selected_mask)
    for row_index in selected_rows:
        occurrence = int(occurrence_ordinals[row_index])
        site_position = selected_site_position[int(site_ordinals[row_index])]
        selected_z1[occurrence, site_position] = z1_values[row_index]
        selected_z2[occurrence, site_position] = z2_values[row_index]

if np.isnan(selected_z1).any() or np.isnan(selected_z2).any():
    raise RuntimeError("Cross-site validation subset is incomplete.")

# Correlation across different sites, measured over independent occurrences.
site_pair_correlations: list[float] = []
for first in range(len(independence_site_ordinals)):
    for second in range(first + 1, len(independence_site_ordinals)):
        site_pair_correlations.append(
            safe_correlation(selected_z1[:, first], selected_z1[:, second])
        )
        site_pair_correlations.append(
            safe_correlation(selected_z2[:, first], selected_z2[:, second])
        )
maximum_abs_cross_site_correlation = float(
    np.max(np.abs(np.asarray(site_pair_correlations, dtype=float)))
)

site_mean_eps_pga = site_mean_eps_pga_sum / EXPECTED_SITES
site_mean_eps_sa = site_mean_eps_sa_sum / EXPECTED_SITES
eta_pga_by_occurrence = final_events.sort_values(
    "controlled_occurrence_ordinal"
)["eta_pga"].to_numpy(dtype=float)
eta_sa_by_occurrence = final_events.sort_values(
    "controlled_occurrence_ordinal"
)["eta_sa0p4"].to_numpy(dtype=float)
maximum_abs_event_site_mean_correlation = max(
    abs(safe_correlation(eta_pga_by_occurrence, site_mean_eps_pga)),
    abs(safe_correlation(eta_sa_by_occurrence, site_mean_eps_sa)),
)

# Exact seed reproducibility check on a deterministic sample of occurrences.
reproduction_sample = np.unique(np.linspace(
    0, expected_occurrences - 1, num=min(16, expected_occurrences), dtype=int
))
field_sample_columns = [
    "controlled_occurrence_ordinal", "site_ordinal",
    "z_site_latent_1", "z_site_latent_2"
]
field_samples: dict[tuple[int, int], tuple[float, float]] = {}
sampled_site_ordinals = [0, 137, 469]
for frame in pd.read_csv(
    CELL18_FINAL_FIELDS_PATH,
    usecols=field_sample_columns,
    chunksize=FINAL_READ_CHUNK_SIZE,
    low_memory=False,
):
    mask = (
        frame["controlled_occurrence_ordinal"].isin(reproduction_sample)
        & frame["site_ordinal"].isin(sampled_site_ordinals)
    )
    for row in frame.loc[mask].itertuples(index=False):
        field_samples[(int(row.controlled_occurrence_ordinal), int(row.site_ordinal))] = (
            float(row.z_site_latent_1), float(row.z_site_latent_2)
        )

maximum_seed_reproduction_difference = 0.0
for occurrence_ordinal in reproduction_sample:
    event_row = final_events.loc[
        final_events["controlled_occurrence_ordinal"].eq(occurrence_ordinal)
    ].iloc[0]
    case_id = str(event_row["control_case_id"])
    rupture_id = str(event_row["rupture_id"])
    realization_index = int(event_row["realization_index"])
    event_seed, _ = stable_seed(
        PIPELINE_VERSION, BASE_RANDOM_SEED, case_id, rupture_id,
        realization_index, "event_latent_pair"
    )
    site_seed, _ = stable_seed(
        PIPELINE_VERSION, BASE_RANDOM_SEED, case_id, rupture_id,
        realization_index, "site_latent_pair"
    )
    event_pair = rng_from_seed(event_seed).standard_normal(2)
    site_pair = rng_from_seed(site_seed).standard_normal((2, EXPECTED_SITES))
    maximum_seed_reproduction_difference = max(
        maximum_seed_reproduction_difference,
        float(abs(event_pair[0] - event_row["z_event_latent_1"])),
        float(abs(event_pair[1] - event_row["z_event_latent_2"])),
    )
    for site_ordinal in sampled_site_ordinals:
        observed = field_samples[(int(occurrence_ordinal), int(site_ordinal))]
        maximum_seed_reproduction_difference = max(
            maximum_seed_reproduction_difference,
            abs(float(site_pair[0, site_ordinal]) - observed[0]),
            abs(float(site_pair[1, site_ordinal]) - observed[1]),
        )

all_chunk_validations = pd.concat(
    [
        pd.read_csv(Path(str(record["validation_output"]))).assign(
            control_case_id=str(record["control_case_id"])
        )
        for record in ordered_records
    ],
    ignore_index=True,
)
all_markers_valid = all(marker_is_valid(record) for record in ordered_records)

checks: list[dict[str, object]] = []
add_check(checks, "Dependencies", "Cell 16 checks passed", int(normalize_boolean_series(cell16_validation["passes"]).sum()), len(cell16_validation), normalize_boolean_series(cell16_validation["passes"]).all())
add_check(checks, "Dependencies", "Cell 17 checks passed", int(normalize_boolean_series(cell17_validation["passes"]).sum()), len(cell17_validation), normalize_boolean_series(cell17_validation["passes"]).all())
add_check(checks, "Dependencies", "Cell 17 output hash matches accepted summary", actual_cell17_hash, expected_cell17_hash, actual_cell17_hash == expected_cell17_hash)
add_check(checks, "Configuration", "Primary branch is epi-off", PRIMARY_BASELINE_BRANCH, "epi-off", PRIMARY_BASELINE_BRANCH == "epi-off")
add_check(checks, "Configuration", "SA period is exact 0.40 s", SEASIDE_W2_PERIOD_S, 0.40, math.isclose(SEASIDE_W2_PERIOD_S, 0.40, abs_tol=PERIOD_MATCH_ATOL))
add_check(checks, "Configuration", "Baker-Jayaram PGA-SA0P4 rho", f"{CROSS_IMT_RHO:.15f}", f"{EXPECTED_PGA_SA0P4_RHO:.15f}", math.isclose(CROSS_IMT_RHO, EXPECTED_PGA_SA0P4_RHO, abs_tol=RHO_FORMULA_ATOL))
add_check(checks, "Templates", "Four controlled rupture templates", control_templates["rupture_id"].nunique(), EXPECTED_CONTROL_RUPTURES, control_templates["rupture_id"].nunique() == EXPECTED_CONTROL_RUPTURES)
add_check(checks, "Templates", "All controlled templates contain 470 sites", int(rupture_site_counts.eq(EXPECTED_SITES).sum()), EXPECTED_CONTROL_RUPTURES, rupture_site_counts.eq(EXPECTED_SITES).all())
add_check(checks, "Restart", "All completion markers valid", int(all_markers_valid), 1, all_markers_valid)
add_check(checks, "Restart", "All chunk validation checks passed", int(normalize_boolean_series(all_chunk_validations["passes"]).sum()), len(all_chunk_validations), normalize_boolean_series(all_chunk_validations["passes"]).all())
add_check(checks, "Coverage", "Controlled event rows", len(final_events), expected_occurrences, len(final_events) == expected_occurrences)
add_check(checks, "Coverage", "Unique controlled occurrence IDs", final_events["controlled_occurrence_id"].nunique(), expected_occurrences, final_events["controlled_occurrence_id"].nunique() == expected_occurrences)
add_check(checks, "Coverage", "Unique controlled occurrence ordinals", final_events["controlled_occurrence_ordinal"].nunique(), expected_occurrences, final_events["controlled_occurrence_ordinal"].nunique() == expected_occurrences)
add_check(checks, "Coverage", "Controlled field rows", field_rows, expected_field_rows, field_rows == expected_field_rows)
add_check(checks, "Coverage", "All occurrence-site pairs present exactly once", int(seen_pairs.sum()), expected_field_rows, seen_pairs.all())
add_check(checks, "Coverage", "Every occurrence has 470 sites", int((occurrence_counts == EXPECTED_SITES).sum()), expected_occurrences, (occurrence_counts == EXPECTED_SITES).all())
add_check(checks, "Coverage", "Every site has all controlled occurrences", int((site_counts == expected_occurrences).sum()), EXPECTED_SITES, (site_counts == expected_occurrences).all())
add_check(checks, "Numeric", "No non-finite field rows", numeric_failures, 0, numeric_failures == 0)
add_check(checks, "Numeric", "No period failures", period_failures, 0, period_failures == 0)
add_check(checks, "Numeric", "All simulated intensities positive", positive_failures, 0, positive_failures == 0)
add_check(checks, "Equation", "PGA reconstruction maximum error", f"{maximum_pga_reconstruction_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", maximum_pga_reconstruction_error <= EXACT_RECONSTRUCTION_ATOL)
add_check(checks, "Equation", "SA0P4 reconstruction maximum error", f"{maximum_sa_reconstruction_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", maximum_sa_reconstruction_error <= EXACT_RECONSTRUCTION_ATOL)
add_check(checks, "Equation", "No equation reconstruction failures", reconstruction_failures, 0, reconstruction_failures == 0)
add_check(checks, "Residual structure", "PGA eta shared exactly across sites", f"max spread={float(np.max(eta_pga_max - eta_pga_min)):.3e}", "0", float(np.max(eta_pga_max - eta_pga_min)) <= EXACT_RECONSTRUCTION_ATOL)
add_check(checks, "Residual structure", "SA0P4 eta shared exactly across sites", f"max spread={float(np.max(eta_sa_max - eta_sa_min)):.3e}", "0", float(np.max(eta_sa_max - eta_sa_min)) <= EXACT_RECONSTRUCTION_ATOL)
add_check(checks, "Residual structure", "Maximum cross-site latent correlation", f"{maximum_abs_cross_site_correlation:.6f}", f"<= {MAX_ABS_CROSS_SITE_CORRELATION}", maximum_abs_cross_site_correlation <= MAX_ABS_CROSS_SITE_CORRELATION, "Measured across 12 deterministic sites and all controlled occurrences; same-site PGA-SA correlation is handled separately.")
add_check(checks, "Residual structure", "Event residual independent of site-mean residual", f"{maximum_abs_event_site_mean_correlation:.6f}", f"<= {MAX_ABS_EVENT_SITE_MEAN_CORRELATION}", maximum_abs_event_site_mean_correlation <= MAX_ABS_EVENT_SITE_MEAN_CORRELATION)

for name in ["z_site_latent_1", "z_site_latent_2", "epsilon_pga", "epsilon_sa0p4"]:
    statistic = moments[name]
    add_check(checks, "Distribution", f"{name} mean", f"{statistic.mean:.6f}", f"abs <= {EPSILON_MEAN_ABS_MAX}", abs(statistic.mean) <= EPSILON_MEAN_ABS_MAX)
    add_check(checks, "Distribution", f"{name} standard deviation", f"{statistic.std:.6f}", f"[{EPSILON_STD_MIN}, {EPSILON_STD_MAX}]", EPSILON_STD_MIN <= statistic.std <= EPSILON_STD_MAX)

for name in ["pga_total_standardized", "sa0p4_total_standardized"]:
    statistic = moments[name]
    add_check(checks, "Distribution", f"{name} mean", f"{statistic.mean:.6f}", f"abs <= {TOTAL_RESIDUAL_MEAN_ABS_MAX}", abs(statistic.mean) <= TOTAL_RESIDUAL_MEAN_ABS_MAX)
    add_check(checks, "Distribution", f"{name} standard deviation", f"{statistic.std:.6f}", f"[{TOTAL_RESIDUAL_STD_MIN}, {TOTAL_RESIDUAL_STD_MAX}]", TOTAL_RESIDUAL_STD_MIN <= statistic.std <= TOTAL_RESIDUAL_STD_MAX)

for name in ["z_event_latent_1", "z_event_latent_2", "eta_pga", "eta_sa0p4"]:
    values = final_events[name].to_numpy(dtype=float)
    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1))
    add_check(checks, "Distribution", f"{name} mean", f"{mean:.6f}", f"abs <= {ETA_MEAN_ABS_MAX}", abs(mean) <= ETA_MEAN_ABS_MAX)
    add_check(checks, "Distribution", f"{name} standard deviation", f"{std:.6f}", f"[{ETA_STD_MIN}, {ETA_STD_MAX}]", ETA_STD_MIN <= std <= ETA_STD_MAX)

event_cross_imt_rho = safe_correlation(
    final_events["eta_pga"].to_numpy(dtype=float),
    final_events["eta_sa0p4"].to_numpy(dtype=float),
)
site_cross_imt_rho = site_cross_imt.correlation
add_check(checks, "Cross IMT", "Event-level PGA-SA0P4 correlation", f"{event_cross_imt_rho:.6f}", f"{CROSS_IMT_RHO:.6f} +/- {EVENT_CROSS_IMT_RHO_ATOL}", abs(event_cross_imt_rho - CROSS_IMT_RHO) <= EVENT_CROSS_IMT_RHO_ATOL)
add_check(checks, "Cross IMT", "Within-event PGA-SA0P4 correlation", f"{site_cross_imt_rho:.6f}", f"{CROSS_IMT_RHO:.6f} +/- {SITE_CROSS_IMT_RHO_ATOL}", abs(site_cross_imt_rho - CROSS_IMT_RHO) <= SITE_CROSS_IMT_RHO_ATOL)
add_check(checks, "Reproducibility", "Random streams reproduce from identifiers", f"max difference={maximum_seed_reproduction_difference:.3e}", "0", maximum_seed_reproduction_difference <= 2.0e-15)

validation = build_validation_frame(checks)
atomic_write_csv(validation, CELL18_VALIDATION_PATH)
if not validation["passes"].all():
    failed = validation.loc[~validation["passes"], "check"].tolist()
    raise RuntimeError(
        "Cell 18 final controlled validation failed:\n"
        + "\n".join(f"  - {item}" for item in failed)
        + f"\n\nValidation file:\n  {CELL18_VALIDATION_PATH}"
    )


# ----------------------------------------------------------------------------
# 13. Reproducibility summary
# ----------------------------------------------------------------------------

control_rupture_summary = (
    control_templates.groupby(
        ["control_case_id", "rupture_id", "source_type"], sort=False
    )
    .agg(
        rupture_ordinal=("rupture_ordinal", "first"),
        magnitude=("magnitude", "first"),
        raw_annual_rate=("raw_annual_rate", "first"),
        minimum_r_rup_km=("r_rup_km", "min"),
        maximum_r_rup_km=("r_rup_km", "max"),
        sites=("site_id", "nunique"),
    )
    .reset_index()
    .to_dict("records")
)

summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": utc_now(),
    "all_checks_passed": True,
    "validation_checks": int(len(validation)),
    "portfolio": "Seaside W2 commercial portfolio",
    "sites": EXPECTED_SITES,
    "period_s": SEASIDE_W2_PERIOD_S,
    "primary_baseline_branch": PRIMARY_BASELINE_BRANCH,
    "controlled_ruptures": EXPECTED_CONTROL_RUPTURES,
    "control_realizations_per_rupture": CONTROL_REALIZATIONS_PER_RUPTURE,
    "controlled_occurrences": expected_occurrences,
    "controlled_field_rows": expected_field_rows,
    "base_random_seed": BASE_RANDOM_SEED,
    "random_generator": "numpy.random.PCG64DXSM",
    "random_specification_path": str(CELL18_RANDOM_SPEC_PATH),
    "random_specification_sha256": sha256_file(CELL18_RANDOM_SPEC_PATH),
    "cross_imt_model": {
        "model": "Baker and Jayaram (2008)",
        "doi": "10.1193/1.2857544",
        "pga_proxy_period_s": PGA_PROXY_PERIOD_S,
        "sa_period_s": SEASIDE_W2_PERIOD_S,
        "target_rho": CROSS_IMT_RHO,
        "empirical_event_rho": event_cross_imt_rho,
        "empirical_within_event_rho": site_cross_imt_rho,
    },
    "aleatory_model": {
        "equation": "ln(IM) = mu_epi_off + tau*eta + phi*epsilon",
        "between_event": "one residual per event occurrence and IMT, shared across all 470 sites",
        "within_event": "independent across sites; cross-correlated between PGA and SA0P4 at the same site",
        "spatial_correlation": "not applied in Phase 1 controlled baseline",
        "epistemic_branch": "epi-off only; epi-lo and epi-hi remain sensitivity branches and are not mixed into occurrence sampling",
    },
    "future_spatial_extension": {
        "same_event_catalog": True,
        "same_between_event_streams": True,
        "same_latent_site_streams": True,
        "method": (
            "Apply future spatial transforms to z_site_latent_1 and "
            "z_site_latent_2 before cross-IMT combination."
        ),
    },
    "control_rupture_summary": control_rupture_summary,
    "numeric_ranges": {
        "pga_simulated_g": {
            "minimum": minimum_pga_g,
            "maximum": maximum_pga_g,
        },
        "sa0p4_simulated_g": {
            "minimum": minimum_sa_g,
            "maximum": maximum_sa_g,
        },
    },
    "validation_diagnostics": {
        "maximum_pga_reconstruction_error": maximum_pga_reconstruction_error,
        "maximum_sa0p4_reconstruction_error": maximum_sa_reconstruction_error,
        "maximum_seed_reproduction_difference": maximum_seed_reproduction_difference,
        "maximum_abs_cross_site_latent_correlation": maximum_abs_cross_site_correlation,
        "maximum_abs_event_site_mean_correlation": maximum_abs_event_site_mean_correlation,
    },
    "source_files": {
        "cell16_summary_path": str(CELL16_SUMMARY_INPUT_PATH),
        "cell16_summary_sha256": sha256_file(CELL16_SUMMARY_INPUT_PATH),
        "cell16_validation_path": str(CELL16_VALIDATION_INPUT_PATH),
        "cell16_validation_sha256": sha256_file(CELL16_VALIDATION_INPUT_PATH),
        "cell17_output_path": str(CELL17_FINAL_OUTPUT_INPUT_PATH),
        "cell17_output_sha256": actual_cell17_hash,
        "cell17_summary_path": str(CELL17_SUMMARY_INPUT_PATH),
        "cell17_summary_sha256": sha256_file(CELL17_SUMMARY_INPUT_PATH),
        "cell17_validation_path": str(CELL17_VALIDATION_INPUT_PATH),
        "cell17_validation_sha256": sha256_file(CELL17_VALIDATION_INPUT_PATH),
        "control_template_path": str(CELL18_CONTROL_TEMPLATE_PATH),
        "control_template_sha256": sha256_file(CELL18_CONTROL_TEMPLATE_PATH),
        "site_order_path": str(CELL18_SITE_ORDER_PATH),
        "site_order_sha256": sha256_file(CELL18_SITE_ORDER_PATH),
    },
    "outputs": {
        "controlled_fields_path": str(CELL18_FINAL_FIELDS_PATH),
        "controlled_fields_size_bytes": CELL18_FINAL_FIELDS_PATH.stat().st_size,
        "controlled_fields_sha256": sha256_file(CELL18_FINAL_FIELDS_PATH),
        "event_residual_ledger_path": str(CELL18_FINAL_EVENTS_PATH),
        "event_residual_ledger_size_bytes": CELL18_FINAL_EVENTS_PATH.stat().st_size,
        "event_residual_ledger_sha256": sha256_file(CELL18_FINAL_EVENTS_PATH),
        "chunk_manifest_path": str(CELL18_MANIFEST_PATH),
        "chunk_manifest_sha256": sha256_file(CELL18_MANIFEST_PATH),
        "validation_path": str(CELL18_VALIDATION_PATH),
        "validation_sha256": sha256_file(CELL18_VALIDATION_PATH),
    },
    "next_step": (
        "Apply the validated identifier-based simulator to every occurrence in "
        "the Notebook 3 annual event catalog, joining occurrence rupture_id to "
        "the Cell 17 epi-off parameters and preserving occurrence-level seeds."
    ),
}
atomic_write_json(CELL18_SUMMARY_PATH, summary)

print("\n" + "=" * 78)
print("CELL 18 CONTROLLED BASELINE STOCHASTIC FIELD VALIDATION COMPLETE")
print("=" * 78)
print(f"Controlled occurrences: {expected_occurrences:,}")
print(f"Controlled field rows:  {expected_field_rows:,}")
print(f"Final validation checks: {len(validation):,}")
print(f"Event PGA-SA0P4 rho:     {event_cross_imt_rho:.6f}")
print(f"Site PGA-SA0P4 rho:      {site_cross_imt_rho:.6f}")
print(f"Maximum cross-site rho:  {maximum_abs_cross_site_correlation:.6f}")
print(f"Maximum equation error:  {max(maximum_pga_reconstruction_error, maximum_sa_reconstruction_error):.3e}")
print(f"\nControlled fields:\n  {CELL18_FINAL_FIELDS_PATH}")
print(f"\nEvent residual ledger:\n  {CELL18_FINAL_EVENTS_PATH}")
print(f"\nValidation:\n  {CELL18_VALIDATION_PATH}")
print(f"\nSummary:\n  {CELL18_SUMMARY_PATH}")
print(
    "\nNext: apply this validated random-stream design to the full annual "
    "event-occurrence catalog."
)


Cell 17 prerequisite passed: 1,845,690 validated epi-off rupture-site parameter rows.
Cross-IMT residual correlation: Baker-Jayaram rho(PGA, SA0P4) = 0.732140990025
CONTROLLED BASELINE STOCHASTIC GROUND-MOTION FIELDS
Controlled ruptures:       4
Realizations per rupture:  128
Controlled occurrences:    512
Seaside W2 sites:          470
Controlled field rows:     240,640
Primary branch:            epi-off
Cross-IMT rho:             0.732140990025
Spatial residual model:    independent across sites

------------------------------------------------------------------------------
CONTROL RUPTURE: interface_nearest
Rupture ID: b18dfe7bfe65277b6a076d34
Controlled chunk interface_nearest passed all 25 checks.

------------------------------------------------------------------------------
CONTROL RUPTURE: interface_farthest
Rupture ID: bab384c6cea3b2009490c76f
Controlled chunk interface_farthest passed all 25 checks.

----------------------------------------------------------------------------

In [36]:
# ============================================================================
# CELL 19
# Full annual-catalog baseline stochastic ground-motion field simulation.
#
# Purpose
# -------
# 1. Read the accepted Notebook 3 event-occurrence catalog.
# 2. Join every occurrence to the validated Cell 17 epi-off GMM parameters.
# 3. Reuse the Cell 18 deterministic random-stream design.
# 4. Generate one shared between-event residual per occurrence and IMT.
# 5. Generate conditionally independent within-event residuals across sites.
# 6. Preserve PGA-SA(0.4 s) cross-IMT residual correlation.
# 7. Process by the existing 16 Cell 17 rupture partitions, with restart markers.
# 8. Validate catalog coverage, field equations, random streams, and outputs.
#
# Phase 1 baseline
# ----------------
# No within-event spatial correlation is applied between different sites.
# The same identifier-based event and site random streams are retained for the
# later spatial-correlation extension.
# ============================================================================

from __future__ import annotations

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import gzip
import hashlib
import json
import math
import os
import shutil
from typing import Iterable

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------------------

PIPELINE_VERSION = "notebook4_cell19_full_baseline_fields_v2"
EXPECTED_RUPTURES = 3_927
EXPECTED_SITES = 470
EXPECTED_CELL17_ROWS = EXPECTED_RUPTURES * EXPECTED_SITES
PRIMARY_BASELINE_BRANCH = "epi-off"
SEASIDE_W2_PERIOD_S = 0.40
PGA_PROXY_PERIOD_S = 0.01
VERIFY_EXISTING_HASHES = True

# This must match Cell 18 so that the production and controlled simulators use
# the same root random-stream convention.
BASE_RANDOM_SEED = int(
    os.environ.get("NOTEBOOK4_CELL19_BASE_SEED", "20260731")
)
FINAL_READ_CHUNK_SIZE = int(
    os.environ.get("NOTEBOOK4_CELL19_FINAL_READ_CHUNK_SIZE", "150000")
)
GZIP_COMPRESSLEVEL = int(
    os.environ.get("NOTEBOOK4_CELL19_GZIP_LEVEL", "1")
)

if FINAL_READ_CHUNK_SIZE <= 0:
    raise ValueError("FINAL_READ_CHUNK_SIZE must be positive.")
if not (1 <= GZIP_COMPRESSLEVEL <= 9):
    raise ValueError("GZIP_COMPRESSLEVEL must lie between 1 and 9.")

PERIOD_MATCH_ATOL = 1.0e-10
EXACT_RECONSTRUCTION_ATOL = 3.0e-12
RHO_FORMULA_ATOL = 2.0e-14

required_variables = ["DATA_DIR", "METADATA_DIR", "EVENT_CATALOG_PATH"]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Cell 19 requires variables created by earlier Notebook 4 cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

DATA_DIR = Path(DATA_DIR).resolve()
METADATA_DIR = Path(METADATA_DIR).resolve()
EVENT_CATALOG_PATH = Path(EVENT_CATALOG_PATH).resolve()


# ----------------------------------------------------------------------------
# 2. Input and output paths
# ----------------------------------------------------------------------------

CELL17_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_bulk_parker_gmm"
CELL17_FINAL_OUTPUT_INPUT_PATH = Path(
    globals().get(
        "CELL17_FINAL_OUTPUT_PATH",
        CELL17_OUTPUT_DIR / "bulk_parker_gmm_parameters.csv.gz",
    )
).resolve()
CELL17_SUMMARY_INPUT_PATH = Path(
    globals().get(
        "CELL17_FINAL_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_summary.json",
    )
).resolve()
CELL17_VALIDATION_INPUT_PATH = Path(
    globals().get(
        "CELL17_FINAL_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_validation.csv",
    )
).resolve()
CELL17_MANIFEST_INPUT_PATH = Path(
    globals().get(
        "CELL17_MANIFEST_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_chunk_manifest.csv",
    )
).resolve()

CELL18_SUMMARY_INPUT_PATH = Path(
    globals().get(
        "CELL18_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_cell_18_controlled_field_summary.json",
    )
).resolve()
CELL18_VALIDATION_INPUT_PATH = Path(
    globals().get(
        "CELL18_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_cell_18_controlled_field_validation.csv",
    )
).resolve()
CELL18_RANDOM_SPEC_INPUT_PATH = Path(
    globals().get(
        "CELL18_RANDOM_SPEC_PATH",
        METADATA_DIR / "notebook_4_cell_18_random_stream_specification.json",
    )
).resolve()
CELL18_SITE_ORDER_INPUT_PATH = Path(
    globals().get(
        "CELL18_SITE_ORDER_PATH",
        METADATA_DIR / "notebook_4_cell_18_site_order.csv",
    )
).resolve()

required_paths = {
    "Notebook 3 event-occurrence catalog": EVENT_CATALOG_PATH,
    "Cell 17 bulk GMM output": CELL17_FINAL_OUTPUT_INPUT_PATH,
    "Cell 17 summary": CELL17_SUMMARY_INPUT_PATH,
    "Cell 17 validation": CELL17_VALIDATION_INPUT_PATH,
    "Cell 17 chunk manifest": CELL17_MANIFEST_INPUT_PATH,
    "Cell 18 summary": CELL18_SUMMARY_INPUT_PATH,
    "Cell 18 validation": CELL18_VALIDATION_INPUT_PATH,
    "Cell 18 random-stream specification": CELL18_RANDOM_SPEC_INPUT_PATH,
    "Cell 18 site order": CELL18_SITE_ORDER_INPUT_PATH,
}
missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]
if missing_paths:
    raise FileNotFoundError(
        "Cell 19 is missing required validated inputs:\n"
        + "\n".join(f"  - {item}" for item in missing_paths)
    )

CELL19_OUTPUT_DIR = DATA_DIR / "processed" / "notebook_4_full_baseline_fields"
CELL19_WORK_DIR = METADATA_DIR / "notebook_4_full_baseline_fields_work"
CELL19_FIELD_CHUNK_DIR = CELL19_WORK_DIR / "field_chunks"
CELL19_EVENT_CHUNK_DIR = CELL19_WORK_DIR / "event_chunks"
CELL19_MARKER_DIR = CELL19_WORK_DIR / "completion_markers"
CELL19_CHUNK_VALIDATION_DIR = CELL19_WORK_DIR / "chunk_validation"

for directory in [
    CELL19_OUTPUT_DIR,
    CELL19_WORK_DIR,
    CELL19_FIELD_CHUNK_DIR,
    CELL19_EVENT_CHUNK_DIR,
    CELL19_MARKER_DIR,
    CELL19_CHUNK_VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

CELL19_OCCURRENCE_INDEX_PATH = (
    METADATA_DIR / "notebook_4_cell_19_event_occurrence_index.csv.gz"
)
CELL19_RANDOM_SPEC_PATH = (
    METADATA_DIR / "notebook_4_cell_19_random_stream_specification.json"
)
CELL19_MANIFEST_PATH = METADATA_DIR / "notebook_4_cell_19_chunk_manifest.csv"
CELL19_FINAL_FIELDS_PATH = (
    CELL19_OUTPUT_DIR / "full_baseline_ground_motion_fields.csv.gz"
)
CELL19_FINAL_EVENTS_PATH = (
    CELL19_OUTPUT_DIR / "full_event_residual_ledger.csv.gz"
)
CELL19_VALIDATION_PATH = (
    METADATA_DIR / "notebook_4_cell_19_full_field_validation.csv"
)
CELL19_SUMMARY_PATH = (
    METADATA_DIR / "notebook_4_cell_19_full_field_summary.json"
)


# ----------------------------------------------------------------------------
# 3. Utility functions
# ----------------------------------------------------------------------------

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            block = file_handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def canonical_json_bytes(value: object) -> bytes:
    return json.dumps(
        value,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=True,
        allow_nan=False,
    ).encode("utf-8")


def normalize_boolean_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    return (
        values.astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def stable_seed(*parts: object) -> tuple[int, str]:
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    raw = hashlib.sha256(payload).digest()[:16]
    return int.from_bytes(raw, byteorder="little", signed=False), raw.hex()


def rng_from_seed(seed: int) -> np.random.Generator:
    return np.random.Generator(np.random.PCG64DXSM(seed))


def atomic_write_json(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def atomic_write_csv(frame: pd.DataFrame, path: Path, **kwargs) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, **kwargs)
    os.replace(temporary, path)


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            fileobj=raw,
            mode="wb",
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as compressed:
            text = frame.to_csv(
                index=False,
                na_rep="",
                lineterminator="\n",
                float_format="%.17g",
            ).encode("utf-8")
            compressed.write(text)
    os.replace(temporary, path)


def combine_gzip_csv_files(source_paths: list[Path], destination_path: Path) -> None:
    temporary = destination_path.with_suffix(destination_path.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_destination:
        with gzip.GzipFile(
            fileobj=raw_destination,
            mode="wb",
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as destination:
            for source_path in source_paths:
                with gzip.open(source_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        destination.write(header)
                    elif header != expected_header:
                        raise RuntimeError(
                            "Chunk CSV headers are inconsistent:\n"
                            f"  {source_path}"
                        )
                    shutil.copyfileobj(source, destination, length=1024 * 1024)
    os.replace(temporary, destination_path)


def safe_correlation(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]
    if x.size < 3 or np.std(x, ddof=1) == 0.0 or np.std(y, ddof=1) == 0.0:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


def build_validation_frame(checks: list[dict[str, object]]) -> pd.DataFrame:
    return pd.DataFrame(
        checks,
        columns=["category", "check", "observed", "expected", "passes", "note"],
    )


def add_check(
    checks: list[dict[str, object]],
    category: str,
    check: str,
    observed: object,
    expected: object,
    passes: bool,
    note: str = "",
) -> None:
    checks.append(
        {
            "category": category,
            "check": check,
            "observed": observed,
            "expected": expected,
            "passes": bool(passes),
            "note": note,
        }
    )


def normalize_manifest_path(value: object) -> Path:
    return Path(str(value)).expanduser().resolve()


class OnlineMoments:
    def __init__(self) -> None:
        self.count = 0
        self.total = 0.0
        self.total_sq = 0.0

    def update(self, values: Iterable[float]) -> None:
        array = np.asarray(values, dtype=float)
        array = array[np.isfinite(array)]
        self.count += int(array.size)
        self.total += float(np.sum(array))
        self.total_sq += float(np.sum(array * array))

    @property
    def mean(self) -> float:
        return self.total / self.count if self.count else float("nan")

    @property
    def std(self) -> float:
        if self.count < 2:
            return float("nan")
        variance = (
            self.total_sq - self.count * self.mean * self.mean
        ) / (self.count - 1)
        return math.sqrt(max(variance, 0.0))


class OnlineBivariate:
    def __init__(self) -> None:
        self.count = 0
        self.sx = 0.0
        self.sy = 0.0
        self.sxx = 0.0
        self.syy = 0.0
        self.sxy = 0.0

    def update(self, x: Iterable[float], y: Iterable[float]) -> None:
        x_array = np.asarray(x, dtype=float)
        y_array = np.asarray(y, dtype=float)
        mask = np.isfinite(x_array) & np.isfinite(y_array)
        x_array = x_array[mask]
        y_array = y_array[mask]
        self.count += int(x_array.size)
        self.sx += float(np.sum(x_array))
        self.sy += float(np.sum(y_array))
        self.sxx += float(np.sum(x_array * x_array))
        self.syy += float(np.sum(y_array * y_array))
        self.sxy += float(np.sum(x_array * y_array))

    @property
    def correlation(self) -> float:
        if self.count < 3:
            return float("nan")
        cov_num = self.sxy - self.sx * self.sy / self.count
        var_x = self.sxx - self.sx * self.sx / self.count
        var_y = self.syy - self.sy * self.sy / self.count
        if var_x <= 0.0 or var_y <= 0.0:
            return float("nan")
        return float(cov_num / math.sqrt(var_x * var_y))


# ----------------------------------------------------------------------------
# 4. Baker and Jayaram (2008) cross-IMT correlation
# ----------------------------------------------------------------------------

def baker_jayaram_2008_correlation(period_1_s: float, period_2_s: float) -> float:
    t_min = min(float(period_1_s), float(period_2_s))
    t_max = max(float(period_1_s), float(period_2_s))
    if t_min <= 0.0:
        t_min = PGA_PROXY_PERIOD_S
    if t_max <= 0.0:
        t_max = PGA_PROXY_PERIOD_S
    if not (0.01 <= t_min <= 10.0 and 0.01 <= t_max <= 10.0):
        raise ValueError("Baker-Jayaram periods must lie in [0.01, 10] s.")

    c1 = 1.0 - math.cos(
        math.pi / 2.0
        - 0.366 * math.log(t_max / max(t_min, 0.109))
    )
    if t_max < 0.2:
        c2 = 1.0 - 0.105 * (
            1.0 - 1.0 / (1.0 + math.exp(100.0 * t_max - 5.0))
        ) * (t_max - t_min) / (t_max - 0.0099)
    else:
        c2 = 0.0
    c3 = c2 if t_max < 0.109 else c1
    c4 = c1 + 0.5 * (math.sqrt(c3) - c3) * (
        1.0 + math.cos(math.pi * t_min / 0.109)
    )

    if t_max <= 0.109:
        rho = c2
    elif t_min > 0.109:
        rho = c1
    elif t_max < 0.2:
        rho = min(c2, c4)
    else:
        rho = c4
    return float(rho)


CROSS_IMT_RHO = baker_jayaram_2008_correlation(
    PGA_PROXY_PERIOD_S,
    SEASIDE_W2_PERIOD_S,
)
EXPECTED_PGA_SA0P4_RHO = 0.7321409900247263
if not math.isclose(
    CROSS_IMT_RHO,
    EXPECTED_PGA_SA0P4_RHO,
    rel_tol=0.0,
    abs_tol=RHO_FORMULA_ATOL,
):
    raise RuntimeError(
        "The Baker-Jayaram PGA-SA0P4 correlation does not reproduce the "
        "accepted Cell 18 value."
    )
CROSS_IMT_COMPLEMENT = math.sqrt(1.0 - CROSS_IMT_RHO**2)


# ----------------------------------------------------------------------------
# 5. Validate Cell 17 and Cell 18 prerequisites
# ----------------------------------------------------------------------------

cell17_summary = json.loads(CELL17_SUMMARY_INPUT_PATH.read_text(encoding="utf-8"))
cell18_summary = json.loads(CELL18_SUMMARY_INPUT_PATH.read_text(encoding="utf-8"))
cell18_random_spec = json.loads(
    CELL18_RANDOM_SPEC_INPUT_PATH.read_text(encoding="utf-8")
)
cell17_validation = pd.read_csv(CELL17_VALIDATION_INPUT_PATH, low_memory=False)
cell18_validation = pd.read_csv(CELL18_VALIDATION_INPUT_PATH, low_memory=False)

for name, summary in [("Cell 17", cell17_summary), ("Cell 18", cell18_summary)]:
    if not bool(summary.get("all_checks_passed", False)):
        raise RuntimeError(f"{name} summary does not report all checks passed.")

for name, validation in [
    ("Cell 17", cell17_validation),
    ("Cell 18", cell18_validation),
]:
    if "passes" not in validation.columns:
        raise RuntimeError(f"{name} validation has no 'passes' column.")
    passes = normalize_boolean_series(validation["passes"])
    if not passes.all():
        failed = validation.loc[~passes, "check"].astype(str).tolist()
        raise RuntimeError(
            f"{name} validation contains failed checks:\n"
            + "\n".join(f"  - {item}" for item in failed)
        )

if int(cell17_summary.get("ruptures", -1)) != EXPECTED_RUPTURES:
    raise RuntimeError("Cell 17 rupture count differs from 3,927.")
if int(cell17_summary.get("sites", -1)) != EXPECTED_SITES:
    raise RuntimeError("Cell 17 site count differs from 470.")
if int(cell17_summary.get("rupture_site_rows", -1)) != EXPECTED_CELL17_ROWS:
    raise RuntimeError("Cell 17 row count differs from 1,845,690.")
if str(cell17_summary.get("primary_baseline_branch")) != PRIMARY_BASELINE_BRANCH:
    raise RuntimeError("Cell 17 primary branch is not epi-off.")
if not math.isclose(
    float(cell17_summary.get("period_s", float("nan"))),
    SEASIDE_W2_PERIOD_S,
    rel_tol=0.0,
    abs_tol=PERIOD_MATCH_ATOL,
):
    raise RuntimeError("Cell 17 period is not exactly 0.40 s.")
if str(cell17_summary.get("period_resolution")) != "exact USGS SA0P4; no interpolation":
    raise RuntimeError("Cell 17 period resolution is not exact USGS SA0P4.")

actual_cell17_hash = sha256_file(CELL17_FINAL_OUTPUT_INPUT_PATH)
if actual_cell17_hash != str(cell17_summary.get("final_output_sha256", "")):
    raise RuntimeError("Cell 17 final output hash differs from its accepted summary.")

cell18_seed = int(cell18_random_spec.get("base_random_seed", -1))
if cell18_seed != BASE_RANDOM_SEED:
    raise RuntimeError(
        "Cell 19 base seed differs from the accepted Cell 18 random-stream "
        f"specification ({cell18_seed})."
    )
if not math.isclose(
    float(cell18_random_spec.get("cross_imt_structure", {}).get("rho", np.nan)),
    CROSS_IMT_RHO,
    rel_tol=0.0,
    abs_tol=RHO_FORMULA_ATOL,
):
    raise RuntimeError("Cell 18 and Cell 19 cross-IMT correlations differ.")

site_order = pd.read_csv(CELL18_SITE_ORDER_INPUT_PATH, low_memory=False)
required_site_order_columns = [
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
]
missing_site_columns = [
    column for column in required_site_order_columns if column not in site_order
]
if missing_site_columns:
    raise RuntimeError(
        "Cell 18 site order is missing columns:\n"
        + "\n".join(f"  - {column}" for column in missing_site_columns)
    )
site_order = site_order[required_site_order_columns].copy()
site_order["site_ordinal"] = pd.to_numeric(
    site_order["site_ordinal"], errors="coerce"
)
if (
    len(site_order) != EXPECTED_SITES
    or not np.array_equal(
        site_order["site_ordinal"].to_numpy(dtype=int),
        np.arange(EXPECTED_SITES, dtype=int),
    )
    or site_order["site_id"].astype(str).duplicated().any()
):
    raise RuntimeError("Cell 18 site order is not the expected 470-site order.")

print(
    "Cell 17 and Cell 18 prerequisites passed: validated epi-off parameters "
    "and deterministic random-stream design."
)
print(
    "Cross-IMT residual correlation: "
    f"Baker-Jayaram rho(PGA, SA0P4) = {CROSS_IMT_RHO:.12f}"
)


# ----------------------------------------------------------------------------
# 6. Load and normalize the Notebook 3 event-occurrence catalog
# ----------------------------------------------------------------------------

catalog = pd.read_csv(EVENT_CATALOG_PATH, low_memory=False)
required_catalog_columns = ["event_id", "rupture_id"]
missing_catalog_columns = [
    column for column in required_catalog_columns if column not in catalog.columns
]
if missing_catalog_columns:
    raise RuntimeError(
        "The Notebook 3 catalog is missing required occurrence columns:\n"
        + "\n".join(f"  - {column}" for column in missing_catalog_columns)
        + "\n\nAvailable columns:\n"
        + "\n".join(f"  - {column}" for column in catalog.columns)
    )

catalog["event_id"] = catalog["event_id"].astype("string").fillna("").astype(str)
catalog["rupture_id"] = (
    catalog["rupture_id"].astype("string").fillna("").astype(str)
)
if catalog.empty:
    raise RuntimeError("The Notebook 3 event-occurrence catalog is empty.")
if catalog["event_id"].eq("").any():
    raise RuntimeError("The Notebook 3 catalog contains blank event IDs.")
if catalog["rupture_id"].eq("").any():
    raise RuntimeError("The Notebook 3 catalog contains blank rupture IDs.")

YEAR_COLUMN_CANDIDATES = [
    "catalog_year",
    "simulation_year",
    "stochastic_year",
    "year",
    "year_id",
    "year_index",
    "catalog_year_index",
    "trial_year",
    "sample_year",
]
year_candidates_present = [
    column for column in YEAR_COLUMN_CANDIDATES if column in catalog.columns
]
if not year_candidates_present:
    raise RuntimeError(
        "Cell 19 requires an explicit stochastic-year column so later annual "
        "loss calculations can preserve zero-event years. None of the "
        "recognized year columns was found.\n\nRecognized names:\n"
        + "\n".join(f"  - {column}" for column in YEAR_COLUMN_CANDIDATES)
        + "\n\nAvailable catalog columns:\n"
        + "\n".join(f"  - {column}" for column in catalog.columns)
    )

YEAR_COLUMN = year_candidates_present[0]
year_values = pd.to_numeric(catalog[YEAR_COLUMN], errors="coerce")
if (
    not np.isfinite(year_values).all()
    or not np.isclose(year_values, np.round(year_values), atol=1.0e-10).all()
):
    raise RuntimeError(f"Catalog year column {YEAR_COLUMN!r} is not integer-valued.")
catalog["catalog_year"] = np.round(year_values).astype(np.int64)

for other_year_column in year_candidates_present[1:]:
    other = pd.to_numeric(catalog[other_year_column], errors="coerce")
    if (
        np.isfinite(other).all()
        and np.isclose(other, np.round(other), atol=1.0e-10).all()
        and not np.array_equal(
            catalog["catalog_year"].to_numpy(dtype=np.int64),
            np.round(other).astype(np.int64).to_numpy(),
        )
    ):
        raise RuntimeError(
            "Multiple recognized catalog-year columns disagree: "
            f"{YEAR_COLUMN!r} and {other_year_column!r}."
        )

catalog["_original_catalog_row"] = np.arange(len(catalog), dtype=np.int64)
if catalog[["catalog_year", "event_id"]].duplicated().any():
    duplicates = catalog.loc[
        catalog[["catalog_year", "event_id"]].duplicated(keep=False),
        ["catalog_year", "event_id", "rupture_id"],
    ].head(30)
    raise RuntimeError(
        "Catalog (year, event_id) pairs are not unique. These pairs are the "
        "required event-occurrence identities.\n\n"
        + duplicates.to_string(index=False)
    )

catalog = catalog.sort_values(
    ["catalog_year", "event_id", "_original_catalog_row"],
    kind="mergesort",
).reset_index(drop=True)
catalog["occurrence_ordinal"] = np.arange(len(catalog), dtype=np.int64)
catalog["occurrence_id"] = (
    "Y"
    + catalog["catalog_year"].astype(str)
    + "|"
    + catalog["event_id"].astype(str)
)
if catalog["occurrence_id"].duplicated().any():
    raise RuntimeError("Normalized occurrence IDs are not unique.")

# Try to preserve the declared simulation duration. This is used later when
# zero-event years are inserted into annual loss tables. The event-field
# simulation itself only needs the years attached to actual occurrences.
def find_declared_catalog_year_count() -> tuple[int | None, str]:
    global_candidates = [
        "CATALOG_YEARS",
        "N_CATALOG_YEARS",
        "SIMULATION_YEARS",
        "N_SIMULATION_YEARS",
        "CATALOG_DURATION_YEARS",
    ]
    for name in global_candidates:
        if name in globals():
            try:
                value = int(globals()[name])
            except (TypeError, ValueError):
                continue
            if value > 0:
                return value, f"notebook global: {name}"

    key_candidates = {
        "catalog_years",
        "simulation_years",
        "n_years",
        "number_of_years",
        "years_simulated",
        "catalog_duration_years",
    }

    def recursive_find(value: object) -> int | None:
        if isinstance(value, dict):
            for key, child in value.items():
                if str(key).lower() in key_candidates:
                    try:
                        parsed = int(child)
                    except (TypeError, ValueError):
                        parsed = -1
                    if parsed > 0:
                        return parsed
                result = recursive_find(child)
                if result is not None:
                    return result
        elif isinstance(value, list):
            for child in value:
                result = recursive_find(child)
                if result is not None:
                    return result
        return None

    metadata_candidates = sorted(
        {
            *EVENT_CATALOG_PATH.parent.glob("*catalog*metadata*.json"),
            *EVENT_CATALOG_PATH.parent.glob("*catalog*summary*.json"),
            *EVENT_CATALOG_PATH.parent.glob("*metadata*.json"),
        }
    )
    for path in metadata_candidates:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except (OSError, json.JSONDecodeError):
            continue
        result = recursive_find(payload)
        if result is not None:
            return result, f"catalog metadata: {path}"
    return None, "not located"


DECLARED_CATALOG_YEARS, DECLARED_CATALOG_YEARS_PROVENANCE = (
    find_declared_catalog_year_count()
)


# ----------------------------------------------------------------------------
# 7. Resolve Cell 17 rupture partitions and map catalog occurrences
# ----------------------------------------------------------------------------

cell17_manifest = pd.read_csv(CELL17_MANIFEST_INPUT_PATH, low_memory=False)
required_manifest_columns = [
    "chunk_number",
    "chunk_id",
    "start",
    "stop",
    "rupture_count",
    "site_count",
    "expected_rows",
    "chunk_index_path",
    "chunk_index_hash",
    "output",
    "status",
]
missing_manifest_columns = [
    column for column in required_manifest_columns if column not in cell17_manifest
]
if missing_manifest_columns:
    raise RuntimeError(
        "Cell 17 manifest is missing required columns:\n"
        + "\n".join(f"  - {column}" for column in missing_manifest_columns)
    )
if len(cell17_manifest) != 16:
    raise RuntimeError(
        f"Expected 16 Cell 17 rupture partitions; observed {len(cell17_manifest)}."
    )

rupture_map_frames: list[pd.DataFrame] = []
normalized_cell17_records: list[dict[str, object]] = []
for _, row in cell17_manifest.sort_values("chunk_number").iterrows():
    chunk_number = int(row["chunk_number"])
    chunk_id = f"{chunk_number:04d}"
    chunk_index_path = normalize_manifest_path(row["chunk_index_path"])
    gmm_output_path = normalize_manifest_path(row["output"])
    if not chunk_index_path.is_file():
        raise FileNotFoundError(f"Missing Cell 17 chunk index: {chunk_index_path}")
    if not gmm_output_path.is_file():
        raise FileNotFoundError(f"Missing Cell 17 chunk output: {gmm_output_path}")
    if sha256_file(chunk_index_path) != str(row["chunk_index_hash"]):
        raise RuntimeError(
            f"Cell 17 chunk-index hash mismatch for partition {chunk_id}."
        )
    chunk_index = pd.read_csv(
        chunk_index_path,
        dtype={"event_id": str, "rupture_id": str, "source_type": str},
        low_memory=False,
    )
    required_index_columns = [
        "rupture_ordinal",
        "event_id",
        "rupture_id",
        "source_type",
        "gmm_name",
    ]
    missing = [column for column in required_index_columns if column not in chunk_index]
    if missing:
        raise RuntimeError(
            f"Cell 17 chunk index {chunk_id} is missing columns: {missing}"
        )
    chunk_index = chunk_index[required_index_columns].copy()
    chunk_index["cell17_chunk_id"] = chunk_id
    rupture_map_frames.append(chunk_index)
    normalized_cell17_records.append(
        {
            "chunk_number": chunk_number,
            "cell17_chunk_id": chunk_id,
            "cell17_chunk_index_path": str(chunk_index_path),
            "cell17_chunk_index_sha256": sha256_file(chunk_index_path),
            "cell17_gmm_output_path": str(gmm_output_path),
            "cell17_gmm_output_sha256": sha256_file(gmm_output_path),
        }
    )

rupture_map = pd.concat(rupture_map_frames, ignore_index=True)
if len(rupture_map) != EXPECTED_RUPTURES:
    raise RuntimeError(
        f"Expected {EXPECTED_RUPTURES:,} rupture-map rows; observed {len(rupture_map):,}."
    )
if rupture_map["rupture_id"].astype(str).duplicated().any():
    raise RuntimeError("Cell 17 rupture-map IDs are not unique.")
rupture_map["rupture_id"] = rupture_map["rupture_id"].astype(str)
rupture_map["rupture_ordinal"] = pd.to_numeric(
    rupture_map["rupture_ordinal"], errors="coerce"
).astype(np.int64)

# Join Cell 17 rupture metadata using collision-proof temporary names. The
# Notebook 3 catalog may already contain columns such as source_type. Using
# temporary names prevents pandas from silently creating _x and _y columns.
rupture_map_for_join = rupture_map[
    [
        "rupture_id",
        "rupture_ordinal",
        "event_id",
        "source_type",
        "gmm_name",
        "cell17_chunk_id",
    ]
].rename(
    columns={
        "rupture_ordinal": "_cell17_rupture_ordinal",
        "event_id": "_cell17_rupture_template_event_id",
        "source_type": "_cell17_source_type",
        "gmm_name": "_cell17_gmm_name",
        "cell17_chunk_id": "_cell17_chunk_id",
    }
)

catalog = catalog.merge(
    rupture_map_for_join,
    on="rupture_id",
    how="left",
    validate="many_to_one",
)

if catalog["_cell17_rupture_ordinal"].isna().any():
    missing_ids = sorted(
        catalog.loc[
            catalog["_cell17_rupture_ordinal"].isna(), "rupture_id"
        ].unique()
    )
    raise RuntimeError(
        "Catalog occurrences reference rupture IDs absent from Cell 17:\n"
        + "\n".join(f"  - {value}" for value in missing_ids[:50])
    )

# If Notebook 3 already supplied source_type, verify it against the
# authoritative Cell 17 rupture classification before replacing it.
if "source_type" in catalog.columns:
    catalog_source_type = (
        catalog["source_type"]
        .astype("string")
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    cell17_source_type = (
        catalog["_cell17_source_type"]
        .astype("string")
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )
    source_type_mismatch = (
        catalog_source_type.ne("")
        & catalog_source_type.ne(cell17_source_type)
    )
    if source_type_mismatch.any():
        mismatch_preview = catalog.loc[
            source_type_mismatch,
            ["catalog_year", "event_id", "rupture_id", "source_type", "_cell17_source_type"],
        ].head(30)
        raise RuntimeError(
            "Notebook 3 source_type values disagree with the authoritative "
            "Cell 17 rupture classification.\n\n"
            + mismatch_preview.to_string(index=False)
        )
    catalog["catalog_source_type"] = catalog["source_type"]

# Apply the authoritative Cell 17 metadata used by the GMM simulation.
catalog["rupture_ordinal"] = pd.to_numeric(
    catalog["_cell17_rupture_ordinal"], errors="raise"
).astype(np.int64)
catalog["rupture_template_event_id"] = catalog[
    "_cell17_rupture_template_event_id"
].astype(str)
catalog["source_type"] = (
    catalog["_cell17_source_type"].astype(str).str.strip().str.upper()
)
catalog["gmm_name"] = catalog["_cell17_gmm_name"].astype(str)
catalog["cell17_chunk_id"] = catalog["_cell17_chunk_id"].astype(str)

if not set(catalog["source_type"].unique()).issubset({"INTERFACE", "SLAB"}):
    raise RuntimeError(
        "Cell 17 supplied unsupported source types: "
        f"{sorted(catalog['source_type'].unique().tolist())}"
    )

catalog = catalog.drop(
    columns=[
        "_cell17_rupture_ordinal",
        "_cell17_rupture_template_event_id",
        "_cell17_source_type",
        "_cell17_gmm_name",
        "_cell17_chunk_id",
    ]
)

occurrence_index_columns = [
    "occurrence_ordinal",
    "occurrence_id",
    "catalog_year",
    "event_id",
    "rupture_id",
    "rupture_ordinal",
    "rupture_template_event_id",
    "source_type",
    "gmm_name",
    "cell17_chunk_id",
]
occurrence_index = catalog[occurrence_index_columns].copy()
write_gzip_csv_deterministic(occurrence_index, CELL19_OCCURRENCE_INDEX_PATH)

EXPECTED_OCCURRENCES = len(occurrence_index)
EXPECTED_FIELD_ROWS = EXPECTED_OCCURRENCES * EXPECTED_SITES
OBSERVED_EVENT_YEARS = occurrence_index["catalog_year"].nunique()
MINIMUM_EVENT_YEAR = int(occurrence_index["catalog_year"].min())
MAXIMUM_EVENT_YEAR = int(occurrence_index["catalog_year"].max())

print("=" * 78)
print("FULL ANNUAL-CATALOG BASELINE STOCHASTIC GROUND-MOTION FIELDS")
print("=" * 78)
print(f"Catalog occurrences:     {EXPECTED_OCCURRENCES:,}")
print(f"Observed event years:    {OBSERVED_EVENT_YEARS:,}")
print(f"Catalog year column:     {YEAR_COLUMN}")
print(f"Event-year range:        {MINIMUM_EVENT_YEAR} to {MAXIMUM_EVENT_YEAR}")
if DECLARED_CATALOG_YEARS is not None:
    print(f"Declared catalog years:  {DECLARED_CATALOG_YEARS:,}")
else:
    print("Declared catalog years:  not recovered; preserve Notebook 3 metadata")
print(f"Seaside W2 sites:        {EXPECTED_SITES:,}")
print(f"Production field rows:   {EXPECTED_FIELD_ROWS:,}")
print(f"Primary branch:          {PRIMARY_BASELINE_BRANCH}")
print(f"Cross-IMT rho:           {CROSS_IMT_RHO:.12f}")
print("Spatial residual model:  independent across sites")


# ----------------------------------------------------------------------------
# 8. Freeze the production random-stream specification
# ----------------------------------------------------------------------------

production_random_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "base_random_seed": BASE_RANDOM_SEED,
    "generator": "numpy.random.PCG64DXSM",
    "seed_derivation": (
        "First 128 bits of SHA-256 over pipe-delimited identifiers. Separate "
        "event-level and site-level streams are created for each catalog "
        "occurrence."
    ),
    "occurrence_identity": "catalog_year + event_id + rupture_id",
    "event_seed_parts": (
        "base_random_seed | cell19 | occurrence_id | rupture_id | event"
    ),
    "site_seed_parts": (
        "base_random_seed | cell19 | occurrence_id | rupture_id | site"
    ),
    "site_order_requirement": (
        "The site generator emits values in Cell 18 site_ordinal order 0..469."
    ),
    "aleatory_equation": (
        "ln(IM_e,s) = mu_epi_off_e,s + tau_e,s*eta_e + phi_e,s*epsilon_e,s"
    ),
    "between_event_structure": (
        "One normalized eta per event occurrence and IMT, repeated across all "
        "470 sites."
    ),
    "within_event_spatial_structure": (
        "Independent latent standard normals across different sites. No "
        "spatial correlation is applied in the Phase 1 baseline."
    ),
    "cross_imt_structure": {
        "model": "Baker and Jayaram (2008) inter-period residual correlation",
        "pga_proxy_period_s": PGA_PROXY_PERIOD_S,
        "sa_period_s": SEASIDE_W2_PERIOD_S,
        "rho": CROSS_IMT_RHO,
        "application": (
            "The same normalized cross-IMT transform is applied separately to "
            "between-event and within-event latent residual pairs."
        ),
    },
    "latent_transform": {
        "pga": "z1",
        "sa0p4": "rho*z1 + sqrt(1-rho^2)*z2",
    },
    "future_spatial_extension": (
        "Regenerate the same two site-level latent vectors from site_seed_hex, "
        "apply the selected spatial transform to each vector, and then apply "
        "the same cross-IMT combination."
    ),
    "catalog_path": str(EVENT_CATALOG_PATH),
    "catalog_sha256": sha256_file(EVENT_CATALOG_PATH),
    "catalog_year_column": YEAR_COLUMN,
    "catalog_occurrences": EXPECTED_OCCURRENCES,
    "declared_catalog_years": DECLARED_CATALOG_YEARS,
    "declared_catalog_years_provenance": DECLARED_CATALOG_YEARS_PROVENANCE,
    "site_order_path": str(CELL18_SITE_ORDER_INPUT_PATH),
    "site_order_sha256": sha256_file(CELL18_SITE_ORDER_INPUT_PATH),
    "cell18_random_specification_sha256": sha256_file(
        CELL18_RANDOM_SPEC_INPUT_PATH
    ),
    "cell17_output_sha256": actual_cell17_hash,
}
random_specification_hash = sha256_bytes(
    canonical_json_bytes(production_random_specification)
)
production_random_specification["specification_sha256"] = (
    random_specification_hash
)
atomic_write_json(CELL19_RANDOM_SPEC_PATH, production_random_specification)


# ----------------------------------------------------------------------------
# 9. Create the 16 production chunk records
# ----------------------------------------------------------------------------

cell17_record_by_id = {
    str(record["cell17_chunk_id"]): record
    for record in normalized_cell17_records
}
manifest_records: list[dict[str, object]] = []
for chunk_number in range(16):
    chunk_id = f"{chunk_number:04d}"
    occurrences = occurrence_index.loc[
        occurrence_index["cell17_chunk_id"].eq(chunk_id)
    ]
    cell17_record = cell17_record_by_id[chunk_id]
    manifest_records.append(
        {
            "chunk_number": chunk_number,
            "chunk_id": chunk_id,
            "catalog_occurrence_count": len(occurrences),
            "expected_field_rows": len(occurrences) * EXPECTED_SITES,
            "cell17_gmm_output_path": cell17_record["cell17_gmm_output_path"],
            "cell17_gmm_output_sha256": cell17_record[
                "cell17_gmm_output_sha256"
            ],
            "event_output": str(
                CELL19_EVENT_CHUNK_DIR / f"full_events_chunk_{chunk_id}.csv.gz"
            ),
            "field_output": str(
                CELL19_FIELD_CHUNK_DIR / f"full_fields_chunk_{chunk_id}.csv.gz"
            ),
            "validation_output": str(
                CELL19_CHUNK_VALIDATION_DIR
                / f"full_fields_chunk_{chunk_id}_validation.csv"
            ),
            "marker_path": str(
                CELL19_MARKER_DIR / f"full_fields_chunk_{chunk_id}.complete.json"
            ),
            "status": "pending",
            "completed_at_utc": "",
        }
    )


def save_manifest() -> None:
    frame = pd.DataFrame(manifest_records).sort_values("chunk_number")
    atomic_write_csv(frame, CELL19_MANIFEST_PATH, lineterminator="\n")


save_manifest()


# ----------------------------------------------------------------------------
# 10. Output schemas and chunk generation
# ----------------------------------------------------------------------------

GMM_TEMPLATE_COLUMNS = [
    "rupture_ordinal",
    "event_id",
    "rupture_id",
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
    "source_type",
    "gmm_name",
    "magnitude",
    "r_rup_km",
    "period_s",
    "vs30_mps",
    "pga_epi_off_mean_ln_g",
    "pga_tau_ln",
    "pga_phi_ln",
    "sa0p4_epi_off_mean_ln_g",
    "sa0p4_tau_ln",
    "sa0p4_phi_ln",
    "calculation_error",
]

FIELD_OUTPUT_COLUMNS = [
    "catalog_year",
    "catalog_event_id",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_ordinal",
    "rupture_template_event_id",
    "rupture_id",
    "source_type",
    "gmm_name",
    "magnitude",
    "site_ordinal",
    "site_id",
    "site_longitude",
    "site_latitude",
    "r_rup_km",
    "vs30_mps",
    "period_s",
    "cross_imt_rho",
    "eta_pga",
    "epsilon_pga",
    "pga_epi_off_mean_ln_g",
    "pga_tau_ln",
    "pga_phi_ln",
    "pga_between_event_shift_ln",
    "pga_within_event_shift_ln",
    "pga_simulated_ln_g",
    "pga_simulated_g",
    "eta_sa0p4",
    "epsilon_sa0p4",
    "sa0p4_epi_off_mean_ln_g",
    "sa0p4_tau_ln",
    "sa0p4_phi_ln",
    "sa0p4_between_event_shift_ln",
    "sa0p4_within_event_shift_ln",
    "sa0p4_simulated_ln_g",
    "sa0p4_simulated_g",
]

EVENT_OUTPUT_COLUMNS = [
    "catalog_year",
    "catalog_event_id",
    "occurrence_ordinal",
    "occurrence_id",
    "rupture_ordinal",
    "rupture_template_event_id",
    "rupture_id",
    "source_type",
    "gmm_name",
    "event_seed_hex",
    "site_seed_hex",
    "z_event_latent_1",
    "z_event_latent_2",
    "eta_pga",
    "eta_sa0p4",
    "cross_imt_rho",
]


def marker_is_valid(record: dict[str, object]) -> bool:
    marker_path = Path(str(record["marker_path"]))
    event_output = Path(str(record["event_output"]))
    field_output = Path(str(record["field_output"]))
    validation_output = Path(str(record["validation_output"]))
    if not all(
        path.is_file()
        for path in [marker_path, event_output, field_output, validation_output]
    ):
        return False
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
        validation = pd.read_csv(validation_output, low_memory=False)
    except (OSError, json.JSONDecodeError, pd.errors.ParserError):
        return False
    required_marker_values = {
        "pipeline_version": PIPELINE_VERSION,
        "random_specification_sha256": random_specification_hash,
        "catalog_sha256": sha256_file(EVENT_CATALOG_PATH),
        "cell17_gmm_output_sha256": str(record["cell17_gmm_output_sha256"]),
        "catalog_occurrence_count": int(record["catalog_occurrence_count"]),
        "expected_field_rows": int(record["expected_field_rows"]),
        "base_random_seed": BASE_RANDOM_SEED,
    }
    for key, value in required_marker_values.items():
        if marker.get(key) != value:
            return False
    if "passes" not in validation.columns:
        return False
    if not normalize_boolean_series(validation["passes"]).all():
        return False
    if VERIFY_EXISTING_HASHES:
        if marker.get("event_output_sha256") != sha256_file(event_output):
            return False
        if marker.get("field_output_sha256") != sha256_file(field_output):
            return False
        if marker.get("validation_sha256") != sha256_file(validation_output):
            return False
    return True


def validate_generated_chunk(
    record: dict[str, object],
    chunk_occurrences: pd.DataFrame,
) -> pd.DataFrame:
    event_output = Path(str(record["event_output"]))
    field_output = Path(str(record["field_output"]))
    expected_occurrences = int(record["catalog_occurrence_count"])
    expected_rows = int(record["expected_field_rows"])

    event_frame = pd.read_csv(
        event_output,
        dtype={
            "catalog_event_id": str,
            "occurrence_id": str,
            "rupture_template_event_id": str,
            "rupture_id": str,
            "source_type": str,
            "gmm_name": str,
            "event_seed_hex": str,
            "site_seed_hex": str,
        },
        low_memory=False,
    )
    field_frame = pd.read_csv(
        field_output,
        dtype={
            "catalog_event_id": str,
            "occurrence_id": str,
            "rupture_template_event_id": str,
            "rupture_id": str,
            "source_type": str,
            "gmm_name": str,
            "site_id": str,
        },
        low_memory=False,
    )

    checks: list[dict[str, object]] = []
    add_check(checks, "Rows", "Event-ledger row count", len(event_frame), expected_occurrences, len(event_frame) == expected_occurrences)
    add_check(checks, "Rows", "Field row count", len(field_frame), expected_rows, len(field_frame) == expected_rows)
    add_check(checks, "Coverage", "Event-ledger occurrence IDs are unique", int(event_frame["occurrence_id"].nunique()), expected_occurrences, event_frame["occurrence_id"].nunique() == expected_occurrences and not event_frame["occurrence_id"].duplicated().any())
    add_check(checks, "Coverage", "Field occurrence-site pairs are unique", int(field_frame[["occurrence_id", "site_id"]].drop_duplicates().shape[0]), expected_rows, field_frame[["occurrence_id", "site_id"]].drop_duplicates().shape[0] == expected_rows)

    occurrence_counts = field_frame.groupby("occurrence_id", sort=False).size()
    add_check(checks, "Coverage", "Every occurrence has 470 sites", int(occurrence_counts.eq(EXPECTED_SITES).sum()), expected_occurrences, len(occurrence_counts) == expected_occurrences and occurrence_counts.eq(EXPECTED_SITES).all())
    add_check(checks, "Coverage", "All expected occurrence IDs are present", len(set(chunk_occurrences["occurrence_id"]) - set(event_frame["occurrence_id"])), 0, set(chunk_occurrences["occurrence_id"]) == set(event_frame["occurrence_id"]))

    seed_unique = (
        event_frame["event_seed_hex"].nunique() == expected_occurrences
        and event_frame["site_seed_hex"].nunique() == expected_occurrences
    )
    add_check(checks, "Random streams", "Event and site seeds are unique by occurrence", int(seed_unique), 1, seed_unique)

    eta_pga_span = field_frame.groupby("occurrence_id")["eta_pga"].agg(lambda values: float(values.max() - values.min()))
    eta_sa_span = field_frame.groupby("occurrence_id")["eta_sa0p4"].agg(lambda values: float(values.max() - values.min()))
    maximum_eta_span = max(float(eta_pga_span.max()) if len(eta_pga_span) else 0.0, float(eta_sa_span.max()) if len(eta_sa_span) else 0.0)
    add_check(checks, "Random streams", "Between-event residuals are constant across sites", f"{maximum_eta_span:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", maximum_eta_span <= EXACT_RECONSTRUCTION_ATOL)

    ledger_eta = event_frame.set_index("occurrence_id")[["eta_pga", "eta_sa0p4"]]
    first_field_eta = field_frame.drop_duplicates("occurrence_id").set_index("occurrence_id")[["eta_pga", "eta_sa0p4"]]
    eta_ledger_difference = float(np.max(np.abs(ledger_eta.sort_index().to_numpy(dtype=float) - first_field_eta.sort_index().to_numpy(dtype=float)))) if expected_occurrences else 0.0
    add_check(checks, "Random streams", "Event ledger and field eta values agree", f"{eta_ledger_difference:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", eta_ledger_difference <= EXACT_RECONSTRUCTION_ATOL)

    pga_reconstructed = (
        field_frame["pga_epi_off_mean_ln_g"]
        + field_frame["pga_tau_ln"] * field_frame["eta_pga"]
        + field_frame["pga_phi_ln"] * field_frame["epsilon_pga"]
    )
    sa_reconstructed = (
        field_frame["sa0p4_epi_off_mean_ln_g"]
        + field_frame["sa0p4_tau_ln"] * field_frame["eta_sa0p4"]
        + field_frame["sa0p4_phi_ln"] * field_frame["epsilon_sa0p4"]
    )
    pga_error = float(np.max(np.abs(pga_reconstructed - field_frame["pga_simulated_ln_g"]))) if expected_rows else 0.0
    sa_error = float(np.max(np.abs(sa_reconstructed - field_frame["sa0p4_simulated_ln_g"]))) if expected_rows else 0.0
    add_check(checks, "Equation", "PGA equation reconstructs exactly", f"{pga_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", pga_error <= EXACT_RECONSTRUCTION_ATOL)
    add_check(checks, "Equation", "SA0P4 equation reconstructs exactly", f"{sa_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", sa_error <= EXACT_RECONSTRUCTION_ATOL)

    pga_exp_error = float(np.max(np.abs(np.exp(field_frame["pga_simulated_ln_g"]) - field_frame["pga_simulated_g"]))) if expected_rows else 0.0
    sa_exp_error = float(np.max(np.abs(np.exp(field_frame["sa0p4_simulated_ln_g"]) - field_frame["sa0p4_simulated_g"]))) if expected_rows else 0.0
    add_check(checks, "Equation", "PGA exponentiation reconstructs exactly", f"{pga_exp_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", pga_exp_error <= EXACT_RECONSTRUCTION_ATOL)
    add_check(checks, "Equation", "SA0P4 exponentiation reconstructs exactly", f"{sa_exp_error:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", sa_exp_error <= EXACT_RECONSTRUCTION_ATOL)

    finite_columns = [
        column for column in FIELD_OUTPUT_COLUMNS
        if column not in {
            "catalog_event_id", "occurrence_id", "rupture_template_event_id",
            "rupture_id", "source_type", "gmm_name", "site_id"
        }
    ]
    numeric_finite = np.isfinite(field_frame[finite_columns].to_numpy(dtype=float)).all() if expected_rows else True
    add_check(checks, "Numeric", "All field numeric values are finite", int(numeric_finite), 1, numeric_finite)
    positive = field_frame["pga_simulated_g"].gt(0.0).all() and field_frame["sa0p4_simulated_g"].gt(0.0).all() if expected_rows else True
    add_check(checks, "Numeric", "All simulated intensities are positive", int(positive), 1, positive)
    period_valid = np.isclose(field_frame["period_s"], SEASIDE_W2_PERIOD_S, rtol=0.0, atol=PERIOD_MATCH_ATOL).all() if expected_rows else True
    add_check(checks, "Inputs", "Every row uses exact period 0.40 s", int(period_valid), 1, period_valid)
    rho_valid = np.isclose(field_frame["cross_imt_rho"], CROSS_IMT_RHO, rtol=0.0, atol=RHO_FORMULA_ATOL).all() if expected_rows else True
    add_check(checks, "Inputs", "Every row uses the accepted cross-IMT rho", int(rho_valid), 1, rho_valid)
    add_check(checks, "Inputs", "Only interface and slab source types are present", sorted(field_frame["source_type"].unique().tolist()) if expected_rows else [], ["INTERFACE", "SLAB"], set(field_frame["source_type"].unique()).issubset({"INTERFACE", "SLAB"}))

    mapping = field_frame.drop_duplicates("occurrence_id")[["occurrence_id", "rupture_id", "catalog_year", "catalog_event_id"]].sort_values("occurrence_id").reset_index(drop=True)
    expected_mapping = chunk_occurrences[["occurrence_id", "rupture_id", "catalog_year", "event_id"]].rename(columns={"event_id": "catalog_event_id"}).sort_values("occurrence_id").reset_index(drop=True)
    mapping_matches = mapping.astype(str).equals(expected_mapping.astype(str))
    add_check(checks, "Catalog mapping", "Field occurrence mapping matches Notebook 3 catalog", int(mapping_matches), 1, mapping_matches)

    return build_validation_frame(checks)


def generate_production_chunk(record: dict[str, object]) -> pd.DataFrame:
    chunk_id = str(record["chunk_id"])
    chunk_occurrences = occurrence_index.loc[
        occurrence_index["cell17_chunk_id"].eq(chunk_id)
    ].sort_values("occurrence_ordinal").reset_index(drop=True)
    event_output = Path(str(record["event_output"]))
    field_output = Path(str(record["field_output"]))
    validation_output = Path(str(record["validation_output"]))
    marker_path = Path(str(record["marker_path"]))
    gmm_output_path = Path(str(record["cell17_gmm_output_path"]))

    if marker_is_valid(record):
        return pd.read_csv(validation_output, low_memory=False)

    for path in [event_output, field_output, validation_output, marker_path]:
        path.unlink(missing_ok=True)

    template = pd.read_csv(
        gmm_output_path,
        usecols=GMM_TEMPLATE_COLUMNS,
        dtype={
            "event_id": str,
            "rupture_id": str,
            "site_id": str,
            "source_type": str,
            "gmm_name": str,
            "calculation_error": str,
        },
        low_memory=False,
    )
    template["calculation_error"] = template["calculation_error"].fillna("").astype(str)
    if template["calculation_error"].ne("").any():
        raise RuntimeError(f"Cell 17 template chunk {chunk_id} contains GMM errors.")
    template = template.sort_values(["rupture_ordinal", "site_ordinal"]).reset_index(drop=True)
    template_counts = template.groupby("rupture_id", sort=False).size()
    if not template_counts.eq(EXPECTED_SITES).all():
        raise RuntimeError(f"A rupture in Cell 17 template chunk {chunk_id} does not have 470 sites.")

    event_rows: list[dict[str, object]] = []
    temporary_field_path = field_output.with_suffix(field_output.suffix + ".tmp")
    header_written = False
    with temporary_field_path.open("wb") as raw:
        with gzip.GzipFile(
            fileobj=raw,
            mode="wb",
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as compressed:
            for occurrence in chunk_occurrences.itertuples(index=False):
                rupture_id = str(occurrence.rupture_id)
                occurrence_id = str(occurrence.occurrence_id)
                rupture_template = template.loc[
                    template["rupture_id"].eq(rupture_id)
                ].copy()
                if len(rupture_template) != EXPECTED_SITES:
                    raise RuntimeError(
                        f"Occurrence {occurrence_id} did not resolve to 470 Cell 17 rows."
                    )
                rupture_template = rupture_template.sort_values("site_ordinal").reset_index(drop=True)
                if not np.array_equal(
                    rupture_template["site_ordinal"].to_numpy(dtype=int),
                    np.arange(EXPECTED_SITES, dtype=int),
                ):
                    raise RuntimeError(
                        f"Occurrence {occurrence_id} has an invalid site order."
                    )

                event_seed, event_seed_hex = stable_seed(
                    BASE_RANDOM_SEED,
                    "cell19",
                    occurrence_id,
                    rupture_id,
                    "event",
                )
                site_seed, site_seed_hex = stable_seed(
                    BASE_RANDOM_SEED,
                    "cell19",
                    occurrence_id,
                    rupture_id,
                    "site",
                )
                event_rng = rng_from_seed(event_seed)
                site_rng = rng_from_seed(site_seed)
                z_event_1, z_event_2 = event_rng.standard_normal(2)
                z_site_1 = site_rng.standard_normal(EXPECTED_SITES)
                z_site_2 = site_rng.standard_normal(EXPECTED_SITES)

                eta_pga = float(z_event_1)
                eta_sa = float(
                    CROSS_IMT_RHO * z_event_1
                    + CROSS_IMT_COMPLEMENT * z_event_2
                )
                epsilon_pga = z_site_1
                epsilon_sa = (
                    CROSS_IMT_RHO * z_site_1
                    + CROSS_IMT_COMPLEMENT * z_site_2
                )

                pga_mu = rupture_template["pga_epi_off_mean_ln_g"].to_numpy(dtype=float)
                pga_tau = rupture_template["pga_tau_ln"].to_numpy(dtype=float)
                pga_phi = rupture_template["pga_phi_ln"].to_numpy(dtype=float)
                sa_mu = rupture_template["sa0p4_epi_off_mean_ln_g"].to_numpy(dtype=float)
                sa_tau = rupture_template["sa0p4_tau_ln"].to_numpy(dtype=float)
                sa_phi = rupture_template["sa0p4_phi_ln"].to_numpy(dtype=float)

                pga_between = pga_tau * eta_pga
                pga_within = pga_phi * epsilon_pga
                pga_ln = pga_mu + pga_between + pga_within
                sa_between = sa_tau * eta_sa
                sa_within = sa_phi * epsilon_sa
                sa_ln = sa_mu + sa_between + sa_within

                field = pd.DataFrame(
                    {
                        "catalog_year": int(occurrence.catalog_year),
                        "catalog_event_id": str(occurrence.event_id),
                        "occurrence_ordinal": int(occurrence.occurrence_ordinal),
                        "occurrence_id": occurrence_id,
                        "rupture_ordinal": rupture_template["rupture_ordinal"].to_numpy(dtype=int),
                        "rupture_template_event_id": rupture_template["event_id"].astype(str).to_numpy(),
                        "rupture_id": rupture_template["rupture_id"].astype(str).to_numpy(),
                        "source_type": rupture_template["source_type"].astype(str).to_numpy(),
                        "gmm_name": rupture_template["gmm_name"].astype(str).to_numpy(),
                        "magnitude": rupture_template["magnitude"].to_numpy(dtype=float),
                        "site_ordinal": rupture_template["site_ordinal"].to_numpy(dtype=int),
                        "site_id": rupture_template["site_id"].astype(str).to_numpy(),
                        "site_longitude": rupture_template["site_longitude"].to_numpy(dtype=float),
                        "site_latitude": rupture_template["site_latitude"].to_numpy(dtype=float),
                        "r_rup_km": rupture_template["r_rup_km"].to_numpy(dtype=float),
                        "vs30_mps": rupture_template["vs30_mps"].to_numpy(dtype=float),
                        "period_s": rupture_template["period_s"].to_numpy(dtype=float),
                        "cross_imt_rho": CROSS_IMT_RHO,
                        "eta_pga": eta_pga,
                        "epsilon_pga": epsilon_pga,
                        "pga_epi_off_mean_ln_g": pga_mu,
                        "pga_tau_ln": pga_tau,
                        "pga_phi_ln": pga_phi,
                        "pga_between_event_shift_ln": pga_between,
                        "pga_within_event_shift_ln": pga_within,
                        "pga_simulated_ln_g": pga_ln,
                        "pga_simulated_g": np.exp(pga_ln),
                        "eta_sa0p4": eta_sa,
                        "epsilon_sa0p4": epsilon_sa,
                        "sa0p4_epi_off_mean_ln_g": sa_mu,
                        "sa0p4_tau_ln": sa_tau,
                        "sa0p4_phi_ln": sa_phi,
                        "sa0p4_between_event_shift_ln": sa_between,
                        "sa0p4_within_event_shift_ln": sa_within,
                        "sa0p4_simulated_ln_g": sa_ln,
                        "sa0p4_simulated_g": np.exp(sa_ln),
                    },
                    columns=FIELD_OUTPUT_COLUMNS,
                )
                encoded = field.to_csv(
                    index=False,
                    header=not header_written,
                    na_rep="",
                    lineterminator="\n",
                    float_format="%.17g",
                ).encode("utf-8")
                compressed.write(encoded)
                header_written = True

                event_rows.append(
                    {
                        "catalog_year": int(occurrence.catalog_year),
                        "catalog_event_id": str(occurrence.event_id),
                        "occurrence_ordinal": int(occurrence.occurrence_ordinal),
                        "occurrence_id": occurrence_id,
                        "rupture_ordinal": int(occurrence.rupture_ordinal),
                        "rupture_template_event_id": str(occurrence.rupture_template_event_id),
                        "rupture_id": rupture_id,
                        "source_type": str(occurrence.source_type),
                        "gmm_name": str(occurrence.gmm_name),
                        "event_seed_hex": event_seed_hex,
                        "site_seed_hex": site_seed_hex,
                        "z_event_latent_1": float(z_event_1),
                        "z_event_latent_2": float(z_event_2),
                        "eta_pga": eta_pga,
                        "eta_sa0p4": eta_sa,
                        "cross_imt_rho": CROSS_IMT_RHO,
                    }
                )

            # Preserve a valid header-only CSV for a partition with no events.
            if not header_written:
                empty = pd.DataFrame(columns=FIELD_OUTPUT_COLUMNS)
                compressed.write(empty.to_csv(index=False, lineterminator="\n").encode("utf-8"))

    os.replace(temporary_field_path, field_output)
    event_frame = pd.DataFrame(event_rows, columns=EVENT_OUTPUT_COLUMNS)
    write_gzip_csv_deterministic(event_frame, event_output)

    validation = validate_generated_chunk(record, chunk_occurrences)
    atomic_write_csv(validation, validation_output, lineterminator="\n")
    if not normalize_boolean_series(validation["passes"]).all():
        failed = validation.loc[~normalize_boolean_series(validation["passes"]), ["category", "check", "observed", "expected", "note"]]
        raise RuntimeError(
            f"Cell 19 production chunk {chunk_id} failed validation.\n\n"
            + failed.to_string(index=False)
            + f"\n\nValidation file:\n  {validation_output}"
        )

    marker = {
        "pipeline_version": PIPELINE_VERSION,
        "chunk_id": chunk_id,
        "completed_at_utc": utc_now(),
        "random_specification_sha256": random_specification_hash,
        "catalog_sha256": sha256_file(EVENT_CATALOG_PATH),
        "cell17_gmm_output_sha256": str(record["cell17_gmm_output_sha256"]),
        "catalog_occurrence_count": int(record["catalog_occurrence_count"]),
        "expected_field_rows": int(record["expected_field_rows"]),
        "base_random_seed": BASE_RANDOM_SEED,
        "event_output_size_bytes": event_output.stat().st_size,
        "event_output_sha256": sha256_file(event_output),
        "field_output_size_bytes": field_output.stat().st_size,
        "field_output_sha256": sha256_file(field_output),
        "validation_sha256": sha256_file(validation_output),
    }
    atomic_write_json(marker_path, marker)
    return validation


# ----------------------------------------------------------------------------
# 11. Run or resume all 16 production chunks
# ----------------------------------------------------------------------------

all_chunk_validations: list[pd.DataFrame] = []
for record in manifest_records:
    chunk_id = str(record["chunk_id"])
    print("\n" + "-" * 78)
    print(f"PRODUCTION FIELD CHUNK {int(record['chunk_number']) + 1} OF 16 [ID {chunk_id}]")
    print(f"Catalog occurrences: {int(record['catalog_occurrence_count']):,}")
    print(f"Expected field rows: {int(record['expected_field_rows']):,}")
    reused = marker_is_valid(record)
    validation = generate_production_chunk(record)
    passes = normalize_boolean_series(validation["passes"])
    all_chunk_validations.append(validation.assign(chunk_id=chunk_id))
    record["status"] = "complete_reused" if reused else "complete_new"
    record["completed_at_utc"] = utc_now()
    save_manifest()
    action = "reused" if reused else "passed"
    print(f"Production chunk {chunk_id} {action} all {len(validation):,} checks.")


# ----------------------------------------------------------------------------
# 12. Concatenate deterministic final outputs
# ----------------------------------------------------------------------------

event_chunk_paths = [Path(str(record["event_output"])) for record in manifest_records]
field_chunk_paths = [Path(str(record["field_output"])) for record in manifest_records]
combine_gzip_csv_files(event_chunk_paths, CELL19_FINAL_EVENTS_PATH)
combine_gzip_csv_files(field_chunk_paths, CELL19_FINAL_FIELDS_PATH)


# ----------------------------------------------------------------------------
# 13. Whole-catalog validation
# ----------------------------------------------------------------------------

final_checks: list[dict[str, object]] = []
event_ledger = pd.read_csv(
    CELL19_FINAL_EVENTS_PATH,
    dtype={
        "catalog_event_id": str,
        "occurrence_id": str,
        "rupture_template_event_id": str,
        "rupture_id": str,
        "source_type": str,
        "gmm_name": str,
        "event_seed_hex": str,
        "site_seed_hex": str,
    },
    low_memory=False,
)

add_check(final_checks, "Rows", "Final event-ledger row count", len(event_ledger), EXPECTED_OCCURRENCES, len(event_ledger) == EXPECTED_OCCURRENCES)
add_check(final_checks, "Coverage", "Final event occurrence IDs are unique", int(event_ledger["occurrence_id"].nunique()), EXPECTED_OCCURRENCES, event_ledger["occurrence_id"].nunique() == EXPECTED_OCCURRENCES and not event_ledger["occurrence_id"].duplicated().any())
add_check(final_checks, "Coverage", "Every Notebook 3 occurrence appears in the event ledger", len(set(occurrence_index["occurrence_id"]) - set(event_ledger["occurrence_id"])), 0, set(occurrence_index["occurrence_id"]) == set(event_ledger["occurrence_id"]))
add_check(final_checks, "Coverage", "Catalog years are preserved in the event ledger", int(event_ledger["catalog_year"].nunique()), OBSERVED_EVENT_YEARS, set(event_ledger["catalog_year"].astype(int)) == set(occurrence_index["catalog_year"].astype(int)))
add_check(final_checks, "Random streams", "Event seed labels are unique", int(event_ledger["event_seed_hex"].nunique()), EXPECTED_OCCURRENCES, event_ledger["event_seed_hex"].nunique() == EXPECTED_OCCURRENCES)
add_check(final_checks, "Random streams", "Site seed labels are unique", int(event_ledger["site_seed_hex"].nunique()), EXPECTED_OCCURRENCES, event_ledger["site_seed_hex"].nunique() == EXPECTED_OCCURRENCES)

eta_pga_moments = OnlineMoments()
eta_sa_moments = OnlineMoments()
eta_cross = OnlineBivariate()
eta_pga_moments.update(event_ledger["eta_pga"].to_numpy(dtype=float))
eta_sa_moments.update(event_ledger["eta_sa0p4"].to_numpy(dtype=float))
eta_cross.update(event_ledger["eta_pga"].to_numpy(dtype=float), event_ledger["eta_sa0p4"].to_numpy(dtype=float))

# Reproduce a deterministic subset of occurrence seeds exactly.
seed_sample = event_ledger.sort_values("occurrence_ordinal").iloc[
    np.unique(np.linspace(0, max(len(event_ledger) - 1, 0), min(25, len(event_ledger)), dtype=int))
]
maximum_seed_difference = 0.0
for row in seed_sample.itertuples(index=False):
    event_seed, event_seed_hex = stable_seed(
        BASE_RANDOM_SEED,
        "cell19",
        str(row.occurrence_id),
        str(row.rupture_id),
        "event",
    )
    site_seed, site_seed_hex = stable_seed(
        BASE_RANDOM_SEED,
        "cell19",
        str(row.occurrence_id),
        str(row.rupture_id),
        "site",
    )
    reproduced = rng_from_seed(event_seed).standard_normal(2)
    maximum_seed_difference = max(
        maximum_seed_difference,
        abs(float(reproduced[0]) - float(row.z_event_latent_1)),
        abs(float(reproduced[1]) - float(row.z_event_latent_2)),
    )
    if event_seed_hex != str(row.event_seed_hex) or site_seed_hex != str(row.site_seed_hex):
        maximum_seed_difference = float("inf")
add_check(final_checks, "Random streams", "Identifier-based event streams reproduce exactly", f"{maximum_seed_difference:.3e}", f"<= {EXACT_RECONSTRUCTION_ATOL:.1e}", maximum_seed_difference <= EXACT_RECONSTRUCTION_ATOL)

occurrence_counts: Counter[int] = Counter()
site_counts: Counter[int] = Counter()
field_occurrence_ids: set[str] = set()
field_rows = 0
reconstruction_failures = 0
maximum_pga_error = 0.0
maximum_sa_error = 0.0
minimum_pga_g = float("inf")
maximum_pga_g = 0.0
minimum_sa_g = float("inf")
maximum_sa_g = 0.0
period_failures = 0
rho_failures = 0
positive_failures = 0
epsilon_pga_moments = OnlineMoments()
epsilon_sa_moments = OnlineMoments()
epsilon_cross = OnlineBivariate()

# Twelve sites provide a direct production check that independent site streams
# did not accidentally become reused across sites.
independence_site_ordinals = np.unique(
    np.linspace(0, EXPECTED_SITES - 1, 12, dtype=int)
)
selected_latent = np.full(
    (EXPECTED_OCCURRENCES, len(independence_site_ordinals)),
    np.nan,
    dtype=float,
)
selected_position = {
    int(site_ordinal): index
    for index, site_ordinal in enumerate(independence_site_ordinals)
}

for frame in pd.read_csv(
    CELL19_FINAL_FIELDS_PATH,
    chunksize=FINAL_READ_CHUNK_SIZE,
    dtype={
        "catalog_event_id": str,
        "occurrence_id": str,
        "rupture_template_event_id": str,
        "rupture_id": str,
        "source_type": str,
        "gmm_name": str,
        "site_id": str,
    },
    low_memory=False,
):
    field_rows += len(frame)
    occurrence_ordinals = frame["occurrence_ordinal"].to_numpy(dtype=int)
    site_ordinals = frame["site_ordinal"].to_numpy(dtype=int)
    occurrence_counts.update(map(int, occurrence_ordinals))
    site_counts.update(map(int, site_ordinals))
    field_occurrence_ids.update(frame["occurrence_id"].astype(str).unique())

    pga_reconstructed = (
        frame["pga_epi_off_mean_ln_g"].to_numpy(dtype=float)
        + frame["pga_tau_ln"].to_numpy(dtype=float) * frame["eta_pga"].to_numpy(dtype=float)
        + frame["pga_phi_ln"].to_numpy(dtype=float) * frame["epsilon_pga"].to_numpy(dtype=float)
    )
    sa_reconstructed = (
        frame["sa0p4_epi_off_mean_ln_g"].to_numpy(dtype=float)
        + frame["sa0p4_tau_ln"].to_numpy(dtype=float) * frame["eta_sa0p4"].to_numpy(dtype=float)
        + frame["sa0p4_phi_ln"].to_numpy(dtype=float) * frame["epsilon_sa0p4"].to_numpy(dtype=float)
    )
    pga_error = np.abs(pga_reconstructed - frame["pga_simulated_ln_g"].to_numpy(dtype=float))
    sa_error = np.abs(sa_reconstructed - frame["sa0p4_simulated_ln_g"].to_numpy(dtype=float))
    maximum_pga_error = max(maximum_pga_error, float(np.max(pga_error)))
    maximum_sa_error = max(maximum_sa_error, float(np.max(sa_error)))
    reconstruction_failures += int(((pga_error > EXACT_RECONSTRUCTION_ATOL) | (sa_error > EXACT_RECONSTRUCTION_ATOL)).sum())

    epsilon_pga = frame["epsilon_pga"].to_numpy(dtype=float)
    epsilon_sa = frame["epsilon_sa0p4"].to_numpy(dtype=float)
    epsilon_pga_moments.update(epsilon_pga)
    epsilon_sa_moments.update(epsilon_sa)
    epsilon_cross.update(epsilon_pga, epsilon_sa)

    period_failures += int((~np.isclose(frame["period_s"].to_numpy(dtype=float), SEASIDE_W2_PERIOD_S, rtol=0.0, atol=PERIOD_MATCH_ATOL)).sum())
    rho_failures += int((~np.isclose(frame["cross_imt_rho"].to_numpy(dtype=float), CROSS_IMT_RHO, rtol=0.0, atol=RHO_FORMULA_ATOL)).sum())
    pga_g = frame["pga_simulated_g"].to_numpy(dtype=float)
    sa_g = frame["sa0p4_simulated_g"].to_numpy(dtype=float)
    positive_failures += int(((pga_g <= 0.0) | (sa_g <= 0.0) | (~np.isfinite(pga_g)) | (~np.isfinite(sa_g))).sum())
    minimum_pga_g = min(minimum_pga_g, float(np.min(pga_g)))
    maximum_pga_g = max(maximum_pga_g, float(np.max(pga_g)))
    minimum_sa_g = min(minimum_sa_g, float(np.min(sa_g)))
    maximum_sa_g = max(maximum_sa_g, float(np.max(sa_g)))

    selected_mask = np.isin(site_ordinals, independence_site_ordinals)
    selected_rows = np.flatnonzero(selected_mask)
    for row_index in selected_rows:
        selected_latent[
            int(occurrence_ordinals[row_index]),
            selected_position[int(site_ordinals[row_index])],
        ] = epsilon_pga[row_index]

add_check(final_checks, "Rows", "Final field row count", field_rows, EXPECTED_FIELD_ROWS, field_rows == EXPECTED_FIELD_ROWS)
add_check(final_checks, "Coverage", "Every occurrence has exactly 470 field rows", sum(value == EXPECTED_SITES for value in occurrence_counts.values()), EXPECTED_OCCURRENCES, len(occurrence_counts) == EXPECTED_OCCURRENCES and all(value == EXPECTED_SITES for value in occurrence_counts.values()))
add_check(final_checks, "Coverage", "Every site has every catalog occurrence", sum(value == EXPECTED_OCCURRENCES for value in site_counts.values()), EXPECTED_SITES, len(site_counts) == EXPECTED_SITES and all(value == EXPECTED_OCCURRENCES for value in site_counts.values()))
add_check(final_checks, "Coverage", "Field occurrence IDs match the event ledger", len(set(event_ledger["occurrence_id"]) ^ field_occurrence_ids), 0, set(event_ledger["occurrence_id"]) == field_occurrence_ids)
add_check(final_checks, "Equation", "All field equations reconstruct within tolerance", reconstruction_failures, 0, reconstruction_failures == 0, f"Maximum PGA error {maximum_pga_error:.3e}; maximum SA0P4 error {maximum_sa_error:.3e}.")
add_check(final_checks, "Inputs", "All rows use exact period 0.40 s", period_failures, 0, period_failures == 0)
add_check(final_checks, "Inputs", "All rows use the accepted cross-IMT rho", rho_failures, 0, rho_failures == 0)
add_check(final_checks, "Numeric", "All simulated intensities are finite and positive", positive_failures, 0, positive_failures == 0)

# Production stochastic diagnostics. The bounds adapt to catalog size and are
# validation thresholds rather than model parameters.
if EXPECTED_OCCURRENCES >= 64:
    eta_mean_bound = max(0.08, 4.0 / math.sqrt(EXPECTED_OCCURRENCES))
    eta_std_bound = max(0.08, 4.0 / math.sqrt(2.0 * (EXPECTED_OCCURRENCES - 1)))
    event_rho_bound = max(0.05, 4.0 / math.sqrt(EXPECTED_OCCURRENCES))
    add_check(final_checks, "Distribution", "PGA between-event residual mean is near zero", f"{eta_pga_moments.mean:.6f}", f"abs <= {eta_mean_bound:.6f}", abs(eta_pga_moments.mean) <= eta_mean_bound)
    add_check(final_checks, "Distribution", "SA0P4 between-event residual mean is near zero", f"{eta_sa_moments.mean:.6f}", f"abs <= {eta_mean_bound:.6f}", abs(eta_sa_moments.mean) <= eta_mean_bound)
    add_check(final_checks, "Distribution", "PGA between-event residual standard deviation is near one", f"{eta_pga_moments.std:.6f}", f"within {eta_std_bound:.6f} of 1", abs(eta_pga_moments.std - 1.0) <= eta_std_bound)
    add_check(final_checks, "Distribution", "SA0P4 between-event residual standard deviation is near one", f"{eta_sa_moments.std:.6f}", f"within {eta_std_bound:.6f} of 1", abs(eta_sa_moments.std - 1.0) <= eta_std_bound)
    add_check(final_checks, "Cross-IMT", "Event residual PGA-SA0P4 correlation matches target", f"{eta_cross.correlation:.6f}", f"{CROSS_IMT_RHO:.6f} ± {event_rho_bound:.6f}", abs(eta_cross.correlation - CROSS_IMT_RHO) <= event_rho_bound)
else:
    add_check(final_checks, "Distribution", "Catalog has enough occurrences for production moment checks", EXPECTED_OCCURRENCES, ">= 64", True, "Moment checks were treated as not applicable because the catalog is small.")

# Site residual checks use millions of rows and therefore have tight bounds.
epsilon_count = epsilon_pga_moments.count
epsilon_mean_bound = max(0.005, 4.0 / math.sqrt(max(epsilon_count, 1)))
epsilon_std_bound = max(0.005, 4.0 / math.sqrt(max(2.0 * (epsilon_count - 1), 1.0)))
add_check(final_checks, "Distribution", "PGA within-event residual mean is near zero", f"{epsilon_pga_moments.mean:.6f}", f"abs <= {epsilon_mean_bound:.6f}", abs(epsilon_pga_moments.mean) <= epsilon_mean_bound)
add_check(final_checks, "Distribution", "SA0P4 within-event residual mean is near zero", f"{epsilon_sa_moments.mean:.6f}", f"abs <= {epsilon_mean_bound:.6f}", abs(epsilon_sa_moments.mean) <= epsilon_mean_bound)
add_check(final_checks, "Distribution", "PGA within-event residual standard deviation is near one", f"{epsilon_pga_moments.std:.6f}", f"within {epsilon_std_bound:.6f} of 1", abs(epsilon_pga_moments.std - 1.0) <= epsilon_std_bound)
add_check(final_checks, "Distribution", "SA0P4 within-event residual standard deviation is near one", f"{epsilon_sa_moments.std:.6f}", f"within {epsilon_std_bound:.6f} of 1", abs(epsilon_sa_moments.std - 1.0) <= epsilon_std_bound)
site_rho_bound = max(0.005, 4.0 / math.sqrt(max(epsilon_count, 1)))
add_check(final_checks, "Cross-IMT", "Site residual PGA-SA0P4 correlation matches target", f"{epsilon_cross.correlation:.6f}", f"{CROSS_IMT_RHO:.6f} ± {site_rho_bound:.6f}", abs(epsilon_cross.correlation - CROSS_IMT_RHO) <= site_rho_bound)

if np.isnan(selected_latent).any():
    maximum_abs_cross_site_correlation = float("inf")
else:
    correlation_matrix = np.corrcoef(selected_latent, rowvar=False)
    off_diagonal = correlation_matrix[~np.eye(correlation_matrix.shape[0], dtype=bool)]
    maximum_abs_cross_site_correlation = float(np.max(np.abs(off_diagonal)))
independence_bound = max(0.10, 4.0 / math.sqrt(max(EXPECTED_OCCURRENCES, 1)))
add_check(final_checks, "Spatial independence", "Selected sites have no material residual correlation", f"{maximum_abs_cross_site_correlation:.6f}", f"<= {independence_bound:.6f}", maximum_abs_cross_site_correlation <= independence_bound, "This checks accidental stream reuse; the Phase 1 model intentionally applies no spatial correlation.")

chunk_validation_frame = pd.concat(all_chunk_validations, ignore_index=True)
add_check(final_checks, "Chunk validation", "All chunk-level checks passed", int(normalize_boolean_series(chunk_validation_frame["passes"]).sum()), len(chunk_validation_frame), normalize_boolean_series(chunk_validation_frame["passes"]).all())

validation = build_validation_frame(final_checks)
atomic_write_csv(validation, CELL19_VALIDATION_PATH, lineterminator="\n")
if not normalize_boolean_series(validation["passes"]).all():
    failed = validation.loc[~normalize_boolean_series(validation["passes"]), ["category", "check", "observed", "expected", "note"]]
    raise RuntimeError(
        "Cell 19 full annual-catalog field validation failed.\n\n"
        + failed.to_string(index=False)
        + f"\n\nValidation file:\n  {CELL19_VALIDATION_PATH}"
    )


# ----------------------------------------------------------------------------
# 14. Reproducibility summary
# ----------------------------------------------------------------------------

source_counts = occurrence_index.groupby("source_type").size().to_dict()
year_event_counts = occurrence_index.groupby("catalog_year").size()
summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": utc_now(),
    "all_checks_passed": True,
    "portfolio": "Seaside W2 commercial portfolio",
    "primary_baseline_branch": PRIMARY_BASELINE_BRANCH,
    "period_s": SEASIDE_W2_PERIOD_S,
    "period_resolution": "exact USGS SA0P4; no interpolation",
    "spatial_residual_model": "conditionally independent across sites",
    "cross_imt_rho": CROSS_IMT_RHO,
    "catalog": {
        "path": str(EVENT_CATALOG_PATH),
        "sha256": sha256_file(EVENT_CATALOG_PATH),
        "year_column": YEAR_COLUMN,
        "occurrences": EXPECTED_OCCURRENCES,
        "observed_event_years": OBSERVED_EVENT_YEARS,
        "minimum_event_year": MINIMUM_EVENT_YEAR,
        "maximum_event_year": MAXIMUM_EVENT_YEAR,
        "declared_catalog_years": DECLARED_CATALOG_YEARS,
        "declared_catalog_years_provenance": DECLARED_CATALOG_YEARS_PROVENANCE,
        "zero_event_year_note": (
            "The occurrence and field files contain rows only for actual event "
            "occurrences. Later annual loss aggregation must restore every "
            "zero-event year using the Notebook 3 declared catalog duration."
        ),
        "minimum_events_in_observed_event_year": int(year_event_counts.min()),
        "maximum_events_in_observed_event_year": int(year_event_counts.max()),
        "source_occurrence_counts": {str(key): int(value) for key, value in source_counts.items()},
    },
    "sites": EXPECTED_SITES,
    "field_rows": EXPECTED_FIELD_ROWS,
    "random_streams": {
        "base_random_seed": BASE_RANDOM_SEED,
        "generator": "numpy.random.PCG64DXSM",
        "specification_path": str(CELL19_RANDOM_SPEC_PATH),
        "specification_sha256": sha256_file(CELL19_RANDOM_SPEC_PATH),
        "future_spatial_reuse": (
            "Regenerate site latent vectors from each occurrence's "
            "site_seed_hex and the frozen Cell 18 site order."
        ),
    },
    "diagnostics": {
        "eta_pga_mean": eta_pga_moments.mean,
        "eta_pga_std": eta_pga_moments.std,
        "eta_sa0p4_mean": eta_sa_moments.mean,
        "eta_sa0p4_std": eta_sa_moments.std,
        "event_cross_imt_rho": eta_cross.correlation,
        "epsilon_pga_mean": epsilon_pga_moments.mean,
        "epsilon_pga_std": epsilon_pga_moments.std,
        "epsilon_sa0p4_mean": epsilon_sa_moments.mean,
        "epsilon_sa0p4_std": epsilon_sa_moments.std,
        "site_cross_imt_rho": epsilon_cross.correlation,
        "maximum_abs_selected_cross_site_rho": maximum_abs_cross_site_correlation,
        "maximum_pga_equation_error": maximum_pga_error,
        "maximum_sa0p4_equation_error": maximum_sa_error,
        "maximum_seed_reproduction_difference": maximum_seed_difference,
    },
    "numeric_ranges": {
        "pga_simulated_g": {"minimum": minimum_pga_g, "maximum": maximum_pga_g},
        "sa0p4_simulated_g": {"minimum": minimum_sa_g, "maximum": maximum_sa_g},
    },
    "source_files": {
        "cell17_output_path": str(CELL17_FINAL_OUTPUT_INPUT_PATH),
        "cell17_output_sha256": actual_cell17_hash,
        "cell17_summary_sha256": sha256_file(CELL17_SUMMARY_INPUT_PATH),
        "cell17_validation_sha256": sha256_file(CELL17_VALIDATION_INPUT_PATH),
        "cell17_manifest_sha256": sha256_file(CELL17_MANIFEST_INPUT_PATH),
        "cell18_summary_sha256": sha256_file(CELL18_SUMMARY_INPUT_PATH),
        "cell18_validation_sha256": sha256_file(CELL18_VALIDATION_INPUT_PATH),
        "cell18_random_specification_sha256": sha256_file(CELL18_RANDOM_SPEC_INPUT_PATH),
        "cell18_site_order_sha256": sha256_file(CELL18_SITE_ORDER_INPUT_PATH),
    },
    "outputs": {
        "occurrence_index_path": str(CELL19_OCCURRENCE_INDEX_PATH),
        "occurrence_index_sha256": sha256_file(CELL19_OCCURRENCE_INDEX_PATH),
        "full_fields_path": str(CELL19_FINAL_FIELDS_PATH),
        "full_fields_size_bytes": CELL19_FINAL_FIELDS_PATH.stat().st_size,
        "full_fields_sha256": sha256_file(CELL19_FINAL_FIELDS_PATH),
        "event_residual_ledger_path": str(CELL19_FINAL_EVENTS_PATH),
        "event_residual_ledger_size_bytes": CELL19_FINAL_EVENTS_PATH.stat().st_size,
        "event_residual_ledger_sha256": sha256_file(CELL19_FINAL_EVENTS_PATH),
        "chunk_manifest_path": str(CELL19_MANIFEST_PATH),
        "chunk_manifest_sha256": sha256_file(CELL19_MANIFEST_PATH),
        "validation_path": str(CELL19_VALIDATION_PATH),
        "validation_sha256": sha256_file(CELL19_VALIDATION_PATH),
    },
    "next_step": (
        "Finalize Notebook 4 with compact event/site field diagnostics and a "
        "downstream handoff table, then use SA0P4 and PGA in Notebook 5 for "
        "damage and ground-up loss calculations."
    ),
}
atomic_write_json(CELL19_SUMMARY_PATH, summary)

print("\n" + "=" * 78)
print("CELL 19 FULL ANNUAL-CATALOG BASELINE FIELD SIMULATION COMPLETE")
print("=" * 78)
print(f"Catalog occurrences: {EXPECTED_OCCURRENCES:,}")
print(f"Production field rows: {EXPECTED_FIELD_ROWS:,}")
print(f"Final validation checks: {len(validation):,}")
print(f"Event PGA-SA0P4 rho: {eta_cross.correlation:.6f}")
print(f"Site PGA-SA0P4 rho:  {epsilon_cross.correlation:.6f}")
print(f"Maximum selected cross-site rho: {maximum_abs_cross_site_correlation:.6f}")
print(f"Maximum equation error: {max(maximum_pga_error, maximum_sa_error):.3e}")
print(f"\nFull baseline fields:\n  {CELL19_FINAL_FIELDS_PATH}")
print(f"\nEvent residual ledger:\n  {CELL19_FINAL_EVENTS_PATH}")
print(f"\nOccurrence index:\n  {CELL19_OCCURRENCE_INDEX_PATH}")
print(f"\nValidation:\n  {CELL19_VALIDATION_PATH}")
print(f"\nSummary:\n  {CELL19_SUMMARY_PATH}")
print(
    "\nNext: create the final Notebook 4 handoff/diagnostic cell, then begin "
    "Notebook 5 damage and ground-up loss calculations."
)


Cell 17 and Cell 18 prerequisites passed: validated epi-off parameters and deterministic random-stream design.
Cross-IMT residual correlation: Baker-Jayaram rho(PGA, SA0P4) = 0.732140990025
FULL ANNUAL-CATALOG BASELINE STOCHASTIC GROUND-MOTION FIELDS
Catalog occurrences:     10,630
Observed event years:    10,593
Catalog year column:     simulation_year
Event-year range:        185 to 1999745
Declared catalog years:  not recovered; preserve Notebook 3 metadata
Seaside W2 sites:        470
Production field rows:   4,996,100
Primary branch:          epi-off
Cross-IMT rho:           0.732140990025
Spatial residual model:  independent across sites

------------------------------------------------------------------------------
PRODUCTION FIELD CHUNK 1 OF 16 [ID 0000]
Catalog occurrences: 5,412
Expected field rows: 2,543,640
Production chunk 0000 passed all 19 checks.

------------------------------------------------------------------------------
PRODUCTION FIELD CHUNK 2 OF 16 [ID 0001]
Cata

In [37]:



from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path

import gzip
import hashlib
import json
import math
import os
import re
from typing import Any, Iterable

import numpy as np
import pandas as pd
from IPython.display import display


# ----------------------------------------------------------------------------
# 1. Configuration
# ----------------------------------------------------------------------------

PIPELINE_VERSION = "notebook4_cell20_final_handoff_v1"
EXPECTED_RUPTURES = 3_927
EXPECTED_SITES = 470
EXPECTED_DISTANCE_ROWS = EXPECTED_RUPTURES * EXPECTED_SITES
EXPECTED_PERIOD_S = 0.40
EXPECTED_PRIMARY_BRANCH = "epi-off"
EXPECTED_SPATIAL_MODEL = "conditionally independent across sites"

# Leave as None when the duration can be recovered from Notebook 3 metadata or
# code. If recovery fails, replace None with the exact authoritative duration,
# for example 2_000_000 only if that is what Notebook 3 actually used.
CATALOG_DURATION_YEARS_OVERRIDE: int | None = None

# Optional environment override. This is useful for reproducible noninteractive
# runs, for example NOTEBOOK4_CATALOG_DURATION_YEARS=2000000.
ENV_DURATION_NAME = "NOTEBOOK4_CATALOG_DURATION_YEARS"
VERIFY_LARGE_FILE_HASHES = True

required_variables = ["DATA_DIR", "METADATA_DIR", "EVENT_CATALOG_PATH"]
missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables:
    raise RuntimeError(
        "Cell 20 requires variables created by earlier Notebook 4 cells:\n"
        + "\n".join(f"  - {name}" for name in missing_variables)
    )

DATA_DIR = Path(DATA_DIR).resolve()
METADATA_DIR = Path(METADATA_DIR).resolve()
EVENT_CATALOG_PATH = Path(EVENT_CATALOG_PATH).resolve()
PROJECT_ROOT = Path(globals().get("PROJECT_ROOT", DATA_DIR.parent)).resolve()

if not EVENT_CATALOG_PATH.is_file():
    raise FileNotFoundError(f"Notebook 3 event catalog not found:\n  {EVENT_CATALOG_PATH}")


# ----------------------------------------------------------------------------
# 2. Accepted Notebook 4 paths
# ----------------------------------------------------------------------------

CELL15_SUMMARY_PATH = Path(
    globals().get(
        "FINAL_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_authoritative_distance_summary.json",
    )
).resolve()
CELL15_VALIDATION_PATH = Path(
    globals().get(
        "FINAL_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_authoritative_distance_validation.csv",
    )
).resolve()

CELL16_SUMMARY_PATH = Path(
    globals().get(
        "SUMMARY_PATH",
        METADATA_DIR / "notebook_4_cell_16_controlled_gmm_summary.json",
    )
).resolve()
CELL16_VALIDATION_PATH = Path(
    globals().get(
        "VALIDATION_PATH",
        METADATA_DIR / "notebook_4_cell_16_controlled_gmm_validation.csv",
    )
).resolve()

CELL17_SUMMARY_PATH = Path(
    globals().get(
        "CELL17_FINAL_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_summary.json",
    )
).resolve()
CELL17_VALIDATION_PATH = Path(
    globals().get(
        "CELL17_FINAL_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_bulk_parker_gmm_validation.csv",
    )
).resolve()

CELL18_SUMMARY_PATH = Path(
    globals().get(
        "CELL18_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_cell_18_controlled_field_summary.json",
    )
).resolve()
CELL18_VALIDATION_PATH = Path(
    globals().get(
        "CELL18_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_cell_18_controlled_field_validation.csv",
    )
).resolve()

CELL19_SUMMARY_PATH = Path(
    globals().get(
        "CELL19_SUMMARY_PATH",
        METADATA_DIR / "notebook_4_cell_19_full_field_summary.json",
    )
).resolve()
CELL19_VALIDATION_PATH = Path(
    globals().get(
        "CELL19_VALIDATION_PATH",
        METADATA_DIR / "notebook_4_cell_19_full_field_validation.csv",
    )
).resolve()
CELL19_OCCURRENCE_INDEX_PATH = Path(
    globals().get(
        "CELL19_OCCURRENCE_INDEX_PATH",
        METADATA_DIR / "notebook_4_cell_19_event_occurrence_index.csv.gz",
    )
).resolve()

required_control_files = {
    "Cell 15 summary": CELL15_SUMMARY_PATH,
    "Cell 15 validation": CELL15_VALIDATION_PATH,
    "Cell 16 summary": CELL16_SUMMARY_PATH,
    "Cell 16 validation": CELL16_VALIDATION_PATH,
    "Cell 17 summary": CELL17_SUMMARY_PATH,
    "Cell 17 validation": CELL17_VALIDATION_PATH,
    "Cell 18 summary": CELL18_SUMMARY_PATH,
    "Cell 18 validation": CELL18_VALIDATION_PATH,
    "Cell 19 summary": CELL19_SUMMARY_PATH,
    "Cell 19 validation": CELL19_VALIDATION_PATH,
    "Cell 19 occurrence index": CELL19_OCCURRENCE_INDEX_PATH,
}
missing_control_files = [
    f"{label}: {path}"
    for label, path in required_control_files.items()
    if not path.is_file()
]
if missing_control_files:
    raise FileNotFoundError(
        "Cell 20 is missing required accepted outputs:\n"
        + "\n".join(f"  - {item}" for item in missing_control_files)
    )

CELL20_OUTPUT_DIR = METADATA_DIR / "notebook_4_final_handoff"
CELL20_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CELL20_VALIDATION_PATH = METADATA_DIR / "notebook_4_final_validation.csv"
CELL20_ARTIFACT_INVENTORY_PATH = (
    CELL20_OUTPUT_DIR / "notebook_4_artifact_inventory.csv"
)
CELL20_OCCUPIED_YEAR_COUNTS_PATH = (
    CELL20_OUTPUT_DIR / "notebook_4_occupied_year_event_counts.csv.gz"
)
CELL20_NOTEBOOK5_HANDOFF_PATH = (
    CELL20_OUTPUT_DIR / "notebook_5_input_handoff.json"
)
CELL20_FINAL_SUMMARY_PATH = METADATA_DIR / "notebook_4_final_summary.json"


# ----------------------------------------------------------------------------
# 3. Utility functions
# ----------------------------------------------------------------------------

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        while True:
            block = file_handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def atomic_write_json(path: Path, payload: Any) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def atomic_write_csv(frame: pd.DataFrame, path: Path) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, lineterminator="\n")
    os.replace(temporary, path)


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    import io

    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw_handle:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_handle,
            compresslevel=6,
            mtime=0,
        ) as gzip_handle:
            with io.TextIOWrapper(gzip_handle, encoding="utf-8", newline="") as text_handle:
                frame.to_csv(text_handle, index=False, lineterminator="\n")
    os.replace(temporary, path)


def load_json(path: Path) -> dict[str, Any]:
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Invalid JSON file: {path}") from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f"Expected a JSON object in: {path}")
    return payload


def normalize_boolean_series(values: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    return (
        values.astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y", "pass", "passed"})
    )


def validation_pass_column(frame: pd.DataFrame, label: str) -> str:
    candidates = ["passes", "passed", "pass", "success", "is_valid"]
    for column in candidates:
        if column in frame.columns:
            return column
    raise RuntimeError(
        f"Could not identify the pass/fail column in {label}. "
        f"Available columns: {list(frame.columns)}"
    )


def nested_get(payload: dict[str, Any], keys: Iterable[str], default: Any = None) -> Any:
    value: Any = payload
    for key in keys:
        if not isinstance(value, dict) or key not in value:
            return default
        value = value[key]
    return value


def add_check(
    checks: list[dict[str, Any]],
    category: str,
    check: str,
    observed: Any,
    expected: Any,
    passes: bool,
    note: str = "",
) -> None:
    checks.append(
        {
            "category": category,
            "check": check,
            "observed": observed,
            "expected": expected,
            "passes": bool(passes),
            "note": note,
        }
    )


def parse_positive_integer(value: Any) -> int | None:
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, np.integer)):
        parsed = int(value)
        return parsed if parsed > 0 else None
    if isinstance(value, (float, np.floating)):
        if math.isfinite(float(value)) and float(value).is_integer() and value > 0:
            return int(value)
        return None
    text = str(value).strip().replace(",", "").replace("_", "")
    if re.fullmatch(r"\d+", text):
        parsed = int(text)
        return parsed if parsed > 0 else None
    return None


# ----------------------------------------------------------------------------
# 4. Load accepted summaries and validations
# ----------------------------------------------------------------------------

summaries = {
    "cell15": load_json(CELL15_SUMMARY_PATH),
    "cell16": load_json(CELL16_SUMMARY_PATH),
    "cell17": load_json(CELL17_SUMMARY_PATH),
    "cell18": load_json(CELL18_SUMMARY_PATH),
    "cell19": load_json(CELL19_SUMMARY_PATH),
}

validation_paths = {
    "cell15": CELL15_VALIDATION_PATH,
    "cell16": CELL16_VALIDATION_PATH,
    "cell17": CELL17_VALIDATION_PATH,
    "cell18": CELL18_VALIDATION_PATH,
    "cell19": CELL19_VALIDATION_PATH,
}
validation_frames: dict[str, pd.DataFrame] = {}

for cell_name, path in validation_paths.items():
    frame = pd.read_csv(path, low_memory=False)
    if frame.empty:
        raise RuntimeError(f"{cell_name.upper()} validation file is empty: {path}")
    pass_column = validation_pass_column(frame, cell_name.upper())
    frame = frame.copy()
    frame["_normalized_pass"] = normalize_boolean_series(frame[pass_column])
    validation_frames[cell_name] = frame


# ----------------------------------------------------------------------------
# 5. Recover the authoritative catalog duration
# ----------------------------------------------------------------------------

DURATION_KEYS = {
    "catalog_years",
    "n_catalog_years",
    "simulation_years",
    "n_simulation_years",
    "catalog_duration_years",
    "number_of_catalog_years",
    "number_of_simulation_years",
    "years_simulated",
}
DURATION_SYMBOLS = {
    "CATALOG_YEARS",
    "N_CATALOG_YEARS",
    "SIMULATION_YEARS",
    "N_SIMULATION_YEARS",
    "CATALOG_DURATION_YEARS",
    "NUMBER_OF_CATALOG_YEARS",
    "NUMBER_OF_SIMULATION_YEARS",
    "YEARS_SIMULATED",
}


def recursively_collect_duration_candidates(
    value: Any,
    provenance: str,
    output: list[tuple[int, str]],
) -> None:
    if isinstance(value, dict):
        for key, child in value.items():
            normalized_key = str(key).strip().lower()
            if normalized_key in DURATION_KEYS:
                parsed = parse_positive_integer(child)
                if parsed is not None:
                    output.append((parsed, f"{provenance} -> {key}"))
            recursively_collect_duration_candidates(child, provenance, output)
    elif isinstance(value, list):
        for child in value:
            recursively_collect_duration_candidates(child, provenance, output)


def duration_candidates_from_text(text: str, provenance: str) -> list[tuple[int, str]]:
    candidates: list[tuple[int, str]] = []
    symbol_pattern = "|".join(sorted(map(re.escape, DURATION_SYMBOLS), key=len, reverse=True))
    key_pattern = "|".join(sorted(map(re.escape, DURATION_KEYS), key=len, reverse=True))
    patterns = [
        re.compile(
            rf"(?im)\b(?P<key>{symbol_pattern})\b\s*(?::|=)\s*(?P<value>\d[\d_,]*)"
        ),
        re.compile(
            rf"(?im)[\"'](?P<key>{key_pattern})[\"']\s*:\s*(?P<value>\d[\d_,]*)"
        ),
    ]
    for pattern in patterns:
        for match in pattern.finditer(text):
            parsed = parse_positive_integer(match.group("value"))
            if parsed is not None:
                candidates.append(
                    (parsed, f"{provenance} -> {match.group('key')}")
                )
    return candidates


def duration_candidates_from_file(path: Path) -> list[tuple[int, str]]:
    suffix = path.suffix.lower()
    candidates: list[tuple[int, str]] = []
    try:
        if suffix == ".json":
            payload = json.loads(path.read_text(encoding="utf-8"))
            recursively_collect_duration_candidates(payload, str(path), candidates)
            return candidates
        if suffix == ".ipynb":
            payload = json.loads(path.read_text(encoding="utf-8"))
            cells = payload.get("cells", []) if isinstance(payload, dict) else []
            code_text = "\n".join(
                "".join(cell.get("source", []))
                for cell in cells
                if isinstance(cell, dict) and cell.get("cell_type") == "code"
            )
            return duration_candidates_from_text(code_text, str(path))
        if suffix == ".csv":
            frame = pd.read_csv(path, low_memory=False)
            for column in frame.columns:
                normalized = str(column).strip().lower()
                if normalized in DURATION_KEYS:
                    for value in frame[column].dropna().tolist():
                        parsed = parse_positive_integer(value)
                        if parsed is not None:
                            candidates.append((parsed, f"{path} -> column {column}"))
            if len(frame.columns) >= 2:
                key_column = frame.columns[0]
                value_column = frame.columns[1]
                for row in frame[[key_column, value_column]].itertuples(index=False, name=None):
                    if str(row[0]).strip().lower() in DURATION_KEYS:
                        parsed = parse_positive_integer(row[1])
                        if parsed is not None:
                            candidates.append((parsed, f"{path} -> row {row[0]}"))
            return candidates
        if suffix in {".py", ".txt", ".md", ".yaml", ".yml", ".toml"}:
            return duration_candidates_from_text(
                path.read_text(encoding="utf-8", errors="ignore"), str(path)
            )
    except (OSError, UnicodeError, json.JSONDecodeError, pd.errors.ParserError):
        return []
    return []


def resolve_catalog_duration() -> tuple[int, str, list[tuple[int, str]]]:
    candidates: list[tuple[int, str]] = []

    # Highest-precedence explicit override in this cell.
    override = parse_positive_integer(CATALOG_DURATION_YEARS_OVERRIDE)
    if CATALOG_DURATION_YEARS_OVERRIDE is not None and override is None:
        raise ValueError("CATALOG_DURATION_YEARS_OVERRIDE must be a positive integer.")
    if override is not None:
        return override, "Cell 20 explicit override", [(override, "Cell 20 explicit override")]

    # Environment override.
    if ENV_DURATION_NAME in os.environ:
        env_value = parse_positive_integer(os.environ[ENV_DURATION_NAME])
        if env_value is None:
            raise ValueError(f"Environment variable {ENV_DURATION_NAME} is not a positive integer.")
        return env_value, f"environment variable: {ENV_DURATION_NAME}", [
            (env_value, f"environment variable: {ENV_DURATION_NAME}")
        ]

    # Notebook globals created by Notebook 3 or earlier cells.
    for symbol in sorted(DURATION_SYMBOLS):
        if symbol in globals():
            parsed = parse_positive_integer(globals()[symbol])
            if parsed is not None:
                candidates.append((parsed, f"notebook global: {symbol}"))

    # Cell 19 may have recovered it even if the current kernel no longer has
    # the original Notebook 3 variable.
    cell19_declared = nested_get(
        summaries["cell19"], ["catalog", "declared_catalog_years"]
    )
    parsed_cell19 = parse_positive_integer(cell19_declared)
    if parsed_cell19 is not None:
        candidates.append(
            (
                parsed_cell19,
                "Cell 19 summary: catalog.declared_catalog_years",
            )
        )

    # Search narrowly targeted metadata and Notebook 3 source files.
    search_paths: set[Path] = set()
    search_directories = {
        EVENT_CATALOG_PATH.parent,
        METADATA_DIR,
        PROJECT_ROOT / "notebooks",
        PROJECT_ROOT,
    }
    filename_patterns = [
        "*catalog*metadata*.json",
        "*catalog*summary*.json",
        "*event*catalog*.json",
        "*catalog*.csv",
        "03_generate_annual_event_catalog.ipynb",
        "*annual*event*catalog*.ipynb",
        "*annual*event*catalog*.py",
    ]
    for directory in search_directories:
        if not directory.is_dir():
            continue
        for pattern in filename_patterns:
            # Use glob first; only use rglob for notebook/source patterns where
            # the project tree may contain a notebooks subdirectory.
            search_paths.update(directory.glob(pattern))
            if directory == PROJECT_ROOT and pattern.endswith((".ipynb", ".py")):
                search_paths.update(directory.rglob(pattern))

    # Avoid scanning Cell 19 and Cell 20 source as duration authority.
    for path in sorted(search_paths):
        if not path.is_file():
            continue
        lower_name = path.name.lower()
        if "notebook_4_cell_19" in lower_name or "notebook_4_cell_20" in lower_name:
            continue
        candidates.extend(duration_candidates_from_file(path))

    # Remove exact duplicate value/provenance pairs.
    candidates = list(dict.fromkeys(candidates))
    unique_values = sorted({value for value, _ in candidates})
    if not unique_values:
        raise RuntimeError(
            "Cell 20 could not recover the authoritative Notebook 3 catalog "
            "duration. The ground-motion fields remain valid, but Notebook 5 "
            "cannot calculate AAL, AEP, OEP, or PML correctly without the full "
            "number of simulated years, including zero-event years.\n\n"
            "Set CATALOG_DURATION_YEARS_OVERRIDE at the top of Cell 20 to the "
            "exact value used in Notebook 3, then rerun Cell 20. Do not use the "
            "largest occupied simulation_year as the duration."
        )
    if len(unique_values) > 1:
        candidate_text = "\n".join(
            f"  - {value:,}: {provenance}" for value, provenance in candidates
        )
        raise RuntimeError(
            "Conflicting catalog-duration values were found. Resolve the "
            "Notebook 3 metadata conflict or set the explicit Cell 20 override.\n\n"
            + candidate_text
        )

    duration = unique_values[0]
    matching_provenance = [
        provenance for value, provenance in candidates if value == duration
    ]
    return duration, matching_provenance[0], candidates


CATALOG_DURATION_YEARS, CATALOG_DURATION_PROVENANCE, DURATION_CANDIDATES = (
    resolve_catalog_duration()
)


# ----------------------------------------------------------------------------
# 6. Read the sparse occurrence index and build occupied-year statistics
# ----------------------------------------------------------------------------

occurrence_index = pd.read_csv(
    CELL19_OCCURRENCE_INDEX_PATH,
    compression="gzip",
    low_memory=False,
    dtype={
        "occurrence_id": str,
        "event_id": str,
        "rupture_id": str,
        "source_type": str,
    },
)

required_occurrence_columns = {
    "occurrence_ordinal",
    "occurrence_id",
    "catalog_year",
    "event_id",
    "rupture_id",
    "source_type",
}
missing_occurrence_columns = sorted(
    required_occurrence_columns.difference(occurrence_index.columns)
)
if missing_occurrence_columns:
    raise RuntimeError(
        "Cell 19 occurrence index is missing required columns:\n"
        + "\n".join(f"  - {column}" for column in missing_occurrence_columns)
    )

catalog_year_numeric = pd.to_numeric(
    occurrence_index["catalog_year"], errors="coerce"
)
if (
    not np.isfinite(catalog_year_numeric).all()
    or not np.isclose(
        catalog_year_numeric, np.round(catalog_year_numeric), atol=1.0e-10
    ).all()
):
    raise RuntimeError("Cell 19 catalog_year values are not finite integers.")
occurrence_index["catalog_year"] = np.round(catalog_year_numeric).astype(np.int64)
occurrence_index["source_type"] = (
    occurrence_index["source_type"].astype(str).str.strip().str.upper()
)

if occurrence_index["occurrence_id"].duplicated().any():
    raise RuntimeError("Cell 19 occurrence IDs are not unique.")
if occurrence_index[["catalog_year", "event_id"]].duplicated().any():
    raise RuntimeError("Cell 19 (catalog_year, event_id) pairs are not unique.")
if not set(occurrence_index["source_type"].unique()).issubset(
    {"INTERFACE", "SLAB"}
):
    raise RuntimeError(
        "Unexpected source types in Cell 19 occurrence index: "
        f"{sorted(occurrence_index['source_type'].unique().tolist())}"
    )

source_year_counts = (
    occurrence_index.assign(_count=1)
    .pivot_table(
        index="catalog_year",
        columns="source_type",
        values="_count",
        aggfunc="sum",
        fill_value=0,
    )
    .rename_axis(columns=None)
)
for source_type in ["INTERFACE", "SLAB"]:
    if source_type not in source_year_counts.columns:
        source_year_counts[source_type] = 0

occupied_year_counts = (
    occurrence_index.groupby("catalog_year", as_index=True)
    .size()
    .rename("event_count")
    .to_frame()
    .join(source_year_counts[["INTERFACE", "SLAB"]], how="left")
    .rename(
        columns={
            "INTERFACE": "interface_event_count",
            "SLAB": "slab_event_count",
        }
    )
    .reset_index()
    .sort_values("catalog_year", kind="mergesort")
    .reset_index(drop=True)
)
occupied_year_counts["has_multiple_events"] = (
    occupied_year_counts["event_count"] > 1
)
occupied_year_counts["catalog_year"] = occupied_year_counts["catalog_year"].astype(
    np.int64
)
for column in ["event_count", "interface_event_count", "slab_event_count"]:
    occupied_year_counts[column] = occupied_year_counts[column].astype(np.int64)

CATALOG_OCCURRENCES = int(len(occurrence_index))
OCCUPIED_YEARS = int(len(occupied_year_counts))
ZERO_EVENT_YEARS = int(CATALOG_DURATION_YEARS - OCCUPIED_YEARS)
MULTIPLE_EVENT_YEARS = int(occupied_year_counts["has_multiple_events"].sum())
MAX_EVENTS_IN_YEAR = int(occupied_year_counts["event_count"].max())
MIN_OCCUPIED_YEAR = int(occupied_year_counts["catalog_year"].min())
MAX_OCCUPIED_YEAR = int(occupied_year_counts["catalog_year"].max())
ANNUAL_EVENT_RATE = CATALOG_OCCURRENCES / CATALOG_DURATION_YEARS
MEAN_INTEREVENT_YEARS = (
    1.0 / ANNUAL_EVENT_RATE if ANNUAL_EVENT_RATE > 0 else math.inf
)

if CATALOG_DURATION_YEARS < OCCUPIED_YEARS:
    raise RuntimeError(
        "Declared catalog duration is smaller than the number of occupied years."
    )
if MAX_OCCUPIED_YEAR > CATALOG_DURATION_YEARS:
    raise RuntimeError(
        "Largest occupied catalog year exceeds the declared catalog duration: "
        f"{MAX_OCCUPIED_YEAR:,} > {CATALOG_DURATION_YEARS:,}."
    )

write_gzip_csv_deterministic(
    occupied_year_counts,
    CELL20_OCCUPIED_YEAR_COUNTS_PATH,
)


# ----------------------------------------------------------------------------
# 7. Resolve and verify authoritative artifacts
# ----------------------------------------------------------------------------

cell15_distance_path = Path(summaries["cell15"]["final_distance_path"]).resolve()
cell15_audit_path = Path(summaries["cell15"]["final_audit_path"]).resolve()
cell16_requested_path = Path(summaries["cell16"]["requested_output_path"]).resolve()
cell17_output_path = Path(summaries["cell17"]["final_output_path"]).resolve()
cell18_site_order_path = Path(
    nested_get(summaries["cell18"], ["source_files", "site_order_path"])
).resolve()
cell18_random_spec_path = Path(
    summaries["cell18"]["random_specification_path"]
).resolve()
cell19_fields_path = Path(
    nested_get(summaries["cell19"], ["outputs", "full_fields_path"])
).resolve()
cell19_event_ledger_path = Path(
    nested_get(summaries["cell19"], ["outputs", "event_residual_ledger_path"])
).resolve()
cell19_random_spec_path = Path(
    nested_get(summaries["cell19"], ["random_streams", "specification_path"])
).resolve()
portfolio_path = Path(summaries["cell17"]["portfolio_path"]).resolve()

artifact_specs = [
    {
        "artifact_id": "notebook3_event_catalog",
        "stage": "Notebook 3",
        "role": "Annual stochastic event-occurrence catalog",
        "path": EVENT_CATALOG_PATH,
        "expected_sha256": nested_get(summaries["cell19"], ["catalog", "sha256"]),
        "required_for_notebook5": True,
    },
    {
        "artifact_id": "seaside_w2_portfolio",
        "stage": "Exposure",
        "role": "470-site Seaside W2 GMM-ready portfolio",
        "path": portfolio_path,
        "expected_sha256": nested_get(summaries["cell17"], ["source_hashes", "portfolio_sha256"]),
        "required_for_notebook5": True,
    },
    {
        "artifact_id": "cell15_distances",
        "stage": "Cell 15",
        "role": "Authoritative USGS rupture-to-site distances",
        "path": cell15_distance_path,
        "expected_sha256": summaries["cell15"].get("final_distance_sha256"),
        "required_for_notebook5": False,
    },
    {
        "artifact_id": "cell15_rupture_audit",
        "stage": "Cell 15",
        "role": "Authoritative rupture-retrieval audit",
        "path": cell15_audit_path,
        "expected_sha256": summaries["cell15"].get("final_audit_sha256"),
        "required_for_notebook5": False,
    },
    {
        "artifact_id": "cell16_controlled_gmm",
        "stage": "Cell 16",
        "role": "Controlled Parker GMM outputs",
        "path": cell16_requested_path,
        "expected_sha256": summaries["cell16"].get("requested_output_sha256"),
        "required_for_notebook5": False,
    },
    {
        "artifact_id": "cell17_bulk_gmm_parameters",
        "stage": "Cell 17",
        "role": "Bulk epi-lo/off/hi Parker GMM parameter matrix",
        "path": cell17_output_path,
        "expected_sha256": summaries["cell17"].get("final_output_sha256"),
        "required_for_notebook5": False,
    },
    {
        "artifact_id": "cell18_site_order",
        "stage": "Cell 18",
        "role": "Frozen 470-site ordering for reproducible residual streams",
        "path": cell18_site_order_path,
        "expected_sha256": nested_get(summaries["cell18"], ["source_files", "site_order_sha256"]),
        "required_for_notebook5": True,
    },
    {
        "artifact_id": "cell18_random_specification",
        "stage": "Cell 18",
        "role": "Controlled random-stream specification",
        "path": cell18_random_spec_path,
        "expected_sha256": summaries["cell18"].get("random_specification_sha256"),
        "required_for_notebook5": False,
    },
    {
        "artifact_id": "cell19_ground_motion_fields",
        "stage": "Cell 19",
        "role": "Full baseline occurrence-site PGA and SA0P4 fields",
        "path": cell19_fields_path,
        "expected_sha256": nested_get(summaries["cell19"], ["outputs", "full_fields_sha256"]),
        "required_for_notebook5": True,
    },
    {
        "artifact_id": "cell19_event_residual_ledger",
        "stage": "Cell 19",
        "role": "Occurrence-level residual seeds and between-event residuals",
        "path": cell19_event_ledger_path,
        "expected_sha256": nested_get(summaries["cell19"], ["outputs", "event_residual_ledger_sha256"]),
        "required_for_notebook5": True,
    },
    {
        "artifact_id": "cell19_occurrence_index",
        "stage": "Cell 19",
        "role": "Sparse annual-catalog occurrence index",
        "path": CELL19_OCCURRENCE_INDEX_PATH,
        "expected_sha256": nested_get(summaries["cell19"], ["outputs", "occurrence_index_sha256"]),
        "required_for_notebook5": True,
    },
    {
        "artifact_id": "cell19_random_specification",
        "stage": "Cell 19",
        "role": "Production random-stream specification",
        "path": cell19_random_spec_path,
        "expected_sha256": nested_get(summaries["cell19"], ["random_streams", "specification_sha256"]),
        "required_for_notebook5": True,
    },
]

inventory_rows: list[dict[str, Any]] = []
for spec in artifact_specs:
    path = Path(spec["path"])
    exists = path.is_file()
    actual_hash = None
    if exists and (VERIFY_LARGE_FILE_HASHES or path.stat().st_size < 100_000_000):
        actual_hash = sha256_file(path)
    expected_hash = spec.get("expected_sha256")
    hash_matches = (
        exists
        and expected_hash is not None
        and actual_hash is not None
        and str(actual_hash).lower() == str(expected_hash).lower()
    )
    inventory_rows.append(
        {
            "artifact_id": spec["artifact_id"],
            "stage": spec["stage"],
            "role": spec["role"],
            "path": str(path),
            "exists": exists,
            "size_bytes": path.stat().st_size if exists else np.nan,
            "expected_sha256": expected_hash,
            "actual_sha256": actual_hash,
            "hash_matches": hash_matches,
            "required_for_notebook5": bool(spec["required_for_notebook5"]),
        }
    )

artifact_inventory = pd.DataFrame(inventory_rows)
atomic_write_csv(artifact_inventory, CELL20_ARTIFACT_INVENTORY_PATH)

if not artifact_inventory["exists"].all():
    missing = artifact_inventory.loc[~artifact_inventory["exists"], ["artifact_id", "path"]]
    raise FileNotFoundError(
        "Notebook 4 artifact inventory contains missing files:\n\n"
        + missing.to_string(index=False)
    )
if not artifact_inventory["hash_matches"].all():
    failed_hashes = artifact_inventory.loc[
        ~artifact_inventory["hash_matches"],
        ["artifact_id", "path", "expected_sha256", "actual_sha256"],
    ]
    raise RuntimeError(
        "Notebook 4 artifact hash verification failed:\n\n"
        + failed_hashes.to_string(index=False)
    )


# ----------------------------------------------------------------------------
# 8. Final validation checks
# ----------------------------------------------------------------------------

checks: list[dict[str, Any]] = []

for cell_name in ["cell15", "cell16", "cell17", "cell18", "cell19"]:
    summary_pass = bool(summaries[cell_name].get("all_checks_passed", False))
    add_check(
        checks,
        "Prerequisites",
        f"{cell_name.upper()} summary reports all checks passed",
        summary_pass,
        True,
        summary_pass,
    )
    frame = validation_frames[cell_name]
    passed_count = int(frame["_normalized_pass"].sum())
    add_check(
        checks,
        "Prerequisites",
        f"{cell_name.upper()} validation file has no failures",
        f"{passed_count}/{len(frame)}",
        f"{len(frame)}/{len(frame)}",
        bool(frame["_normalized_pass"].all()),
    )

add_check(
    checks,
    "Geometry",
    "Cell 15 rupture count",
    summaries["cell15"].get("unique_ruptures"),
    EXPECTED_RUPTURES,
    int(summaries["cell15"].get("unique_ruptures", -1)) == EXPECTED_RUPTURES,
)
add_check(
    checks,
    "Geometry",
    "Cell 15 site count",
    summaries["cell15"].get("sites"),
    EXPECTED_SITES,
    int(summaries["cell15"].get("sites", -1)) == EXPECTED_SITES,
)
add_check(
    checks,
    "Geometry",
    "Cell 15 distance rows",
    summaries["cell15"].get("distance_rows"),
    EXPECTED_DISTANCE_ROWS,
    int(summaries["cell15"].get("distance_rows", -1)) == EXPECTED_DISTANCE_ROWS,
)

add_check(
    checks,
    "GMM",
    "Cell 17 period is exact SA0P4",
    summaries["cell17"].get("period_s"),
    EXPECTED_PERIOD_S,
    math.isclose(
        float(summaries["cell17"].get("period_s", np.nan)),
        EXPECTED_PERIOD_S,
        rel_tol=0.0,
        abs_tol=1.0e-12,
    )
    and "exact" in str(summaries["cell17"].get("period_resolution", "")).lower(),
)
add_check(
    checks,
    "GMM",
    "Cell 17 primary branch is epi-off",
    summaries["cell17"].get("primary_baseline_branch"),
    EXPECTED_PRIMARY_BRANCH,
    str(summaries["cell17"].get("primary_baseline_branch"))
    == EXPECTED_PRIMARY_BRANCH,
)
add_check(
    checks,
    "GMM",
    "Cell 17 rupture-site parameter rows",
    summaries["cell17"].get("rupture_site_rows"),
    EXPECTED_DISTANCE_ROWS,
    int(summaries["cell17"].get("rupture_site_rows", -1))
    == EXPECTED_DISTANCE_ROWS,
)

cell19_catalog = summaries["cell19"].get("catalog", {})
cell19_fields_rows = int(summaries["cell19"].get("field_rows", -1))
add_check(
    checks,
    "Catalog",
    "Occurrence-index row count agrees with Cell 19",
    CATALOG_OCCURRENCES,
    int(cell19_catalog.get("occurrences", -1)),
    CATALOG_OCCURRENCES == int(cell19_catalog.get("occurrences", -1)),
)
add_check(
    checks,
    "Catalog",
    "Occupied-year count agrees with Cell 19",
    OCCUPIED_YEARS,
    int(cell19_catalog.get("observed_event_years", -1)),
    OCCUPIED_YEARS == int(cell19_catalog.get("observed_event_years", -1)),
)
add_check(
    checks,
    "Catalog",
    "Declared duration is at least the largest occupied year",
    CATALOG_DURATION_YEARS,
    f">= {MAX_OCCUPIED_YEAR}",
    CATALOG_DURATION_YEARS >= MAX_OCCUPIED_YEAR,
    CATALOG_DURATION_PROVENANCE,
)
add_check(
    checks,
    "Catalog",
    "Zero-event-year count is nonnegative",
    ZERO_EVENT_YEARS,
    ">= 0",
    ZERO_EVENT_YEARS >= 0,
)
add_check(
    checks,
    "Catalog",
    "Occupied plus zero-event years equals duration",
    OCCUPIED_YEARS + ZERO_EVENT_YEARS,
    CATALOG_DURATION_YEARS,
    OCCUPIED_YEARS + ZERO_EVENT_YEARS == CATALOG_DURATION_YEARS,
)
add_check(
    checks,
    "Fields",
    "Cell 19 field-row count equals occurrences times sites",
    cell19_fields_rows,
    CATALOG_OCCURRENCES * EXPECTED_SITES,
    cell19_fields_rows == CATALOG_OCCURRENCES * EXPECTED_SITES,
)
add_check(
    checks,
    "Fields",
    "Cell 19 baseline remains spatially independent",
    summaries["cell19"].get("spatial_residual_model"),
    EXPECTED_SPATIAL_MODEL,
    str(summaries["cell19"].get("spatial_residual_model", "")).lower()
    == EXPECTED_SPATIAL_MODEL.lower(),
)

# Verify the field and event-ledger schemas without loading millions of rows.
field_columns = pd.read_csv(cell19_fields_path, compression="gzip", nrows=0).columns.tolist()
event_ledger_columns = pd.read_csv(
    cell19_event_ledger_path, compression="gzip", nrows=0
).columns.tolist()
required_field_columns = {
    "catalog_year",
    "occurrence_id",
    "rupture_id",
    "site_id",
    "source_type",
    "magnitude",
    "pga_simulated_g",
    "sa0p4_simulated_g",
}
required_ledger_columns = {
    "catalog_year",
    "occurrence_id",
    "rupture_id",
    "event_seed_hex",
    "site_seed_hex",
    "eta_pga",
    "eta_sa0p4",
}
add_check(
    checks,
    "Handoff schema",
    "Full field file has required Notebook 5 columns",
    len(required_field_columns.intersection(field_columns)),
    len(required_field_columns),
    required_field_columns.issubset(field_columns),
    "Required: " + ", ".join(sorted(required_field_columns)),
)
add_check(
    checks,
    "Handoff schema",
    "Event ledger preserves reproducible random-stream columns",
    len(required_ledger_columns.intersection(event_ledger_columns)),
    len(required_ledger_columns),
    required_ledger_columns.issubset(event_ledger_columns),
    "Required: " + ", ".join(sorted(required_ledger_columns)),
)

add_check(
    checks,
    "Integrity",
    "All inventoried artifacts exist",
    int(artifact_inventory["exists"].sum()),
    len(artifact_inventory),
    bool(artifact_inventory["exists"].all()),
)
add_check(
    checks,
    "Integrity",
    "All inventoried artifact hashes match",
    int(artifact_inventory["hash_matches"].sum()),
    len(artifact_inventory),
    bool(artifact_inventory["hash_matches"].all()),
)

validation = pd.DataFrame(checks)
atomic_write_csv(validation, CELL20_VALIDATION_PATH)

if not validation["passes"].all():
    failed = validation.loc[~validation["passes"]]
    raise RuntimeError(
        "Cell 20 final Notebook 4 validation failed:\n\n"
        + failed.to_string(index=False)
        + f"\n\nValidation file:\n  {CELL20_VALIDATION_PATH}"
    )


# ----------------------------------------------------------------------------
# 9. Write the Notebook 5 handoff contract
# ----------------------------------------------------------------------------

source_occurrence_counts = (
    occurrence_index.groupby("source_type").size().astype(int).to_dict()
)
events_per_year_frequency = (
    occupied_year_counts.groupby("event_count").size().astype(int).to_dict()
)

notebook5_handoff = {
    "schema_version": "notebook5_input_handoff_v1",
    "created_at_utc": utc_now(),
    "notebook4_pipeline_version": PIPELINE_VERSION,
    "notebook4_complete": True,
    "portfolio": {
        "name": "Seaside W2 commercial portfolio",
        "sites": EXPECTED_SITES,
        "structural_type": "W2",
        "period_s": EXPECTED_PERIOD_S,
        "vs30_note": (
            "The Notebook 4 portfolio uses the portfolio vs30_mps field; the "
            "accepted summaries identify the 365 m/s values as inferred rather "
            "than site-specific measurements."
        ),
        "path": str(portfolio_path),
        "sha256": sha256_file(portfolio_path),
    },
    "annual_catalog": {
        "path": str(EVENT_CATALOG_PATH),
        "sha256": sha256_file(EVENT_CATALOG_PATH),
        "declared_duration_years": CATALOG_DURATION_YEARS,
        "duration_provenance": CATALOG_DURATION_PROVENANCE,
        "duration_candidates": [
            {"years": value, "provenance": provenance}
            for value, provenance in DURATION_CANDIDATES
        ],
        "occurrences": CATALOG_OCCURRENCES,
        "occupied_years": OCCUPIED_YEARS,
        "zero_event_years": ZERO_EVENT_YEARS,
        "multiple_event_years": MULTIPLE_EVENT_YEARS,
        "maximum_events_in_one_year": MAX_EVENTS_IN_YEAR,
        "minimum_occupied_year_label": MIN_OCCUPIED_YEAR,
        "maximum_occupied_year_label": MAX_OCCUPIED_YEAR,
        "annual_event_occurrence_rate": ANNUAL_EVENT_RATE,
        "mean_interevent_time_years": MEAN_INTEREVENT_YEARS,
        "source_occurrence_counts": {
            str(key): int(value) for key, value in source_occurrence_counts.items()
        },
        "occupied_year_event_count_frequency": {
            str(key): int(value) for key, value in events_per_year_frequency.items()
        },
        "zero_year_rule": (
            "Notebook 5 annual loss tables must use the declared duration as "
            "the denominator and must explicitly include all zero-event years "
            "as zero annual loss. Do not use only occupied years."
        ),
        "occupied_year_count_path": str(CELL20_OCCUPIED_YEAR_COUNTS_PATH),
        "occupied_year_count_sha256": sha256_file(CELL20_OCCUPIED_YEAR_COUNTS_PATH),
    },
    "ground_motion_model": {
        "interface_gmm": summaries["cell17"].get("interface_gmm"),
        "slab_gmm": summaries["cell17"].get("slab_gmm"),
        "primary_epistemic_branch": EXPECTED_PRIMARY_BRANCH,
        "retained_epistemic_branches": summaries["cell17"].get(
            "epistemic_branches_retained", ["epi-lo", "epi-off", "epi-hi"]
        ),
        "baseline_branch_rule": (
            "Use epi-off for the Phase 1 baseline. Epi-lo and epi-hi are "
            "alternative epistemic sensitivity branches and must not be mixed "
            "as additive earthquake occurrences."
        ),
        "imts": ["PGA", "SA0P4"],
        "period_resolution": "exact USGS SA0P4; no interpolation",
    },
    "aleatory_model": {
        "equation": "ln(IM) = mu_epi_off + tau*eta + phi*epsilon",
        "between_event": (
            "One occurrence-level residual per IMT shared by all 470 sites."
        ),
        "within_event": (
            "Conditionally independent across different sites in the Phase 1 "
            "baseline; PGA and SA0P4 are cross-correlated at the same site."
        ),
        "cross_imt_rho": float(summaries["cell19"].get("cross_imt_rho")),
        "spatial_correlation": "not applied in the Phase 1 baseline",
        "random_stream_reuse": (
            "The event and site seeds in the event residual ledger are frozen "
            "for future independent-versus-spatially-correlated comparisons."
        ),
    },
    "notebook5_primary_input": {
        "path": str(cell19_fields_path),
        "sha256": sha256_file(cell19_fields_path),
        "row_granularity": "one event occurrence and one Seaside W2 site",
        "rows": cell19_fields_rows,
        "year_column": "catalog_year",
        "occurrence_key": "occurrence_id",
        "rupture_key": "rupture_id",
        "site_key": "site_id",
        "source_column": "source_type",
        "magnitude_column": "magnitude",
        "structural_damage_im_column": "sa0p4_simulated_g",
        "structural_damage_im_period_s": EXPECTED_PERIOD_S,
        "supporting_pga_column": "pga_simulated_g",
        "pga_use_note": (
            "PGA is retained for diagnostics or damage components explicitly "
            "parameterized by PGA. Do not assign a component to PGA unless the "
            "selected vulnerability or fragility model requires it."
        ),
        "available_columns": field_columns,
    },
    "supporting_inputs": {
        "occurrence_index": {
            "path": str(CELL19_OCCURRENCE_INDEX_PATH),
            "sha256": sha256_file(CELL19_OCCURRENCE_INDEX_PATH),
        },
        "event_residual_ledger": {
            "path": str(cell19_event_ledger_path),
            "sha256": sha256_file(cell19_event_ledger_path),
            "available_columns": event_ledger_columns,
        },
        "site_order": {
            "path": str(cell18_site_order_path),
            "sha256": sha256_file(cell18_site_order_path),
        },
        "production_random_specification": {
            "path": str(cell19_random_spec_path),
            "sha256": sha256_file(cell19_random_spec_path),
        },
        "bulk_gmm_parameters": {
            "path": str(cell17_output_path),
            "sha256": sha256_file(cell17_output_path),
        },
    },
    "modeling_boundaries": [
        "This handoff represents the Phase 1 no-spatial-correlation baseline.",
        "The same annual event catalog must be reused for later correlation cases.",
        "The same occurrence and latent site random streams must be reused where possible.",
        "Catalog logic-tree alternatives must not be treated as additive sources unless the USGS model defines them as additive.",
        "Source-scale factors must remain separate from epistemic branch weights.",
        "Annual loss statistics must include zero-event years.",
    ],
    "validation": {
        "path": str(CELL20_VALIDATION_PATH),
        "sha256": sha256_file(CELL20_VALIDATION_PATH),
        "checks": int(len(validation)),
        "all_checks_passed": True,
    },
    "artifact_inventory": {
        "path": str(CELL20_ARTIFACT_INVENTORY_PATH),
        "sha256": sha256_file(CELL20_ARTIFACT_INVENTORY_PATH),
    },
    "next_notebook": "05_calculate_ground_up_losses.ipynb",
    "next_task": (
        "Join occurrence-site SA0P4 fields to the Seaside W2 exposure and "
        "validated vulnerability/fragility assumptions, sample damage and "
        "ground-up loss, then aggregate event and annual losses while retaining "
        "all zero-event years."
    ),
}
atomic_write_json(CELL20_NOTEBOOK5_HANDOFF_PATH, notebook5_handoff)


# ----------------------------------------------------------------------------
# 10. Final Notebook 4 summary
# ----------------------------------------------------------------------------

final_summary = {
    "pipeline_version": PIPELINE_VERSION,
    "completed_at_utc": utc_now(),
    "notebook4_complete": True,
    "all_checks_passed": True,
    "validation_checks": int(len(validation)),
    "catalog_duration_years": CATALOG_DURATION_YEARS,
    "catalog_duration_provenance": CATALOG_DURATION_PROVENANCE,
    "catalog_occurrences": CATALOG_OCCURRENCES,
    "occupied_years": OCCUPIED_YEARS,
    "zero_event_years": ZERO_EVENT_YEARS,
    "multiple_event_years": MULTIPLE_EVENT_YEARS,
    "annual_event_occurrence_rate": ANNUAL_EVENT_RATE,
    "mean_interevent_time_years": MEAN_INTEREVENT_YEARS,
    "sites": EXPECTED_SITES,
    "field_rows": cell19_fields_rows,
    "primary_branch": EXPECTED_PRIMARY_BRANCH,
    "primary_damage_im": "SA0P4",
    "primary_damage_im_column": "sa0p4_simulated_g",
    "phase1_spatial_model": EXPECTED_SPATIAL_MODEL,
    "outputs": {
        "final_validation_path": str(CELL20_VALIDATION_PATH),
        "final_validation_sha256": sha256_file(CELL20_VALIDATION_PATH),
        "artifact_inventory_path": str(CELL20_ARTIFACT_INVENTORY_PATH),
        "artifact_inventory_sha256": sha256_file(CELL20_ARTIFACT_INVENTORY_PATH),
        "occupied_year_counts_path": str(CELL20_OCCUPIED_YEAR_COUNTS_PATH),
        "occupied_year_counts_sha256": sha256_file(CELL20_OCCUPIED_YEAR_COUNTS_PATH),
        "notebook5_handoff_path": str(CELL20_NOTEBOOK5_HANDOFF_PATH),
        "notebook5_handoff_sha256": sha256_file(CELL20_NOTEBOOK5_HANDOFF_PATH),
    },
    "next_notebook": "05_calculate_ground_up_losses.ipynb",
}
atomic_write_json(CELL20_FINAL_SUMMARY_PATH, final_summary)

# Export convenient variables for any subsequent cells in the current kernel.
NOTEBOOK4_COMPLETE = True
NOTEBOOK4_CATALOG_DURATION_YEARS = CATALOG_DURATION_YEARS
NOTEBOOK5_HANDOFF_PATH = CELL20_NOTEBOOK5_HANDOFF_PATH
NOTEBOOK5_GROUND_MOTION_FIELDS_PATH = cell19_fields_path
NOTEBOOK5_OCCURRENCE_INDEX_PATH = CELL19_OCCURRENCE_INDEX_PATH
NOTEBOOK5_EVENT_LEDGER_PATH = cell19_event_ledger_path

print("=" * 78)
print("NOTEBOOK 4 FINAL VALIDATION AND NOTEBOOK 5 HANDOFF COMPLETE")
print("=" * 78)
print(f"Final validation checks:  {len(validation):,}")
print(f"Catalog duration:        {CATALOG_DURATION_YEARS:,} years")
print(f"Duration provenance:     {CATALOG_DURATION_PROVENANCE}")
print(f"Catalog occurrences:     {CATALOG_OCCURRENCES:,}")
print(f"Occupied years:          {OCCUPIED_YEARS:,}")
print(f"Zero-event years:        {ZERO_EVENT_YEARS:,}")
print(f"Multiple-event years:    {MULTIPLE_EVENT_YEARS:,}")
print(f"Annual event rate:       {ANNUAL_EVENT_RATE:.8f}")
print(f"Mean interevent time:    {MEAN_INTEREVENT_YEARS:.3f} years")
print(f"Seaside W2 sites:        {EXPECTED_SITES:,}")
print(f"Ground-motion rows:      {cell19_fields_rows:,}")
print(f"Primary damage IM:       SA(0.4 s), column sa0p4_simulated_g")
print(f"Baseline spatial model:  independent within-event site residuals")
print(f"\nFinal validation:\n  {CELL20_VALIDATION_PATH}")
print(f"\nArtifact inventory:\n  {CELL20_ARTIFACT_INVENTORY_PATH}")
print(f"\nOccupied-year counts:\n  {CELL20_OCCUPIED_YEAR_COUNTS_PATH}")
print(f"\nNotebook 5 handoff:\n  {CELL20_NOTEBOOK5_HANDOFF_PATH}")
print(f"\nFinal summary:\n  {CELL20_FINAL_SUMMARY_PATH}")
print(
    "\nNotebook 4 is complete. Next: open "
    "05_calculate_ground_up_losses.ipynb and read the Notebook 5 handoff JSON."
)

display(validation)


NOTEBOOK 4 FINAL VALIDATION AND NOTEBOOK 5 HANDOFF COMPLETE
Final validation checks:  27
Catalog duration:        2,000,000 years
Duration provenance:     C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\03_generate_annual_event_catalog.ipynb -> CATALOG_YEARS
Catalog occurrences:     10,630
Occupied years:          10,593
Zero-event years:        1,989,407
Multiple-event years:    36
Annual event rate:       0.00531500
Mean interevent time:    188.147 years
Seaside W2 sites:        470
Ground-motion rows:      4,996,100
Primary damage IM:       SA(0.4 s), column sa0p4_simulated_g
Baseline spatial model:  independent within-event site residuals

Final validation:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_final_validation.csv

Artifact inventory:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\notebook_4_final_handoff\notebook_4_artifact_inventory.csv

Occupied-year counts:
  C:\Users\US

,category,check,observed,expected,passes,note
0,Prerequisites,CELL15 summary reports all checks passed,True,True,True,
1,Prerequisites,CELL15 validation file has no failures,15/15,15/15,True,
2,Prerequisites,CELL16 summary reports all checks passed,True,True,True,
3,Prerequisites,CELL16 validation file has no failures,30/30,30/30,True,
4,Prerequisites,CELL17 summary reports all checks passed,True,True,True,
5,Prerequisites,CELL17 validation file has no failures,21/21,21/21,True,
6,Prerequisites,CELL18 summary reports all checks passed,True,True,True,
7,Prerequisites,CELL18 validation file has no failures,50/50,50/50,True,
8,Prerequisites,CELL19 summary reports all checks passed,True,True,True,
9,Prerequisites,CELL19 validation file has no failures,27/27,27/27,True,
